## Load Prepared Research Corpus

1.1 Import Required Libraries

In [ ]:
# Import libraries for loading the prepared research corpus

import pickle
from pathlib import Path
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


1.2 Connect Google Drive

In [ ]:
# Connect Google Drive to access the prepared research corpus

from google.colab import drive

drive.mount('/content/drive')

folder_path = Path(
    "/content/drive/MyDrive/NLP-Project/CyberThreatResearch"
)

data_folder = folder_path / "processed_data"

print("Google Drive connected.")
print("Prepared data folder exists:", data_folder.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive connected.
Prepared data folder exists: True


1.3 Load the saved corpus and inventories

In [ ]:
# Load the prepared page records and research inventories

with open(data_folder / "page_documents.pkl", "rb") as file:
    page_documents = pickle.load(file)

screening_df = pd.read_csv(
    data_folder / "screening_inventory.csv"
)

metadata_df = pd.read_csv(
    data_folder / "metadata_inventory.csv"
)

print("Prepared corpus loaded successfully.")
print("Page-level records:", len(page_documents))
print("Screening records:", len(screening_df))
print("Metadata records:", len(metadata_df))

Prepared corpus loaded successfully.
Page-level records: 3071
Screening records: 162
Metadata records: 162


1.4 Verify the loaded corpus

In [ ]:
# Verify the loaded corpus and its article-level coverage

article_ids = set(
    record["article_id"] for record in page_documents
)

print("Unique articles in page records:", len(article_ids))
print("Articles in screening inventory:", screening_df["filename"].nunique())
print("Articles in metadata inventory:", metadata_df["filename"].nunique())

print("\nSample page record:")
print(page_documents[0])

Unique articles in page records: 162
Articles in screening inventory: 162
Articles in metadata inventory: 162

Sample page record:
{'article_id': '1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf', 'page_number': 1, 'text': 'Torabi\xa0et\xa0al. Cybersecurity             (2023) 6:1  \nhttps://doi.org/10.1186/s42400-022-00134-9\nRESEARCH\n© The Author(s) 2023. Open Access This article is licensed under a Creative Commons Attribution 4.0 International License, which \npermits use, sharing, adaptation, distribution and reproduction in any medium or format, as long as you give appropriate credit to the \noriginal author(s) and the source, provide a link to the Creative Commons licence, and indicate if changes were made. The images or \nother third party material in this article are included in the article’s Creative Commons licence, unless indicated otherwise in a credit line \nto the material. If material is not included in the article’s Creative Co

## 2.Document Chunking

2.1 Prepare page-level text

In [ ]:
# Clean extracted page text while preserving article and page information

import re
import unicodedata

clean_page_documents = []

for record in page_documents:
    text = unicodedata.normalize("NFKC", record["text"])

    # Remove invisible zero-width characters
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    if text:
        clean_page_documents.append({
            "article_id": record["article_id"],
            "page_number": record["page_number"],
            "text": text
        })

print("Original page records:", len(page_documents))
print("Cleaned page records:", len(clean_page_documents))
print("Unique articles:", len(set(
    record["article_id"] for record in clean_page_documents
)))

Original page records: 3071
Cleaned page records: 3071
Unique articles: 162


2.2 Examine text length

In [ ]:
# Examine the text length distribution of cleaned page records

page_lengths = [
    len(record["text"]) for record in clean_page_documents
]

print("Page text length statistics:")
print(pd.Series(page_lengths).describe())

print("\nShortest page text:", min(page_lengths))
print("Longest page text:", max(page_lengths))
print("Total extracted characters:", sum(page_lengths))

Page text length statistics:
count     3071.000000
mean      3950.185282
std       1889.951271
min          1.000000
25%       2617.000000
50%       3761.000000
75%       5224.500000
max      12100.000000
dtype: float64

Shortest page text: 1
Longest page text: 12100
Total extracted characters: 12131019


2.3 Identify pages with very little text

In [ ]:
# Identify pages with very little extracted text

short_pages = [
    record for record in clean_page_documents
    if len(record["text"]) < 100
]

print("Pages with fewer than 100 characters:", len(short_pages))

print("\nExamples:")
for record in short_pages[:10]:
    print(
        record["article_id"],
        "| Page:", record["page_number"],
        "| Text:", repr(record["text"][:100])
    )

Pages with fewer than 100 characters: 3

Examples:
Applications_of_deep_learning_for_phishing_detection.pdf | Page: 10 | Text: '1466 Catal et al. Fig. 6 SLR process used in this study 123'
Hybrid_machine_learning_model_for_phishing_detection.pdf | Page: 7 | Text: '7'
Security Audit of IoT Device Networks A Reproducible Machine Learning Framework for Threat Detection and Performance Benchmarking.pdf | Page: 11 | Text: 'Sensors 2025, 25, 7519 11 of 41 Figure 5. Numeric Distributions.'


2.4 Check the distribution of page lengths

In [ ]:
# Group page records by text length

length_groups = {
    "Below 100 characters": 0,
    "100–499 characters": 0,
    "500–999 characters": 0,
    "1,000–2,999 characters": 0,
    "3,000 characters or more": 0
}

for length in page_lengths:
    if length < 100:
        length_groups["Below 100 characters"] += 1
    elif length < 500:
        length_groups["100–499 characters"] += 1
    elif length < 1000:
        length_groups["500–999 characters"] += 1
    elif length < 3000:
        length_groups["1,000–2,999 characters"] += 1
    else:
        length_groups["3,000 characters or more"] += 1

for group, count in length_groups.items():
    print(f"{group}: {count}")

Below 100 characters: 3
100–499 characters: 59
500–999 characters: 72
1,000–2,999 characters: 836
3,000 characters or more: 2101


### 2.5 Set up the two chunking strategies

For a fair comparison, we will test:
1. Fixed-size chunking: Split text into chunks of 1,200 characters with 200 characters of overlap.
2. Sentence-based chunking: Group complete sentences into chunks of approximately 1,200 characters, with controlled overlap and a maximum size.


Both strategies will preserve article IDs and PDF page numbers. We will compare their chunk counts, lengths, and text quality before selecting the final strategy.

In [ ]:
# Define fixed-size chunking with overlapping text

chunk_size = 1200
chunk_overlap = 200

fixed_chunks = []

for record in clean_page_documents:
    text = record["text"]
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end].strip()

        if chunk_text:
            fixed_chunks.append({
                "article_id": record["article_id"],
                "page_number": record["page_number"],
                "text": chunk_text
            })

        start += chunk_size - chunk_overlap

print("Fixed-size chunks created:", len(fixed_chunks))
print("Sample chunk:")
print(fixed_chunks[0])

Fixed-size chunks created: 13669
Sample chunk:
{'article_id': '1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf', 'page_number': 1, 'text': 'Torabi et al. Cybersecurity (2023) 6:1 https://doi.org/10.1186/s42400-022-00134-9 RESEARCH © The Author(s) 2023. Open Access This article is licensed under a Creative Commons Attribution 4.0 International License, which permits use, sharing, adaptation, distribution and reproduction in any medium or format, as long as you give appropriate credit to the original author(s) and the source, provide a link to the Creative Commons licence, and indicate if changes were made. The images or other third party material in this article are included in the article’s Creative Commons licence, unless indicated otherwise in a credit line to the material. If material is not included in the article’s Creative Commons licence and your intended use is not permitted by statutory regulation or exceeds the permitted use, you will

2.6 Prepare sentence-based chunking

In [ ]:
# Install and load NLTK sentence tokenization tools

import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.tokenize import sent_tokenize

print("Sentence tokenizer ready.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...


Sentence tokenizer ready.


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


2.7 Test sentence splitting

In [ ]:
# Test sentence splitting on one extracted PDF page

sample_text = clean_page_documents[0]["text"]

sample_sentences = sent_tokenize(sample_text)

print("Number of sentences:", len(sample_sentences))

print("\nFirst five sentences:")
for sentence in sample_sentences[:5]:
    print("-", sentence)

Number of sentences: 33

First five sentences:
- Torabi et al.
- Cybersecurity (2023) 6:1 https://doi.org/10.1186/s42400-022-00134-9 RESEARCH © The Author(s) 2023.
- Open Access This article is licensed under a Creative Commons Attribution 4.0 International License, which permits use, sharing, adaptation, distribution and reproduction in any medium or format, as long as you give appropriate credit to the original author(s) and the source, provide a link to the Creative Commons licence, and indicate if changes were made.
- The images or other third party material in this article are included in the article’s Creative Commons licence, unless indicated otherwise in a credit line to the material.
- If material is not included in the article’s Creative Commons licence and your intended use is not permitted by statutory regulation or exceeds the permitted use, you will need to obtain permission directly from the copyright holder.


2.8 Implement sentence-based chunking



In [ ]:
# Create sentence-based chunks with a maximum size and sentence-boundary overlap

sentence_chunks = []

chunk_size = 1200
chunk_overlap = 200

for record in clean_page_documents:
    sentences = sent_tokenize(record["text"])
    current_sentences = []

    for sentence in sentences:

        # Handle sentences longer than the maximum chunk size
        if len(sentence) > chunk_size:

            if current_sentences:
                sentence_chunks.append({
                    "article_id": record["article_id"],
                    "page_number": record["page_number"],
                    "text": " ".join(current_sentences)
                })
                current_sentences = []

            for start in range(0, len(sentence), chunk_size):
                part = sentence[start:start + chunk_size]

                sentence_chunks.append({
                    "article_id": record["article_id"],
                    "page_number": record["page_number"],
                    "text": part.strip()
                })

            continue

        # Add the sentence if it fits in the current chunk
        candidate = " ".join(current_sentences + [sentence])

        if len(candidate) <= chunk_size:
            current_sentences.append(sentence)

        else:
            # Save the current chunk
            sentence_chunks.append({
                "article_id": record["article_id"],
                "page_number": record["page_number"],
                "text": " ".join(current_sentences)
            })

            # Retain previous complete sentences for overlap
            overlap_sentences = []

            for previous_sentence in reversed(current_sentences):
                proposed_overlap = [previous_sentence] + overlap_sentences

                if len(" ".join(proposed_overlap)) <= chunk_overlap:
                    overlap_sentences = proposed_overlap
                else:
                    break

            # Remove overlap sentences if necessary to fit the new sentence
            while (
                overlap_sentences
                and len(" ".join(overlap_sentences + [sentence])) > chunk_size
            ):
                overlap_sentences.pop(0)

            current_sentences = overlap_sentences + [sentence]

    # Save the remaining sentences
    if current_sentences:
        sentence_chunks.append({
            "article_id": record["article_id"],
            "page_number": record["page_number"],
            "text": " ".join(current_sentences)
        })

print("Sentence-based chunks created:", len(sentence_chunks))
print("Maximum chunk length:", max(
    len(chunk["text"]) for chunk in sentence_chunks
))

Sentence-based chunks created: 13266
Maximum chunk length: 1200


In [ ]:
# Verify the updated sentence-based chunks

print("Total chunks:", len(sentence_chunks))

print("Maximum chunk length:", max(
    len(chunk["text"]) for chunk in sentence_chunks
))

print("Chunks below 100 characters:", sum(
    1 for chunk in sentence_chunks
    if len(chunk["text"]) < 100
))

print("Articles represented:", len(set(
    chunk["article_id"] for chunk in sentence_chunks
)))

print("Missing metadata:", sum(
    1 for chunk in sentence_chunks
    if not chunk["article_id"] or not chunk["page_number"]
))

Total chunks: 13266
Maximum chunk length: 1200
Chunks below 100 characters: 78
Articles represented: 162
Missing metadata: 0


In [ ]:
# Check overlap between consecutive sentence-based chunks

for i in range(3):
    first_chunk = sentence_chunks[i]["text"]
    second_chunk = sentence_chunks[i + 1]["text"]

    print(f"\nChunk {i + 1} and Chunk {i + 2}")
    print("Same article:",
          sentence_chunks[i]["article_id"] == sentence_chunks[i + 1]["article_id"])
    print("Same page:",
          sentence_chunks[i]["page_number"] == sentence_chunks[i + 1]["page_number"])

    print("Previous chunk ending:")
    print(first_chunk[-250:])

    print("\nNext chunk beginning:")
    print(second_chunk[:250])
    print("-" * 60)


Chunk 1 and Chunk 2
Same article: True
Same page: True
Previous chunk ending:
nce and your intended use is not permitted by statutory regulation or exceeds the permitted use, you will need to obtain permission directly from the copyright holder. To view a copy of this licence, visit http://creativecommons.org/licenses/by/4.0/.

Next chunk beginning:
To view a copy of this licence, visit http://creativecommons.org/licenses/by/4.0/. Open Access Cybersecurity Practical autoencoder based anomaly detection by using vector reconstruction error Hasan Torabi1, Seyedeh Leili Mirtaheri1* and Sergio Greco2
------------------------------------------------------------

Chunk 2 and Chunk 3
Same article: True
Same page: True
Previous chunk ending:
. Network anomaly detection systems have gained much attention in intelligently monitoring network traffic using machine learning methods. This paper presents an efficient model based on autoencoders for anomaly detection in cloud computing networks.

Next

2.9 Compare the two strategies

In [ ]:
# Compare the chunk counts and lengths of both strategies

fixed_lengths = [len(chunk["text"]) for chunk in fixed_chunks]
sentence_lengths = [len(chunk["text"]) for chunk in sentence_chunks]

comparison_df = pd.DataFrame({
    "Metric": [
        "Total chunks",
        "Average chunk length",
        "Minimum chunk length",
        "Maximum chunk length"
    ],
    "Fixed-size": [
        len(fixed_chunks),
        round(sum(fixed_lengths) / len(fixed_lengths)),
        min(fixed_lengths),
        max(fixed_lengths)
    ],
    "Sentence-based": [
        len(sentence_chunks),
        round(sum(sentence_lengths) / len(sentence_lengths)),
        min(sentence_lengths),
        max(sentence_lengths)
    ]
})

comparison_df

,Metric,Fixed-size,Sentence-based
0,Total chunks,13669,13266
1,Average chunk length,1038,1001
2,Minimum chunk length,1,1
3,Maximum chunk length,1200,1200


2.10 Inspect chunk quality

In [ ]:
# Display sample chunks from both chunking strategies

for i in range(3):
    print(f"\n{'=' * 70}")
    print(f"FIXED-SIZE CHUNK {i + 1}")
    print(f"{'=' * 70}")
    print(f"Article: {fixed_chunks[i]['article_id']}")
    print(f"Page: {fixed_chunks[i]['page_number']}")
    print(f"Length: {len(fixed_chunks[i]['text'])}")
    print(fixed_chunks[i]["text"][:500])

    print(f"\n{'=' * 70}")
    print(f"SENTENCE-BASED CHUNK {i + 1}")
    print(f"{'=' * 70}")
    print(f"Article: {sentence_chunks[i]['article_id']}")
    print(f"Page: {sentence_chunks[i]['page_number']}")
    print(f"Length: {len(sentence_chunks[i]['text'])}")
    print(sentence_chunks[i]["text"][:500])


FIXED-SIZE CHUNK 1
Article: 1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf
Page: 1
Length: 1199
Torabi et al. Cybersecurity (2023) 6:1 https://doi.org/10.1186/s42400-022-00134-9 RESEARCH © The Author(s) 2023. Open Access This article is licensed under a Creative Commons Attribution 4.0 International License, which permits use, sharing, adaptation, distribution and reproduction in any medium or format, as long as you give appropriate credit to the original author(s) and the source, provide a link to the Creative Commons licence, and indicate if changes were made. The images or other third p

SENTENCE-BASED CHUNK 1
Article: 1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf
Page: 1
Length: 963
Torabi et al. Cybersecurity (2023) 6:1 https://doi.org/10.1186/s42400-022-00134-9 RESEARCH © The Author(s) 2023. Open Access This article is licensed under a Creative Commons Attribution 4.0 International License, wh

2.11 Check metadata and chunk integrity

In [ ]:
# Check metadata completeness and short chunks in both strategies

for name, chunks in [
    ("Fixed-size", fixed_chunks),
    ("Sentence-based", sentence_chunks)
]:
    missing_metadata = sum(
        1 for chunk in chunks
        if not chunk["article_id"] or not chunk["page_number"]
    )

    short_chunks = sum(
        1 for chunk in chunks
        if len(chunk["text"]) < 100
    )

    print(f"\n{name} chunking")
    print("Total chunks:", len(chunks))
    print("Missing metadata:", missing_metadata)
    print("Chunks below 100 characters:", short_chunks)
    print("Articles represented:", len(set(
        chunk["article_id"] for chunk in chunks
    )))


Fixed-size chunking
Total chunks: 13669
Missing metadata: 0
Chunks below 100 characters: 285
Articles represented: 162

Sentence-based chunking
Total chunks: 13266
Missing metadata: 0
Chunks below 100 characters: 78
Articles represented: 162


2.12 Check page-level coverage

In [ ]:
# Verify page-level coverage in both chunking strategies

original_pages = set(
    (record["article_id"], record["page_number"])
    for record in clean_page_documents
)

fixed_pages = set(
    (chunk["article_id"], chunk["page_number"])
    for chunk in fixed_chunks
)

sentence_pages = set(
    (chunk["article_id"], chunk["page_number"])
    for chunk in sentence_chunks
)

print("Original article-page pairs:", len(original_pages))
print("Fixed-size article-page pairs:", len(fixed_pages))
print("Sentence-based article-page pairs:", len(sentence_pages))

print("\nMissing pages in fixed-size:",
      len(original_pages - fixed_pages))

print("Missing pages in sentence-based:",
      len(original_pages - sentence_pages))

Original article-page pairs: 3071
Fixed-size article-page pairs: 3071
Sentence-based article-page pairs: 3071

Missing pages in fixed-size: 0
Missing pages in sentence-based: 0



### 2.13 Comparison and Selection of Chunking Strategies

Two chunking strategies were implemented and compared: fixed-size chunking and sentence-based chunking.

**Fixed-size chunking** divides extracted text into chunks of 1,200 characters with a 200-character overlap. Although it provides consistent chunk lengths, it can split sentences and words across chunk boundaries.

**Sentence-based chunking** groups complete sentences into chunks with a maximum size of 1,200 characters. It retains previous complete sentences to provide an overlap of approximately 200 characters. Exceptionally long sentences are split when necessary to respect the maximum chunk size.

Both strategies preserved all 162 articles and maintained article IDs and page numbers.

Sentence-based chunking was selected as the preferred strategy because it generally preserves sentence boundaries, improves passage readability, and maintains more meaningful textual context. Fixed-size chunking was retained as a baseline for subsequent retrieval experiments.

The final choice will also be evaluated through retrieval performance rather than relying exclusively on text readability and chunk statistics.


Part 3 — Enrich Chunks with Article Metadata

3.1 Examine available metadata columns

In [ ]:
# Examine the available article metadata fields

print("Metadata columns:")
print(metadata_df.columns.tolist())

print("\nSample article metadata:")
print(metadata_df.head(3).to_string(index=False))

Metadata columns:
['filename', 'doi_candidate', 'title', 'authors', 'year', 'venue', 'crossref_status']

Sample article metadata:
                                                                                 filename               doi_candidate                                                                              title                                                                                                                 authors   year                     venue crossref_status
1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf  10.1186/s42400-022-00134-9 Practical autoencoder based anomaly detection by using vector reconstruction error                                                                     Hasan Torabi; Seyedeh Leili Mirtaheri; Sergio Greco 2023.0             Cybersecurity       Retrieved
 11. A network intrusion detection method based on deep learning with higher accuracy.pdf 10.1016/j.procs.2020.06.055   A network 

3.2 Create an article metadata lookup

In [ ]:
# Create a lookup dictionary for article metadata

article_metadata = {}

for _, row in metadata_df.iterrows():
    article_metadata[row["filename"]] = {
        "title": row["title"],
        "authors": row["authors"],
        "year": row["year"],
        "venue": row["venue"],
        "doi": row["doi_candidate"]
    }

print("Article metadata lookup created.")
print("Articles in lookup:", len(article_metadata))

print("\nSample metadata:")
print(article_metadata[next(iter(article_metadata))])

Article metadata lookup created.
Articles in lookup: 162

Sample metadata:
{'title': 'Practical autoencoder based anomaly detection by using vector reconstruction error', 'authors': 'Hasan Torabi; Seyedeh Leili Mirtaheri; Sergio Greco', 'year': 2023.0, 'venue': 'Cybersecurity', 'doi': '10.1186/s42400-022-00134-9'}


3.3 Add metadata to both chunking strategies

In [ ]:
# Add article metadata to fixed-size and sentence-based chunks

def add_article_metadata(chunks):
    enriched_chunks = []

    for chunk in chunks:
        article_info = article_metadata.get(chunk["article_id"], {})

        enriched_chunks.append({
            "article_id": chunk["article_id"],
            "title": article_info.get("title"),
            "authors": article_info.get("authors"),
            "year": article_info.get("year"),
            "venue": article_info.get("venue"),
            "doi": article_info.get("doi"),
            "page_number": chunk["page_number"],
            "text": chunk["text"]
        })

    return enriched_chunks


fixed_chunks = add_article_metadata(fixed_chunks)
sentence_chunks = add_article_metadata(sentence_chunks)

print("Metadata added successfully.")
print("Fixed-size chunks:", len(fixed_chunks))
print("Sentence-based chunks:", len(sentence_chunks))

Metadata added successfully.
Fixed-size chunks: 13669
Sentence-based chunks: 13266


3.4 Verify metadata completeness

In [ ]:
# Verify metadata completeness in both chunking strategies

for name, chunks in [
    ("Fixed-size", fixed_chunks),
    ("Sentence-based", sentence_chunks)
]:
    missing_fields = {
        field: sum(
            1 for chunk in chunks
            if pd.isna(chunk[field]) or str(chunk[field]).strip() == ""
        )
        for field in [
            "article_id", "title", "authors",
            "year", "venue", "doi", "page_number"
        ]
    }

    print(f"\n{name} chunks")
    print("Total chunks:", len(chunks))

    for field, count in missing_fields.items():
        print(f"Missing {field}: {count}")


Fixed-size chunks
Total chunks: 13669
Missing article_id: 0
Missing title: 84
Missing authors: 84
Missing year: 84
Missing venue: 84
Missing doi: 113
Missing page_number: 0

Sentence-based chunks
Total chunks: 13266
Missing article_id: 0
Missing title: 79
Missing authors: 79
Missing year: 79
Missing venue: 79
Missing doi: 108
Missing page_number: 0


3.5 Inspect an enriched chunk

In [ ]:
# Inspect one enriched sentence-based chunk

sample_chunk = sentence_chunks[1]

for field, value in sample_chunk.items():
    print(f"{field}: {value}")

article_id: 1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf
title: Practical autoencoder based anomaly detection by using vector reconstruction error
authors: Hasan Torabi; Seyedeh Leili Mirtaheri; Sergio Greco
year: 2023.0
venue: Cybersecurity
doi: 10.1186/s42400-022-00134-9
page_number: 1
text: To view a copy of this licence, visit http://creativecommons.org/licenses/by/4.0/. Open Access Cybersecurity Practical autoencoder based anomaly detection by using vector reconstruction error Hasan Torabi1, Seyedeh Leili Mirtaheri1* and Sergio Greco2 Abstract Nowadays, cloud computing provides easy access to a set of variable and configurable computing resources based on user demand through the network. Cloud computing services are available through common internet protocols and network standards. In addition to the unique benefits of cloud computing, insecure communication and attacks on cloud networks cannot be ignored. There are several techniques f

3.6 Identify articles with missing metadata

In [ ]:
# Identify articles with missing metadata

missing_metadata_articles = [
    article_id
    for article_id, info in article_metadata.items()
    if any(
        pd.isna(info[field]) or str(info[field]).strip() == ""
        for field in ["title", "authors", "year", "venue"]
    )
]

print("Articles with incomplete metadata:", len(missing_metadata_articles))

print("\nExamples:")
for article_id in missing_metadata_articles[:15]:
    print("\nFilename:", article_id)
    print("Metadata:", article_metadata[article_id])

Articles with incomplete metadata: 1

Examples:

Filename: nlp006.pdf
Metadata: {'title': nan, 'authors': nan, 'year': nan, 'venue': nan, 'doi': nan}


3.7 Check whether chunk filenames match metadata filenames

In [ ]:
# Compare article filenames in chunks and metadata inventory

chunk_article_ids = set(
    chunk["article_id"] for chunk in sentence_chunks
)

metadata_filenames = set(metadata_df["filename"])

unmatched_filenames = chunk_article_ids - metadata_filenames

print("Unique articles in chunks:", len(chunk_article_ids))
print("Unique metadata filenames:", len(metadata_filenames))
print("Unmatched filenames:", len(unmatched_filenames))

print("\nExamples of unmatched filenames:")
for filename in sorted(unmatched_filenames)[:15]:
    print(filename)

Unique articles in chunks: 162
Unique metadata filenames: 162
Unmatched filenames: 0

Examples of unmatched filenames:


3.8 Remove the excluded article from both chunk collections

In [ ]:
# Exclude the previously screened-out article from both chunk collections

excluded_article = "nlp006.pdf"

fixed_chunks = [
    chunk for chunk in fixed_chunks
    if chunk["article_id"] != excluded_article
]

sentence_chunks = [
    chunk for chunk in sentence_chunks
    if chunk["article_id"] != excluded_article
]

print("Excluded article:", excluded_article)
print("\nRemaining fixed-size chunks:", len(fixed_chunks))
print("Remaining sentence-based chunks:", len(sentence_chunks))

print("\nArticles in fixed-size chunks:",
      len(set(chunk["article_id"] for chunk in fixed_chunks)))

print("Articles in sentence-based chunks:",
      len(set(chunk["article_id"] for chunk in sentence_chunks)))

Excluded article: nlp006.pdf

Remaining fixed-size chunks: 13585
Remaining sentence-based chunks: 13187

Articles in fixed-size chunks: 161
Articles in sentence-based chunks: 161


3.9 Verify metadata completeness again

In [ ]:
# Recheck required metadata after excluding the ineligible article

required_fields = [
    "article_id", "title", "authors",
    "year", "venue", "page_number"
]

for name, chunks in [
    ("Fixed-size", fixed_chunks),
    ("Sentence-based", sentence_chunks)
]:
    missing = sum(
        1 for chunk in chunks
        if any(
            pd.isna(chunk[field]) or str(chunk[field]).strip() == ""
            for field in required_fields
        )
    )

    print(f"\n{name}")
    print("Total chunks:", len(chunks))
    print("Chunks with missing required metadata:", missing)


Fixed-size
Total chunks: 13585
Chunks with missing required metadata: 0

Sentence-based
Total chunks: 13187
Chunks with missing required metadata: 0


3.10 Save the enriched chunks

In [ ]:
# Save both enriched chunking strategies

with open(data_folder / "fixed_size_chunks.pkl", "wb") as file:
    pickle.dump(fixed_chunks, file)

with open(data_folder / "sentence_based_chunks.pkl", "wb") as file:
    pickle.dump(sentence_chunks, file)

print("Enriched chunks saved successfully.")
print("Fixed-size chunks:", len(fixed_chunks))
print("Sentence-based chunks:", len(sentence_chunks))

Enriched chunks saved successfully.
Fixed-size chunks: 13585
Sentence-based chunks: 13187


In [ ]:
# Verify that both chunk files were saved

fixed_file = data_folder / "fixed_size_chunks.pkl"
sentence_file = data_folder / "sentence_based_chunks.pkl"

print("Fixed-size file exists:", fixed_file.exists())
print("Sentence-based file exists:", sentence_file.exists())

print("Fixed-size file size (MB):",
      round(fixed_file.stat().st_size / (1024 * 1024), 2))

print("Sentence-based file size (MB):",
      round(sentence_file.stat().st_size / (1024 * 1024), 2))

Fixed-size file exists: True
Sentence-based file exists: True
Fixed-size file size (MB): 14.39
Sentence-based file size (MB): 13.5


Part 4 — Text Embedding and Vector Index Preparation

4.1 Install required libraries

In [ ]:
# Install libraries for sentence embeddings and vector search

!pip install -q sentence-transformers faiss-cpu

print("Required libraries installed successfully.")

Required libraries installed successfully.


4.2 Import embedding and vector-search libraries

In [ ]:
# Import libraries for generating embeddings and building vector indexes

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Embedding and vector-search libraries imported successfully.")

Embedding and vector-search libraries imported successfully.


4.3 Select and load the embedding model

In [ ]:
# Load the sentence-transformer embedding model

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")
print("Model:", "all-MiniLM-L6-v2")
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.
Model: all-MiniLM-L6-v2
Embedding dimension: 384


/tmp/ipykernel_2379/1563278249.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


4.4 — Generate embeddings for the research chunks

In [ ]:
# Generate embeddings for the sentence-based research chunks

sentence_texts = [chunk["text"] for chunk in sentence_chunks]

sentence_embeddings = embedding_model.encode(
    sentence_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Sentence-based embeddings generated successfully.")
print("Number of embeddings:", sentence_embeddings.shape[0])
print("Embedding dimension:", sentence_embeddings.shape[1])

Batches:   0%|          | 0/413 [00:00<?, ?it/s]

Sentence-based embeddings generated successfully.
Number of embeddings: 13187
Embedding dimension: 384


4.5 Verify the generated embeddings

In [ ]:
# Verify the shape, data type, and normalization of sentence embeddings

print("Embedding shape:", sentence_embeddings.shape)
print("Data type:", sentence_embeddings.dtype)
print("All values finite:", np.isfinite(sentence_embeddings).all())
print("First embedding norm:", np.linalg.norm(sentence_embeddings[0]))

Embedding shape: (13187, 384)
Data type: float32
All values finite: True
First embedding norm: 1.0


4.6 — Build the FAISS vector index

In [ ]:
# Build a FAISS index using normalized sentence embeddings

embedding_dimension = sentence_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)

faiss_index.add(sentence_embeddings)

print("FAISS vector index created successfully.")
print("Number of indexed chunks:", faiss_index.ntotal)
print("Embedding dimension:", faiss_index.d)

FAISS vector index created successfully.
Number of indexed chunks: 13187
Embedding dimension: 384


4.7 — Save the FAISS index and chunk metadata

In [ ]:
# Save the FAISS index and corresponding sentence-based chunks

index_path = data_folder / "sentence_based_faiss.index"
chunks_path = data_folder / "sentence_based_chunks.pkl"

faiss.write_index(faiss_index, str(index_path))

with open(chunks_path, "wb") as file:
    pickle.dump(sentence_chunks, file)

print("FAISS index and chunk metadata saved successfully.")
print("Index file exists:", index_path.exists())
print("Chunk metadata file exists:", chunks_path.exists())

FAISS index and chunk metadata saved successfully.
Index file exists: True
Chunk metadata file exists: True


4.8 — Test semantic search

In [ ]:
# Test semantic search using a research-related query

query = "What machine learning techniques are used for phishing website detection?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = faiss_index.search(query_embedding, 5)

for rank, (score, index) in enumerate(zip(scores[0], indices[0]), start=1):
    chunk = sentence_chunks[index]

    print(f"\nRank: {rank}")
    print(f"Similarity score: {score:.4f}")
    print(f"Article: {chunk['title']}")
    print(f"Year: {chunk['year']}")
    print(f"PDF page: {chunk['page_number']}")
    print(f"Passage: {chunk['text'][:500]}")
    print("-" * 80)


Rank: 1
Similarity score: 0.8530
Article: Modeling Hybrid Feature-Based Phishing Websites Detection Using Machine Learning Techniques
Year: 2024.0
PDF page: 2
Passage: [9–11]. Thus in this paper, we aim to build an effective data-driven solution using machine learning techniques to determine whether or not a website is phishing. Most of the machine learning based phishing detection approaches extract the features from the 123
--------------------------------------------------------------------------------

Rank: 2
Similarity score: 0.8343
Article: A systematic literature review on phishing website detection techniques
Year: 2023.0
PDF page: 2
Passage: Machine Learning: Nowadays, Machine Learning is a prevalent approach for detecting phishing websites (Sindhu et al., 2020). Common attributes such as URL information, website structure, and JavaScript features are collected to represent phishing URLs Fig. 1. Types of phishing attacks. A. Safiand S. Singh Journal of King Saud University –

4.9 — Test a different research query

In [ ]:
# Test semantic search with an intrusion detection research query

query = "How are deep learning models used for network intrusion detection?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = faiss_index.search(query_embedding, 5)

for rank, (score, index) in enumerate(zip(scores[0], indices[0]), start=1):
    chunk = sentence_chunks[index]

    print(f"\nRank: {rank}")
    print(f"Similarity score: {score:.4f}")
    print(f"Article: {chunk['title']}")
    print(f"Year: {chunk['year']}")
    print(f"PDF page: {chunk['page_number']}")
    print(f"Passage: {chunk['text'][:500]}")
    print("-" * 80)


Rank: 1
Similarity score: 0.7779
Article: Recurrent nonsymmetric deep auto encoder approach for network intrusion detection system
Year: 2022.0
PDF page: 2
Passage: Machine learning techniques for intrusion detection systems are being examined, according to a framework for software analysis. Intrusion detection systems are being evaluated for their ability to identify deep learning algorithms, according to an analysis of available methods. There is evidence that machine learning approaches are being Fig. 1. A simple autoencoder. S. M. and R. Kalpana
--------------------------------------------------------------------------------

Rank: 2
Similarity score: 0.7707
Article: A Novel Two-Stage Deep Learning Model for Network Intrusion Detection: LSTM-AE
Year: 2023.0
PDF page: 3
Passage: V. Hnamte et al. : Novel Two-Stage Deep Learning Model for Network Intrusion Detection: LSTM-AE FIGURE 1. Proposed framework—LSTM-AE Model. II. RELATED WORKS Since the invention of computer architectures, t

4.10 — Save the semantic search test results

In [ ]:
# Prepare semantic search results for both test queries

semantic_test_results = []

test_queries = [
    "What machine learning techniques are used for phishing website detection?",
    "How are deep learning models used for network intrusion detection?"
]

for query in test_queries:
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, 5)

    for rank, (score, index) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk = sentence_chunks[index]

        semantic_test_results.append({
            "query": query,
            "rank": rank,
            "similarity_score": float(score),
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "year": chunk["year"],
            "page_number": chunk["page_number"],
            "text": chunk["text"]
        })

print("Semantic search results prepared successfully.")
print("Total retrieved passages:", len(semantic_test_results))

Semantic search results prepared successfully.
Total retrieved passages: 10


4.11 — Save semantic search results

In [ ]:
# Save semantic search results for later retrieval evaluation

semantic_results_df = pd.DataFrame(semantic_test_results)

results_path = data_folder / "semantic_search_test_results.csv"

semantic_results_df.to_csv(results_path, index=False)

print("Semantic search results saved successfully.")
print("File exists:", results_path.exists())
print("Number of saved results:", len(semantic_results_df))

Semantic search results saved successfully.
File exists: True
Number of saved results: 10


Part 5 — BM25 Keyword-Based Retrieval

Objective: Implement BM25 to retrieve research passages based on keyword matching.

Our system will now have two complementary retrieval approaches:
1. Dense retrieval (completed): Finds semantically similar passages using sentence embeddings.
2. Sparse retrieval (upcoming): Finds passages based on matching words and their importance using BM25.

5.1 — Install the BM25 library

In [ ]:
# Install the library for BM25 keyword-based retrieval

!pip install -q rank-bm25

print("BM25 library installed successfully.")

BM25 library installed successfully.


In [ ]:
# Import the BM25 implementation for keyword-based retrieval

from rank_bm25 import BM25Okapi

print("BM25 library imported successfully.")

BM25 library imported successfully.


5.3 — Prepare documents for BM25 indexing

In [ ]:
# Tokenize sentence-based research chunks for BM25 indexing

import re

bm25_tokenized_corpus = [
    re.findall(r"\b\w+\b", chunk["text"].lower())
    for chunk in sentence_chunks
]

print("BM25 corpus prepared successfully.")
print("Number of tokenized chunks:", len(bm25_tokenized_corpus))
print("Example tokens:", bm25_tokenized_corpus[0][:20])

BM25 corpus prepared successfully.
Number of tokenized chunks: 13187
Example tokens: ['torabi', 'et', 'al', 'cybersecurity', '2023', '6', '1', 'https', 'doi', 'org', '10', '1186', 's42400', '022', '00134', '9', 'research', 'the', 'author', 's']


5.4 — Build the BM25 index

In [ ]:
# Build the BM25 index from the tokenized research chunks

bm25_index = BM25Okapi(bm25_tokenized_corpus)

print("BM25 index created successfully.")
print("Number of indexed chunks:", len(bm25_tokenized_corpus))

BM25 index created successfully.
Number of indexed chunks: 13187


5.5 — Test BM25 keyword-based retrieval

In [ ]:
# Test BM25 keyword-based retrieval with a phishing detection query

query = "What machine learning techniques are used for phishing website detection?"

query_tokens = re.findall(r"\b\w+\b", query.lower())

bm25_scores = bm25_index.get_scores(query_tokens)

top_indices = np.argsort(bm25_scores)[::-1][:5]

for rank, index in enumerate(top_indices, start=1):
    chunk = sentence_chunks[index]

    print(f"\nRank: {rank}")
    print(f"BM25 score: {bm25_scores[index]:.4f}")
    print(f"Article: {chunk['title']}")
    print(f"Year: {chunk['year']}")
    print(f"PDF page: {chunk['page_number']}")
    print(f"Passage: {chunk['text'][:500]}")
    print("-" * 80)


Rank: 1
BM25 score: 23.4011
Article: Phishing website detection: How effective are deep <scp>learning‐based</scp> models and hyperparameter optimization?
Year: 2022.0
PDF page: 2
Passage: Lately, researchers have been exploring machine learning-/deep learning- based techniques to detect malware, intru- sion, phishing and website defacement attacks. Deep learning has a wide variety of applications—it is used extensively in image recognition, natural language processing, speech recognition and navigation of self-driving cars.3 In military and aerospace domain, it is used to detect objects from satellites and other identified areas of responsibility that have high importance for the
--------------------------------------------------------------------------------

Rank: 2
BM25 score: 23.2774
Article: Applications of deep learning for phishing detection: a systematic literature review
Year: 2022.0
PDF page: 17
Passage: Applications of deep learning for phishing detection: a... 1473 Table 2

5.6 — Test BM25 with an intrusion detection query

In [ ]:
# Test BM25 keyword-based retrieval with an intrusion detection query

query = "How are deep learning models used for network intrusion detection?"

query_tokens = re.findall(r"\b\w+\b", query.lower())

bm25_scores = bm25_index.get_scores(query_tokens)

top_indices = np.argsort(bm25_scores)[::-1][:5]

for rank, index in enumerate(top_indices, start=1):
    chunk = sentence_chunks[index]

    print(f"\nRank: {rank}")
    print(f"BM25 score: {bm25_scores[index]:.4f}")
    print(f"Article: {chunk['title']}")
    print(f"Year: {chunk['year']}")
    print(f"PDF page: {chunk['page_number']}")
    print(f"Passage: {chunk['text'][:500]}")
    print("-" * 80)


Rank: 1
BM25 score: 18.5349
Article: A Systematic Literature Review of Methods and Datasets for Anomaly-Based Network Intrusion Detection
Year: 2022.0
PDF page: 4
Passage: Z. Yang, X. Liu, T. Li et al. Computers & Security 116 (2022) 102675 Table 2 Research questions . RQ1 Application domains (a) What domains are covered by intrusion detection techniques? (b) How are the studies distributed among the different areas? (c) What are the reasons for this distribution? (d) How does research in these domains differ from country to country? RQ2 Data preprocessing methods (a) What are the common data preprocessing techniques used in network intrusion detection? (b) How ar
--------------------------------------------------------------------------------

Rank: 2
BM25 score: 18.3018
Article: SmartVille: A Framework for Realistic Deep Learning-Based Online Network Intrusion Detection
Year: 2026.0
PDF page: 1
Passage: Representative use cases show how the framework can be used to analyse cur­ ricu

5.7 — Save BM25 test results

In [ ]:
# Prepare BM25 results for both test queries

bm25_test_results = []

test_queries = [
    "What machine learning techniques are used for phishing website detection?",
    "How are deep learning models used for network intrusion detection?"
]

for query in test_queries:
    query_tokens = re.findall(r"\b\w+\b", query.lower())
    bm25_scores = bm25_index.get_scores(query_tokens)
    top_indices = np.argsort(bm25_scores)[::-1][:5]

    for rank, index in enumerate(top_indices, start=1):
        chunk = sentence_chunks[index]

        bm25_test_results.append({
            "query": query,
            "rank": rank,
            "bm25_score": float(bm25_scores[index]),
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "year": chunk["year"],
            "page_number": chunk["page_number"],
            "text": chunk["text"]
        })

print("BM25 test results prepared successfully.")
print("Total retrieved passages:", len(bm25_test_results))

BM25 test results prepared successfully.
Total retrieved passages: 10


5.8 — Save BM25 test results

In [ ]:
# Save BM25 test results for later retrieval evaluation

bm25_results_df = pd.DataFrame(bm25_test_results)

bm25_results_path = data_folder / "bm25_search_test_results.csv"

bm25_results_df.to_csv(bm25_results_path, index=False)

print("BM25 test results saved successfully.")
print("File exists:", bm25_results_path.exists())
print("Number of saved results:", len(bm25_results_df))

BM25 test results saved successfully.
File exists: True
Number of saved results: 10


6 — Hybrid Retrieval Using BM25 and FAISS

Objective: Combine keyword-based retrieval (BM25) and semantic retrieval (FAISS) to improve the retrieval of relevant research passages.

We will use Reciprocal Rank Fusion (RRF) to combine the rankings from both retrieval methods.

6.1 — Define the hybrid retrieval function

In [ ]:
# Define hybrid retrieval using Reciprocal Rank Fusion

def hybrid_retrieval(query, top_k=5, candidate_k=10, rrf_k=60):

    # Retrieve candidates using FAISS
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    dense_scores, dense_indices = faiss_index.search(
        query_embedding, candidate_k
    )

    # Retrieve candidates using BM25
    query_tokens = re.findall(r"\b\w+\b", query.lower())
    sparse_scores = bm25_index.get_scores(query_tokens)

    sparse_indices = np.argsort(sparse_scores)[::-1][:candidate_k]

    # Combine rankings using Reciprocal Rank Fusion
    rrf_scores = {}

    for rank, index in enumerate(dense_indices[0], start=1):
        rrf_scores[index] = rrf_scores.get(index, 0) + 1 / (rrf_k + rank)

    for rank, index in enumerate(sparse_indices, start=1):
        rrf_scores[index] = rrf_scores.get(index, 0) + 1 / (rrf_k + rank)

    # Sort chunks by their combined RRF scores
    ranked_indices = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:top_k]

    results = []

    for rank, index in enumerate(ranked_indices, start=1):
        chunk = sentence_chunks[index]

        results.append({
            "rank": rank,
            "rrf_score": rrf_scores[index],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "year": chunk["year"],
            "page_number": chunk["page_number"],
            "text": chunk["text"]
        })

    return results

print("Hybrid retrieval function defined successfully.")

Hybrid retrieval function defined successfully.


6.2 — Test hybrid retrieval

In [ ]:
# Test hybrid retrieval with a phishing website detection query

query = "What machine learning techniques are used for phishing website detection?"

hybrid_results = hybrid_retrieval(query, top_k=5)

for result in hybrid_results:
    print(f"\nRank: {result['rank']}")
    print(f"RRF score: {result['rrf_score']:.4f}")
    print(f"Article: {result['title']}")
    print(f"Year: {result['year']}")
    print(f"PDF page: {result['page_number']}")
    print(f"Passage: {result['text'][:500]}")
    print("-" * 80)


Rank: 1
RRF score: 0.0164
Article: Modeling Hybrid Feature-Based Phishing Websites Detection Using Machine Learning Techniques
Year: 2024.0
PDF page: 2
Passage: [9–11]. Thus in this paper, we aim to build an effective data-driven solution using machine learning techniques to determine whether or not a website is phishing. Most of the machine learning based phishing detection approaches extract the features from the 123
--------------------------------------------------------------------------------

Rank: 2
RRF score: 0.0164
Article: Phishing website detection: How effective are deep <scp>learning‐based</scp> models and hyperparameter optimization?
Year: 2022.0
PDF page: 2
Passage: Lately, researchers have been exploring machine learning-/deep learning- based techniques to detect malware, intru- sion, phishing and website defacement attacks. Deep learning has a wide variety of applications—it is used extensively in image recognition, natural language processing, speech recognition and

6.3 — Test hybrid retrieval with intrusion detection

In [ ]:
# Test hybrid retrieval with a network intrusion detection query

query = "How are deep learning models used for network intrusion detection?"

hybrid_results = hybrid_retrieval(query, top_k=5)

for result in hybrid_results:
    print(f"\nRank: {result['rank']}")
    print(f"RRF score: {result['rrf_score']:.4f}")
    print(f"Article: {result['title']}")
    print(f"Year: {result['year']}")
    print(f"PDF page: {result['page_number']}")
    print(f"Passage: {result['text'][:500]}")
    print("-" * 80)


Rank: 1
RRF score: 0.0306
Article: Optimizing intrusion detection using a hybrid CNN LSTM deep learning framework on the NSL KDD dataset
Year: 2026.0
PDF page: 3
Passage: Kim et al. specifically emphasize the process of feature learning in Deep Learning for the purpose of network intrusion detection [4]. Their book delves into the exploration of how Deep Learning models may autonomously extract pertinent features from network traffic data, hence obviating the necessity for manual feature engineering. The accuracy and efficacy of IDS might be improved using this approach. Chockwanich and Visoottiviseth introduce an IDS that utilizes deep learning techniques [5]. T
--------------------------------------------------------------------------------

Rank: 2
RRF score: 0.0164
Article: Recurrent nonsymmetric deep auto encoder approach for network intrusion detection system
Year: 2022.0
PDF page: 2
Passage: Machine learning techniques for intrusion detection systems are being examined, accordi

6.4 — Save hybrid retrieval test results

In [ ]:
# Prepare hybrid retrieval results for both research queries

hybrid_test_results = []

test_queries = [
    "What machine learning techniques are used for phishing website detection?",
    "How are deep learning models used for network intrusion detection?"
]

for query in test_queries:
    results = hybrid_retrieval(query, top_k=5)

    for result in results:
        hybrid_test_results.append({
            "query": query,
            "rank": result["rank"],
            "rrf_score": result["rrf_score"],
            "article_id": result["article_id"],
            "title": result["title"],
            "year": result["year"],
            "page_number": result["page_number"],
            "text": result["text"]
        })

print("Hybrid retrieval results prepared successfully.")
print("Total retrieved passages:", len(hybrid_test_results))

Hybrid retrieval results prepared successfully.
Total retrieved passages: 10


6.5 — Save hybrid retrieval results

In [ ]:
# Save hybrid retrieval results for later evaluation

hybrid_results_df = pd.DataFrame(hybrid_test_results)

hybrid_results_path = data_folder / "hybrid_search_test_results.csv"

hybrid_results_df.to_csv(hybrid_results_path, index=False)

print("Hybrid retrieval results saved successfully.")
print("File exists:", hybrid_results_path.exists())
print("Number of saved results:", len(hybrid_results_df))

Hybrid retrieval results saved successfully.
File exists: True
Number of saved results: 10


7 — Transformer-Based Cross-Encoder Reranking

7.1 Install the required library


In [ ]:
# Install the library required for Cross-Encoder reranking
!pip install -q sentence-transformers

print("Cross-Encoder library is ready.")

Cross-Encoder library is ready.


7.2 — Load the Cross-Encoder model

In [ ]:
# Load the pretrained Cross-Encoder reranking model
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Cross-Encoder model loaded successfully.")
print("Model: cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder model loaded successfully.
Model: cross-encoder/ms-marco-MiniLM-L-6-v2


7.3 — Test Cross-Encoder reranking

In [ ]:
# Test Cross-Encoder scoring on hybrid retrieval results

test_query = "What machine learning techniques are used for phishing website detection?"

candidate_results = hybrid_retrieval(
    test_query,
    top_k=10
)

# Prepare query-passage pairs for the Cross-Encoder
reranking_pairs = [
    [test_query, result["text"]]
    for result in candidate_results
]

# Calculate relevance scores
reranking_scores = reranker.predict(reranking_pairs)

# Add scores and sort passages by relevance
reranked_results = []

for result, score in zip(candidate_results, reranking_scores):
    reranked_results.append({
        **result,
        "cross_encoder_score": float(score)
    })

reranked_results = sorted(
    reranked_results,
    key=lambda x: x["cross_encoder_score"],
    reverse=True
)

print("Cross-Encoder reranking test completed.")
print("Number of passages reranked:", len(reranked_results))
print("\nTop 5 reranked passages:")

for result in reranked_results[:5]:
    print("\nTitle:", result["title"])
    print("Year:", result["year"])
    print("Page:", result["page_number"])
    print("Cross-Encoder score:", round(result["cross_encoder_score"], 4))
    print("Passage:", result["text"][:300])

Cross-Encoder reranking test completed.
Number of passages reranked: 10

Top 5 reranked passages:

Title: Applications of deep learning for phishing detection: a systematic literature review
Year: 2022.0
Page: 17
Cross-Encoder score: 8.2007
Passage: Applications of deep learning for phishing detection: a... 1473 Table 2 (continued) Study Review type Focus Description Singh and Meenu [62] Non-systematic Phishing website detection using machine learning Discusses the applications of ML techniques for phishing website detection along with various 

Title: Modeling Hybrid Feature-Based Phishing Websites Detection Using Machine Learning Techniques
Year: 2024.0
Page: 2
Cross-Encoder score: 7.9868
Passage: [9–11]. Thus in this paper, we aim to build an effective data-driven solution using machine learning techniques to determine whether or not a website is phishing. Most of the machine learning based phishing detection approaches extract the features from the 123

Title: A systematic literatu

7.4 — Create a reusable reranking function


In [ ]:
# Define a reusable function for Cross-Encoder reranking

def rerank_results(query, candidate_results, top_k=5):
    # Prepare query-passage pairs
    pairs = [
        [query, result["text"]]
        for result in candidate_results
    ]

    # Calculate Cross-Encoder relevance scores
    scores = reranker.predict(pairs)

    # Attach scores to retrieved passages
    reranked = []

    for result, score in zip(candidate_results, scores):
        reranked.append({
            **result,
            "cross_encoder_score": float(score)
        })

    # Sort passages by Cross-Encoder score
    reranked.sort(
        key=lambda x: x["cross_encoder_score"],
        reverse=True
    )

    return reranked[:top_k]


print("Cross-Encoder reranking function defined successfully.")

Cross-Encoder reranking function defined successfully.


7.5 — Test reranking with both research queries

In [ ]:
# Test Cross-Encoder reranking on phishing and intrusion detection queries

reranking_test_queries = [
    "What machine learning techniques are used for phishing website detection?",
    "How are deep learning models used for network intrusion detection?"
]

reranking_test_results = []

for query in reranking_test_queries:
    # Retrieve 10 hybrid candidates
    candidates = hybrid_retrieval(
        query,
        top_k=10
    )

    # Rerank candidates and keep the top 5
    results = rerank_results(
        query,
        candidates,
        top_k=5
    )

    # Store the results
    for result in results:
        reranking_test_results.append({
            "query": query,
            "rank": result["rank"],
            "article_id": result["article_id"],
            "title": result["title"],
            "year": result["year"],
            "page_number": result["page_number"],
            "rrf_score": result["rrf_score"],
            "cross_encoder_score": result["cross_encoder_score"],
            "text": result["text"]
        })

print("Reranking tests completed successfully.")
print("Total reranked passages:", len(reranking_test_results))

Reranking tests completed successfully.
Total reranked passages: 10


7.6 — Correct and organize reranking results

In [ ]:
# Organize results using the final Cross-Encoder ranking

reranking_results_df = pd.DataFrame(reranking_test_results)

# Preserve the original hybrid rank
reranking_results_df = reranking_results_df.rename(
    columns={"rank": "hybrid_rank"}
)

# Sort by query and Cross-Encoder relevance score
reranking_results_df = reranking_results_df.sort_values(
    ["query", "cross_encoder_score"],
    ascending=[True, False]
).reset_index(drop=True)

# Assign new Cross-Encoder ranking positions
reranking_results_df["rerank_rank"] = (
    reranking_results_df.groupby("query").cumcount() + 1
)

print("Reranking results organized successfully.")
print("Total passages:", len(reranking_results_df))
print("\nRanking comparison:")

print(
    reranking_results_df[
        ["query", "hybrid_rank", "rerank_rank", "title", "cross_encoder_score"]
    ].to_string(index=False)
)

Reranking results organized successfully.
Total passages: 10

Ranking comparison:
                                                                    query  hybrid_rank  rerank_rank                                                                                                                                                title  cross_encoder_score
       How are deep learning models used for network intrusion detection?            7            1                                                Optimizing intrusion detection using a hybrid CNN LSTM deep learning framework on the NSL KDD dataset             8.531256
       How are deep learning models used for network intrusion detection?            4            2                                                                       A Novel Two-Stage Deep Learning Model for Network Intrusion Detection: LSTM-AE             8.001167
       How are deep learning models used for network intrusion detection?            1            3         

7.6 — Save reranking results

In [ ]:
# Save the organized Cross-Encoder reranking results

reranking_results_path = data_folder / "cross_encoder_reranking_results.csv"

reranking_results_df.to_csv(
    reranking_results_path,
    index=False
)

print("Cross-Encoder reranking results saved successfully.")
print("File exists:", reranking_results_path.exists())
print("Number of saved results:", len(reranking_results_df))

Cross-Encoder reranking results saved successfully.
File exists: True
Number of saved results: 10


8 — LLM-Based Evidence-Grounded Answer Generation

8.2 — Install the official Gemini Python SDK

In [ ]:
# Install the official Google Gen AI Python SDK
!pip install -q --upgrade google-genai

print("Google Gemini SDK installed successfully.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.5.0 which is incompatible.
Google Gemini SDK installed successfully.


In [ ]:
# Install compatible Gemini SDK and Google authentication versions
!pip install -q "google-genai>=2.12.1,<3" "google-auth==2.49.0"

print("Compatible Gemini SDK and authentication versions installed.")

Compatible Gemini SDK and authentication versions installed.


8.2 — Verify package versions

In [ ]:
# Verify Gemini SDK and Google authentication package versions
import importlib.metadata as metadata

print("Google Gen AI SDK:", metadata.version("google-genai"))
print("Google authentication:", metadata.version("google-auth"))

Google Gen AI SDK: 2.12.1
Google authentication: 2.49.0


8.3 — Connect to Gemini securely

In [ ]:
# Connect to Gemini using the API key stored in Colab Secrets
from google.colab import userdata
from google import genai

gemini_api_key = userdata.get("GEMINI_API_KEY")

if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY was not found in Colab Secrets.")

gemini_client = genai.Client(api_key=gemini_api_key)

print("Gemini API client initialized successfully.")

Gemini API client initialized successfully.


8.4 — Test Gemini response generation

In [ ]:
# Test Gemini 3.5 Flash-Lite for the NLP research project

test_model = "gemini-3.5-flash-lite"

try:
    interaction = gemini_client.interactions.create(
        model=test_model,
        input="Explain phishing website detection in two simple sentences."
    )

    print("Model:", test_model)
    print("\nGenerated response:")
    print(interaction.output_text)

except Exception as e:
    print("Gemini model test failed:")
    print(str(e))

Gemini model test failed:
name 'gemini_client' is not defined


In [ ]:
# Test Gemini gemini-3.5-flash-lite using the Interactions API
interaction = gemini_client.interactions.create(
    model="gemini-3.5-flash-lite",
    input="Explain the purpose of a literature review in academic research in two sentences."
)

print("Gemini response:")
print(interaction.output_text)

Gemini response:
A literature review synthesizes existing scholarly work on a specific topic to identify gaps, contradictions, or trends in current research. Its primary purpose is to establish a theoretical foundation and justify the need for the researcher's own study within the broader academic conversation.


In [ ]:
# Load the saved chunks and FAISS vector index from Google Drive
import pickle
import faiss
import numpy as np

with open(data_folder / "sentence_based_chunks.pkl", "rb") as f:
    sentence_chunks = pickle.load(f)

with open(data_folder / "fixed_size_chunks.pkl", "rb") as f:
    fixed_chunks = pickle.load(f)

faiss_index = faiss.read_index(str(data_folder / "sentence_based_faiss.index"))

print("Saved research data restored successfully.")
print("Sentence-based chunks:", len(sentence_chunks))
print("Fixed-size chunks:", len(fixed_chunks))
print("FAISS indexed chunks:", faiss_index.ntotal)

Saved research data restored successfully.
Sentence-based chunks: 13187
Fixed-size chunks: 13585
FAISS indexed chunks: 13187


In [ ]:
!pip install faiss-gpu
import faiss


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 MB 59.0 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 11.8 MB/s  0:00:27
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 51.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 36.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 43.6 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [faiss-gpu]


In [ ]:
# Install and import the libraries required for retrieval and Gemini
!pip install -q sentence-transformers faiss-cpu rank-bm25 "google-genai>=2.12.1,<3" "google-auth==2.49.0"

import re
import pickle
import numpy as np
import pandas as pd
import faiss

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from google.colab import userdata
from google import genai

print("Required libraries installed and imported successfully.")

Required libraries installed and imported successfully.


In [ ]:
# Restore the embedding model, BM25 index, and Cross-Encoder reranker
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

bm25_tokenized_corpus = [
    re.findall(r"\b\w+\b", chunk["text"].lower())
    for chunk in sentence_chunks
]
bm25_index = BM25Okapi(bm25_tokenized_corpus)

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Retrieval models and BM25 index restored successfully.")
print("Embedding model: all-MiniLM-L6-v2")
print("BM25 indexed chunks:", len(bm25_tokenized_corpus))
print("Cross-Encoder: cross-encoder/ms-marco-MiniLM-L-6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (12 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Retrieval models and BM25 index restored successfully.
Embedding model: all-MiniLM-L6-v2
BM25 indexed chunks: 13187
Cross-Encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


In [ ]:
# Restore hybrid retrieval using Reciprocal Rank Fusion
def hybrid_retrieval(query, top_k=5, candidate_k=10, rrf_k=60):
    # Retrieve candidates using FAISS
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    dense_scores, dense_indices = faiss_index.search(
        query_embedding, candidate_k
    )

    # Retrieve candidates using BM25
    query_tokens = re.findall(r"\b\w+\b", query.lower())
    sparse_scores = bm25_index.get_scores(query_tokens)
    sparse_indices = np.argsort(sparse_scores)[::-1][:candidate_k]

    # Combine rankings using Reciprocal Rank Fusion
    rrf_scores = {}

    for rank, index in enumerate(dense_indices[0], start=1):
        rrf_scores[index] = rrf_scores.get(index, 0) + 1 / (rrf_k + rank)

    for rank, index in enumerate(sparse_indices, start=1):
        rrf_scores[index] = rrf_scores.get(index, 0) + 1 / (rrf_k + rank)

    # Sort chunks by combined RRF scores
    ranked_indices = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:top_k]

    results = []

    for rank, index in enumerate(ranked_indices, start=1):
        chunk = sentence_chunks[index]

        results.append({
            "rank": rank,
            "rrf_score": rrf_scores[index],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "year": chunk["year"],
            "page_number": chunk["page_number"],
            "text": chunk["text"]
        })

    return results

print("Hybrid retrieval function restored successfully.")

Hybrid retrieval function restored successfully.


Evidence-Grounded Answer Generation

In [ ]:
# Define a reusable function for Cross-Encoder reranking
def rerank_results(query, candidate_results, top_k=5):
    pairs = [
        [query, result["text"]]
        for result in candidate_results
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for result, score in zip(candidate_results, scores):
        reranked.append({
            **result,
            "cross_encoder_score": float(score)
        })

    reranked.sort(
        key=lambda x: x["cross_encoder_score"],
        reverse=True
    )

    return reranked[:top_k]

print("Cross-Encoder reranking function defined successfully.")

Cross-Encoder reranking function defined successfully.


In [ ]:
# Define the improved evidence-grounded answer generation function

def generate_evidence_grounded_answer(query, top_k=5):

    # Retrieve relevant passages using hybrid retrieval
    candidates = hybrid_retrieval(
        query,
        top_k=10,
        candidate_k=20
    )

    # Rerank the retrieved passages
    evidence = rerank_results(
        query,
        candidates,
        top_k=top_k
    )

    # Prepare evidence with complete article metadata
    context = ""

    for i, passage in enumerate(evidence, start=1):

        year = passage["year"]

        if pd.notna(year):
            year = int(float(year))

        context += f"""
Evidence {i}
Article ID: {passage['article_id']}
Article Title: {passage['title']}
Publication Year: {year}
PDF Page Number: {passage['page_number']}
Supporting Passage:
{passage['text']}
"""

    # Prepare the improved evidence-grounded prompt
    prompt = f"""
You are an academic research assistant.

Answer the research question using only the supplied research evidence.

STRICT REQUIREMENTS:

1. Use only claims supported by the supplied passages.
2. Do not invent algorithms, findings, statistics, or references.
3. Every factual research claim must have a citation.
4. Cite each source separately. Never combine multiple articles inside one citation.
5. Follow this exact citation format:

[Article ID: filename, Title: article title, Year: year, PDF Page: page number]

6. Copy article IDs, titles, years, and page numbers exactly from the evidence.
7. Do not describe a result from a review as a general result for all studies.
8. Clearly distinguish findings reported by individual papers from broader synthesis.
9. If evidence is insufficient, state what cannot be established.
10. Use clear academic language.

Research Question:
{query}

Research Evidence:
{context}

Write a structured answer with appropriate citations.
"""

    # Generate the answer using Gemini Flash-Lite
    try:
        interaction = gemini_client.interactions.create(
            model="gemini-3.5-flash-lite",
            input=prompt
        )

        answer = interaction.output_text

    except Exception as e:
        answer = f"Gemini API request failed: {str(e)}"

    return {
        "question": query,
        "answer": answer,
        "evidence": evidence
    }

print("Improved evidence-grounded answer generation function defined successfully.")

Improved evidence-grounded answer generation function defined successfully.


In [ ]:
# Test the improved evidence-grounded answer generation function

test_question = "What machine learning techniques are used for phishing website detection?"

test_result = generate_evidence_grounded_answer(test_question)

print("Research Question:")
print(test_result["question"])

print("\nGenerated Answer:")
print(test_result["answer"])

Research Question:
What machine learning techniques are used for phishing website detection?

Generated Answer:
Based on the provided research evidence, machine learning techniques are widely used and prevalent approaches for detecting phishing websites [Article ID: A_systematic_literature_review_on_phishing_website_detection_techniques.pdf, Title: A systematic literature review on phishing website detection techniques, Year: 2023, PDF Page: 2]. 

Specific findings and aspects of machine learning techniques for phishing website detection include the following:

* **General Performance and Characteristics:** Machine learning-based methods are generally fast and effective, allow for on-the-fly detection of phishing web pages, and cope well with zero-hour attacks [Article ID: Phishing_or_Not_Phishing_A_Survey_on_the_Detection_of_Phishing_Websites.pdf, Title: Phishing or Not Phishing? A Survey on the Detection of Phishing Websites, Year: 2023, PDF Page: 17]. 
* **Performance Factors:** The

In [ ]:
# Retrieve evidence for a question that may be outside the research corpus

unanswerable_test_question = (
    "What was the exact financial loss caused by phishing attacks "
    "in Bhutan in 2025?"
)

# Retrieve and rerank the evidence
unanswerable_candidates = hybrid_retrieval(
    unanswerable_test_question,
    top_k=10,
    candidate_k=20
)

unanswerable_evidence = rerank_results(
    unanswerable_test_question,
    unanswerable_candidates,
    top_k=5
)

# Inspect the retrieved passages
print("Research Question:")
print(unanswerable_test_question)

print("\nRetrieved Evidence:")

for i, passage in enumerate(unanswerable_evidence, start=1):
    print(f"\nEvidence {i}")
    print("Article:", passage["title"])
    print("Article ID:", passage["article_id"])
    print("Page:", passage["page_number"])
    print("Passage:", passage["text"][:500])

Research Question:
What was the exact financial loss caused by phishing attacks in Bhutan in 2025?

Retrieved Evidence:

Evidence 1
Article: CANTINA+
Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Page: 2
Passage: 21:2 G. Xiang et al. 1. INTRODUCTION Phishing is a form of identity theft, in which criminals build replicas of target Web sites and lure unsuspecting victims to disclose their sensitive information like passwords, personal identification numbers (PINs), etc. According to one survey by the Gartner Group [McCall 2007], phishing attacks in the United States caused $3.2 billion loss in 2007, with about 3.6 million victims falling for the attacks, a huge increase from the 2.3 million the year before

Evidence 2
Article: Heuristic machine learning approaches for identifying phishing threats across web and email platforms
Article ID: Heuristic_Machine_Learning_Approaches_for_Identifying_Phishing_Threats_Across_Web_and_Email_Platforms.pdf


In [ ]:
# Test whether Gemini abstains when the retrieved evidence is insufficient

context = ""

for i, passage in enumerate(unanswerable_evidence, start=1):

    year = passage["year"]

    if pd.notna(year):
        year = int(float(year))

    context += f"""
Evidence {i}
Article ID: {passage['article_id']}
Article Title: {passage['title']}
Publication Year: {year}
PDF Page Number: {passage['page_number']}
Supporting Passage:
{passage['text']}
"""

# Prepare the abstention test prompt
abstention_prompt = f"""
You are an academic research assistant.

Answer the research question using only the supplied research evidence.

IMPORTANT INSTRUCTIONS:

1. Do not invent or estimate missing information.
2. Do not use statistics from another country or year as an answer.
3. If the supplied evidence does not directly answer the question,
   clearly state that the information is unavailable in the supplied corpus.
4. Explain briefly why the evidence is insufficient.
5. Do not present unrelated statistics as the answer.
6. Cite evidence only when discussing information actually supported
   by the passages.
7. Use clear academic language.

Research Question:
{unanswerable_test_question}

Research Evidence:
{context}

Provide your answer.
"""

# Generate the response using Gemini Flash-Lite
try:
    abstention_interaction = gemini_client.interactions.create(
        model="gemini-3.5-flash-lite",
        input=abstention_prompt
    )

    abstention_answer = abstention_interaction.output_text

except Exception as e:
    abstention_answer = f"Gemini API request failed: {str(e)}"

print("Research Question:")
print(unanswerable_test_question)

print("\nGenerated Answer:")
print(abstention_answer)

Research Question:
What was the exact financial loss caused by phishing attacks in Bhutan in 2025?

Generated Answer:
Based on the supplied research evidence, the information regarding the exact financial loss caused by phishing attacks in Bhutan in 2025 is unavailable. 

The evidence is insufficient because none of the provided articles or passages contain data, statistics, or mentions regarding Bhutan or financial losses for the year 2025; the texts only discuss general phishing trends, detection frameworks, or historical statistics from other regions (such as the United States in 2007) and time periods.


In [ ]:
# Inspect the research passages used to generate the answer
for i, passage in enumerate(test_result["evidence"], start=1):

    print(f"\n{'=' * 80}")
    print(f"PASSAGE {i}")
    print(f"{'=' * 80}")

    print("Article ID:", passage["article_id"])
    print("Title:", passage["title"])
    print("Year:", passage["year"])
    print("PDF Page:", passage["page_number"])
    print("Cross-Encoder Score:", round(passage["cross_encoder_score"], 4))

    print("\nSupporting Text:")
    print(passage["text"])


PASSAGE 1
Article ID: Applications_of_deep_learning_for_phishing_detection.pdf
Title: Applications of deep learning for phishing detection: a systematic literature review
Year: 2022.0
PDF Page: 17
Cross-Encoder Score: 8.2007

Supporting Text:
Applications of deep learning for phishing detection: a... 1473 Table 2 (continued) Study Review type Focus Description Singh and Meenu [62] Non-systematic Phishing website detection using machine learning Discusses the applications of ML techniques for phishing website detection along with various approaches protection approaches They found that the ML algorithms have achieved an approximate 99% accuracy for phishing website detection by combining 30 features Benavides et al. [12] Systematic Phishing attack solutions using Deep Learning Discusses the applications of DL techniques for phishing website detection Classified and analyzed various DL-based solutions DL algorithms have not been explored enough for phishing website detection Still, ther

9 — Research Question Dataset and Ground Truth

9.1 — Initialize the 80-question dataset

In [ ]:
# Create the structured evaluation dataset for 80 research questions

import pandas as pd
import os

# Define the four question categories
categories = (
    ["Factual"] * 20 +
    ["Conceptual"] * 20 +
    ["Multi-paper Synthesis"] * 20 +
    ["Unanswerable"] * 20
)

# Define the dataset columns
evaluation_columns = [
    "question_id",
    "question_category",
    "question",
    "gold_answer",
    "answerable",
    "evidence_article_ids",
    "evidence_pages",
    "verification_notes"
]

# Create the empty evaluation dataset
evaluation_questions = pd.DataFrame({
    "question_id": [f"Q{i:03d}" for i in range(1, 81)],
    "question_category": categories,
    "question": [""] * 80,
    "gold_answer": [""] * 80,
    "answerable": [""] * 80,
    "evidence_article_ids": [""] * 80,
    "evidence_pages": [""] * 80,
    "verification_notes": [""] * 80
})

# Save the dataset template to Google Drive
evaluation_file = os.path.join(
    data_folder,
    "evaluation_questions.csv"
)

evaluation_questions.to_csv(evaluation_file, index=False)

# Display the dataset structure
print("Evaluation dataset created successfully.")
print("Total questions:", len(evaluation_questions))
print("Saved to:", evaluation_file)

display(evaluation_questions.head())

Evaluation dataset created successfully.
Total questions: 80
Saved to: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/evaluation_questions.csv


,question_id,question_category,question,gold_answer,answerable,evidence_article_ids,evidence_pages,verification_notes
0,Q001,Factual,,,,,,
1,Q002,Factual,,,,,,
2,Q003,Factual,,,,,,
3,Q004,Factual,,,,,,
4,Q005,Factual,,,,,,


A. Inspect the available research papers

In [ ]:
# Load and inspect the research paper inventory

screening_inventory = pd.read_csv(
    os.path.join(data_folder, "screening_inventory.csv")
)

metadata_inventory = pd.read_csv(
    os.path.join(data_folder, "metadata_inventory.csv")
)

print("SCREENING INVENTORY")
print("Total records:", len(screening_inventory))
print("Available columns:", screening_inventory.columns.tolist())

display(screening_inventory.head(5))

print("\nMETADATA INVENTORY")
print("Total records:", len(metadata_inventory))
print("Available columns:", metadata_inventory.columns.tolist())

display(metadata_inventory.head(5))

# Display topic distribution if a topic column is available
topic_columns = [
    col for col in screening_inventory.columns
    if "topic" in col.lower() or "category" in col.lower()
]

for col in topic_columns:
    print(f"\nDistribution by {col}:")
    display(screening_inventory[col].value_counts(dropna=False))

SCREENING INVENTORY
Total records: 162
Available columns: ['filename', 'doi_candidate', 'title', 'authors', 'year', 'venue', 'crossref_status', 'topic', 'subtopic', 'screening_decision', 'exclusion_reason', 'publication_verification', 'verification_source', 'doi_normalized']


,filename,doi_candidate,title,authors,year,venue,crossref_status,topic,subtopic,screening_decision,exclusion_reason,publication_verification,verification_source,doi_normalized
0,1. Practical autoencoder based anomaly detecti...,10.1186/s42400-022-00134-9,Practical autoencoder based anomaly detection ...,Hasan Torabi; Seyedeh Leili Mirtaheri; Sergio ...,2023.0,Cybersecurity,Retrieved,Cybersecurity and Threat Detection,Anomaly Detection,Pending,NaN,Peer-reviewed journal verified,Springer Nature journal peer-review policy,10.1186/s42400-022-00134-9
1,11. A network intrusion detection method based...,10.1016/j.procs.2020.06.055,A network intrusion detection method based on ...,Yuening Zhang; Yiming Zhang; Nan Zhang; Mingzh...,2020.0,Procedia Computer Science,Retrieved,Cybersecurity and Threat Detection,Intrusion Detection,Pending,NaN,Publication record verified,Elsevier ScienceDirect,10.1016/j.procs.2020.06.055
2,12. Network intrusion detection using feature ...,10.1186/s40537-023-00834-0,Network intrusion detection using feature fusi...,Abiodun Ayantayo; Amrit Kaur; Anit Kour; Xavie...,2023.0,Journal of Big Data,Retrieved,Cybersecurity and Threat Detection,Intrusion Detection,Pending,NaN,Peer-reviewed journal verified,DOAJ journal record and Springer publisher policy,10.1186/s40537-023-00834-0
3,13. A sequential deep learning framework for a...,10.1016/j.cose.2024.103928,A sequential deep learning framework for a rob...,Soumyadeep Hore; Jalal Ghadermazi; Ankit Shah;...,2024.0,Computers & Security,Retrieved,Cybersecurity and Threat Detection,Intrusion Detection,Pending,NaN,Peer-review policy verified,Official Elsevier peer-review policy,10.1016/j.cose.2024.103928
4,14. End-to-end deep learning for smart maritim...,10.1038/s41598-025-19450-4,End-to-end deep learning for smart maritime th...,R. Anuja; J. Annrose,2025.0,Scientific Reports,Retrieved,Cybersecurity and Threat Detection,Anomaly Detection,Pending,NaN,Peer-reviewed journal verified,Official Scientific Reports journal website,10.1038/s41598-025-19450-4



METADATA INVENTORY
Total records: 162
Available columns: ['filename', 'doi_candidate', 'title', 'authors', 'year', 'venue', 'crossref_status']


,filename,doi_candidate,title,authors,year,venue,crossref_status
0,1. Practical autoencoder based anomaly detecti...,10.1186/s42400-022-00134-9,Practical autoencoder based anomaly detection ...,Hasan Torabi; Seyedeh Leili Mirtaheri; Sergio ...,2023.0,Cybersecurity,Retrieved
1,11. A network intrusion detection method based...,10.1016/j.procs.2020.06.055,A network intrusion detection method based on ...,Yuening Zhang; Yiming Zhang; Nan Zhang; Mingzh...,2020.0,Procedia Computer Science,Retrieved
2,12. Network intrusion detection using feature ...,10.1186/s40537-023-00834-0,Network intrusion detection using feature fusi...,Abiodun Ayantayo; Amrit Kaur; Anit Kour; Xavie...,2023.0,Journal of Big Data,Retrieved
3,13. A sequential deep learning framework for a...,10.1016/j.cose.2024.103928,A sequential deep learning framework for a rob...,Soumyadeep Hore; Jalal Ghadermazi; Ankit Shah;...,2024.0,Computers & Security,Retrieved
4,14. End-to-end deep learning for smart maritim...,10.1038/s41598-025-19450-4,End-to-end deep learning for smart maritime th...,R. Anuja; J. Annrose,2025.0,Scientific Reports,Retrieved



Distribution by topic:


,count
topic,
Cybersecurity and Threat Detection,162



Distribution by subtopic:


,count
subtopic,
Intrusion Detection,65
Phishing Detection,38
Anomaly Detection,32
Datasets and Benchmarking,26
Excluded,1


9B — Inspect the candidate research papers

In [ ]:
# Inspect screening decisions and candidate research papers

# Display the current screening decision counts
print("SCREENING DECISION COUNTS")
display(
    screening_inventory["screening_decision"]
    .value_counts(dropna=False)
    .to_frame("Number of Articles")
)

# Display publication verification counts
print("\nPUBLICATION VERIFICATION COUNTS")
display(
    screening_inventory["publication_verification"]
    .value_counts(dropna=False)
    .to_frame("Number of Articles")
)

# Exclude records already marked as excluded
candidate_papers = screening_inventory[
    screening_inventory["screening_decision"] != "Excluded"
].copy()

# Display the number of candidate papers
print("\nCandidate papers:", len(candidate_papers))

# Show candidate papers grouped by research subtopic
for topic, group in candidate_papers.groupby("subtopic"):
    print(f"\n{topic}: {len(group)} papers")

    display(
        group[["filename", "title", "year", "venue"]]
        .head(5)
    )

SCREENING DECISION COUNTS


,Number of Articles
screening_decision,
Pending,161
Excluded,1



PUBLICATION VERIFICATION COUNTS


,Number of Articles
publication_verification,
Peer-reviewed journal verified,76
Peer-review policy verified,37
Publication record verified,25
Peer-reviewed journal policy verified,13
NaN,10
Not checked,1



Candidate papers: 161

Anomaly Detection: 32 papers


,filename,title,year,venue
0,1. Practical autoencoder based anomaly detecti...,Practical autoencoder based anomaly detection ...,2023.0,Cybersecurity
4,14. End-to-end deep learning for smart maritim...,End-to-end deep learning for smart maritime th...,2025.0,Scientific Reports
9,20. A cross-attention CNN–LSTM fusion model fo...,A cross-attention CNN–LSTM fusion model for ne...,2026.0,Scientific Reports
13,26. Analysis of Autoencoders for Network Intru...,Analysis of Autoencoders for Network Intrusion...,2021.0,Sensors
14,27. Network intrusion detection for cyber secu...,Network intrusion detection for cyber security...,2017.0,2017 IEEE National Aerospace and Electronics C...


... (2 outputs left out)


Intrusion Detection: 65 papers


,filename,title,year,venue
1,11. A network intrusion detection method based...,A network intrusion detection method based on ...,2020.0,Procedia Computer Science
2,12. Network intrusion detection using feature ...,Network intrusion detection using feature fusi...,2023.0,Journal of Big Data
3,13. A sequential deep learning framework for a...,A sequential deep learning framework for a rob...,2024.0,Computers & Security
5,15. Intrusion detection in the internet of thi...,Intrusion detection in the internet of things ...,2025.0,Cybersecurity
6,17. Deep learning enabled intrusion detection ...,Deep learning enabled intrusion detection syst...,2025.0,EURASIP Journal on Wireless Communications and...



Phishing Detection: 38 papers


,filename,title,year,venue
40,A_Feature_Rich_Machine_Learning_Framework_for_...,CANTINA+,2011.0,ACM Transactions on Information and System Sec...
41,A_Hybrid_Approach_for_Phishing_Email_Detection...,A Hybrid Approach for Phishing Email Detection...,2025.0,2025 5th International Conference on Pervasive...
43,A_State-of-the-Art_Review_on_Phishing_Website_...,A State-of-the-Art Review on Phishing Website ...,2024.0,IEEE Access
44,A_data_centric_and_interpretable_PCA_based_fra...,A data centric and interpretable PCA based fra...,2026.0,Journal of Electrical Systems and Information ...
46,A_systematic_literature_review_on_phishing_web...,A systematic literature review on phishing web...,2023.0,Journal of King Saud University - Computer and...


In [ ]:
# Display actual research passages for evaluation question development

print("Total sentence-based chunks:", len(sentence_chunks))

print("\nAvailable fields:")
print(sentence_chunks[0].keys())

# Display five sample passages
for i, passage in enumerate(sentence_chunks[:5], start=1):

    print(f"\n--- Passage {i} ---")
    print("Article ID:", passage["article_id"])
    print("Title:", passage["title"])
    print("Year:", passage["year"])
    print("PDF Page:", passage["page_number"])
    print("Text:", passage["text"][:500])

Total sentence-based chunks: 13187

Available fields:
dict_keys(['article_id', 'title', 'authors', 'year', 'venue', 'doi', 'page_number', 'text'])

--- Passage 1 ---
Article ID: 1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf
Title: Practical autoencoder based anomaly detection by using vector reconstruction error
Year: 2023.0
PDF Page: 1
Text: Torabi et al. Cybersecurity (2023) 6:1 https://doi.org/10.1186/s42400-022-00134-9 RESEARCH © The Author(s) 2023. Open Access This article is licensed under a Creative Commons Attribution 4.0 International License, which permits use, sharing, adaptation, distribution and reproduction in any medium or format, as long as you give appropriate credit to the original author(s) and the source, provide a link to the Creative Commons licence, and indicate if changes were made. The images or other third p

--- Passage 2 ---
Article ID: 1. Practical autoencoder based anomaly detection by using vector reconstruction

Inspect representative research articles

In [ ]:
# Display representative passages from different cybersecurity topics

topics_to_inspect = [
    "phishing",
    "intrusion detection",
    "anomaly detection",
    "dataset"
]

for topic in topics_to_inspect:

    print("\n" + "=" * 80)
    print("TOPIC:", topic.upper())
    print("=" * 80)

    matching_passages = [
        passage for passage in sentence_chunks
        if topic.lower() in (
            passage["title"] + " " + passage["text"][:500]
        ).lower()
    ]

    # Select passages from different articles
    seen_articles = set()
    displayed = 0

    for passage in matching_passages:

        if passage["article_id"] in seen_articles:
            continue

        seen_articles.add(passage["article_id"])

        print("\nArticle:", passage["title"])
        print("Article ID:", passage["article_id"])
        print("Year:", passage["year"])
        print("PDF Page:", passage["page_number"])
        print("Passage:", passage["text"][:700])

        displayed += 1

        if displayed == 3:
            break


TOPIC: PHISHING

Article: Online Self-Supervised Deep Learning for Intrusion Detection Systems
Article ID: 30. Online Self-Supervised Deep Learning for Intrusion Detection Systems (2024).pdf
Year: 2024.0
PDF Page: 3
Passage: 5670 IEEE TRANSACTIONS ON INFORMATION FORENSICS AND SECURITY, VOL. 19, 2024 (CNN) for phishing and using Long Short-Term Memory (LSTM) for Botnet attacks. The authors evaluated the perfor- mance of this method also on the N-BaIoT dataset achieving 94.8% accuracy. CNN was also used by Liu et al. [32] with features that are processed by the triangle area maps based multivariate correlation analysis algorithm. In recent work [33], Bovenzi et al. employed DL models, specifically using Auto Encoders (AEs) and KitNET, for unsupervised early anomaly detection in IoT datasets, namely IoT-23 [34] and Kitsune [17]. The results of [33] demonstrated the potential for early anomaly detection in IoT netwo

Article: A survey of intrusion detection datasets for communication netw

9.3 — Select articles using the screening inventory

In [ ]:
# Check the article categories recorded in the screening inventory

import os
import pandas as pd

inventory_path = os.path.join(data_folder, "screening_inventory.csv")
screening_df = pd.read_csv(inventory_path)

print("Available inventory columns:")
print(screening_df.columns.tolist())

print("\nArticle counts by subtopic:")
print(screening_df["subtopic"].value_counts(dropna=False))

print("\nSample articles from each subtopic:")

for topic in screening_df["subtopic"].dropna().unique():

    print("\n" + "=" * 70)
    print("SUBTOPIC:", topic)
    print("=" * 70)

    selected = screening_df[
        screening_df["subtopic"] == topic
    ][["filename", "title", "year", "screening_decision"]].head(3)

    print(selected.to_string(index=False))

Available inventory columns:
['filename', 'doi_candidate', 'title', 'authors', 'year', 'venue', 'crossref_status', 'topic', 'subtopic', 'screening_decision', 'exclusion_reason', 'publication_verification', 'verification_source', 'doi_normalized']

Article counts by subtopic:
subtopic
Intrusion Detection          65
Phishing Detection           38
Anomaly Detection            32
Datasets and Benchmarking    26
Excluded                      1
Name: count, dtype: int64

Sample articles from each subtopic:

SUBTOPIC: Anomaly Detection
                                                                                          filename                                                                                       title   year screening_decision
         1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf          Practical autoencoder based anomaly detection by using vector reconstruction error 2023.0            Pending
14. End-to-end deep learning 

9.4 — Inspect passages from correctly classified articles

In [ ]:
# Inspect passages from articles selected by their assigned subtopic

topic_keywords = {
    "Phishing Detection": [
        "phishing", "malicious website", "phishing email",
        "URL", "website detection"
    ],
    "Intrusion Detection": [
        "intrusion detection", "network traffic",
        "intrusion detection system", "IDS"
    ],
    "Anomaly Detection": [
        "anomaly detection", "anomalies",
        "reconstruction error", "outlier"
    ],
    "Datasets and Benchmarking": [
        "dataset", "benchmark", "NSL-KDD",
        "CIC-IDS", "UNSW-NB15"
    ]
}

for topic, keywords in topic_keywords.items():

    print("\n" + "=" * 90)
    print("SUBTOPIC:", topic)
    print("=" * 90)

    selected_articles = screening_df[
        screening_df["subtopic"] == topic
    ]["filename"].dropna().head(2).tolist()

    for article_id in selected_articles:

        article_passages = [
            passage for passage in sentence_chunks
            if passage["article_id"] == article_id
        ]

        relevant_passages = [
            passage for passage in article_passages
            if any(
                keyword.lower() in passage["text"].lower()
                for keyword in keywords
            )
        ]

        print("\nARTICLE:", article_id)
        print("Matching passages:", len(relevant_passages))

        for passage in relevant_passages[:2]:

            print("\nPDF Page:", passage["page_number"])
            print("Passage:", passage["text"][:700])


SUBTOPIC: Phishing Detection

ARTICLE: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Matching passages: 87

PDF Page: 1
Passage: 21 CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites GUANG XIANG, JASON HONG, CAROLYN P. ROSE, and LORRIE CRANOR, Carnegie Mellon University Phishing is a plague in cyberspace. Typically, phish detection methods either use human-verified URL blacklists or exploit Web page features via machine learning techniques. However, the former is frail in terms of new phish, and the latter suffers from the scarcity of effective features and the high false positive rate (FP). To alleviate those problems, we propose a layered anti-phishing solution that aims at (1) exploiting the expressiveness of a rich set of features with machine learning to achieve a high true posit

PDF Page: 1
Passage: Moreover, we designed two filters to help reduce FP and achieve runtime speedup. The first is a near-duplicate phi

9.5 — Add the first eight factual questions

In [ ]:
# Add the first eight evidence-grounded factual questions

factual_questions = [
    {
        "question_id": "Q001",
        "question_category": "Factual",
        "question": "How many phishing and legitimate webpages were used to evaluate CANTINA+?",
        "gold_answer": "The evaluation used 8,118 phishing webpages and 4,883 legitimate webpages.",
        "answerable": True,
        "evidence_article_ids": "A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage describing the CANTINA+ evaluation corpus."
    },
    {
        "question_id": "Q002",
        "question_category": "Factual",
        "question": "What true-positive and false-positive rates did CANTINA+ report in its randomized evaluation?",
        "gold_answer": "CANTINA+ achieved over 92% true positive on unique testing phishing pages, over 99% on near-duplicate testing phishing pages, and about 0.4% false positive with 10% training phishing pages.",
        "answerable": True,
        "evidence_article_ids": "A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage describing the randomized evaluation results."
    },
    {
        "question_id": "Q003",
        "question_category": "Factual",
        "question": "Which cryptographic protocols are discussed for validating sender identity and data integrity in the hybrid phishing email detection approach?",
        "gold_answer": "The protocols are SPF, DKIM, DMARC, S/MIME, and PGP.",
        "answerable": True,
        "evidence_article_ids": "A_Hybrid_Approach_for_Phishing_Email_Detection_using_AI_and_Cryptography.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage listing the cryptographic protocols."
    },
    {
        "question_id": "Q004",
        "question_category": "Factual",
        "question": "What three feature-fusion deep learning models are proposed for network intrusion detection by Ayantayo et al. (2023)?",
        "gold_answer": "The three models are early-fusion, late-fusion, and late-ensemble learning models.",
        "answerable": True,
        "evidence_article_ids": "12. Network intrusion detection using feature fusion with deep learning.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage describing the proposed feature-fusion models."
    },
    {
        "question_id": "Q005",
        "question_category": "Factual",
        "question": "Which two datasets are used to evaluate the feature-fusion models in Ayantayo et al. (2023)?",
        "gold_answer": "The UNSW-NB15 and NSL-KDD datasets.",
        "answerable": True,
        "evidence_article_ids": "12. Network intrusion detection using feature fusion with deep learning.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage identifying the datasets used for evaluation."
    },
    {
        "question_id": "Q006",
        "question_category": "Factual",
        "question": "How is reconstruction error used in the autoencoder-based anomaly detection method by Torabi et al. (2023)?",
        "gold_answer": "Reconstruction error is used as an anomaly detection or classification metric.",
        "answerable": True,
        "evidence_article_ids": "1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage explaining the role of reconstruction error."
    },
    {
        "question_id": "Q007",
        "question_category": "Factual",
        "question": "Which three deep learning components are combined in the maritime threat detection framework?",
        "gold_answer": "The framework combines an Autoencoder (AE), a Convolutional Neural Network (CNN), and a Long Short-Term Memory (LSTM) network.",
        "answerable": True,
        "evidence_article_ids": "14. End-to-end deep learning for smart maritime threat detection an AE–CNN–LSTM-based approach.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage describing the proposed AE-CNN-LSTM framework."
    },
    {
        "question_id": "Q008",
        "question_category": "Factual",
        "question": "Which dataset is described as widely used for evaluating anomaly detection systems in the paper on the KDD Cup 99 dataset?",
        "gold_answer": "The KDD Cup 99 dataset.",
        "answerable": True,
        "evidence_article_ids": "A Detailed Analysis of the KDD CUP 99 Data Set.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage describing KDD CUP 99 as a widely used evaluation dataset."
    }
]

# Load the existing evaluation question template
evaluation_df = pd.read_csv(
    os.path.join(data_folder, "evaluation_questions.csv")
)

# Update only the first eight factual question rows
for question in factual_questions:

    row_index = evaluation_df.index[
        evaluation_df["question_id"] == question["question_id"]
    ]

    if len(row_index) == 1:
        for column, value in question.items():
            evaluation_df.loc[row_index[0], column] = value

# Save the updated evaluation dataset
evaluation_df.to_csv(
    os.path.join(data_folder, "evaluation_questions.csv"),
    index=False
)

print("First eight factual questions saved successfully.")
print("\nCompleted questions:")
print(evaluation_df[
    evaluation_df["question"].notna()
    & (evaluation_df["question"].astype(str).str.strip() != "")
][["question_id", "question", "answerable"]].to_string(index=False))

print("\nTotal populated questions:",
      evaluation_df["question"].notna().sum())

First eight factual questions saved successfully.

Completed questions:
question_id                                                                                                                                       question answerable
       Q001                                                                      How many phishing and legitimate webpages were used to evaluate CANTINA+?       True
       Q002                                                  What true-positive and false-positive rates did CANTINA+ report in its randomized evaluation?       True
       Q003 Which cryptographic protocols are discussed for validating sender identity and data integrity in the hybrid phishing email detection approach?       True
       Q004                         What three feature-fusion deep learning models are proposed for network intrusion detection by Ayantayo et al. (2023)?       True
       Q005                                                   Which two datasets are used to evalu

/tmp/ipykernel_718/780073791.py:100: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'How many phishing and legitimate webpages were used to evaluate CANTINA+?' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evaluation_df.loc[row_index[0], column] = value
/tmp/ipykernel_718/780073791.py:100: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The evaluation used 8,118 phishing webpages and 4,883 legitimate webpages.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evaluation_df.loc[row_index[0], column] = value
/tmp/ipykernel_718/780073791.py:100: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a co

9.6 — Fix the CSV column data types

In [ ]:
# Set evaluation dataset columns to appropriate data types

evaluation_path = os.path.join(
    data_folder, "evaluation_questions.csv"
)

evaluation_df = pd.read_csv(evaluation_path)

text_columns = [
    "question_id",
    "question_category",
    "question",
    "gold_answer",
    "evidence_article_ids",
    "evidence_pages",
    "verification_notes"
]

for column in text_columns:
    evaluation_df[column] = evaluation_df[column].astype("string")

evaluation_df["answerable"] = evaluation_df["answerable"].astype("boolean")

evaluation_df.to_csv(evaluation_path, index=False)

print("Column data types corrected.")
print("\nData types:")
print(evaluation_df.dtypes)

print("\nPopulated questions:",
      evaluation_df["question"].notna().sum())

Column data types corrected.

Data types:
question_id             string[python]
question_category       string[python]
question                string[python]
gold_answer             string[python]
answerable                     boolean
evidence_article_ids    string[python]
evidence_pages          string[python]
verification_notes      string[python]
dtype: object

Populated questions: 8


9.7 — Find evidence for Q009–Q020

In [ ]:
# Retrieve additional passages from articles not yet used in factual questions

used_articles = set(
    evaluation_df.loc[
        evaluation_df["question"].notna()
        & (evaluation_df["question"].str.strip() != ""),
        "evidence_article_ids"
    ].dropna().str.split(";").explode().str.strip()
)

topics_to_inspect = [
    "Phishing Detection",
    "Intrusion Detection",
    "Anomaly Detection",
    "Datasets and Benchmarking"
]

for topic in topics_to_inspect:

    print("\n" + "=" * 90)
    print("SUBTOPIC:", topic)
    print("=" * 90)

    remaining_articles = screening_df[
        (screening_df["subtopic"] == topic)
        & (~screening_df["filename"].isin(used_articles))
    ]["filename"].dropna().head(3).tolist()

    for article_id in remaining_articles:

        article_passages = [
            passage for passage in sentence_chunks
            if passage["article_id"] == article_id
        ]

        print("\nARTICLE:", article_id)

        displayed = 0

        for passage in article_passages:

            text = passage["text"].strip()

            # Skip very short passages and copyright-only text
            if len(text) < 250:
                continue

            print("\nPDF Page:", passage["page_number"])
            print("Passage:", text[:800])

            displayed += 1

            if displayed == 2:
                break


SUBTOPIC: Phishing Detection

ARTICLE: A_State-of-the-Art_Review_on_Phishing_Website_Detection_Techniques.pdf

PDF Page: 1
Passage: Received 18 November 2024, accepted 6 December 2024, date of publication 11 December 2024, date of current version 19 December 2024. Digital Object Identifier 10.1109/ACCESS.2024.3514972 A State-of-the-Art Review on Phishing Website Detection Techniques WENHAO LI 1, (Graduate Student Member, IEEE), SELVAKUMAR MANICKAM 1, (Member, IEEE), YUNG-WEY CHONG 2, WEILAN LENG3, AND PRIYADARSI NANDA 4, (Senior Member, IEEE) 1Cybersecurity Research Centre, Universiti Sains Malaysia, George Town, Penang 11800, Malaysia 2School of Computer Sciences, Universiti Sains Malaysia, George Town, Penang 11800, Malaysia 3Research Institute of Drilling and Production Engineering Technology, Chuanqing Drilling Engineering Company Ltd., CNPC, Guanghan 618300, China 4Faculty of Engineering and IT, University of Tech

PDF Page: 1
Passage: The rapid evolution of phishing tactics has 

In [ ]:
# Add factual questions Q009 to Q020 using the inspected research passages

new_factual_questions = [
    {
        "question_id": "Q009",
        "question_category": "Factual",
        "question": "Which three emerging approaches to phishing website detection are highlighted in the 2024 state-of-the-art review?",
        "gold_answer": "The review highlights graph-based approaches, large language model (LLM)-based approaches, and phishing kit-based detection methods.",
        "answerable": True,
        "evidence_article_ids": "A_State-of-the-Art_Review_on_Phishing_Website_Detection_Techniques.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the abstract describing emerging phishing detection methods."
    },
    {
        "question_id": "Q010",
        "question_category": "Factual",
        "question": "How many scientific papers did the systematic literature review on phishing website detection examine?",
        "gold_answer": "The systematic literature review examined 80 scientific papers.",
        "answerable": True,
        "evidence_article_ids": "A_systematic_literature_review_on_phishing_website_detection_techniques.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage describing the scope of the systematic literature survey."
    },
    {
        "question_id": "Q011",
        "question_category": "Factual",
        "question": "How many sequential deep neural network architectures are included in the multistage intrusion detection framework proposed by Hore et al. (2024)?",
        "gold_answer": "The framework contains three sequential deep neural network architectures: one classifier and two specific autoencoders.",
        "answerable": True,
        "evidence_article_ids": "13. A sequential deep learning framework for a robust and resilient network intrusion detection system.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage describing the framework architecture."
    },
    {
        "question_id": "Q012",
        "question_category": "Factual",
        "question": "Which three types of attacks does the multistage intrusion detection framework aim to handle?",
        "gold_answer": "The framework aims to handle zero-day attacks, out-of-distribution (OOD) attacks, and adversarial evasion attacks.",
        "answerable": True,
        "evidence_article_ids": "13. A sequential deep learning framework for a robust and resilient network intrusion detection system.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage describing the framework's intended detection capabilities."
    },
    {
        "question_id": "Q013",
        "question_category": "Factual",
        "question": "Which two unsupervised deep learning techniques are used for feature extraction and dimensionality reduction in the 2017 network intrusion detection study?",
        "gold_answer": "The study uses an Autoencoder (AE) and a Restricted Boltzmann Machine (RBM).",
        "answerable": True,
        "evidence_article_ids": "27. Network intrusion detection for cyber security using unsupervised deep learning approaches (2017).pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage describing the unsupervised techniques."
    },
    {
        "question_id": "Q014",
        "question_category": "Factual",
        "question": "What detection accuracy values are reported for the Autoencoder and RBM with K-means approaches in the 2017 unsupervised intrusion detection study?",
        "gold_answer": "The study reports approximately 91.86% detection accuracy using the Autoencoder and 92.12% using RBM with K-means.",
        "answerable": True,
        "evidence_article_ids": "27. Network intrusion detection for cyber security using unsupervised deep learning approaches (2017).pdf",
        "evidence_pages": "1",
        "verification_notes": "The passage reports the two accuracy values in the order AE and RBM with K-means, respectively."
    },
    {
        "question_id": "Q015",
        "question_category": "Factual",
        "question": "Which two datasets are used to evaluate the cross-attention CNN-LSTM network traffic anomaly detection model?",
        "gold_answer": "The model is evaluated using the ToN-IoT and CIC-IoT 2023 datasets.",
        "answerable": True,
        "evidence_article_ids": "20. A cross-attention CNN–LSTM fusion model for network traffic anomaly detection (2026).pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage describing the evaluation datasets."
    },
    {
        "question_id": "Q016",
        "question_category": "Factual",
        "question": "What accuracy does the cross-attention CNN-LSTM model achieve on the ToN-IoT dataset?",
        "gold_answer": "The model achieves 99.42% accuracy on the ToN-IoT dataset.",
        "answerable": True,
        "evidence_article_ids": "20. A cross-attention CNN–LSTM fusion model for network traffic anomaly detection (2026).pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the reported experimental accuracy."
    },
    {
        "question_id": "Q017",
        "question_category": "Factual",
        "question": "What accuracy does the cross-attention CNN-LSTM model achieve on the CIC-IoT 2023 dataset?",
        "gold_answer": "The model achieves 99.44% accuracy on the CIC-IoT 2023 dataset.",
        "answerable": True,
        "evidence_article_ids": "20. A cross-attention CNN–LSTM fusion model for network traffic anomaly detection (2026).pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the reported experimental accuracy."
    },
    {
        "question_id": "Q018",
        "question_category": "Factual",
        "question": "What AUC values does the cross-attention CNN-LSTM model report on the ToN-IoT and CIC-IoT 2023 datasets?",
        "gold_answer": "The model reports an AUC of 0.9997 on ToN-IoT and 0.9988 on CIC-IoT 2023.",
        "answerable": True,
        "evidence_article_ids": "20. A cross-attention CNN–LSTM fusion model for network traffic anomaly detection (2026).pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage reporting AUC values for both datasets."
    },
    {
        "question_id": "Q019",
        "question_category": "Factual",
        "question": "Which four techniques are incorporated into the cross-attention CNN-LSTM model to improve generalization and reduce overfitting?",
        "gold_answer": "The model incorporates Gaussian noise injection at the input, weight regularization, dropout, and label smoothing.",
        "answerable": True,
        "evidence_article_ids": "20. A cross-attention CNN–LSTM fusion model for network traffic anomaly detection (2026).pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage describing the model's regularization techniques."
    },
    {
        "question_id": "Q020",
        "question_category": "Factual",
        "question": "How many studies in the systematic literature review on phishing website detection applied machine learning techniques?",
        "gold_answer": "The review reports that machine learning techniques were applied in 57 studies.",
        "answerable": True,
        "evidence_article_ids": "A_systematic_literature_review_on_phishing_website_detection_techniques.pdf",
        "evidence_pages": "1",
        "verification_notes": "Based on the passage reporting the number of studies using machine learning techniques."
    }
]

# Load the current evaluation dataset
evaluation_path = os.path.join(data_folder, "evaluation_questions.csv")
evaluation_df = pd.read_csv(evaluation_path)

# Ensure text columns can store text without dtype warnings
text_columns = [
    "question_id",
    "question_category",
    "question",
    "gold_answer",
    "evidence_article_ids",
    "evidence_pages",
    "verification_notes"
]

for column in text_columns:
    evaluation_df[column] = evaluation_df[column].astype("string")

# Update the existing question rows
for question in new_factual_questions:

    row_index = evaluation_df.index[
        evaluation_df["question_id"] == question["question_id"]
    ]

    if len(row_index) != 1:
        raise ValueError(
            f"Expected exactly one row for {question['question_id']}"
        )

    for column, value in question.items():
        evaluation_df.loc[row_index[0], column] = value

# Save the updated evaluation dataset
evaluation_df.to_csv(evaluation_path, index=False)

# Display the completed factual questions
completed = evaluation_df[
    evaluation_df["question_category"] == "Factual"
].dropna(subset=["question"])

print("Questions Q009–Q020 saved successfully.")
print("\nTotal populated factual questions:", len(completed))
print("\nQuestion IDs:")
print(completed["question_id"].tolist())

print("\nTotal populated questions in the dataset:",
      evaluation_df["question"].notna().sum())

Questions Q009–Q020 saved successfully.

Total populated factual questions: 20

Question IDs:
['Q001', 'Q002', 'Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 'Q019', 'Q020']

Total populated questions in the dataset: 20


9.9 — Validate the 20 factual questions

In [ ]:
# Validate the first 20 factual evaluation questions

import pandas as pd

evaluation_path = os.path.join(
    data_folder, "evaluation_questions.csv"
)

evaluation_df = pd.read_csv(evaluation_path)

factual_df = evaluation_df[
    evaluation_df["question_category"] == "Factual"
].copy()

# Check for missing required fields
required_columns = [
    "question_id",
    "question",
    "gold_answer",
    "answerable",
    "evidence_article_ids",
    "evidence_pages"
]

print("1. Missing required values:")

for column in required_columns:
    missing = factual_df[column].isna().sum()
    print(f"{column}: {missing}")

# Check duplicate question IDs
print("\n2. Duplicate question IDs:")
print(factual_df[
    factual_df["question_id"].duplicated(keep=False)
][["question_id", "question"]])

# Build article and page references from the corpus
article_pages = {}

for passage in sentence_chunks:
    article_id = passage["article_id"]
    page = str(int(passage["page_number"]))

    article_pages.setdefault(article_id, set()).add(page)

# Check whether every evidence reference exists
invalid_references = []

for _, row in factual_df.iterrows():

    article_ids = [
        item.strip()
        for item in str(row["evidence_article_ids"]).split(";")
    ]

    pages = [
        item.strip()
        for item in str(row["evidence_pages"]).split(";")
    ]

    if len(article_ids) != len(pages):
        invalid_references.append(
            (row["question_id"], "Article/page count mismatch")
        )
        continue

    for article_id, page in zip(article_ids, pages):

        if article_id not in article_pages:
            invalid_references.append(
                (row["question_id"], f"Article not found: {article_id}")
            )

        elif page not in article_pages[article_id]:
            invalid_references.append(
                (row["question_id"], f"Page {page} not found: {article_id}")
            )

print("\n3. Invalid article/page references:")
if invalid_references:
    for issue in invalid_references:
        print(issue)
else:
    print("No invalid article/page references found.")

print("\n4. Final factual question count:", len(factual_df))
print("5. All questions marked answerable:",
      factual_df["answerable"].eq(True).all())

1. Missing required values:
question_id: 0
question: 0
gold_answer: 0
answerable: 0
evidence_article_ids: 0
evidence_pages: 0

2. Duplicate question IDs:
Empty DataFrame
Columns: [question_id, question]
Index: []

3. Invalid article/page references:
('Q001', 'Page 1.0 not found: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf')
('Q002', 'Page 1.0 not found: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf')
('Q003', 'Page 1.0 not found: A_Hybrid_Approach_for_Phishing_Email_Detection_using_AI_and_Cryptography.pdf')
('Q004', 'Page 1.0 not found: 12. Network intrusion detection using feature fusion with deep learning.pdf')
('Q005', 'Page 1.0 not found: 12. Network intrusion detection using feature fusion with deep learning.pdf')
('Q006', 'Page 1.0 not found: 1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf')
('Q007', 'Page 1.0 not found: 14. End-to-end deep learning for smart maritime thr

In [ ]:
# Check article IDs and page numbers using the saved list-based corpus.

import pandas as pd

# Load the evaluation questions.
evaluation_df = pd.read_csv(f"{data_folder}/evaluation_questions.csv")

# Inspect the sentence-based chunk structure.
print("SENTENCE CHUNKS")
print("Type:", type(sentence_chunks))
print("Number of chunks:", len(sentence_chunks))
print("First chunk:", sentence_chunks[0])

# Inspect the first question.
test_article = str(evaluation_df.loc[0, "evidence_article_ids"]).strip()
test_page = evaluation_df.loc[0, "evidence_pages"]

print("\nFIRST QUESTION")
print("Question ID:", evaluation_df.loc[0, "question_id"])
print("Article ID:", test_article)
print("Page reference:", repr(test_page))

# Find the article in the original page-level records.
matching_pages = [
    record for record in page_records
    if str(record.get("article_id", "")).strip() == test_article
]

print("\nPAGE-LEVEL RECORD MATCH")
print("Matching page records:", len(matching_pages))

if matching_pages:
    print("Available PDF pages:")
    print(sorted(set(record["page_number"] for record in matching_pages)))
else:
    print("No exact article ID match in page_documents.pkl")

# Find the article in the sentence-based chunks.
matching_chunks = [
    record for record in sentence_chunks
    if str(record.get("article_id", "")).strip() == test_article
]

print("\nSENTENCE-CHUNK MATCH")
print("Matching chunks:", len(matching_chunks))

if matching_chunks:
    print("Available chunk page numbers:")
    print(sorted(set(record["page_number"] for record in matching_chunks)))
else:
    print("No exact article ID match in sentence_based_chunks.pkl")

# Check whether the article filename appears under a slightly different name.
if not matching_pages:
    print("\nSIMILAR ARTICLE IDs IN PAGE RECORDS:")
    for article_id in sorted(set(
        str(record.get("article_id", "")) for record in page_records
    )):
        if "Feature_Rich_Machine_Learning" in article_id:
            print(article_id)

SENTENCE CHUNKS
Type: <class 'list'>
Number of chunks: 13187
First chunk: {'article_id': '1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf', 'title': 'Practical autoencoder based anomaly detection by using vector reconstruction error', 'authors': 'Hasan Torabi; Seyedeh Leili Mirtaheri; Sergio Greco', 'year': 2023.0, 'venue': 'Cybersecurity', 'doi': '10.1186/s42400-022-00134-9', 'page_number': 1, 'text': 'Torabi et al. Cybersecurity (2023) 6:1 https://doi.org/10.1186/s42400-022-00134-9 RESEARCH © The Author(s) 2023. Open Access This article is licensed under a Creative Commons Attribution 4.0 International License, which permits use, sharing, adaptation, distribution and reproduction in any medium or format, as long as you give appropriate credit to the original author(s) and the source, provide a link to the Creative Commons licence, and indicate if changes were made. The images or other third party material in this article are included in the a

In [ ]:
# Validate evaluation questions against the original page-level corpus.

import pandas as pd
import re

# Convert the page-level records into a set of valid article-page pairs.
valid_article_pages = {
    (
        str(record["article_id"]).strip(),
        int(record["page_number"])
    )
    for record in page_records
    if record.get("article_id") is not None
    and record.get("page_number") is not None
}

# Load the evaluation dataset.
evaluation_df = pd.read_csv(
    f"{data_folder}/evaluation_questions.csv"
)

# Check required values.
required_columns = [
    "question_id",
    "question",
    "gold_answer",
    "answerable",
    "evidence_article_ids",
    "evidence_pages"
]

missing_values = evaluation_df[required_columns].isna().sum()

print("1. MISSING REQUIRED VALUES")
print(missing_values)

# Check duplicate question IDs.
duplicates = evaluation_df[
    evaluation_df["question_id"].duplicated(keep=False)
]

print("\n2. DUPLICATE QUESTION IDs")
print(duplicates[["question_id", "question"]])

# Normalize article and page references.
def split_references(value):
    if pd.isna(value):
        return []
    return [
        item.strip()
        for item in re.split(r"[;\n]", str(value))
        if item.strip()
    ]

def normalize_page(value):
    number = pd.to_numeric(value, errors="coerce")
    if pd.isna(number) or not float(number).is_integer():
        return None
    return int(number)

# Validate every article-page reference.
invalid_references = []

for _, row in evaluation_df.iterrows():
    article_ids = split_references(row["evidence_article_ids"])
    pages = split_references(row["evidence_pages"])

    if len(article_ids) != len(pages):
        invalid_references.append(
            (row["question_id"], "Article/page reference count mismatch")
        )
        continue

    for article_id, page_value in zip(article_ids, pages):
        page_number = normalize_page(page_value)

        if page_number is None:
            invalid_references.append(
                (row["question_id"], f"Invalid page number: {page_value}")
            )
        elif (article_id, page_number) not in valid_article_pages:
            invalid_references.append(
                (row["question_id"], f"Article/page not found: {article_id}, page {page_number}")
            )

print("\n3. INVALID ARTICLE/PAGE REFERENCES")
if invalid_references:
    for item in invalid_references:
        print(item)
else:
    print("None")

# Summarize the factual questions.
factual_df = evaluation_df[
    evaluation_df["question_id"].astype(str).str.match(r"Q\d{3}")
    & evaluation_df["question_id"].astype(str).str.extract(r"Q(\d{3})")[0].astype(float).between(1, 20)
]

print("\n4. FINAL VALIDATION SUMMARY")
print("Total dataset rows:", len(evaluation_df))
print("Factual questions Q001–Q020:", len(factual_df))
print("All factual questions answerable:",
      factual_df["answerable"].astype(str).str.lower().eq("true").all())
print("Missing required values:", int(missing_values.sum()))
print("Duplicate question IDs:", len(duplicates))
print("Invalid references:", len(invalid_references))

1. MISSING REQUIRED VALUES
question_id              0
question                60
gold_answer             60
answerable              60
evidence_article_ids    60
evidence_pages          60
dtype: int64

2. DUPLICATE QUESTION IDs
Empty DataFrame
Columns: [question_id, question]
Index: []

3. INVALID ARTICLE/PAGE REFERENCES
None

4. FINAL VALIDATION SUMMARY
Total dataset rows: 80
Factual questions Q001–Q020: 20
All factual questions answerable: True
Missing required values: 300
Duplicate question IDs: 0
Invalid references: 0


Part 9.10 — Conceptual question evidence inspection (Q021–Q040).

In [ ]:
# Inspect conceptual passages from the four research subtopics.

import pandas as pd

# Identify articles already used in Q001–Q020.
used_questions = evaluation_df[
    evaluation_df["question_id"].isin(
        [f"Q{i:03d}" for i in range(1, 21)]
    )
]

used_article_ids = set(
    used_questions["evidence_article_ids"]
    .dropna()
    .astype(str)
    .str.split(";")
    .explode()
    .str.strip()
)

# Select unused articles from each subtopic.
subtopics = [
    "Phishing Detection",
    "Intrusion Detection",
    "Anomaly Detection",
    "Datasets and Benchmarking"
]

for subtopic in subtopics:
    print("\n" + "=" * 90)
    print("SUBTOPIC:", subtopic)
    print("=" * 90)

    selected_articles = screening_df[
        (screening_df["subtopic"] == subtopic)
        & (~screening_df["filename"].isin(used_article_ids))
        & (screening_df["screening_decision"] != "Excluded")
    ].head(3)

    for _, article in selected_articles.iterrows():
        article_id = article["filename"]

        article_chunks = [
            chunk for chunk in sentence_chunks
            if chunk["article_id"] == article_id
        ]

        print("\nARTICLE:", article_id)
        print("TITLE:", article["title"])

        # Display the first two available chunks for inspection.
        for chunk in article_chunks[:2]:
            print("\nPDF PAGE:", chunk["page_number"])
            print("PASSAGE:", chunk["text"][:1800])
            print("-" * 70)


SUBTOPIC: Phishing Detection

ARTICLE: A_data_centric_and_interpretable_PCA_based_framework_for_phishing_email_detection.pdf
TITLE: A data centric and interpretable PCA based framework for phishing email detection

PDF PAGE: 1
PASSAGE: Journal of Electrical Systems and Information Technology RESEARCH Open Access Adeyemo et al. Journal of Electrical Systems and Information Technology (2026) 13:56 https://doi.org/10.1186/s43067-026-00359-5 A data centric and interpretable PCA based framework for phishing email detection Adetoye A. Adeyemo1* , Oladayo G. Atanda2, Adedoyin I. Oyebade3, Busolami E. Oluwadamilare2, Olugbemiga S. Popoola2, Olufemi M. Oyelami3 and Oluwaseyi S. Erinfolami4 *Correspondence: Adetoye A. Adeyemo adetoye.adeyemo@tech-u.edu.ng 1Computer Engineering Department, Faculty of Engineering and Technology, Abiola Ajimobi Technical University, Ibadan, Oyo State, Nigeria 2Cyber Security Programme, College of Computing and Communication Studies, Bowen University, Iwo, Osun Sta

9.11 — Save Q021–Q040

In [ ]:
# Add 20 evidence-based conceptual questions to the evaluation dataset.

import pandas as pd

# Prepare the 20 conceptual questions.
conceptual_questions = [
    {
        "question_id": "Q021",
        "question_category": "Conceptual",
        "question": "Why does the PhishBench framework evaluate phishing detection features under identical experimental conditions?",
        "gold_answer": "Using the same system specifications, datasets, classifiers, and evaluation metrics allows existing phishing detection features to be compared systematically and fairly.",
        "answerable": True,
        "evidence_article_ids": "An_In-Depth_Benchmarking_and_Evaluation_of_Phishing_Detection_Research_for_Security_Needs.pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the description of PhishBench and its unified experimental conditions."
    },
    {
        "question_id": "Q022",
        "question_category": "Conceptual",
        "question": "Why should researchers consider class imbalance and changing legitimate-to-phishing ratios when evaluating phishing detection systems?",
        "gold_answer": "Because the imbalanced nature of phishing attacks and changes in the ratio of legitimate to phishing samples can affect classification performance.",
        "answerable": True,
        "evidence_article_ids": "An_In-Depth_Benchmarking_and_Evaluation_of_Phishing_Detection_Research_for_Security_Needs.pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the benchmarking study's discussion of dataset characteristics and classification performance."
    },
    {
        "question_id": "Q023",
        "question_category": "Conceptual",
        "question": "What does the finding that retraining alone is insufficient to defeat new phishing attacks imply for evaluating detection methods?",
        "gold_answer": "Detection methods should also be evaluated for their robustness and scalability on new and diverse datasets rather than relying only on retraining.",
        "answerable": True,
        "evidence_article_ids": "An_In-Depth_Benchmarking_and_Evaluation_of_Phishing_Detection_Research_for_Security_Needs.pdf",
        "evidence_pages": "1",
        "verification_notes": "The paper reports that retraining alone is insufficient and describes testing methods on new and diverse datasets to assess robustness and scalability."
    },
    {
        "question_id": "Q024",
        "question_category": "Conceptual",
        "question": "Why is explainability important when using deep learning for phishing detection?",
        "gold_answer": "Deep learning models can behave as black boxes, making their decisions difficult to understand and limiting user trust and operational transparency. Explainable AI methods such as LIME and SHAP can help interpret model predictions.",
        "answerable": True,
        "evidence_article_ids": "An_Intelligent_Phishing_Detection_System_Using_Deep_Learning_and_Explainable_AI.pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the paper's discussion of black-box limitations and explainable AI."
    },
    {
        "question_id": "Q025",
        "question_category": "Conceptual",
        "question": "What data-related problems motivate the development of a data-centric framework for phishing email detection?",
        "gold_answer": "Existing approaches can be affected by outdated and imbalanced datasets, which are identified as shortcomings that motivate improved data-centric detection approaches.",
        "answerable": True,
        "evidence_article_ids": "A_data_centric_and_interpretable_PCA_based_framework_for_phishing_email_detection.pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the available abstract passage on page 1."
    },
    {
        "question_id": "Q026",
        "question_category": "Conceptual",
        "question": "What limitations of traditional network intrusion detection methods motivated the proposed Auto-Encoder and LSTM approach?",
        "gold_answer": "Traditional methods can have difficulty capturing long-distance dependencies and contextual information. High data dimensionality and complex feature extraction also create challenges for real-time detection and accuracy.",
        "answerable": True,
        "evidence_article_ids": "11. A network intrusion detection method based on deep learning with higher accuracy.pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the abstract passage describing the motivation for the Auto-Encoder and LSTM method."
    },
    {
        "question_id": "Q027",
        "question_category": "Conceptual",
        "question": "Why is interpretability a concern when machine learning and deep learning models are used in intrusion detection systems?",
        "gold_answer": "As models become more complex, security analysts may find it difficult to understand the reasoning behind specific alerts, making interpretation of detection decisions challenging.",
        "answerable": True,
        "evidence_article_ids": "15. Intrusion detection in the internet of things using convolutional neural networks an explainable AI approach.pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the abstract's discussion of model complexity and interpretability."
    },
    {
        "question_id": "Q028",
        "question_category": "Conceptual",
        "question": "How do CNNs and GRUs complement each other in the proposed hybrid intrusion detection system for IoT networks?",
        "gold_answer": "CNNs extract spatial features from network traffic, while GRUs model temporal dependencies. Combining these capabilities is suitable for detecting dynamic and complex IoT intrusions.",
        "answerable": True,
        "evidence_article_ids": "17. Deep learning enabled intrusion detection system for IoT security (2025).pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the description of CNN and GRU capabilities in the abstract."
    },
    {
        "question_id": "Q029",
        "question_category": "Conceptual",
        "question": "Why may traditional machine learning approaches be insufficient for intrusion detection in large-scale IoT environments?",
        "gold_answer": "Traditional machine learning approaches may not adequately handle the complexity of large-scale and unstructured data found in IoT environments.",
        "answerable": True,
        "evidence_article_ids": "17. Deep learning enabled intrusion detection system for IoT security (2025).pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the abstract's explanation of the limitations of traditional machine learning."
    },
    {
        "question_id": "Q030",
        "question_category": "Conceptual",
        "question": "What is the main security role of an intrusion detection system in IoT environments?",
        "gold_answer": "An intrusion detection system helps mitigate the impact of cybercrime and safeguard systems against different malicious attacks.",
        "answerable": True,
        "evidence_article_ids": "17. Deep learning enabled intrusion detection system for IoT security (2025).pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the introduction discussing the role of IDS."
    },
    {
        "question_id": "Q031",
        "question_category": "Conceptual",
        "question": "Why have autoencoders been explored for detecting unknown or zero-day network attacks?",
        "gold_answer": "Autoencoders can help detect unknown attack types, including zero-day attacks, while also reducing the need for laborious data labelling.",
        "answerable": True,
        "evidence_article_ids": "26. Analysis of Autoencoders for Network Intrusion Detection (2021).pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the abstract's explanation of autoencoder applications in NIDS."
    },
    {
        "question_id": "Q032",
        "question_category": "Conceptual",
        "question": "What practical challenge makes it difficult to develop an effective autoencoder architecture for network intrusion detection?",
        "gold_answer": "Finding the optimal autoencoder architecture and hyperparameter settings requires considerable time and effort, even though autoencoders can help detect unknown attacks.",
        "answerable": True,
        "evidence_article_ids": "26. Analysis of Autoencoders for Network Intrusion Detection (2021).pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the abstract's discussion of architecture and hyperparameter selection."
    },
    {
        "question_id": "Q033",
        "question_category": "Conceptual",
        "question": "How does network pruning make a stacked autoencoder-based intrusion detection system more suitable for resource-constrained edge devices?",
        "gold_answer": "Pruning removes relatively ineffective neurons based on their importance, reducing the number of model parameters and computational complexity while maintaining similar attack detection quality.",
        "answerable": True,
        "evidence_article_ids": "33. Stacked Autoencoder-Based Probabilistic Feature Extraction for On-Device Network Intrusion Detection (2022 issue).pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the description of probabilistic pruning and the reported reduction in computational complexity."
    },
    {
        "question_id": "Q034",
        "question_category": "Conceptual",
        "question": "Why is the pruned and pretrained encoder connected to a separate classifier network in the proposed on-device intrusion detection approach?",
        "gold_answer": "The separate classifier performs attack type inference while allowing the pruned and pretrained encoder to be reused, avoiding the need for full retraining from scratch.",
        "answerable": True,
        "evidence_article_ids": "33. Stacked Autoencoder-Based Probabilistic Feature Extraction for On-Device Network Intrusion Detection (2022 issue).pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the abstract's description of the encoder and separate classifier."
    },
    {
        "question_id": "Q035",
        "question_category": "Conceptual",
        "question": "What optimization problem does the Artificial Bee Colony algorithm address in neural network training, and why does the proposed approach modify it?",
        "gold_answer": "Conventional back-propagation can become trapped in local minima, while global search strategies can be slow and may have low detection rates. The Artificial Bee Colony algorithm helps avoid local minima, but the study notes that modifications are needed.",
        "answerable": True,
        "evidence_article_ids": "28. Network anomaly detection using Deep Autoencoder and parallel Artificial Bee Colony algorithm-trained neural network (2024).pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the discussion of ANN optimization and the proposed modified ABC approach."
    },
    {
        "question_id": "Q036",
        "question_category": "Conceptual",
        "question": "Why are datasets important when evaluating network intrusion detection systems?",
        "gold_answer": "Datasets provide the data needed to evaluate how well an intrusion detection system can identify intrusion behaviour.",
        "answerable": True,
        "evidence_article_ids": "A Detailed Analysis of Benchmark Datasets for Network Intrusion Detection System.pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the abstract explaining the role of datasets in NIDS evaluation."
    },
    {
        "question_id": "Q037",
        "question_category": "Conceptual",
        "question": "Why is access to good historical network data important for developing intrusion detection systems?",
        "gold_answer": "Developing intrusion detection systems relies on access to good historical network data, which supports the creation and development of systems that identify anomalous network behaviour.",
        "answerable": True,
        "evidence_article_ids": "A review on intrusion detection datasets tools, processes, and features.pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the review's abstract."
    },
    {
        "question_id": "Q038",
        "question_category": "Conceptual",
        "question": "How do network traffic analysis tools contribute to the creation of intrusion detection datasets?",
        "gold_answer": "Network traffic analysis tools are used in the dataset creation process. The reviewed work categorizes these tools and describes how dataset creation can combine one or more tool categories.",
        "answerable": True,
        "evidence_article_ids": "A review on intrusion detection datasets tools, processes, and features.pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the review's description of traffic analysis tools and dataset creation."
    },
    {
        "question_id": "Q039",
        "question_category": "Conceptual",
        "question": "Why should researchers consider dataset features when developing or evaluating network intrusion detection systems?",
        "gold_answer": "Dataset features influence the effectiveness of network intrusion detection systems, so understanding their impact is important when developing and evaluating these systems.",
        "answerable": True,
        "evidence_article_ids": "A review on intrusion detection datasets tools, processes, and features.pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the review's discussion of the impact of dataset features."
    },
    {
        "question_id": "Q040",
        "question_category": "Conceptual",
        "question": "According to the review of intrusion detection datasets, what security weaknesses can make computer systems more vulnerable to network attacks?",
        "gold_answer": "System loopholes, ineffective security policies, and a lack of knowledge about attacks and cybercrime can increase the vulnerability of computer systems to network attacks.",
        "answerable": True,
        "evidence_article_ids": "A Review of the Advancement in Intrusion Detection Datasets.pdf",
        "evidence_pages": "1",
        "verification_notes": "Supported by the introduction passage discussing factors that increase network attack targets."
    }
]

# Load the existing evaluation dataset.
evaluation_df = pd.read_csv(
    f"{data_folder}/evaluation_questions.csv"
)

# Ensure text columns can store strings.
text_columns = [
    "question_id",
    "question_category",
    "question",
    "gold_answer",
    "evidence_article_ids",
    "evidence_pages",
    "verification_notes"
]

for column in text_columns:
    evaluation_df[column] = evaluation_df[column].astype("string")

# Update the existing Q021–Q040 rows.
new_questions_df = pd.DataFrame(conceptual_questions)

for _, question_row in new_questions_df.iterrows():
    question_id = question_row["question_id"]
    row_index = evaluation_df.index[
        evaluation_df["question_id"] == question_id
    ]

    if len(row_index) != 1:
        raise ValueError(
            f"Expected exactly one row for {question_id}, found {len(row_index)}"
        )

    for column in new_questions_df.columns:
        evaluation_df.loc[row_index[0], column] = question_row[column]

# Save the updated evaluation dataset.
evaluation_df.to_csv(
    f"{data_folder}/evaluation_questions.csv",
    index=False
)

print("Questions Q021–Q040 saved successfully.")
print("\nConceptual questions saved:", len(new_questions_df))
print("\nSubtopic distribution:")

print(
    pd.Series({
        "Phishing Detection": 5,
        "Intrusion Detection": 5,
        "Anomaly Detection": 5,
        "Datasets and Benchmarking": 5
    })
)

print("\nTotal populated questions:",
      evaluation_df["question"].notna().sum())

print("\nSaved question IDs:")
print(
    evaluation_df[
        evaluation_df["question_id"].isin(
            [f"Q{i:03d}" for i in range(21, 41)]
        )
    ]["question_id"].tolist()
)

Questions Q021–Q040 saved successfully.

Conceptual questions saved: 20

Subtopic distribution:
Phishing Detection           5
Intrusion Detection          5
Anomaly Detection            5
Datasets and Benchmarking    5
dtype: int64

Total populated questions: 40

Saved question IDs:
['Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'Q026', 'Q027', 'Q028', 'Q029', 'Q030', 'Q031', 'Q032', 'Q033', 'Q034', 'Q035', 'Q036', 'Q037', 'Q038', 'Q039', 'Q040']


In [ ]:
# Validate the newly saved conceptual questions.

import pandas as pd
import re

evaluation_df = pd.read_csv(
    f"{data_folder}/evaluation_questions.csv"
)

# Select Q021–Q040.
conceptual_df = evaluation_df[
    evaluation_df["question_id"].isin(
        [f"Q{i:03d}" for i in range(21, 41)]
    )
].copy()

# Check required fields.
required_columns = [
    "question_id",
    "question",
    "gold_answer",
    "answerable",
    "evidence_article_ids",
    "evidence_pages"
]

print("1. CONCEPTUAL QUESTION COUNT:", len(conceptual_df))
print("\n2. MISSING VALUES:")
print(conceptual_df[required_columns].isna().sum())

# Check duplicate IDs.
print("\n3. DUPLICATE QUESTION IDs:")
print(conceptual_df[
    conceptual_df["question_id"].duplicated(keep=False)
][["question_id", "question"]])

# Build valid article-page references from the original corpus.
valid_article_pages = {
    (
        str(record["article_id"]).strip(),
        int(record["page_number"])
    )
    for record in page_records
    if record.get("article_id") is not None
    and record.get("page_number") is not None
}

# Validate the references.
invalid_references = []

for _, row in conceptual_df.iterrows():
    article_ids = [
        x.strip() for x in re.split(
            r"[;\n]", str(row["evidence_article_ids"])
        ) if x.strip()
    ]

    pages = [
        x.strip() for x in re.split(
            r"[;\n]", str(row["evidence_pages"])
        ) if x.strip()
    ]

    if len(article_ids) != len(pages):
        invalid_references.append(
            (row["question_id"], "Article/page count mismatch")
        )
        continue

    for article_id, page in zip(article_ids, pages):
        page_number = pd.to_numeric(page, errors="coerce")

        if pd.isna(page_number) or not float(page_number).is_integer():
            invalid_references.append(
                (row["question_id"], f"Invalid page: {page}")
            )
        elif (article_id, int(page_number)) not in valid_article_pages:
            invalid_references.append(
                (row["question_id"], f"Invalid reference: {article_id}, page {page}")
            )

print("\n4. INVALID ARTICLE/PAGE REFERENCES:")
print(invalid_references if invalid_references else "None")

print("\n5. ANSWERABILITY:")
print(conceptual_df["answerable"].value_counts(dropna=False))

print("\n6. TOTAL POPULATED QUESTIONS:",
      evaluation_df["question"].notna().sum())

1. CONCEPTUAL QUESTION COUNT: 20

2. MISSING VALUES:
question_id             0
question                0
gold_answer             0
answerable              0
evidence_article_ids    0
evidence_pages          0
dtype: int64

3. DUPLICATE QUESTION IDs:
Empty DataFrame
Columns: [question_id, question]
Index: []

4. INVALID ARTICLE/PAGE REFERENCES:
None

5. ANSWERABILITY:
answerable
True    20
Name: count, dtype: int64

6. TOTAL POPULATED QUESTIONS: 40


In [ ]:
# Find the exact article filename for Q040 in the saved corpus.

search_term = "advancement in intrusion detection datasets"

matching_ids = sorted({
    record["article_id"]
    for record in page_records
    if search_term.lower() in record["article_id"].lower()
})

print("Matching article IDs:")

for article_id in matching_ids:
    print(article_id)

    available_pages = sorted({
        record["page_number"]
        for record in page_records
        if record["article_id"] == article_id
    })

    print("Available pages:", available_pages)
    print("-" * 60)

Matching article IDs:
A review of the advancement in intrusion detection datasets.pdf
Available pages: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
------------------------------------------------------------


In [ ]:
# Correct the article filename in Q040 and save the evaluation dataset.

q040_index = evaluation_df.index[
    evaluation_df["question_id"] == "Q040"
][0]

evaluation_df.loc[q040_index, "evidence_article_ids"] = (
    "A review of the advancement in intrusion detection datasets.pdf"
)

evaluation_df.to_csv(
    f"{data_folder}/evaluation_questions.csv",
    index=False
)

print("Q040 reference corrected successfully.")
print(
    evaluation_df.loc[
        evaluation_df["question_id"] == "Q040",
        ["question_id", "evidence_article_ids", "evidence_pages"]
    ].to_string(index=False)
)

Q040 reference corrected successfully.
question_id                                            evidence_article_ids  evidence_pages
       Q040 A review of the advancement in intrusion detection datasets.pdf             1.0


In [ ]:
# Identify unused articles for multi-paper synthesis questions.

# Get article IDs already used in the first 40 questions.
used_article_ids = set()

for article_list in evaluation_df[
    evaluation_df["question_id"].isin(
        [f"Q{i:03d}" for i in range(1, 41)]
    )
]["evidence_article_ids"].dropna():

    for article_id in str(article_list).split(";"):
        article_id = article_id.strip()

        if article_id:
            used_article_ids.add(article_id)

# Identify articles that have not been used yet.
unused_articles = screening_df[
    ~screening_df["filename"].isin(used_article_ids)
].copy()

# Display unused articles by subtopic.
print("UNUSED ARTICLES BY SUBTOPIC")
print(unused_articles["subtopic"].value_counts(dropna=False))

print("\nREPRESENTATIVE UNUSED ARTICLES")

for subtopic in [
    "Phishing Detection",
    "Intrusion Detection",
    "Anomaly Detection",
    "Datasets and Benchmarking"
]:
    print(f"\n{subtopic}")

    selected_articles = unused_articles[
        unused_articles["subtopic"] == subtopic
    ][["filename", "title", "year"]].head(5)

    print(selected_articles.to_string(index=False))

UNUSED ARTICLES BY SUBTOPIC
subtopic
Intrusion Detection          60
Phishing Detection           31
Anomaly Detection            25
Datasets and Benchmarking    22
Excluded                      1
Name: count, dtype: int64

REPRESENTATIVE UNUSED ARTICLES

Phishing Detection
                                                                                                                   filename                                                                                                                   title   year
An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf An explainable deep learning framework for multi-class malicious URL classification using priority-coefficient analysis 2026.0
                        An_intelligent_identification_and_classification_system_for_malicious_uniform_resource_locators.pdf                  An intelligent identification and classification system for malicious uniform reso

In [ ]:
# Inspect passages from unused articles for multi-paper synthesis questions.

# Select three unused articles from each relevant subtopic.
selected_synthesis_articles = []

for subtopic in [
    "Phishing Detection",
    "Intrusion Detection",
    "Anomaly Detection",
    "Datasets and Benchmarking"
]:
    candidates = unused_articles[
        (unused_articles["subtopic"] == subtopic) &
        (unused_articles["screening_decision"] != "Excluded")
    ].head(3)

    selected_synthesis_articles.extend(
        candidates.to_dict("records")
    )

# Display the first-page passages from each selected article.
for article in selected_synthesis_articles:
    article_id = article["filename"]

    article_chunks = [
        chunk for chunk in sentence_chunks
        if chunk["article_id"] == article_id
        and int(chunk["page_number"]) == 1
    ]

    print("\n" + "=" * 100)
    print("SUBTOPIC:", article["subtopic"])
    print("TITLE:", article["title"])
    print("YEAR:", article["year"])
    print("ARTICLE ID:", article_id)
    print("=" * 100)

    if not article_chunks:
        print("No first-page chunks found.")
        continue

    for i, chunk in enumerate(article_chunks[:3], start=1):
        print(f"\nPASSAGE {i}:")
        print(chunk["text"][:1500])


SUBTOPIC: Phishing Detection
TITLE: An explainable deep learning framework for multi-class malicious URL classification using priority-coefficient analysis
YEAR: 2026.0
ARTICLE ID: An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf

PASSAGE 1:
Discover Artificial Intelligence Nhu et al. Discover Artificial Intelligence (2026) 6:1491 https://doi.org/10.1007/s44163-026-02458-8 RESEARCH Open Access An explainable deep learning framework for multi-class malicious URL classification using priority-coefficient analysis Nguyen Gia Nhu1, Tran Minh Bao2, Arshpreet Kaur3, Kumar Shashvat3 and Dac-Nhuong Le4* *Correspondence: Dac-Nhuong Le nhuongld@dhhp.edu.vn 1School of Computer Science and AI, Duy Tan University, Danang 55000, Vietnam 2Ho Chi Minh University of Industry and Trade, Ho Chi Minh 10000, Vietnam 3Manipal University Jaipur, Rajasthan 303007, India 4Haiphong University, Haiphong 18000, Vietnam © The Author(s) 202

9.12 — Creating the multi-paper synthesis questions

In [ ]:
# Prepare 20 multi-paper synthesis questions using the inspected research passages.

synthesis_questions = [

    # Phishing Detection: Q041-Q045
    {
        "question_id": "Q041",
        "question_category": "Multi-paper Synthesis",
        "question": "How do the classification scopes of the X-URL framework and the intelligent malicious URL detection system differ?",
        "gold_answer": "The X-URL framework performs four-class classification covering benign, defacement, malware, and phishing URLs. The intelligent malicious URL system uses a two-stage approach: first distinguishing benign URLs from malware, then classifying URLs into five classes: benign, spam, phishing, malware, and defacement.",
        "answerable": True,
        "evidence_article_ids": "An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf;An_intelligent_identification_and_classification_system_for_malicious_uniform_resource_locators.pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Combines the classification categories and task structures described in both article abstracts."
    },
    {
        "question_id": "Q042",
        "question_category": "Multi-paper Synthesis",
        "question": "How do the modelling approaches used by X-URL and the intelligent malicious URL detection system differ?",
        "gold_answer": "X-URL uses a character-level BiLSTM with class-frequency weighted cross-entropy and Bayesian model selection. The other system evaluates four ensemble approaches: bagging trees, k-nearest neighbors, boosted decision trees, and subspace discriminators. Thus, the first uses a deep sequential neural network, while the second investigates ensemble learning methods.",
        "answerable": True,
        "evidence_article_ids": "An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf;An_intelligent_identification_and_classification_system_for_malicious_uniform_resource_locators.pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Compares the model architectures and algorithms explicitly identified in both abstracts."
    },
    {
        "question_id": "Q043",
        "question_category": "Multi-paper Synthesis",
        "question": "How does the systematic literature review help contextualize the supervised deep learning approach used in the X-URL framework?",
        "gold_answer": "The systematic review examined 43 journal articles and found that all but one of the reviewed models used supervised deep learning. X-URL provides a specific example of this direction through its trained character-level BiLSTM classifier. The review therefore places the X-URL approach within a broader pattern of supervised deep learning in phishing detection.",
        "answerable": True,
        "evidence_article_ids": "Applications_of_deep_learning_for_phishing_detection.pdf;An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Combines the systematic review finding with the supervised BiLSTM approach described by X-URL."
    },
    {
        "question_id": "Q044",
        "question_category": "Multi-paper Synthesis",
        "question": "What different evaluation measures are reported by the X-URL framework and the intelligent malicious URL detection system?",
        "gold_answer": "X-URL reports stratified five-fold cross-validation accuracy, hold-out accuracy, and weighted F1-score, and uses validation macro-F1 for Bayesian model selection. The intelligent malicious URL detection system evaluates accuracy, precision, recall, F-score, and detection time. Together, these studies illustrate the use of both classification-quality measures and detection-speed measures.",
        "answerable": True,
        "evidence_article_ids": "An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf;An_intelligent_identification_and_classification_system_for_malicious_uniform_resource_locators.pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Lists only evaluation measures explicitly mentioned in the two article passages."
    },
    {
        "question_id": "Q045",
        "question_category": "Multi-paper Synthesis",
        "question": "How do the findings of the systematic review and the X-URL framework illustrate the use of deep learning for phishing-related URL detection?",
        "gold_answer": "The systematic review found that supervised deep learning was used by all but one of the 43 reviewed journal studies. X-URL demonstrates one implementation of this approach by using a character-level BiLSTM for four-class URL classification and reporting 98.53% mean cross-validation accuracy for its optimized weighted BiLSTM. The review provides the broader research context, while X-URL presents a specific framework and experimental result.",
        "answerable": True,
        "evidence_article_ids": "Applications_of_deep_learning_for_phishing_detection.pdf;An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Combines the review's finding with the specific architecture and cross-validation result reported by X-URL."
    },

    # Intrusion Detection: Q046-Q050
    {
        "question_id": "Q046",
        "question_category": "Multi-paper Synthesis",
        "question": "How do the network traffic inputs considered by the 2024 intrusion detection study differ from those considered by the 2026 deployment-efficient CNN-LSTM framework?",
        "gold_answer": "The 2024 study combines network traffic and host traffic features and uses a CNN after feature selection and dimensionality reduction. The 2026 deployment-efficient framework evaluates heterogeneous traffic domains, including laboratory-generated CICIDS-2017 traffic, highly imbalanced UNSW-NB15 traffic, and real-world encrypted CERNET-2020 traffic. The studies therefore address different traffic characteristics and evaluation settings.",
        "answerable": True,
        "evidence_article_ids": "18. Enhancing network intrusion detection systems with combined network and host traffic features using deep learning deep learning and IoT perspective (2024).pdf;21. A deployment-efficient CNN-LSTM framework for real-time malicious traffic detection across heterogeneous, imbalanced, and encrypted network traffic (2026).pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Compares the input features and traffic domains stated in both abstracts."
    },
    {
        "question_id": "Q047",
        "question_category": "Multi-paper Synthesis",
        "question": "How do the two intrusion detection studies address the challenge of detecting evolving or unfamiliar network threats?",
        "gold_answer": "The 2024 study identifies the dependence of traditional signature-based NIDS on predetermined signatures as a limitation when detecting new and unanticipated intrusions. The 2026 deployment-efficient framework evaluates its model across heterogeneous traffic domains and reports partial cross-dataset transferability, achieving over 80% accuracy without retraining on an aligned 22-feature subset. These approaches address the challenge from different perspectives, but the reported transfer result is limited to the stated feature subset.",
        "answerable": True,
        "evidence_article_ids": "18. Enhancing network intrusion detection systems with combined network and host traffic features using deep learning deep learning and IoT perspective (2024).pdf;21. A deployment-efficient CNN-LSTM framework for real-time malicious traffic detection across heterogeneous, imbalanced, and encrypted network traffic (2026).pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Combines the stated limitation of signature-based systems and the cross-dataset transfer result. Does not claim complete zero-day detection."
    },
    {
        "question_id": "Q048",
        "question_category": "Multi-paper Synthesis",
        "question": "What are the main architectural differences between the CNN-based approach in the 2024 study and the hybrid CNN-LSTM approach in the 2026 NSL-KDD study?",
        "gold_answer": "The 2024 study employs a convolutional neural network with feature selection and dimensionality reduction to analyze combined network and host traffic features. The 2026 NSL-KDD study proposes a hybrid CNN-LSTM model and compares it with nine machine learning and deep learning classifiers. The latter explicitly combines CNN and LSTM components, whereas the former describes a CNN-based approach.",
        "answerable": True,
        "evidence_article_ids": "18. Enhancing network intrusion detection systems with combined network and host traffic features using deep learning deep learning and IoT perspective (2024).pdf;19. Optimizing intrusion detection using a hybrid CNN LSTM deep learning framework on the NSL KDD dataset (2026).pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Compares the architectures and experimental descriptions in both article passages."
    },
    {
        "question_id": "Q049",
        "question_category": "Multi-paper Synthesis",
        "question": "How do the dataset evaluation strategies of the 2026 NSL-KDD CNN-LSTM study and the deployment-efficient CNN-LSTM study differ?",
        "gold_answer": "The NSL-KDD study uses two explicit evaluation protocols: a stratified in-distribution split of KDDTrain+ and the official setting that trains on KDDTrain+ and tests on KDDTest+. The deployment-efficient study evaluates three heterogeneous traffic domains, CICIDS-2017, UNSW-NB15, and CERNET-2020, and also examines cross-dataset transfer using an aligned 22-feature subset. The studies therefore use different strategies to examine model performance and generalization.",
        "answerable": True,
        "evidence_article_ids": "19. Optimizing intrusion detection using a hybrid CNN LSTM deep learning framework on the NSL KDD dataset (2026).pdf;21. A deployment-efficient CNN-LSTM framework for real-time malicious traffic detection across heterogeneous, imbalanced, and encrypted network traffic (2026).pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Uses the two NSL-KDD protocols and the three traffic domains and transfer setting described in the abstracts."
    },
    {
        "question_id": "Q050",
        "question_category": "Multi-paper Synthesis",
        "question": "What different performance findings are reported by the 2024 network-and-host traffic study and the 2026 deployment-efficient CNN-LSTM study?",
        "gold_answer": "The 2024 network-and-host traffic study reports 98.5% detection accuracy for its CNN-based NIDS. The 2026 deployment-efficient study reports 98.1% accuracy, 97.6% F1-score, and 0.99 AUC on the encrypted CERNET dataset, and 99.2% accuracy on CICIDS-2017. These are results from different experimental settings and datasets, so the values should not be treated as a direct head-to-head comparison.",
        "answerable": True,
        "evidence_article_ids": "18. Enhancing network intrusion detection systems with combined network and host traffic features using deep learning deep learning and IoT perspective (2024).pdf;21. A deployment-efficient CNN-LSTM framework for real-time malicious traffic detection across heterogeneous, imbalanced, and encrypted network traffic (2026).pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Reports the stated results while explicitly avoiding a direct performance ranking across different datasets."
    },

    # Anomaly Detection: Q051-Q055
    {
        "question_id": "Q051",
        "question_category": "Multi-paper Synthesis",
        "question": "How do the anomaly detection mechanisms of AI2DS and the LSTM-based autoencoder study differ?",
        "gold_answer": "AI2DS trains an autoencoder on normal data and uses reconstruction errors with adaptive thresholding to identify deviations from normal network behaviour. The LSTM-based autoencoder study uses an LSTM to capture temporal dependencies in sequential network traffic and distinguish normal from abnormal activity. Both use autoencoder-based learning, but the second explicitly emphasizes temporal patterns.",
        "answerable": True,
        "evidence_article_ids": "35. AI2DS Advanced Deep Autoencoder-Driven Method for Intelligent Network Intrusion Detection Systems (2025).pdf;38. Enhanced Network Security using LSTM-Based Autoencoder Models (2024).pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Combines AI2DS reconstruction-error detection and the temporal modelling role of LSTM."
    },
    {
        "question_id": "Q052",
        "question_category": "Multi-paper Synthesis",
        "question": "How do the LSTM-based autoencoder and GTAE-IDS represent different patterns in network traffic?",
        "gold_answer": "The LSTM-based autoencoder uses LSTM networks to capture temporal dependencies and long-term patterns in sequential network traffic. GTAE-IDS uses packet-based graphs and graph embeddings to represent network communications, while graph autoencoders and transformers capture structural and contextual patterns. The first emphasizes temporal dependencies, whereas the second emphasizes graph structure and context.",
        "answerable": True,
        "evidence_article_ids": "38. Enhanced Network Security using LSTM-Based Autoencoder Models (2024).pdf;39. GTAE-IDS Graph Transformer-Based Autoencoder Framework for Real-Time Network Intrusion Detection (2025).pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Compares temporal sequence modelling with graph-based structural and contextual modelling."
    },
    {
        "question_id": "Q053",
        "question_category": "Multi-paper Synthesis",
        "question": "What limitations of existing intrusion detection approaches motivate the LSTM-based autoencoder study and GTAE-IDS?",
        "gold_answer": "The LSTM-based autoencoder study notes that signature-based detection is less effective against new and unknown attacks, while anomaly detection can produce high false-positive rates. GTAE-IDS identifies limitations in flat vector or grid representations, which may fail to capture structural and contextual details, and in supervised or semi-supervised approaches that require extensive labelled data. The studies therefore identify different challenges involving unknown attacks, false positives, data representation, and labelled-data requirements.",
        "answerable": True,
        "evidence_article_ids": "38. Enhanced Network Security using LSTM-Based Autoencoder Models (2024).pdf;39. GTAE-IDS Graph Transformer-Based Autoencoder Framework for Real-Time Network Intrusion Detection (2025).pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Combines the stated limitations in both introductions and abstracts."
    },
    {
        "question_id": "Q054",
        "question_category": "Multi-paper Synthesis",
        "question": "How do AI2DS and GTAE-IDS use unsupervised or normal-data learning for intrusion detection?",
        "gold_answer": "AI2DS trains its autoencoder using normal data and identifies anomalies through reconstruction errors and adaptive thresholding. GTAE-IDS uses an unsupervised packet-based graph framework that learns structural and global patterns from benign network traffic without requiring labelled graph data. Both approaches use learned representations of benign or normal behaviour to support anomaly detection.",
        "answerable": True,
        "evidence_article_ids": "35. AI2DS Advanced Deep Autoencoder-Driven Method for Intelligent Network Intrusion Detection Systems (2025).pdf;39. GTAE-IDS Graph Transformer-Based Autoencoder Framework for Real-Time Network Intrusion Detection (2025).pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Combines the normal-data training description in AI2DS and the unsupervised benign-graph learning description in GTAE-IDS."
    },
    {
        "question_id": "Q055",
        "question_category": "Multi-paper Synthesis",
        "question": "What performance findings do AI2DS and GTAE-IDS report, and why should their results be interpreted in their respective experimental contexts?",
        "gold_answer": "AI2DS reports average improvements of 6.61%, 31.11%, 11.46%, and 16.91% in precision, recall, F1-score, and accuracy, respectively, over state-of-the-art methods. GTAE-IDS reports over 98% accuracy on benchmark network intrusion datasets. These findings describe different performance measures and experimental evaluations, so they do not establish that one framework is better than the other.",
        "answerable": True,
        "evidence_article_ids": "35. AI2DS Advanced Deep Autoencoder-Driven Method for Intelligent Network Intrusion Detection Systems (2025).pdf;39. GTAE-IDS Graph Transformer-Based Autoencoder Framework for Real-Time Network Intrusion Detection (2025).pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Uses the reported performance statements and avoids unsupported direct comparison."
    },

    # Datasets and Benchmarking: Q056-Q060
    {
        "question_id": "Q056",
        "question_category": "Multi-paper Synthesis",
        "question": "How do the scopes of the 2026 intrusion detection dataset survey and the 2025 Advanced IDS benchmark study differ?",
        "gold_answer": "The 2026 survey examines 124 intrusion detection datasets and characterizes them across 20 dimensions, including attack diversity, temporal coverage, data formats, and availability. The 2025 Advanced IDS study presents a benchmark using 14 recent preprocessed network-flow datasets and evaluates seven categories of algorithms. The survey provides a broad characterization of datasets, while Advanced IDS focuses on a more specific comparative benchmark.",
        "answerable": True,
        "evidence_article_ids": "A survey of intrusion detection datasets for communication networks.pdf;Advanced IDS a comparative study of datasets and machine learning algorithms for network flow-based intrusion detection systems.pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Compares the dataset counts, dimensions, and stated purposes of both studies."
    },
    {
        "question_id": "Q057",
        "question_category": "Multi-paper Synthesis",
        "question": "What dataset and benchmarking problems are identified by both the 2026 dataset survey and the 2025 Advanced IDS study?",
        "gold_answer": "The 2026 survey identifies limited real-world traffic representation, inconsistent documentation, dataset bias, evaluation risks, and underrepresentation of emerging attack vectors. The 2025 Advanced IDS study notes that researchers still commonly use old datasets and that the absence of a standardized benchmark makes comparisons between models difficult. Together, these studies highlight concerns about dataset realism, quality, and fair model evaluation.",
        "answerable": True,
        "evidence_article_ids": "A survey of intrusion detection datasets for communication networks.pdf;Advanced IDS a comparative study of datasets and machine learning algorithms for network flow-based intrusion detection systems.pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Combines explicitly stated dataset and benchmarking limitations from both papers."
    },
    {
        "question_id": "Q058",
        "question_category": "Multi-paper Synthesis",
        "question": "How do the testing methodology proposed in 1996 and the benchmark framework presented in Advanced IDS support systematic evaluation of intrusion detection systems?",
        "gold_answer": "The 1996 methodology adapts software-testing techniques for IDS evaluation and describes general performance objectives, test-case selection strategies, and testing procedures. Advanced IDS provides 14 preprocessed datasets with training, validation, and test splits and compares algorithm categories using Macro F1. Both support systematic evaluation, but one emphasizes testing procedures while the other emphasizes standardized datasets and algorithm comparison.",
        "answerable": True,
        "evidence_article_ids": "A_methodology_for_testing_intrusion_detection_systems.pdf;Advanced IDS a comparative study of datasets and machine learning algorithms for network flow-based intrusion detection systems.pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Contrasts the procedural testing methodology with the dataset-based benchmark framework."
    },
    {
        "question_id": "Q059",
        "question_category": "Multi-paper Synthesis",
        "question": "Which considerations from the 2026 dataset survey can be connected to the dataset selection and benchmarking concerns discussed in Advanced IDS?",
        "gold_answer": "The 2026 survey highlights attack diversity, temporal coverage, data formats, availability, realism, dataset quality, and benchmarking risks as important dimensions. Advanced IDS emphasizes the need for recent network-flow datasets and notes the problems caused by continued reliance on old datasets and the lack of standardized benchmarks. Together, these findings indicate that dataset selection should consider both the characteristics and relevance of the data and whether the datasets support fair and meaningful model comparisons.",
        "answerable": True,
        "evidence_article_ids": "A survey of intrusion detection datasets for communication networks.pdf;Advanced IDS a comparative study of datasets and machine learning algorithms for network flow-based intrusion detection systems.pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Synthesizes dataset dimensions from the survey with dataset recency and benchmarking concerns from Advanced IDS."
    },
    {
        "question_id": "Q060",
        "question_category": "Multi-paper Synthesis",
        "question": "How do the recommendations of the 2026 dataset survey and the practical resources in Advanced IDS relate to reproducible intrusion detection research?",
        "gold_answer": "The 2026 survey recommends dataset standardization, high-fidelity data collection, improved labelling, quantitative dataset assessment, and privacy-preserving sharing. Advanced IDS provides a centralized source of preprocessed datasets with training, validation, and test splits to support straightforward and fair comparisons. These contributions relate to reproducibility by encouraging better documented datasets and more consistent experimental evaluation.",
        "answerable": True,
        "evidence_article_ids": "A survey of intrusion detection datasets for communication networks.pdf;Advanced IDS a comparative study of datasets and machine learning algorithms for network flow-based intrusion detection systems.pdf",
        "evidence_pages": "1;1",
        "verification_notes": "Combines the survey recommendations with the centralized preprocessed datasets and splits in Advanced IDS."
    }
]

# Update only Q041-Q060 in the existing evaluation dataset.
for question in synthesis_questions:
    question_id = question["question_id"]

    row_index = evaluation_df.index[
        evaluation_df["question_id"] == question_id
    ]

    if len(row_index) == 0:
        raise ValueError(f"Question ID {question_id} was not found in the template.")

    for column, value in question.items():
        evaluation_df.loc[row_index[0], column] = value

# Save the updated evaluation dataset.
evaluation_df.to_csv(
    f"{data_folder}/evaluation_questions.csv",
    index=False
)

print("Q041-Q060 saved successfully.")
print("\nMulti-paper synthesis questions saved:", len(synthesis_questions))

print("\nSubtopic distribution:")
print(
    pd.Series([
        "Phishing Detection",
        "Phishing Detection",
        "Phishing Detection",
        "Phishing Detection",
        "Phishing Detection",
        "Intrusion Detection",
        "Intrusion Detection",
        "Intrusion Detection",
        "Intrusion Detection",
        "Intrusion Detection",
        "Anomaly Detection",
        "Anomaly Detection",
        "Anomaly Detection",
        "Anomaly Detection",
        "Anomaly Detection",
        "Datasets and Benchmarking",
        "Datasets and Benchmarking",
        "Datasets and Benchmarking",
        "Datasets and Benchmarking",
        "Datasets and Benchmarking"
    ]).value_counts()
)

print("\nTotal populated questions:",
      evaluation_df["question"].notna().sum())

print("\nSaved question IDs:")
print([question["question_id"] for question in synthesis_questions])

Q041-Q060 saved successfully.

Multi-paper synthesis questions saved: 20

Subtopic distribution:
Phishing Detection           5
Intrusion Detection          5
Anomaly Detection            5
Datasets and Benchmarking    5
Name: count, dtype: int64

Total populated questions: 60

Saved question IDs:
['Q041', 'Q042', 'Q043', 'Q044', 'Q045', 'Q046', 'Q047', 'Q048', 'Q049', 'Q050', 'Q051', 'Q052', 'Q053', 'Q054', 'Q055', 'Q056', 'Q057', 'Q058', 'Q059', 'Q060']


/tmp/ipykernel_718/2946519599.py:226: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1;1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evaluation_df.loc[row_index[0], column] = value


In [ ]:
# Correct column data types and validate the multi-paper synthesis questions.

import pandas as pd
import re

# Convert text columns to string type.
text_columns = [
    "question_id",
    "question_category",
    "question",
    "gold_answer",
    "evidence_article_ids",
    "evidence_pages",
    "verification_notes"
]

for column in text_columns:
    evaluation_df[column] = evaluation_df[column].astype("string")

# Save the corrected evaluation dataset.
evaluation_df.to_csv(
    f"{data_folder}/evaluation_questions.csv",
    index=False
)

# Select Q041-Q060.
synthesis_df = evaluation_df[
    evaluation_df["question_id"].isin(
        [f"Q{i:03d}" for i in range(41, 61)]
    )
].copy()

# Build valid article-page combinations from the corpus.
valid_article_pages = {
    (
        str(record["article_id"]).strip(),
        int(record["page_number"])
    )
    for record in page_records
    if record.get("article_id") is not None
    and record.get("page_number") is not None
}

# Check question references.
invalid_references = []
insufficient_articles = []

for _, row in synthesis_df.iterrows():

    article_ids = [
        item.strip()
        for item in str(row["evidence_article_ids"]).split(";")
        if item.strip()
    ]

    pages = [
        item.strip()
        for item in str(row["evidence_pages"]).split(";")
        if item.strip()
    ]

    # Each question must use at least two distinct articles.
    if len(set(article_ids)) < 2:
        insufficient_articles.append(row["question_id"])

    # Article IDs and page references must match in number.
    if len(article_ids) != len(pages):
        invalid_references.append(
            (row["question_id"], "Article/page count mismatch")
        )
        continue

    # Check whether every article and page exists.
    for article_id, page in zip(article_ids, pages):

        page_number = pd.to_numeric(page, errors="coerce")

        if pd.isna(page_number) or not float(page_number).is_integer():
            invalid_references.append(
                (row["question_id"], f"Invalid page: {page}")
            )

        elif (article_id, int(page_number)) not in valid_article_pages:
            invalid_references.append(
                (row["question_id"], f"Invalid reference: {article_id}, page {page}")
            )

# Display validation results.
required_columns = [
    "question_id",
    "question",
    "gold_answer",
    "answerable",
    "evidence_article_ids",
    "evidence_pages"
]

print("1. SYNTHESIS QUESTION COUNT:", len(synthesis_df))

print("\n2. MISSING VALUES:")
print(synthesis_df[required_columns].isna().sum())

print("\n3. DUPLICATE QUESTION IDs:")
print(
    synthesis_df[
        synthesis_df["question_id"].duplicated(keep=False)
    ][["question_id", "question"]]
)

print("\n4. QUESTIONS WITH FEWER THAN TWO DISTINCT ARTICLES:")
print(insufficient_articles if insufficient_articles else "None")

print("\n5. INVALID ARTICLE/PAGE REFERENCES:")
print(invalid_references if invalid_references else "None")

print("\n6. ANSWERABILITY:")
print(synthesis_df["answerable"].value_counts(dropna=False))

print("\n7. TOTAL POPULATED QUESTIONS:",
      evaluation_df["question"].notna().sum())

print("\n8. CORRECTED CSV SAVED SUCCESSFULLY.")

1. SYNTHESIS QUESTION COUNT: 20

2. MISSING VALUES:
question_id             0
question                0
gold_answer             0
answerable              0
evidence_article_ids    0
evidence_pages          0
dtype: int64

3. DUPLICATE QUESTION IDs:
Empty DataFrame
Columns: [question_id, question]
Index: []

4. QUESTIONS WITH FEWER THAN TWO DISTINCT ARTICLES:
None

5. INVALID ARTICLE/PAGE REFERENCES:
None

6. ANSWERABILITY:
answerable
True    20
Name: count, dtype: int64

7. TOTAL POPULATED QUESTIONS: 60

8. CORRECTED CSV SAVED SUCCESSFULLY.


In [ ]:
# Confirm the current evaluation dataset and research corpus before preparing unanswerable questions.

print("EVALUATION DATASET")
print("Total rows:", len(evaluation_df))
print("Populated questions:", evaluation_df["question"].notna().sum())

print("\nQUESTION CATEGORY DISTRIBUTION")
print(
    evaluation_df[
        evaluation_df["question"].notna()
    ]["question_category"].value_counts()
)

print("\nUNANSWERED QUESTION SLOTS")
print(
    evaluation_df[
        evaluation_df["question_id"].isin(
            [f"Q{i:03d}" for i in range(61, 81)]
        )
    ][["question_id", "question", "answerable"]].to_string(index=False)
)

print("\nRESEARCH CORPUS")
print("Page records:", len(page_records))
print("Sentence-based chunks:", len(sentence_chunks))

print("\nRETAINED ARTICLE SUBTOPICS")
print(screening_df["subtopic"].value_counts(dropna=False))

EVALUATION DATASET
Total rows: 80
Populated questions: 60

QUESTION CATEGORY DISTRIBUTION
question_category
Factual                  20
Conceptual               20
Multi-paper Synthesis    20
Name: count, dtype: Int64

UNANSWERED QUESTION SLOTS
question_id question answerable
       Q061     <NA>        NaN
       Q062     <NA>        NaN
       Q063     <NA>        NaN
       Q064     <NA>        NaN
       Q065     <NA>        NaN
       Q066     <NA>        NaN
       Q067     <NA>        NaN
       Q068     <NA>        NaN
       Q069     <NA>        NaN
       Q070     <NA>        NaN
       Q071     <NA>        NaN
       Q072     <NA>        NaN
       Q073     <NA>        NaN
       Q074     <NA>        NaN
       Q075     <NA>        NaN
       Q076     <NA>        NaN
       Q077     <NA>        NaN
       Q078     <NA>        NaN
       Q079     <NA>        NaN
       Q080     <NA>        NaN

RESEARCH CORPUS
Page records: 3071
Sentence-based chunks: 13187

RETAINED ARTICLE 

In [ ]:

# Test a balanced set of candidate unanswerable questions against the research corpus.

unanswerable_candidates = [
    # Q061-Q065: Bhutan-specific cybersecurity information
    "What was the exact financial loss caused by phishing attacks in Bhutan in 2025?",
    "How many phishing websites were reported in Bhutan during 2025?",
    "Which Bhutanese government agency recorded the highest number of phishing incidents in 2025?",
    "What percentage of Bhutanese internet users experienced phishing attacks in 2025?",
    "How many organizations in Bhutan experienced ransomware attacks in 2025?",

    # Q066-Q070: Missing experimental results and performance metrics
    "What was the exact training time in seconds reported for the hybrid CNN-LSTM model evaluated on the NSL-KDD dataset in the 2026 study?",
    "What was the exact inference latency per network flow reported by the AI2DS autoencoder-based intrusion detection method?",
    "What was the exact false positive rate of the GTAE-IDS framework on each benchmark dataset?",
    "What was the exact memory consumption of the stacked autoencoder model during its edge-device evaluation?",
    "What was the exact statistical significance value reported when comparing the X-URL model with the other URL classification models?",

    # Q071-Q075: Unreported technical details
    "What exact batch size was used to train the CNN-GRU intrusion detection model evaluated on NF-UNSW-NB15?",
    "What exact learning rate was used to train the hybrid CNN-LSTM model evaluated on NSL-KDD?",
    "What was the exact number of transformer attention heads used in the GTAE-IDS framework?",
    "What exact hardware configuration was used for the real-time deployment evaluation of the AI2DS method?",
    "What was the exact number of hidden neurons in every layer of the autoencoder used in the practical reconstruction-error anomaly detection study?",

    # Q076-Q080: Unsupported real-world claims and information outside the corpus
    "Which cybersecurity organization currently manages Bhutan's national cyber incident response operations?",
    "How many confirmed ransomware incidents were reported worldwide during November 2026?",
    "Which commercial cybersecurity product currently has the highest phishing detection rate in independent real-world testing?",
    "What is the current annual budget allocated to national cybersecurity operations in Bhutan?",
    "Which country's national cybersecurity strategy has reduced phishing losses by the largest percentage in 2026?"
]

# Retrieve and rerank evidence for each candidate question.
candidate_retrieval_results = []

for index, question in enumerate(unanswerable_candidates, start=61):

    candidates = hybrid_retrieval(
        question,
        top_k=10,
        candidate_k=20
    )

    reranked = rerank_results(
        question,
        candidates,
        top_k=3
    )

    candidate_retrieval_results.append({
        "question_id": f"Q{index:03d}",
        "question": question,
        "results": reranked
    })

    print("\n" + "=" * 90)
    print(f"Q{index:03d}: {question}")

    for rank, result in enumerate(reranked, start=1):
        print(f"\nResult {rank}")
        print("Article ID:", result.get("article_id"))
        print("Title:", result.get("title"))
        print("Year:", result.get("year"))
        print("PDF Page:", result.get("page_number"))
        print("Passage:", str(result.get("text", ""))[:700])

print("\nCandidate retrieval testing completed.")



Q061: What was the exact financial loss caused by phishing attacks in Bhutan in 2025?

Result 1
Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Title: CANTINA+
Year: 2011.0
PDF Page: 2
Passage: 21:2 G. Xiang et al. 1. INTRODUCTION Phishing is a form of identity theft, in which criminals build replicas of target Web sites and lure unsuspecting victims to disclose their sensitive information like passwords, personal identification numbers (PINs), etc. According to one survey by the Gartner Group [McCall 2007], phishing attacks in the United States caused $3.2 billion loss in 2007, with about 3.6 million victims falling for the attacks, a huge increase from the 2.3 million the year before. Moore and Clayton [2007] reported that the loss to consumers and businesses in 2007 in the US alone was around $2 billion. A significant proportion of those losses was caused by one particularly inf

Result 2
Article ID: Heuristic_Machine_Learning_Approaches_f

In [ ]:

# Inspect relevant passages in the original papers before deciding whether technical questions are unanswerable.

papers_to_inspect = {
    "Q066_Q072": "19. Optimizing intrusion detection using a hybrid CNN LSTM deep learning framework on the NSL KDD dataset (2026).pdf",
    "Q067_Q074": "35. AI2DS Advanced Deep Autoencoder-Driven Method for Intelligent Network Intrusion Detection Systems (2025).pdf",
    "Q068_Q073": "39. GTAE-IDS Graph Transformer-Based Autoencoder Framework for Real-Time Network Intrusion Detection (2025).pdf",
    "Q069": "33. Stacked Autoencoder-Based Probabilistic Feature Extraction for On-Device Network Intrusion Detection (2022 issue).pdf",
    "Q070": "An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf",
    "Q071": "17. Deep learning enabled intrusion detection system for IoT security (2025).pdf",
    "Q075": "1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf"
}

technical_terms = [
    "training time", "inference time", "latency",
    "batch size", "learning rate", "optimizer",
    "attention head", "number of heads",
    "false positive rate", "FPR",
    "memory consumption", "memory utilization",
    "hidden neurons", "hidden layer",
    "statistical significance", "p-value",
    "hardware configuration", "GPU", "CPU",
    "training configuration", "hyperparameter"
]

for question_group, article_id in papers_to_inspect.items():

    matching_chunks = [
        chunk for chunk in sentence_chunks
        if chunk.get("article_id") == article_id
    ]

    print("\n" + "=" * 100)
    print("QUESTION GROUP:", question_group)
    print("ARTICLE ID:", article_id)
    print("TOTAL CHUNKS:", len(matching_chunks))

    relevant_chunks = []

    for chunk in matching_chunks:
        passage = str(chunk.get("text", ""))
        passage_lower = passage.lower()

        matched_terms = [
            term for term in technical_terms
            if term.lower() in passage_lower
        ]

        if matched_terms:
            relevant_chunks.append({
                "page": chunk.get("page_number"),
                "terms": matched_terms,
                "text": passage
            })

    print("CHUNKS MATCHING TECHNICAL TERMS:", len(relevant_chunks))

    for item in relevant_chunks:
        print("\n" + "-" * 80)
        print("PDF PAGE:", item["page"])
        print("MATCHED TERMS:", ", ".join(item["terms"]))
        print("PASSAGE:", item["text"][:1500])



QUESTION GROUP: Q066_Q072
ARTICLE ID: 19. Optimizing intrusion detection using a hybrid CNN LSTM deep learning framework on the NSL KDD dataset (2026).pdf
TOTAL CHUNKS: 63
CHUNKS MATCHING TECHNICAL TERMS: 11

--------------------------------------------------------------------------------
PDF PAGE: 2
MATCHED TERMS: hyperparameter
PASSAGE: To reduce the risk that conclusions are overly dependent on a single benchmark, we com­ plement the NSL-KDD evaluation with validation on a more recent dataset (CSE-CIC- IDS2018). This additional experiment is intended to probe whether the proposed hybrid CNN–LSTM detector maintains discriminative capability under a newer traffic distribu­ tion and a different benign-to-attack composition. The objective of this research study is to assess the efficacy of deep learning architectures specifically CNN, LSTM networks, and hybrid CNN+LSTM models, in improving the detection abilities of IDS using the NSL-KDD dataset. The purpose of these studies is to prov

In [ ]:

# Part 9.13.4: Identify explicit limitations and unreported evaluations across the corpus

import re
from collections import defaultdict

limitation_patterns = [
    r"not evaluated",
    r"not tested",
    r"not reported",
    r"not measured",
    r"not conducted",
    r"not investigated",
    r"was not assessed",
    r"were not assessed",
    r"remains unevaluated",
    r"remain unevaluated",
    r"future work",
    r"future research",
    r"further investigation",
    r"limitations of this study",
    r"was beyond the scope",
    r"were beyond the scope"
]

pattern = re.compile("|".join(limitation_patterns), re.IGNORECASE)

limitation_results = defaultdict(list)

for chunk in sentence_chunks:
    passage = str(chunk.get("text", ""))
    matches = pattern.findall(passage)

    if matches:
        article_id = chunk.get("article_id", "Unknown")

        # Keep no more than two relevant passages per article.
        if len(limitation_results[article_id]) < 2:
            limitation_results[article_id].append({
                "title": chunk.get("title", ""),
                "year": chunk.get("year", ""),
                "page": chunk.get("page_number", ""),
                "matched_terms": ", ".join(sorted(set(matches), key=str.lower)),
                "passage": passage[:1800]
            })

print("Articles containing explicit limitation statements:",
      len(limitation_results))

for article_id, passages in limitation_results.items():
    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)

    for item in passages:
        print("\nTitle:", item["title"])
        print("Year:", item["year"])
        print("PDF Page:", item["page"])
        print("Matched terms:", item["matched_terms"])
        print("Passage:", item["passage"])
        print("-" * 100)


Articles containing explicit limitation statements: 130

ARTICLE ID: 1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf

Title: Practical autoencoder based anomaly detection by using vector reconstruction error
Year: 2023.0
PDF Page: 12
Matched terms: further investigation
Passage: By analyzing the evaluation results and comparing them with some of the recent existing methods, it was shown that the proposed anomaly detec- tion had a better performance than the existing ones. The results confirmed that the proposed approach could be a promising solution, with high efficiency for construct- ing autoencoder-based anomaly detection for cloud net- works. The codes used for the evaluation are publicly available and could be used by researchers for further investigations. Although the proposed method performs well in simulations and benchmark datasets, issues still appear in practical cases. This research could be resumed by applying the proposed algorit

In [ ]:

# Search the full research corpus for Bhutan-related evidence.

bhutan_keywords = [
    "bhutan",
    "bhutanese",
    "thimphu",
    "druk",
    "bhutan telecom",
    "bhutan internet"
]

bhutan_matches = []

for chunk in sentence_chunks:
    text_content = str(chunk.get("text", ""))
    text_lower = text_content.lower()

    matched_terms = [
        term for term in bhutan_keywords
        if term in text_lower
    ]

    if matched_terms:
        bhutan_matches.append({
            "article_id": chunk.get("article_id"),
            "title": chunk.get("title"),
            "year": chunk.get("year"),
            "page_number": chunk.get("page_number"),
            "matched_terms": ", ".join(matched_terms),
            "text": text_content[:1000]
        })

bhutan_matches_df = pd.DataFrame(bhutan_matches)

print("Total matching chunks:", len(bhutan_matches_df))

if not bhutan_matches_df.empty:
    print("\nMatching articles:")
    display(
        bhutan_matches_df[
            ["article_id", "title", "year", "page_number",
             "matched_terms"]
        ].drop_duplicates(subset=["article_id"])
    )

    print("\nMatching passages:")
    display(bhutan_matches_df.head(20))
else:
    print("No Bhutan-related keyword matches found.")


Total matching chunks: 163

Matching articles:


,article_id,title,year,page_number,matched_terms
0,An Interpretable Network Intrusion Detection M...,An Interpretable Network Intrusion Detection M...,2025.0,2,bhutan
21,Phishing_URL_Detection_and_Interpretability_Wi...,Phishing\n <scp>URL</scp>\n...,2026.0,2,bhutan
41,Phishing_website_detection_How_effective_are_d...,Phishing website detection: How effective are ...,2022.0,2,bhutan
60,Transactions on Emerging Telecommunications Te...,Network intrusion detection system: A systemat...,2021.0,2,bhutan
90,Applied Computational Intelligence and Soft Co...,A Comprehensive Survey of Intrusion Detection ...,2026.0,2,bhutan
124,Meta‐analysis and systematic review for anomal...,Meta-Analysis and Systematic Review for Anomal...,2024.0,2,bhutan



Matching passages:


,article_id,title,year,page_number,matched_terms,text
0,An Interpretable Network Intrusion Detection M...,An Interpretable Network Intrusion Detection M...,2025.0,2,bhutan,"|| Scallops? || I do not even like scallops, a..."
1,An Interpretable Network Intrusion Detection M...,An Interpretable Network Intrusion Detection M...,2025.0,3,bhutan,"For example, LIME [17] explains the prediction..."
2,An Interpretable Network Intrusion Detection M...,An Interpretable Network Intrusion Detection M...,2025.0,4,bhutan,"Then, the single feature vector is fed into a ..."
3,An Interpretable Network Intrusion Detection M...,An Interpretable Network Intrusion Detection M...,2025.0,5,bhutan,"As shown in Figure 3a, the leaf node vL2 repre..."
4,An Interpretable Network Intrusion Detection M...,An Interpretable Network Intrusion Detection M...,2025.0,5,bhutan,"IET Information Security 5 ietis, 2025, 1, Dow..."
5,An Interpretable Network Intrusion Detection M...,An Interpretable Network Intrusion Detection M...,2025.0,6,bhutan,K 1⁄4 W K ð ÞE; Q 1⁄4 W Q ð ÞE; V = W V ð ÞE: ...
6,An Interpretable Network Intrusion Detection M...,An Interpretable Network Intrusion Detection M...,2025.0,7,bhutan,FIGURE 4: The interpretation strategy of TAN. ...
7,An Interpretable Network Intrusion Detection M...,An Interpretable Network Intrusion Detection M...,2025.0,8,bhutan,The set of decision rules rs = Q.trace_paths(x...
8,An Interpretable Network Intrusion Detection M...,An Interpretable Network Intrusion Detection M...,2025.0,9,bhutan,"In these baselines, LR, DT, RF, and MLP-SHAP a..."
9,An Interpretable Network Intrusion Detection M...,An Interpretable Network Intrusion Detection M...,2025.0,10,bhutan,"10 IET Information Security ietis, 2025, 1, Do..."


In [ ]:

# Inspect the context surrounding Bhutan-related keyword matches.

import re

for article_id in bhutan_matches_df["article_id"].dropna().unique():
    article_chunks = bhutan_matches_df[
        bhutan_matches_df["article_id"] == article_id
    ]

    print("\n" + "=" * 90)
    print("ARTICLE:", article_id)
    print("MATCHING CHUNKS:", len(article_chunks))

    shown = 0

    for _, row in article_chunks.iterrows():
        text_content = str(row["text"])
        matches = list(re.finditer(r"bhutan", text_content, re.IGNORECASE))

        if not matches:
            continue

        for match in matches:
            start = max(0, match.start() - 250)
            end = min(len(text_content), match.end() + 250)

            print("\nPDF PAGE:", row["page_number"])
            print("CONTEXT:", text_content[start:end].replace("\n", " "))

            shown += 1
            if shown >= 3:
                break

        if shown >= 3:
            break



ARTICLE: An Interpretable Network Intrusion Detection Model viaDecision Tree Enhanced Deep Attention Network.pdf
MATCHING CHUNKS: 21

PDF PAGE: 2
CONTEXT: (b) the interpretation for a sentiment classification task, and (c) the interpretation for an NID task. 2 IET Information Security ietis, 2025, 1, Downloaded from https://ietresearch.onlinelibrary.wiley.com/doi/10.1049/ise2/5552833 by INASP/HINARI - BHUTAN, Wiley Online Library on [28/10/2026]. See the Terms and Conditions (https://onlinelibrary.wiley.com/terms-and-conditions) on Wiley Online Library for rule

PDF PAGE: 3
CONTEXT: re a weight for a particular prediction from any deep learning model by calculating the Shapley values. IET Information Security 3 ietis, 2025, 1, Downloaded from https://ietresearch.onlinelibrary.wiley.com/doi/10.1049/ise2/5552833 by INASP/HINARI - BHUTAN, Wiley Online Library on [28/10/2026]. See the Terms and Conditions (https://onlinelibrary.wiley.com/terms-and-conditions) on Wiley Online Library for r

In [ ]:

# Run retrieval using the actual Bhutan-specific evaluation questions.

bhutan_questions = {
    "Q061": "What was the exact financial loss caused by phishing attacks in Bhutan in 2025?",
    "Q062": "How many phishing websites were reported in Bhutan during 2025?",
    "Q063": "Which Bhutanese government agency recorded the highest number of phishing incidents in 2025?",
    "Q064": "What percentage of Bhutanese internet users experienced phishing attacks in 2025?",
    "Q065": "How many organizations in Bhutan experienced ransomware attacks in 2025?"
}

bhutan_retrieval_results = {}

for question_id, question in bhutan_questions.items():

    candidates = hybrid_retrieval(
        question,
        top_k=10,
        candidate_k=20
    )

    reranked = rerank_results(
        question,
        candidates,
        top_k=5
    )

    bhutan_retrieval_results[question_id] = reranked

    print("\n" + "=" * 100)
    print("QUESTION:", question_id)
    print(question)

    for rank, result in enumerate(reranked, start=1):
        print(f"\n--- Result {rank} ---")
        print("Article:", result.get("article_id"))
        print("Title:", result.get("title"))
        print("Page:", result.get("page_number"))
        print("Passage:", str(result.get("text", ""))[:700])



QUESTION: Q061
What was the exact financial loss caused by phishing attacks in Bhutan in 2025?

--- Result 1 ---
Article: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Title: CANTINA+
Page: 2
Passage: 21:2 G. Xiang et al. 1. INTRODUCTION Phishing is a form of identity theft, in which criminals build replicas of target Web sites and lure unsuspecting victims to disclose their sensitive information like passwords, personal identification numbers (PINs), etc. According to one survey by the Gartner Group [McCall 2007], phishing attacks in the United States caused $3.2 billion loss in 2007, with about 3.6 million victims falling for the attacks, a huge increase from the 2.3 million the year before. Moore and Clayton [2007] reported that the loss to consumers and businesses in 2007 in the US alone was around $2 billion. A significant proportion of those losses was caused by one particularly inf

--- Result 2 ---
Article: Heuristic_Machine_Learning_Approaches

In [ ]:
# Retrieve and rerank evidence for the five questions about missing experimental results.

experimental_questions = {
    "Q066": "What detection accuracy did the Practical Autoencoder model achieve when evaluated on real operational networks?",
    "Q067": "What exact generalisation score did the feature-fusion NIDS achieve on previously unseen attack characteristics?",
    "Q068": "What detection accuracy did the IoT CNN-based IDS achieve across heterogeneous IoT environments with different communication protocols?",
    "Q069": "What accuracy did the hybrid CNN–LSTM model achieve in a multi-day evaluation of CSE-CIC-IDS2018?",
    "Q070": "What zero-day detection rate did the RNDAE model achieve on real-world backbone network traffic?"
}

experimental_retrieval_results = {}

for question_id, question in experimental_questions.items():

    candidates = hybrid_retrieval(
        question,
        top_k=10,
        candidate_k=20
    )

    reranked = rerank_results(
        question,
        candidates,
        top_k=5
    )

    experimental_retrieval_results[question_id] = reranked

    print("\n" + "=" * 100)
    print("QUESTION:", question_id)
    print(question)

    for rank, result in enumerate(reranked, start=1):
        print(f"\n--- Result {rank} ---")
        print("Article:", result.get("article_id"))
        print("Title:", result.get("title"))
        print("Page:", result.get("page_number"))
        print("Passage:", str(result.get("text", ""))[:700])


QUESTION: Q066
What detection accuracy did the Practical Autoencoder model achieve when evaluated on real operational networks?

--- Result 1 ---
Article: 7. A convolutional autoencoder architecture for robust network intrusion detection in embedded systems.pdf
Title: A convolutional autoencoder architecture for robust network intrusion detection in embedded systems
Page: 4
Passage: As such, our proposed method requires neither feature selection during training nor feature extraction at runtime. While there exist some prior solutions [16,18] that investigated how to perform detection using the whole packet, they did not achieve the same level of accuracy as that of the proposed approach. 2.2.2. Timing performance When deploying an IDS in a real environment, especially in CPS settings, we should not only evaluate the detection capabilities but also the time required to perform such detection and its implications. Kathareios et al. [7] developed a multi-stage solution for anomaly detect

In [ ]:
# Retrieve and rerank evidence for the five technical and operational evaluation questions.

technical_questions = {
    "Q071": "What was the 95th-percentile end-to-end inference latency of the X-URL model under live gateway or DNS traffic?",
    "Q072": "What adversarial robustness score did X-URL achieve against Unicode homoglyph and character-edit attacks?",
    "Q073": "What performance did X-URL achieve when detecting URLs from previously unseen time periods?",
    "Q074": "What was the expert or security-analyst agreement rate for the SHAP explanations generated by X-URL?",
    "Q075": "What percentage of errors made by the deployment-efficient CNN–LSTM model was attributable to individual attack families and encrypted benign–malicious traffic similarity?"
}

technical_retrieval_results = {}

for question_id, question in technical_questions.items():

    candidates = hybrid_retrieval(
        question,
        top_k=10,
        candidate_k=20
    )

    reranked = rerank_results(
        question,
        candidates,
        top_k=5
    )

    technical_retrieval_results[question_id] = reranked

    print("\n" + "=" * 100)
    print("QUESTION:", question_id)
    print(question)

    for rank, result in enumerate(reranked, start=1):
        print(f"\n--- Result {rank} ---")
        print("Article:", result.get("article_id"))
        print("Title:", result.get("title"))
        print("Page:", result.get("page_number"))
        print("Passage:", str(result.get("text", ""))[:700])


QUESTION: Q071
What was the 95th-percentile end-to-end inference latency of the X-URL model under live gateway or DNS traffic?

--- Result 1 ---
Article: Classical_First_Selective_Cascades_for_Resource_Constrained_Phishing_Email_Detection.pdf
Title: Classical-First Selective Cascades for Resource-Constrained Phishing Email Detection
Page: 16
Passage: Electronics 2026, 15, 3051 16 of 35 79% of the F1 gap to MiniLM and 75% of the gap to DistilBERT. End-to-end inference is 2.3× faster than MiniLM and 4.4× faster than DistilBERT, and the compact model trains end- to-end in 7.7 s from raw text versus 118.9 s (MiniLM) or 661 s (DistilBERT) for the encoder embedding pass alone. Scoring 106 emails on a 2-core CPU takes ≈2 h 13 min (compact), ≈5 h 09 min (MiniLM), and ≈9 h 51 min (DistilBERT). The latency profile is particularly rel- evant to low-cost operational servers and edge-adjacent IoT/WSN deployments, where full transformer inference for every message may be unnecessary or economically

In [ ]:
# Retrieve and rerank evidence for the final five unanswerable questions.

deployment_questions = {
    "Q076": "What detection performance did the cross-attention CNN–LSTM model achieve in additional real-world network deployments beyond its two benchmark datasets?",
    "Q077": "What confirmed effect did increasing autoencoder depth have across a wider range of model configurations and datasets?",
    "Q078": "What exact inference-speed improvement did the DAE–ABC model achieve when additional hardware resources were provided?",
    "Q079": "What zero-day detection accuracy did the FI-AE model achieve when detecting attack classes excluded during training?",
    "Q080": "What detection performance did the SSID framework achieve when adapting a pretrained IDS to a different network whose traffic had not previously been learned?"
}

deployment_retrieval_results = {}

for question_id, question in deployment_questions.items():

    candidates = hybrid_retrieval(
        question,
        top_k=10,
        candidate_k=20
    )

    reranked = rerank_results(
        question,
        candidates,
        top_k=5
    )

    deployment_retrieval_results[question_id] = reranked

    print("\n" + "=" * 100)
    print("QUESTION:", question_id)
    print(question)

    for rank, result in enumerate(reranked, start=1):
        print(f"\n--- Result {rank} ---")
        print("Article:", result.get("article_id"))
        print("Title:", result.get("title"))
        print("Page:", result.get("page_number"))
        print("Passage:", str(result.get("text", ""))[:700])


QUESTION: Q076
What detection performance did the cross-attention CNN–LSTM model achieve in additional real-world network deployments beyond its two benchmark datasets?

--- Result 1 ---
Article: 20. A cross-attention CNN–LSTM fusion model for network traffic anomaly detection (2026).pdf
Title: A cross-attention CNN–LSTM fusion model for network traffic anomaly detection
Page: 2
Passage: These complementary strengths have motivated a growing body of work on hybrid CNN–LSTM architectures for intrusion detection, particularly in IoT environ­ ments19–24. Despite steady performance gains, several limitations remain in the current literature. Many studies are evaluated on a single benchmark dataset, rely on simplistic preprocessing pipelines, and provide limited evi­ dence of generalization across heterogeneous network environments7,25. Moreover, existing solutions often com­ bine CNN and LSTM features through naive concatenation or sequential stacking, without explicit mechanisms for the 

In [ ]:
# Retrieve and rerank evidence for the final three unanswerable questions.

final_questions = {
    "Q078": "What exact inference-speed improvement did the DAE–ABC model achieve when additional hardware resources were provided?",
    "Q079": "What zero-day detection accuracy did the FI-AE model achieve when detecting attack classes excluded during training?",
    "Q080": "What detection performance did the SSID framework achieve when adapting a pretrained IDS to a different network whose traffic had not previously been learned?"
}

final_retrieval_results = {}

for question_id, question in final_questions.items():

    candidates = hybrid_retrieval(
        question,
        top_k=10,
        candidate_k=20
    )

    reranked = rerank_results(
        question,
        candidates,
        top_k=5
    )

    final_retrieval_results[question_id] = reranked

    print("\n" + "=" * 100)
    print("QUESTION:", question_id)
    print(question)

    for rank, result in enumerate(reranked, start=1):
        print(f"\n--- Result {rank} ---")
        print("Article:", result.get("article_id"))
        print("Title:", result.get("title"))
        print("Page:", result.get("page_number"))
        print("Passage:", str(result.get("text", ""))[:700])


QUESTION: Q078
What exact inference-speed improvement did the DAE–ABC model achieve when additional hardware resources were provided?

--- Result 1 ---
Article: 28. Network anomaly detection using Deep Autoencoder and parallel Artificial Bee Colony algorithm-trained neural network (2024).pdf
Title: Network anomaly detection using Deep Autoencoder and parallel Artificial Bee Colony algorithm-trained neural network
Page: 31
Passage: However, there are still certain aspects of the DAE-based ABC-ANN that need improvement. Our proposed approach surpasses conventional methods in computational efficiency and classification metrics by leveraging the proposed optimization technique. Nevertheless, the model encounters limitations in acceleration due to insufficient hardware, notwithstanding its high hardware requirements. Exploring additional hardware resources may lead to improved results. Future work includes the investigation of hybrid models to improve the anomaly detection performance. By 

In [ ]:
# Inspect the existing evaluation dataset before adding Q061-Q080.

evaluation_path = (
    "/content/drive/MyDrive/NLP-Project/CyberThreatResearch/"
    "processed_data/evaluation_questions.csv"
)

evaluation_df = pd.read_csv(evaluation_path)

print("Dataset shape:", evaluation_df.shape)
print("\nColumns:")
print(evaluation_df.columns.tolist())

print("\nData types:")
print(evaluation_df.dtypes)

print("\nLast five existing questions:")
display(evaluation_df.tail(5))

print("\nQuestion IDs and categories:")
display(
    evaluation_df.groupby("question_category")["question_id"]
    .agg(["count", "min", "max"])
)

Dataset shape: (80, 8)

Columns:
['question_id', 'question_category', 'question', 'gold_answer', 'answerable', 'evidence_article_ids', 'evidence_pages', 'verification_notes']

Data types:
question_id             object
question_category       object
question                object
gold_answer             object
answerable              object
evidence_article_ids    object
evidence_pages          object
verification_notes      object
dtype: object

Last five existing questions:


,question_id,question_category,question,gold_answer,answerable,evidence_article_ids,evidence_pages,verification_notes
75,Q076,Unanswerable,NaN,NaN,NaN,NaN,NaN,NaN
76,Q077,Unanswerable,NaN,NaN,NaN,NaN,NaN,NaN
77,Q078,Unanswerable,NaN,NaN,NaN,NaN,NaN,NaN
78,Q079,Unanswerable,NaN,NaN,NaN,NaN,NaN,NaN
79,Q080,Unanswerable,NaN,NaN,NaN,NaN,NaN,NaN



Question IDs and categories:


,count,min,max
question_category,,,
Conceptual,20,Q021,Q040
Factual,20,Q001,Q020
Multi-paper Synthesis,20,Q041,Q060
Unanswerable,20,Q061,Q080


In [ ]:
# Populate the 20 reviewed unanswerable questions while preserving Q001-Q060.

unanswerable_data = [
    {
        "question_id": "Q061",
        "question": "What was the exact financial loss caused by phishing attacks in Bhutan in 2025?",
        "gold_answer": "The supplied corpus does not establish the exact financial loss caused by phishing attacks in Bhutan in 2025.",
        "verification_notes": "Hybrid retrieval returned general phishing information and historical or international statistics, but no Bhutan-specific financial-loss figure for 2025. A corpus keyword scan found Bhutan-related matches only in document download footers, not research findings."
    },
    {
        "question_id": "Q062",
        "question": "How many phishing websites were reported in Bhutan in 2025?",
        "gold_answer": "The supplied corpus does not report the number of phishing websites reported in Bhutan in 2025.",
        "verification_notes": "Retrieved passages included global phishing statistics but did not establish a Bhutan-specific count for 2025. Bhutan-related keyword matches in the corpus were document download footers."
    },
    {
        "question_id": "Q063",
        "question": "Which Bhutanese government agency recorded the highest number of phishing incidents in 2025?",
        "gold_answer": "The supplied corpus does not identify a Bhutanese government agency with the highest number of phishing incidents in 2025.",
        "verification_notes": "Retrieval returned general and international phishing statistics, with no Bhutanese government agency comparison or ranking for 2025. Bhutan-related keyword matches were document download footers."
    },
    {
        "question_id": "Q064",
        "question": "What percentage of Bhutanese internet users experienced phishing attacks in 2025?",
        "gold_answer": "The supplied corpus does not establish the percentage of Bhutanese internet users who experienced phishing attacks in 2025.",
        "verification_notes": "Retrieved passages contained general or global phishing information but no Bhutan-specific user prevalence percentage for 2025. Bhutan-related keyword matches were document download footers."
    },
    {
        "question_id": "Q065",
        "question": "How many organizations in Bhutan experienced ransomware attacks in 2025?",
        "gold_answer": "The supplied corpus does not establish how many organizations in Bhutan experienced ransomware attacks in 2025.",
        "verification_notes": "Retrieval returned general phishing and IoT attack statistics, not a Bhutan-specific ransomware incident count for 2025. Bhutan-related keyword matches were document download footers."
    },
    {
        "question_id": "Q066",
        "question": "What detection accuracy did the Practical Autoencoder model achieve when evaluated on real operational networks?",
        "gold_answer": "The supplied corpus does not report a detection accuracy for the Practical Autoencoder model on real operational networks.",
        "evidence_article_ids": "1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf",
        "evidence_pages": "12",
        "verification_notes": "The Practical Autoencoder paper discusses the need for practical evaluation on real networks as a limitation. Retrieval did not establish an accuracy result from such an evaluation."
    },
    {
        "question_id": "Q067",
        "question": "What exact generalisation score did the feature-fusion NIDS achieve on previously unseen attack characteristics?",
        "gold_answer": "The supplied corpus does not report an exact generalisation score for the feature-fusion NIDS on previously unseen attack characteristics.",
        "evidence_article_ids": "12. Network intrusion detection using feature fusion with deep learning.pdf",
        "evidence_pages": "3;21",
        "verification_notes": "The target paper identifies further investigation of generalisation and efficacy as necessary. Its retrieved passages do not provide the requested exact score for previously unseen attack characteristics."
    },
    {
        "question_id": "Q068",
        "question": "In the explainable CNN-based IoT intrusion detection study, what detection accuracy did the model achieve when evaluated across heterogeneous IoT environments using different communication protocols?",
        "gold_answer": "The supplied corpus does not establish a detection accuracy from an evaluation across heterogeneous IoT environments using different communication protocols for the specified explainable CNN-based study.",
        "evidence_pages": "21",
        "verification_notes": "The target study identifies evaluation across heterogeneous IoT environments as future work. A retrieved survey passage reports results from another study, but it does not establish the requested result for this target study."
    },
    {
        "question_id": "Q069",
        "question": "What accuracy did the hybrid CNN–LSTM model achieve in a multi-day evaluation of CSE-CIC-IDS2018?",
        "gold_answer": "The supplied corpus does not establish the model's accuracy in a multi-day evaluation of CSE-CIC-IDS2018. A reported result for a specific dataset subset is not evidence of multi-day performance.",
        "evidence_pages": "16",
        "verification_notes": "The retrieved target-paper passage reports 99.996% accuracy on the specific 02-14-2018.csv subset. This does not establish performance across a multi-day evaluation, which is what the question asks."
    },
    {
        "question_id": "Q070",
        "question": "What zero-day detection rate did the RNDAE model achieve on real-world backbone network traffic?",
        "gold_answer": "The supplied corpus does not report a zero-day detection rate for the RNDAE model on real-world backbone network traffic.",
        "evidence_pages": "6",
        "verification_notes": "The RNDAE paper identifies zero-day capability assessment and evaluation using real-world backbone traffic as future work. The requested evaluation result is therefore not established by the paper."
    },
    {
        "question_id": "Q071",
        "question": "What was the 95th-percentile end-to-end inference latency of the X-URL model under live gateway or DNS traffic?",
        "gold_answer": "The supplied corpus does not report the 95th-percentile end-to-end inference latency of X-URL under live gateway or DNS traffic.",
        "evidence_article_ids": "An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf",
        "evidence_pages": "17;23",
        "verification_notes": "The X-URL paper identifies production latency, throughput, memory under load and live gateway or DNS integration as unevaluated. No 95th-percentile end-to-end latency measurement is reported."
    },
    {
        "question_id": "Q072",
        "question": "What adversarial robustness score did X-URL achieve against Unicode homoglyph and character-edit attacks?",
        "gold_answer": "The supplied corpus does not report an adversarial robustness score for X-URL against Unicode homoglyph or character-edit attacks.",
        "evidence_article_ids": "An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf",
        "evidence_pages": "23;24",
        "verification_notes": "The X-URL paper states that Unicode homoglyph substitutions and character insertion, deletion or permutation attacks were not tested."
    },
    {
        "question_id": "Q073",
        "question": "What performance did X-URL achieve when detecting URLs from previously unseen time periods?",
        "gold_answer": "The supplied corpus does not establish X-URL's performance on URLs from previously unseen time periods.",
        "evidence_article_ids": "An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf",
        "evidence_pages": "13;24",
        "verification_notes": "The X-URL evaluation used random splitting rather than chronological splitting. The paper identifies temporal novelty and concept drift as unevaluated, so its reported scores do not establish future-period generalisation."
    },
    {
        "question_id": "Q074",
        "question": "What was the expert or security-analyst agreement rate for the SHAP explanations generated by X-URL?",
        "gold_answer": "The supplied corpus does not report an expert or security-analyst agreement rate for X-URL's SHAP explanations.",
        "evidence_article_ids": "An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf",
        "evidence_pages": "17;22",
        "verification_notes": "The X-URL paper describes SHAP as an offline audit or decision-support method and does not report analyst agreement or quantitative validation of explanations."
    },
    {
        "question_id": "Q075",
        "question": "What percentage of the deployment-efficient CNN–LSTM model's misclassifications came from each individual attack family in its encrypted-traffic evaluation?",
        "gold_answer": "The supplied corpus does not report the percentage of misclassifications attributable to each individual attack family in the model's encrypted-traffic evaluation.",
        "evidence_pages": "37",
        "verification_notes": "The deployment-efficient CNN–LSTM paper reports binary evaluation and does not provide per-attack-category results. Therefore, the requested distribution of errors across individual attack families cannot be established."
    },
    {
        "question_id": "Q076",
        "question": "What detection performance did the cross-attention CNN–LSTM model achieve in additional real-world network deployments beyond its two benchmark datasets?",
        "gold_answer": "The supplied corpus does not report detection performance from additional real-world network deployments beyond the two benchmark datasets for the specified cross-attention CNN–LSTM model.",
        "evidence_article_ids": "20. A cross-attention CNN–LSTM fusion model for network traffic anomaly detection (2026).pdf",
        "evidence_pages": "18",
        "verification_notes": "The target paper's evaluation is limited to two IoT benchmark datasets and identifies additional real-world deployment as future work. Results from other CNN–LSTM studies do not establish results for this model."
    },
    {
        "question_id": "Q077",
        "question": "What effect did increasing autoencoder depth have when tested across a broader range of model configurations and datasets beyond those evaluated in the study?",
        "gold_answer": "The supplied corpus does not establish the effect of increasing autoencoder depth across a broader range of configurations and datasets beyond those evaluated in the study.",
        "evidence_article_ids": "26. Analysis of Autoencoders for Network Intrusion Detection (2021).pdf",
        "evidence_pages": "20",
        "verification_notes": "The paper reports findings for its tested configurations, but identifies the limited configuration range and need for further examination of larger autoencoders. The question asks about additional configurations and datasets beyond the study."
    },
    {
        "question_id": "Q078",
        "question": "What exact inference-speed improvement did the DAE–ABC model achieve when additional hardware resources were provided?",
        "gold_answer": "The supplied corpus does not report an exact inference-speed improvement from providing additional hardware resources to the DAE-based ABC-ANN model.",
        "evidence_article_ids": "28. Network anomaly detection using Deep Autoencoder and parallel Artificial Bee Colony algorithm-trained neural network (2024).pdf",
        "evidence_pages": "31",
        "verification_notes": "The paper states that insufficient hardware limits acceleration and suggests that additional hardware may improve results. It does not report an experiment or exact speed improvement resulting from additional hardware."
    },
    {
        "question_id": "Q079",
        "question": "What zero-day detection accuracy did the FI-AE model achieve when detecting attack classes excluded during training?",
        "gold_answer": "The supplied corpus does not report zero-day detection accuracy for FI-AE on attack classes excluded during training.",
        "evidence_article_ids": "44. Feature importance guided autoencoder for dimensionality reduction in intrusion detection systems.pdf",
        "evidence_pages": "9;13",
        "verification_notes": "The FI-AE paper focuses on labelled benchmark datasets and explains that zero-day detection requires a distinct evaluation, such as holding out attack classes during training. The requested zero-day evaluation is not reported."
    },
    {
        "question_id": "Q080",
        "question": "What detection performance did the SSID framework achieve when adapting a pretrained IDS to a different network whose traffic had not previously been learned?",
        "gold_answer": "The supplied corpus does not report detection performance for SSID adapting a pretrained IDS to a different network with previously unseen traffic.",
        "evidence_article_ids": "30. Online Self-Supervised Deep Learning for Intrusion Detection Systems (2024).pdf",
        "evidence_pages": "13",
        "verification_notes": "The SSID paper explicitly identifies adaptation of a pretrained IDS across different networks with previously unlearned traffic as future work. Its existing experiments do not establish the requested cross-network adaptation result."
    }
]

# Update only Q061-Q080; all earlier questions remain unchanged.
for item in unanswerable_data:
    row_index = evaluation_df.index[
        evaluation_df["question_id"] == item["question_id"]
    ]

    if len(row_index) != 1:
        raise ValueError(
            f"Expected exactly one row for {item['question_id']}, "
            f"but found {len(row_index)}."
        )

    for column in [
        "question",
        "gold_answer",
        "evidence_article_ids",
        "evidence_pages",
        "verification_notes"
    ]:
        if column in item:
            evaluation_df.loc[row_index[0], column] = item[column]

    evaluation_df.loc[row_index[0], "answerable"] = False

# Save the updated dataset.
evaluation_df.to_csv(evaluation_path, index=False)

print("Updated evaluation dataset saved.")
print("Questions populated:", len(unanswerable_data))
print("File:", evaluation_path)

Updated evaluation dataset saved.
Questions populated: 20
File: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/evaluation_questions.csv


In [ ]:
# Validate the completed 80-question evaluation dataset.

evaluation_df = pd.read_csv(evaluation_path)

expected_categories = {
    "Factual": 20,
    "Conceptual": 20,
    "Multi-paper Synthesis": 20,
    "Unanswerable": 20
}

print("Dataset shape:", evaluation_df.shape)
print("\nCategory counts:")
print(evaluation_df["question_category"].value_counts())

print("\nUnique question IDs:", evaluation_df["question_id"].nunique())
print("Missing question text:", evaluation_df["question"].isna().sum())
print("Missing gold answers:", evaluation_df["gold_answer"].isna().sum())
print("Missing verification notes:", evaluation_df["verification_notes"].isna().sum())

print("\nUnanswerable questions marked False:")
print(
    (evaluation_df.loc[
        evaluation_df["question_id"].between("Q061", "Q080"),
        "answerable"
    ].astype(str).str.lower() == "false").sum()
)

print("\nCategory counts correct:",
      evaluation_df["question_category"].value_counts().to_dict()
      == expected_categories)

print("\nQuestion IDs complete:",
      set(evaluation_df["question_id"])
      == {f"Q{i:03d}" for i in range(1, 81)})

display(
    evaluation_df.loc[
        evaluation_df["question_id"].between("Q061", "Q080"),
        ["question_id", "question", "answerable",
         "evidence_article_ids", "evidence_pages"]
    ]
)

Dataset shape: (80, 8)

Category counts:
question_category
Factual                  20
Conceptual               20
Multi-paper Synthesis    20
Unanswerable             20
Name: count, dtype: int64

Unique question IDs: 80
Missing question text: 0
Missing gold answers: 0
Missing verification notes: 0

Unanswerable questions marked False:
20

Category counts correct: True

Question IDs complete: True


,question_id,question,answerable,evidence_article_ids,evidence_pages
60,Q061,What was the exact financial loss caused by ph...,False,NaN,NaN
61,Q062,How many phishing websites were reported in Bh...,False,NaN,NaN
62,Q063,Which Bhutanese government agency recorded the...,False,NaN,NaN
63,Q064,What percentage of Bhutanese internet users ex...,False,NaN,NaN
64,Q065,How many organizations in Bhutan experienced r...,False,NaN,NaN
65,Q066,What detection accuracy did the Practical Auto...,False,1. Practical autoencoder based anomaly detecti...,12
66,Q067,What exact generalisation score did the featur...,False,12. Network intrusion detection using feature ...,3;21
67,Q068,In the explainable CNN-based IoT intrusion det...,False,NaN,21
68,Q069,What accuracy did the hybrid CNN–LSTM model ac...,False,NaN,16
69,Q070,What zero-day detection rate did the RNDAE mod...,False,NaN,6


In [ ]:
# Identify corpus article IDs for the four unanswerable questions with missing references.

target_keywords = {
    "Q068": ["explainable", "CNN", "IoT"],
    "Q069": ["hybrid", "CNN", "LSTM", "CSE-CIC-IDS2018"],
    "Q070": ["RNDAE"],
    "Q075": ["deployment-efficient", "CNN", "LSTM"]
}

# Build a unique article catalogue from the existing sentence chunks.
article_catalogue = {}

for chunk in sentence_chunks:
    article_id = chunk.get("article_id")
    title = chunk.get("title", "")

    if article_id:
        article_catalogue[article_id] = title

# Find candidate articles using the question-specific keywords.
for question_id, keywords in target_keywords.items():
    print("\n" + "=" * 90)
    print("QUESTION:", question_id)

    for article_id, title in article_catalogue.items():
        searchable_text = f"{article_id} {title}".lower()

        if any(keyword.lower() in searchable_text for keyword in keywords):
            print("Article ID:", article_id)
            print("Title:", title)


QUESTION: Q068
Article ID: 14. End-to-end deep learning for smart maritime threat detection an AE–CNN–LSTM-based approach.pdf
Title: End-to-end deep learning for smart maritime threat detection: an AE–CNN–LSTM-based approach
Article ID: 15. Intrusion detection in the internet of things using convolutional neural networks an explainable AI approach.pdf
Title: Intrusion detection in the internet of things using convolutional neural networks: an explainable AI approach
Article ID: 17. Deep learning enabled intrusion detection system for IoT security (2025).pdf
Title: Deep learning enabled intrusion detection system for IoT security
Article ID: 18. Enhancing network intrusion detection systems with combined network and host traffic features using deep learning deep learning and IoT perspective (2024).pdf
Title: Enhancing network intrusion detection systems with combined network and host traffic features using deep learning: deep learning and IoT perspective
Article ID: 19. Optimizing intr

In [ ]:
# Locate the RNDAE paper using its name and related terms in the corpus.

search_terms = [
    "RNDAE",
    "random neural deep autoencoder",
    "real-world backbone",
    "backbone network traffic"
]

matched_articles = {}

for chunk in sentence_chunks:
    text_content = str(chunk.get("text", ""))
    searchable_text = (
        str(chunk.get("title", "")) + " " + text_content
    ).lower()

    if any(term.lower() in searchable_text for term in search_terms):
        article_id = chunk.get("article_id")

        if article_id:
            if article_id not in matched_articles:
                matched_articles[article_id] = {
                    "title": chunk.get("title"),
                    "pages": set(),
                    "matched_terms": set()
                }

            matched_articles[article_id]["pages"].add(
                str(chunk.get("page_number"))
            )

            for term in search_terms:
                if term.lower() in searchable_text:
                    matched_articles[article_id]["matched_terms"].add(term)

for article_id, details in matched_articles.items():
    print("\nArticle ID:", article_id)
    print("Title:", details["title"])
    print("Pages:", sorted(details["pages"]))
    print("Matched terms:", sorted(details["matched_terms"]))


Article ID: 18. Enhancing network intrusion detection systems with combined network and host traffic features using deep learning deep learning and IoT perspective (2024).pdf
Title: Enhancing network intrusion detection systems with combined network and host traffic features using deep learning: deep learning and IoT perspective
Pages: ['18']
Matched terms: ['backbone network traffic']

Article ID: 43. Recurrent nonsymmetric deep auto encoder approach for network intrusion detection system.pdf
Title: Recurrent nonsymmetric deep auto encoder approach for network intrusion detection system
Pages: ['1', '2', '3', '4', '5', '6']
Matched terms: ['RNDAE', 'backbone network traffic', 'real-world backbone']

Article ID: DCNN A novel binary and multi-class network intrusion detection model via deep convolutional neural network (2024).pdf
Title: DCNN: a novel binary and multi-class network intrusion detection model via deep convolutional neural network
Pages: ['18']
Matched terms: ['RNDAE']


In [ ]:
# Fill the four missing evidence article IDs in the evaluation dataset.

article_id_updates = {
    "Q068": "15. Intrusion detection in the internet of things using convolutional neural networks an explainable AI approach.pdf",
    "Q069": "19. Optimizing intrusion detection using a hybrid CNN LSTM deep learning framework on the NSL KDD dataset (2026).pdf",
    "Q070": "43. Recurrent nonsymmetric deep auto encoder approach for network intrusion detection system.pdf",
    "Q075": "21. A deployment-efficient CNN-LSTM framework for real-time malicious traffic detection across heterogeneous, imbalanced, and encrypted network traffic (2026).pdf"
}

for question_id, article_id in article_id_updates.items():
    row_index = evaluation_df.index[
        evaluation_df["question_id"] == question_id
    ]

    if len(row_index) != 1:
        raise ValueError(f"Unexpected number of rows for {question_id}")

    evaluation_df.loc[row_index[0], "evidence_article_ids"] = article_id

# Save the updated CSV.
evaluation_df.to_csv(evaluation_path, index=False)

print("Updated evidence references:")
display(
    evaluation_df[
        evaluation_df["question_id"].isin(article_id_updates)
    ][["question_id", "evidence_article_ids", "evidence_pages"]]
)

Updated evidence references:


,question_id,evidence_article_ids,evidence_pages
67,Q068,15. Intrusion detection in the internet of thi...,21
68,Q069,19. Optimizing intrusion detection using a hyb...,16
69,Q070,43. Recurrent nonsymmetric deep auto encoder a...,6
74,Q075,21. A deployment-efficient CNN-LSTM framework ...,37


In [ ]:
# Check that every populated evidence article ID exists in the corpus.

corpus_article_ids = {
    chunk.get("article_id")
    for chunk in sentence_chunks
    if chunk.get("article_id")
}

referenced_rows = evaluation_df[
    evaluation_df["evidence_article_ids"].notna()
]

invalid_references = []

for _, row in referenced_rows.iterrows():
    article_ids = [
        item.strip()
        for item in str(row["evidence_article_ids"]).split(";")
        if item.strip()
    ]

    for article_id in article_ids:
        if article_id not in corpus_article_ids:
            invalid_references.append(
                (row["question_id"], article_id)
            )

print("Questions with populated evidence references:",
      len(referenced_rows))
print("Invalid article references:", len(invalid_references))

if invalid_references:
    display(pd.DataFrame(
        invalid_references,
        columns=["question_id", "invalid_article_id"]
    ))
else:
    print("All populated article IDs exist in the corpus.")

Questions with populated evidence references: 75
Invalid article references: 0
All populated article IDs exist in the corpus.


In [ ]:
# Validate that every cited PDF page exists in its referenced article.

import pandas as pd

# Build a map of article IDs to their available PDF pages.
article_page_map = {}

for chunk in sentence_chunks:
    article_id = chunk.get("article_id")
    page_number = chunk.get("page_number")

    if article_id and pd.notna(page_number):
        try:
            page_number = int(float(page_number))
            article_page_map.setdefault(article_id, set()).add(page_number)
        except (ValueError, TypeError):
            continue

# Check every populated evidence reference.
invalid_page_references = []
missing_page_references = []
checked_page_references = 0

for _, row in evaluation_df.iterrows():

    if pd.isna(row["evidence_article_ids"]):
        continue

    article_ids = [
        item.strip()
        for item in str(row["evidence_article_ids"]).split(";")
        if item.strip()
    ]

    if pd.isna(row["evidence_pages"]):
        missing_page_references.append(row["question_id"])
        continue

    pages = [
        item.strip()
        for item in str(row["evidence_pages"]).split(";")
        if item.strip()
    ]

    if len(article_ids) != len(pages):
        invalid_page_references.append({
            "question_id": row["question_id"],
            "problem": "Number of article IDs does not match number of pages",
            "article_ids": article_ids,
            "pages": pages
        })
        continue

    for article_id, page in zip(article_ids, pages):
        try:
            page_number = int(float(page))
            checked_page_references += 1

            available_pages = article_page_map.get(article_id, set())

            if page_number not in available_pages:
                invalid_page_references.append({
                    "question_id": row["question_id"],
                    "article_id": article_id,
                    "page": page_number,
                    "problem": "Page not found in corpus"
                })

        except (ValueError, TypeError):
            invalid_page_references.append({
                "question_id": row["question_id"],
                "article_id": article_id,
                "page": page,
                "problem": "Invalid page number"
            })

print("Page references checked:", checked_page_references)
print("Questions with missing page references:",
      missing_page_references)
print("Invalid page references:", len(invalid_page_references))

if invalid_page_references:
    display(pd.DataFrame(invalid_page_references))
else:
    print("All populated evidence page references exist in the corpus.")

Page references checked: 89
Questions with missing page references: []
Invalid page references: 6


,question_id,problem,article_ids,pages
0,Q067,Number of article IDs does not match number of...,[12. Network intrusion detection using feature...,"[3, 21]"
1,Q071,Number of article IDs does not match number of...,[An_explainable_deep_learning_framework_for_mu...,"[17, 23]"
2,Q072,Number of article IDs does not match number of...,[An_explainable_deep_learning_framework_for_mu...,"[23, 24]"
3,Q073,Number of article IDs does not match number of...,[An_explainable_deep_learning_framework_for_mu...,"[13, 24]"
4,Q074,Number of article IDs does not match number of...,[An_explainable_deep_learning_framework_for_mu...,"[17, 22]"
5,Q079,Number of article IDs does not match number of...,[44. Feature importance guided autoencoder for...,"[9, 13]"


In [ ]:
# Validate multiple page references belonging to a single article.

invalid_page_references = []
checked_page_references = 0
missing_page_references = []

for _, row in evaluation_df.iterrows():

    if pd.isna(row["evidence_article_ids"]):
        continue

    article_ids = [
        item.strip()
        for item in str(row["evidence_article_ids"]).split(";")
        if item.strip()
    ]

    if pd.isna(row["evidence_pages"]):
        missing_page_references.append(row["question_id"])
        continue

    pages = [
        item.strip()
        for item in str(row["evidence_pages"]).split(";")
        if item.strip()
    ]

    # One article can have several supporting pages.
    if len(article_ids) == 1:
        page_pairs = [
            (article_ids[0], page) for page in pages
        ]

    # When multiple articles are listed, pair them with their pages.
    elif len(article_ids) == len(pages):
        page_pairs = list(zip(article_ids, pages))

    else:
        invalid_page_references.append({
            "question_id": row["question_id"],
            "problem": "Ambiguous article-to-page mapping",
            "article_ids": article_ids,
            "pages": pages
        })
        continue

    for article_id, page in page_pairs:
        try:
            page_number = int(float(page))
            checked_page_references += 1

            available_pages = article_page_map.get(article_id, set())

            if page_number not in available_pages:
                invalid_page_references.append({
                    "question_id": row["question_id"],
                    "article_id": article_id,
                    "page": page_number,
                    "problem": "Page not found in corpus"
                })

        except (ValueError, TypeError):
            invalid_page_references.append({
                "question_id": row["question_id"],
                "article_id": article_id,
                "page": page,
                "problem": "Invalid page number"
            })

print("Page references checked:", checked_page_references)
print("Questions with missing page references:",
      missing_page_references)
print("Invalid page references:", len(invalid_page_references))

if invalid_page_references:
    display(pd.DataFrame(invalid_page_references))
else:
    print("All populated evidence page references exist in the corpus.")

Page references checked: 101
Questions with missing page references: []
Invalid page references: 0
All populated evidence page references exist in the corpus.


Final step: validate the complete 80-question dataset

In [ ]:
# Perform final structural and reference-integrity validation.

evaluation_df = pd.read_csv(evaluation_path)

expected_categories = {
    "Factual": 20,
    "Conceptual": 20,
    "Multi-paper Synthesis": 20,
    "Unanswerable": 20
}

expected_ids = {f"Q{i:03d}" for i in range(1, 81)}

category_counts = evaluation_df["question_category"].value_counts().to_dict()

missing_values = evaluation_df[
    ["question_id", "question_category", "question",
     "gold_answer", "answerable", "verification_notes"]
].isna().sum()

duplicate_ids = evaluation_df[
    evaluation_df["question_id"].duplicated(keep=False)
]

answerability_errors = []

for _, row in evaluation_df.iterrows():
    expected_answerable = row["question_id"] <= "Q060"
    actual_answerable = str(row["answerable"]).strip().lower() == "true"

    if expected_answerable != actual_answerable:
        answerability_errors.append(row["question_id"])

print("DATASET SUMMARY")
print("-" * 50)
print("Total questions:", len(evaluation_df))
print("Unique question IDs:", evaluation_df["question_id"].nunique())
print("Category counts:", category_counts)
print("Question IDs complete:", set(evaluation_df["question_id"]) == expected_ids)
print("Category counts correct:", category_counts == expected_categories)
print("Duplicate IDs:", len(duplicate_ids))
print("Missing required values:", missing_values.sum())
print("Answerability errors:", answerability_errors)

print("\nREFERENCE VALIDATION")
print("-" * 50)
print("Populated article references:",
      evaluation_df["evidence_article_ids"].notna().sum())
print("Populated page references:",
      evaluation_df["evidence_pages"].notna().sum())
print("Invalid article references:", len(invalid_references))
print("Invalid page references:", len(invalid_page_references))

all_checks_passed = (
    len(evaluation_df) == 80
    and evaluation_df["question_id"].nunique() == 80
    and set(evaluation_df["question_id"]) == expected_ids
    and category_counts == expected_categories
    and len(duplicate_ids) == 0
    and missing_values.sum() == 0
    and len(answerability_errors) == 0
    and len(invalid_references) == 0
    and len(invalid_page_references) == 0
)

print("\n" + "=" * 50)

if all_checks_passed:
    print("ALL STRUCTURAL AND REFERENCE CHECKS PASSED.")
else:
    print("SOME CHECKS FAILED. Review the results above.")

print("=" * 50)

DATASET SUMMARY
--------------------------------------------------
Total questions: 80
Unique question IDs: 80
Category counts: {'Factual': 20, 'Conceptual': 20, 'Multi-paper Synthesis': 20, 'Unanswerable': 20}
Question IDs complete: True
Category counts correct: True
Duplicate IDs: 0
Missing required values: 0
Answerability errors: []

REFERENCE VALIDATION
--------------------------------------------------
Populated article references: 75
Populated page references: 75
Invalid article references: 0
Invalid page references: 0

ALL STRUCTURAL AND REFERENCE CHECKS PASSED.


10: Evaluation

10.1: Back up the evaluation dataset

In [ ]:
# Create a backup of the completed evaluation dataset.

import shutil
import os

backup_path = os.path.join(
    os.path.dirname(evaluation_path),
    "evaluation_questions_backup.csv"
)

shutil.copy2(evaluation_path, backup_path)

print("Evaluation dataset backup created successfully.")
print("Backup location:", backup_path)
print("Backup file exists:", os.path.exists(backup_path))

Evaluation dataset backup created successfully.
Backup location: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/evaluation_questions_backup.csv
Backup file exists: True


10.2 — Load evaluation questions

In [ ]:
# Load the evaluation dataset and inspect its basic structure.

evaluation_df = pd.read_csv(evaluation_path)

print("Evaluation dataset loaded successfully.")
print("Dataset shape:", evaluation_df.shape)

print("\nQuestion categories:")
print(evaluation_df["question_category"].value_counts())

print("\nAnswerability distribution:")
print(evaluation_df["answerable"].value_counts())

print("\nColumn data types:")
print(evaluation_df.dtypes)

print("\nFirst five evaluation questions:")
display(evaluation_df[
    ["question_id", "question_category", "question", "answerable"]
].head())


Evaluation dataset loaded successfully.
Dataset shape: (80, 8)

Question categories:
question_category
Factual                  20
Conceptual               20
Multi-paper Synthesis    20
Unanswerable             20
Name: count, dtype: int64

Answerability distribution:
answerable
True     60
False    20
Name: count, dtype: int64

Column data types:
question_id             object
question_category       object
question                object
gold_answer             object
answerable                bool
evidence_article_ids    object
evidence_pages          object
verification_notes      object
dtype: object

First five evaluation questions:


,question_id,question_category,question,answerable
0,Q001,Factual,How many phishing and legitimate webpages were...,True
1,Q002,Factual,What true-positive and false-positive rates di...,True
2,Q003,Factual,Which cryptographic protocols are discussed fo...,True
3,Q004,Factual,What three feature-fusion deep learning models...,True
4,Q005,Factual,Which two datasets are used to evaluate the fe...,True


10.3: Prepare the ground-truth evidence references

In [ ]:
# Prepare and inspect the ground-truth article references.

ground_truth = {}

for _, row in evaluation_df.iterrows():
    question_id = row["question_id"]

    if pd.notna(row["evidence_article_ids"]):
        article_ids = [
            article.strip()
            for article in str(row["evidence_article_ids"]).split(";")
            if article.strip()
        ]
    else:
        article_ids = []

    ground_truth[question_id] = article_ids

answerable_questions = [
    qid for qid, refs in ground_truth.items() if refs
]

unanswerable_questions = [
    qid for qid, refs in ground_truth.items() if not refs
]

print("GROUND-TRUTH REFERENCE SUMMARY")
print("-" * 50)
print("Total questions:", len(ground_truth))
print("Questions with evidence references:", len(answerable_questions))
print("Questions without evidence references:", len(unanswerable_questions))

print("\nSAMPLE GROUND-TRUTH REFERENCES")
print("-" * 50)

for question_id in ["Q001", "Q041", "Q061", "Q066"]:
    print(f"{question_id}: {ground_truth[question_id]}")

GROUND-TRUTH REFERENCE SUMMARY
--------------------------------------------------
Total questions: 80
Questions with evidence references: 75
Questions without evidence references: 5

SAMPLE GROUND-TRUTH REFERENCES
--------------------------------------------------
Q001: ['A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf']
Q041: ['An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf', 'An_intelligent_identification_and_classification_system_for_malicious_uniform_resource_locators.pdf']
Q061: []
Q066: ['1. Practical autoencoder based anomaly detection by using vector reconstruction error.pdf']


10.4: Check the existing retrieval functions

In [ ]:
# Check that the previously implemented retrieval functions are available.

required_functions = [
    "bm25_retrieval",
    "semantic_retrieval",
    "hybrid_retrieval",
    "rerank_results",
    "generate_evidence_grounded_answer"
]

for function_name in required_functions:
    available = function_name in globals() and callable(globals()[function_name])
    print(f"{function_name}: {'Available' if available else 'Not available'}")

bm25_retrieval: Not available
semantic_retrieval: Not available
hybrid_retrieval: Available
rerank_results: Available
generate_evidence_grounded_answer: Available


10.5: Inspect the existing retrieval implementation

In [ ]:
# Inspect the existing retrieval functions and related variables.

import inspect

print("HYBRID RETRIEVAL FUNCTION")
print("-" * 50)
print("Signature:", inspect.signature(hybrid_retrieval))
print(inspect.getsource(hybrid_retrieval))

print("\nRELATED RETRIEVAL FUNCTIONS AND VARIABLES")
print("-" * 50)

for name in globals():
    if any(term in name.lower() for term in ["bm25", "semantic", "faiss", "embedding"]):
        print(name, ":", type(globals()[name]).__name__)

HYBRID RETRIEVAL FUNCTION
--------------------------------------------------
Signature: (query, top_k=5, candidate_k=10, rrf_k=60)
def hybrid_retrieval(query, top_k=5, candidate_k=10, rrf_k=60):
    # Retrieve candidates using FAISS
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    dense_scores, dense_indices = faiss_index.search(
        query_embedding, candidate_k
    )

    # Retrieve candidates using BM25
    query_tokens = re.findall(r"\b\w+\b", query.lower())
    sparse_scores = bm25_index.get_scores(query_tokens)
    sparse_indices = np.argsort(sparse_scores)[::-1][:candidate_k]

    # Combine rankings using Reciprocal Rank Fusion
    rrf_scores = {}

    for rank, index in enumerate(dense_indices[0], start=1):
        rrf_scores[index] = rrf_scores.get(index, 0) + 1 / (rrf_k + rank)

    for rank, index in enumerate(sparse_indices, start=1):
        rrf_scores[index] = rr

10.6: Identify the existing retrieval variables

In [ ]:
# Identify the existing retrieval models and indexes.

retrieval_variables = [
    name for name in list(globals().keys())
    if any(term in name.lower()
           for term in ["bm25", "semantic", "faiss", "embedding"])
]

for name in retrieval_variables:
    print(f"{name}: {type(globals()[name]).__name__}")

faiss: module
faiss_index: IndexFlatIP
BM25Okapi: type
embedding_model: SentenceTransformer
bm25_tokenized_corpus: list
bm25_index: BM25Okapi


10.7: Prepare individual retrieval functions

In [ ]:
# Display the current retrieval-related function definitions.

import inspect

print("AVAILABLE RETRIEVAL FUNCTIONS")
print("-" * 50)

for name, obj in list(globals().items()):
    if inspect.isfunction(obj) and any(
        term in name.lower()
        for term in ["retriev", "search", "rank"]
    ):
        print(f"\nFunction: {name}")
        print("Signature:", inspect.signature(obj))

AVAILABLE RETRIEVAL FUNCTIONS
--------------------------------------------------

Function: hybrid_retrieval
Signature: (query, top_k=5, candidate_k=10, rrf_k=60)

Function: rerank_results
Signature: (query, candidate_results, top_k=5)


10.8: Verify index alignment

In [ ]:
# Verify that the retrieval indexes align with the sentence-based chunks.

print("RETRIEVAL INDEX ALIGNMENT")
print("-" * 50)

print("Total sentence chunks:", len(sentence_chunks))
print("FAISS indexed vectors:", faiss_index.ntotal)
print("BM25 tokenized documents:", len(bm25_tokenized_corpus))
print("BM25 indexed documents:", len(bm25_index.doc_len))

alignment_check = (
    len(sentence_chunks) == faiss_index.ntotal
    == len(bm25_tokenized_corpus)
    == len(bm25_index.doc_len)
)

print("\nIndex alignment:", alignment_check)

if alignment_check:
    print("All retrieval indexes are aligned with the sentence chunks.")
else:
    print("WARNING: Retrieval index sizes do not match.")

RETRIEVAL INDEX ALIGNMENT
--------------------------------------------------
Total sentence chunks: 13187
FAISS indexed vectors: 13187
BM25 tokenized documents: 13187
BM25 indexed documents: 13187

Index alignment: True
All retrieval indexes are aligned with the sentence chunks.


10.9: Implement individual retrieval functions

In [ ]:
# Define BM25-only and dense-only retrieval using the existing indexes.

def bm25_only_retrieval(query, top_k=5):
    query_tokens = re.findall(r"\b\w+\b", query.lower())

    scores = bm25_index.get_scores(query_tokens)
    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(ranked_indices, start=1):
        chunk = sentence_chunks[index]

        results.append({
            "rank": rank,
            "score": float(scores[index]),
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "year": chunk["year"],
            "page_number": chunk["page_number"],
            "text": chunk["text"]
        })

    return results


def dense_only_retrieval(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []

    for rank, index in enumerate(indices[0], start=1):
        if index == -1:
            continue

        chunk = sentence_chunks[index]

        results.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "year": chunk["year"],
            "page_number": chunk["page_number"],
            "text": chunk["text"]
        })

    return results


print("BM25-only retrieval function created.")
print("Dense-only retrieval function created.")

print("\nFunction signatures:")
print("BM25:", inspect.signature(bm25_only_retrieval))
print("Dense:", inspect.signature(dense_only_retrieval))

BM25-only retrieval function created.
Dense-only retrieval function created.

Function signatures:
BM25: (query, top_k=5)
Dense: (query, top_k=5)


10.10: Test the retrieval functions

In [ ]:
# Test BM25, dense and hybrid retrieval using the first evaluation question.

test_question = evaluation_df.loc[
    evaluation_df["question_id"] == "Q001", "question"
].iloc[0]

print("TEST QUESTION:")
print(test_question)

print("\n" + "=" * 60)

retrieval_methods = {
    "BM25": bm25_only_retrieval,
    "Dense": dense_only_retrieval,
    "Hybrid": hybrid_retrieval
}

for method_name, retrieval_function in retrieval_methods.items():

    results = retrieval_function(test_question, top_k=5)

    print(f"\n{method_name.upper()} RETRIEVAL")
    print("-" * 60)

    for result in results:
        print(
            f"Rank: {result['rank']} | "
            f"Article: {result['article_id']} | "
            f"Page: {result['page_number']}"
        )

TEST QUESTION:
How many phishing and legitimate webpages were used to evaluate CANTINA+?


BM25 RETRIEVAL
------------------------------------------------------------
Rank: 1 | Article: Phishing_website_detection_How_effective_are_deeplearning_based_models_and_hyperparameter_optimization.pdf | Page: 10
Rank: 2 | Article: CANTINA_A_Content_Based_Approach_to_Dectecting_Phishing_web_sites.pdf | Page: 4
Rank: 3 | Article: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf | Page: 20
Rank: 4 | Article: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf | Page: 19
Rank: 5 | Article: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf | Page: 5

DENSE RETRIEVAL
------------------------------------------------------------
Rank: 1 | Article: Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf | Page: 3
Rank: 2 | Article: A_Feature_Ri

10.11: Test Cross-Encoder reranking

In [ ]:
# Test hybrid retrieval followed by Cross-Encoder reranking.

test_candidates = hybrid_retrieval(
    test_question,
    top_k=10,
    candidate_k=20
)

reranked_results = rerank_results(
    test_question,
    test_candidates,
    top_k=5
)

print("HYBRID + CROSS-ENCODER RERANKING")
print("-" * 60)

for result in reranked_results:
    print(
        f"Rank: {result['rank']} | "
        f"Article: {result['article_id']} | "
        f"Page: {result['page_number']}"
    )

HYBRID + CROSS-ENCODER RERANKING
------------------------------------------------------------
Rank: 3 | Article: Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf | Page: 3
Rank: 8 | Article: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf | Page: 20
Rank: 10 | Article: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf | Page: 5
Rank: 7 | Article: CANTINA_A_Content_Based_Approach_to_Dectecting_Phishing_web_sites.pdf | Page: 4
Rank: 5 | Article: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf | Page: 18


10.12: Inspect the Cross-Encoder reranking function

In [ ]:
# Inspect the existing Cross-Encoder reranking implementation.

import inspect

print("RERANKING FUNCTION")
print("-" * 50)

print("Signature:", inspect.signature(rerank_results))
print(inspect.getsource(rerank_results))

RERANKING FUNCTION
--------------------------------------------------
Signature: (query, candidate_results, top_k=5)
def rerank_results(query, candidate_results, top_k=5):
    pairs = [
        [query, result["text"]]
        for result in candidate_results
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for result, score in zip(candidate_results, scores):
        reranked.append({
            **result,
            "cross_encoder_score": float(score)
        })

    reranked.sort(
        key=lambda x: x["cross_encoder_score"],
        reverse=True
    )

    return reranked[:top_k]



10.13: Verify the corrected ranking

In [ ]:
# Assign new sequential ranks to the Cross-Encoder reranked results.

corrected_reranked_results = []

for new_rank, result in enumerate(reranked_results, start=1):
    corrected_result = result.copy()
    corrected_result["rank"] = new_rank
    corrected_reranked_results.append(corrected_result)

print("CORRECTED CROSS-ENCODER RANKING")
print("-" * 60)

for result in corrected_reranked_results:
    print(
        f"Rank: {result['rank']} | "
        f"Cross-Encoder score: {result['cross_encoder_score']:.4f} | "
        f"Article: {result['article_id']} | "
        f"Page: {result['page_number']}"
    )

CORRECTED CROSS-ENCODER RANKING
------------------------------------------------------------
Rank: 1 | Cross-Encoder score: 5.4023 | Article: Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf | Page: 3
Rank: 2 | Cross-Encoder score: 4.6830 | Article: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf | Page: 20
Rank: 3 | Cross-Encoder score: 4.1403 | Article: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf | Page: 5
Rank: 4 | Cross-Encoder score: 4.0285 | Article: CANTINA_A_Content_Based_Approach_to_Dectecting_Phishing_web_sites.pdf | Page: 4
Rank: 5 | Cross-Encoder score: 3.7658 | Article: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf | Page: 18


10.14: Finalise the retrieval evaluation scope

In [ ]:
# Summarise ground-truth references by question category and answerability.

evaluation_df["has_evidence_reference"] = (
    evaluation_df["question_id"].map(
        lambda qid: len(ground_truth[qid]) > 0
    )
)

reference_summary = (
    evaluation_df.groupby(
        ["question_category", "answerable", "has_evidence_reference"]
    )
    .size()
    .reset_index(name="question_count")
)

display(reference_summary)

print("\nOVERALL RETRIEVAL EVALUATION SCOPE")
print("-" * 50)

print(
    "Questions with evidence references:",
    evaluation_df["has_evidence_reference"].sum()
)

print(
    "Questions without evidence references:",
    (~evaluation_df["has_evidence_reference"]).sum()
)

print(
    "Answerable questions with references:",
    (
        evaluation_df["answerable"]
        & evaluation_df["has_evidence_reference"]
    ).sum()
)

print(
    "Unanswerable questions with limitation references:",
    (
        ~evaluation_df["answerable"]
        & evaluation_df["has_evidence_reference"]
    ).sum()
)

,question_category,answerable,has_evidence_reference,question_count
0,Conceptual,True,True,20
1,Factual,True,True,20
2,Multi-paper Synthesis,True,True,20
3,Unanswerable,False,False,5
4,Unanswerable,False,True,15



OVERALL RETRIEVAL EVALUATION SCOPE
--------------------------------------------------
Questions with evidence references: 75
Questions without evidence references: 5
Answerable questions with references: 60
Unanswerable questions with limitation references: 15


10.16: Test article-level deduplication

In [ ]:
# Test article-level deduplication using the Q001 retrieval results.

def deduplicate_by_article(results):
    unique_results = []
    seen_articles = set()

    for result in results:
        article_id = result["article_id"]

        if article_id not in seen_articles:
            unique_results.append(result)
            seen_articles.add(article_id)

    return unique_results


print("ARTICLE-LEVEL DEDUPLICATION TEST")
print("-" * 50)

for method_name, results in [
    ("BM25", bm25_only_retrieval(test_question, top_k=10)),
    ("Dense", dense_only_retrieval(test_question, top_k=10)),
    ("Hybrid", hybrid_retrieval(test_question, top_k=10)),
    ("Hybrid + Cross-Encoder", corrected_reranked_results)
]:
    unique_results = deduplicate_by_article(results)

    print(f"\n{method_name}")
    print("Original chunks:", len(results))
    print("Unique articles:", len(unique_results))

    for rank, result in enumerate(unique_results, start=1):
        print(f"Article rank {rank}: {result['article_id']}")

ARTICLE-LEVEL DEDUPLICATION TEST
--------------------------------------------------

BM25
Original chunks: 10
Unique articles: 5
Article rank 1: Phishing_website_detection_How_effective_are_deeplearning_based_models_and_hyperparameter_optimization.pdf
Article rank 2: CANTINA_A_Content_Based_Approach_to_Dectecting_Phishing_web_sites.pdf
Article rank 3: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf
Article rank 4: nlp002.pdf
Article rank 5: Detecting_phishing_web_pages_by_exploiting_raw_URL_and_HTML_characteristics.pdf

Dense
Original chunks: 10
Unique articles: 4
Article rank 1: Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf
Article rank 2: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Article rank 3: CANTINA_A_Content_Based_Approach_to_Dectecting_Phishing_web_sites.pdf
Article rank 4: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques

10.17: Test a larger candidate pool

In [ ]:
# Test retrieval with a larger candidate pool for article-level evaluation.

candidate_k = 100

bm25_test = bm25_only_retrieval(test_question, top_k=candidate_k)

dense_test = dense_only_retrieval(test_question, top_k=candidate_k)

hybrid_test = hybrid_retrieval(
    test_question,
    top_k=candidate_k,
    candidate_k=candidate_k
)

reranked_test = rerank_results(
    test_question,
    hybrid_test,
    top_k=candidate_k
)

retrieval_test_results = {
    "BM25": bm25_test,
    "Dense": dense_test,
    "Hybrid": hybrid_test,
    "Hybrid + Cross-Encoder": reranked_test
}

print("LARGE CANDIDATE POOL TEST")
print("-" * 50)

for method_name, results in retrieval_test_results.items():
    unique_results = deduplicate_by_article(results)

    print(f"\n{method_name}")
    print("Retrieved chunks:", len(results))
    print("Unique articles:", len(unique_results))
    print("Enough articles for Recall@5:", len(unique_results) >= 5)
    print("Enough articles for nDCG@10:", len(unique_results) >= 10)

LARGE CANDIDATE POOL TEST
--------------------------------------------------

BM25
Retrieved chunks: 100
Unique articles: 21
Enough articles for Recall@5: True
Enough articles for nDCG@10: True

Dense
Retrieved chunks: 100
Unique articles: 20
Enough articles for Recall@5: True
Enough articles for nDCG@10: True

Hybrid
Retrieved chunks: 100
Unique articles: 23
Enough articles for Recall@5: True
Enough articles for nDCG@10: True

Hybrid + Cross-Encoder
Retrieved chunks: 100
Unique articles: 23
Enough articles for Recall@5: True
Enough articles for nDCG@10: True


10.18: Prepare article-level rankings

In [ ]:
# Prepare article-level rankings for the four retrieval methods.

article_level_test_results = {}

for method_name, results in retrieval_test_results.items():
    unique_results = deduplicate_by_article(results)

    ranked_articles = []

    for rank, result in enumerate(unique_results, start=1):
        article = result.copy()
        article["rank"] = rank
        ranked_articles.append(article)

    article_level_test_results[method_name] = ranked_articles

    print(f"\n{method_name}")
    print("Top 5 unique articles:")

    for article in ranked_articles[:5]:
        print(
            f"Rank {article['rank']}: "
            f"{article['article_id']}"
        )


BM25
Top 5 unique articles:
Rank 1: Phishing_website_detection_How_effective_are_deeplearning_based_models_and_hyperparameter_optimization.pdf
Rank 2: CANTINA_A_Content_Based_Approach_to_Dectecting_Phishing_web_sites.pdf
Rank 3: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf
Rank 4: nlp002.pdf
Rank 5: Detecting_phishing_web_pages_by_exploiting_raw_URL_and_HTML_characteristics.pdf

Dense
Top 5 unique articles:
Rank 1: Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf
Rank 2: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Rank 3: CANTINA_A_Content_Based_Approach_to_Dectecting_Phishing_web_sites.pdf
Rank 4: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf
Rank 5: Phishing_Website_Detection_Using_Deep_Learning_Models.pdf

Hybrid
Top 5 unique articles:
Rank 1: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_De

10.19: Define evaluation metrics

In [ ]:
# Define article-level Recall@5 and nDCG@10 evaluation metrics.

import math

def calculate_recall_at_k(results, relevant_articles, k=5):
    if not relevant_articles:
        return None

    retrieved_articles = {
        result["article_id"]
        for result in results[:k]
    }

    relevant_retrieved = retrieved_articles.intersection(
        set(relevant_articles)
    )

    return len(relevant_retrieved) / len(relevant_articles)


def calculate_ndcg_at_k(results, relevant_articles, k=10):
    if not relevant_articles:
        return None

    relevant_set = set(relevant_articles)

    relevance_scores = [
        1 if result["article_id"] in relevant_set else 0
        for result in results[:k]
    ]

    dcg = sum(
        relevance / math.log2(rank + 1)
        for rank, relevance in enumerate(relevance_scores, start=1)
    )

    ideal_relevance = [1] * min(len(relevant_set), k)

    idcg = sum(
        relevance / math.log2(rank + 1)
        for rank, relevance in enumerate(ideal_relevance, start=1)
    )

    return dcg / idcg if idcg > 0 else 0


# Test both metrics using Q001.
q001_relevant_articles = ground_truth["Q001"]

print("Q001 METRIC TEST")
print("-" * 50)

for method_name, results in article_level_test_results.items():
    recall = calculate_recall_at_k(
        results,
        q001_relevant_articles,
        k=5
    )

    ndcg = calculate_ndcg_at_k(
        results,
        q001_relevant_articles,
        k=10
    )

    print(f"\n{method_name}")
    print(f"Recall@5: {recall:.4f}")
    print(f"nDCG@10: {ndcg:.4f}")

Q001 METRIC TEST
--------------------------------------------------

BM25
Recall@5: 0.0000
nDCG@10: 0.3562

Dense
Recall@5: 1.0000
nDCG@10: 0.6309

Hybrid
Recall@5: 1.0000
nDCG@10: 0.5000

Hybrid + Cross-Encoder
Recall@5: 1.0000
nDCG@10: 1.0000


10.20: Evaluate all 60 answerable questions

In [ ]:
# Evaluate article-level retrieval metrics for all 60 answerable questions.

evaluation_results = []

answerable_df = evaluation_df[
    evaluation_df["question_id"].isin(
        [f"Q{i:03d}" for i in range(1, 61)]
    )
].copy()

for _, row in answerable_df.iterrows():
    question_id = row["question_id"]
    query = row["question"]
    relevant_articles = ground_truth[question_id]

    # Retrieve 100 candidate chunks using each method.
    bm25_results = bm25_only_retrieval(query, top_k=100)

    dense_results = dense_only_retrieval(query, top_k=100)

    hybrid_results = hybrid_retrieval(
        query,
        top_k=100,
        candidate_k=100
    )

    reranked_results = rerank_results(
        query,
        hybrid_results,
        top_k=100
    )

    methods = {
        "BM25": bm25_results,
        "Dense": dense_results,
        "Hybrid": hybrid_results,
        "Hybrid + Cross-Encoder": reranked_results
    }

    for method_name, results in methods.items():
        unique_results = deduplicate_by_article(results)

        recall = calculate_recall_at_k(
            unique_results,
            relevant_articles,
            k=5
        )

        ndcg = calculate_ndcg_at_k(
            unique_results,
            relevant_articles,
            k=10
        )

        evaluation_results.append({
            "question_id": question_id,
            "question_category": row["question_category"],
            "method": method_name,
            "relevant_article_count": len(relevant_articles),
            "retrieved_unique_articles": len(unique_results),
            "recall_at_5": recall,
            "ndcg_at_10": ndcg
        })

    print(f"Completed {question_id}")

retrieval_evaluation_df = pd.DataFrame(evaluation_results)

print("\nEVALUATION COMPLETED")
print("Total evaluation records:", len(retrieval_evaluation_df))
print("Unique questions:", retrieval_evaluation_df["question_id"].nunique())
print("\nRecords per method:")
print(retrieval_evaluation_df["method"].value_counts())

Completed Q001
Completed Q002
Completed Q003
Completed Q004
Completed Q005
Completed Q006
Completed Q007
Completed Q008
Completed Q009
Completed Q010
Completed Q011
Completed Q012
Completed Q013
Completed Q014
Completed Q015
Completed Q016
Completed Q017
Completed Q018
Completed Q019
Completed Q020
Completed Q021
Completed Q022
Completed Q023
Completed Q024
Completed Q025
Completed Q026
Completed Q027
Completed Q028
Completed Q029
Completed Q030
Completed Q031
Completed Q032
Completed Q033
Completed Q034
Completed Q035
Completed Q036
Completed Q037
Completed Q038
Completed Q039
Completed Q040
Completed Q041
Completed Q042
Completed Q043
Completed Q044
Completed Q045
Completed Q046
Completed Q047
Completed Q048
Completed Q049
Completed Q050
Completed Q051
Completed Q052
Completed Q053
Completed Q054
Completed Q055
Completed Q056
Completed Q057
Completed Q058
Completed Q059
Completed Q060

EVALUATION COMPLETED
Total evaluation records: 240
Unique questions: 60

Records per method:
method

10.21: Calculate overall retrieval performance

In [ ]:
# Calculate average retrieval performance for each method.

overall_retrieval_performance = (
    retrieval_evaluation_df
    .groupby("method")
    .agg(
        evaluated_questions=("question_id", "nunique"),
        average_recall_at_5=("recall_at_5", "mean"),
        average_ndcg_at_10=("ndcg_at_10", "mean")
    )
    .reset_index()
)

overall_retrieval_performance[
    ["average_recall_at_5", "average_ndcg_at_10"]
] = overall_retrieval_performance[
    ["average_recall_at_5", "average_ndcg_at_10"]
].round(4)

print("OVERALL RETRIEVAL PERFORMANCE")
print("-" * 65)

display(overall_retrieval_performance)

OVERALL RETRIEVAL PERFORMANCE
-----------------------------------------------------------------


,method,evaluated_questions,average_recall_at_5,average_ndcg_at_10
0,BM25,60,0.7500,0.7108
1,Dense,60,0.7333,0.6639
2,Hybrid,60,0.7917,0.7552
3,Hybrid + Cross-Encoder,60,0.8583,0.8135


In [ ]:
# Save retrieval evaluation results to Google Drive.

import os

results_folder = "/content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data"

files_to_save = {
    "retrieval_evaluation_results.csv": retrieval_evaluation_df,
    "overall_retrieval_performance.csv": overall_retrieval_performance,
    "category_retrieval_performance.csv": category_performance
}

for filename, dataframe in files_to_save.items():
    path = os.path.join(results_folder, filename)
    dataframe.to_csv(path, index=False)
    print(f"Saved: {filename} | Rows: {len(dataframe)} | Exists: {os.path.exists(path)}")

print("\nAll retrieval evaluation files saved.")

Saved: retrieval_evaluation_results.csv | Rows: 240 | Exists: True
Saved: overall_retrieval_performance.csv | Rows: 4 | Exists: True
Saved: category_retrieval_performance.csv | Rows: 12 | Exists: True

All retrieval evaluation files saved.


10.22: Examine performance by question category

In [ ]:
# Compare retrieval performance across the three answerable question categories.

category_performance = (
    retrieval_evaluation_df
    .groupby(["question_category", "method"])
    .agg(
        evaluated_questions=("question_id", "nunique"),
        average_recall_at_5=("recall_at_5", "mean"),
        average_ndcg_at_10=("ndcg_at_10", "mean")
    )
    .reset_index()
)

category_performance[
    ["average_recall_at_5", "average_ndcg_at_10"]
] = category_performance[
    ["average_recall_at_5", "average_ndcg_at_10"]
].round(4)

print("RETRIEVAL PERFORMANCE BY QUESTION CATEGORY")
print("-" * 65)

display(category_performance)

RETRIEVAL PERFORMANCE BY QUESTION CATEGORY
-----------------------------------------------------------------


,question_category,method,evaluated_questions,average_recall_at_5,average_ndcg_at_10
0,Conceptual,BM25,20,0.750,0.7014
1,Conceptual,Dense,20,0.750,0.6076
2,Conceptual,Hybrid,20,0.800,0.7604
3,Conceptual,Hybrid + Cross-Encoder,20,0.850,0.7559
4,Factual,BM25,20,0.900,0.8872
5,Factual,Dense,20,0.850,0.7504
6,Factual,Hybrid,20,0.950,0.8497
7,Factual,Hybrid + Cross-Encoder,20,1.000,0.9815
8,Multi-paper Synthesis,BM25,20,0.600,0.5440
9,Multi-paper Synthesis,Dense,20,0.600,0.6338


10.23: Prepare answer-generation evaluation

In [ ]:
# Check the existing evidence-grounded answer generation function.

print("Answer-generation function available:",
      "generate_evidence_grounded_answer" in globals())

if "generate_evidence_grounded_answer" in globals():
    import inspect

    print("\nFunction signature:")
    print(inspect.signature(generate_evidence_grounded_answer))

    print("\nFunction documentation:")
    print(generate_evidence_grounded_answer.__doc__)

Answer-generation function available: True

Function signature:
(query, top_k=5)

Function documentation:
None


10.24: Inspect the function's implementation

In [ ]:
# Inspect the existing answer-generation function without executing it.

import inspect

print(inspect.getsource(generate_evidence_grounded_answer))

def generate_evidence_grounded_answer(query, top_k=5):

    # Retrieve relevant passages using hybrid retrieval
    candidates = hybrid_retrieval(
        query,
        top_k=10,
        candidate_k=20
    )

    # Rerank the retrieved passages
    evidence = rerank_results(
        query,
        candidates,
        top_k=top_k
    )

    # Prepare evidence with complete article metadata
    context = ""

    for i, passage in enumerate(evidence, start=1):

        year = passage["year"]

        if pd.notna(year):
            year = int(float(year))

        context += f"""
Evidence {i}
Article ID: {passage['article_id']}
Article Title: {passage['title']}
Publication Year: {year}
PDF Page Number: {passage['page_number']}
Supporting Passage:
{passage['text']}
"""

    # Prepare the improved evidence-grounded prompt
    prompt = f"""
You are an academic research assistant.

Answer the research question using only the supplied research evidence.

STRICT REQUIREMENTS:

1. Use only claim

In [ ]:
# Check for previously generated answer results that can be reused.

existing_answer_results = []

for variable_name, variable_value in list(globals().items()):
    if (
        isinstance(variable_value, dict)
        and {"question", "answer", "evidence"}.issubset(variable_value.keys())
    ):
        existing_answer_results.append(variable_name)

print("Previously generated answer result variables:")

if existing_answer_results:
    for variable_name in existing_answer_results:
        print("-", variable_name)
else:
    print("No reusable answer result found.")

Previously generated answer result variables:
- test_result


In [ ]:
print("TEST QUESTION:")
print(test_result["question"])

print("\nGENERATED ANSWER:")
print(test_result["answer"])

print("\nRETRIEVED EVIDENCE:")
print("Number of evidence passages:", len(test_result["evidence"]))

for i, passage in enumerate(test_result["evidence"], start=1):
    print(f"\nEvidence {i}")
    print("Article ID:", passage["article_id"])
    print("Title:", passage["title"])
    print("Year:", passage["year"])
    print("PDF Page:", passage["page_number"])


TEST QUESTION:
What machine learning techniques are used for phishing website detection?

GENERATED ANSWER:
Based on the provided research evidence, machine learning techniques are widely used and prevalent approaches for detecting phishing websites [Article ID: A_systematic_literature_review_on_phishing_website_detection_techniques.pdf, Title: A systematic literature review on phishing website detection techniques, Year: 2023, PDF Page: 2] [Article ID: Applications_of_deep_learning_for_phishing_detection.pdf, Title: Applications of deep learning for phishing detection: a systematic literature review, Year: 2022, PDF Page: 17]. 

The characteristics and findings regarding these techniques include:

* **General Performance and Characteristics:** Machine learning-based methods are generally characterized as fast, effective, capable of coping well with zero-hour attacks, and allowing for on-the-fly detection of phishing web pages [Article ID: Phishing_or_Not_Phishing_A_Survey_on_the_Detec

10.27: Check citation validity automatically

In [ ]:
article_ids_in_corpus = {
    chunk["article_id"] for chunk in sentence_chunks
}

citation_validation = []

for passage in test_result["evidence"]:
    article_id = passage["article_id"]
    page_number = int(float(passage["page_number"]))

    matching_pages = [
        int(float(chunk["page_number"]))
        for chunk in sentence_chunks
        if chunk["article_id"] == article_id
    ]

    citation_validation.append({
        "article_id": article_id,
        "article_exists": article_id in article_ids_in_corpus,
        "page_number": page_number,
        "page_exists": page_number in matching_pages
    })

citation_validation_df = pd.DataFrame(citation_validation)

display(citation_validation_df)

,article_id,article_exists,page_number,page_exists
0,Applications_of_deep_learning_for_phishing_det...,True,17,True
1,Modeling_Hybrid_Feature_Based_Phishing_Website...,True,2,True
2,A_systematic_literature_review_on_phishing_web...,True,2,True
3,Phishing_or_Not_Phishing_A_Survey_on_the_Detec...,True,17,True
4,Phishing_website_detection_How_effective_are_d...,True,2,True


10.28: Check citation metadata consistency

In [ ]:
metadata_validation = []

for passage in test_result["evidence"]:
    article_id = passage["article_id"]

    matching_metadata = metadata_inventory[
        metadata_inventory["filename"] == article_id
    ]

    if len(matching_metadata) == 1:
        metadata = matching_metadata.iloc[0]

        metadata_validation.append({
            "article_id": article_id,
            "title_matches": (
                str(passage["title"]).strip()
                == str(metadata["title"]).strip()
            ),
            "year_matches": (
                int(float(passage["year"]))
                == int(float(metadata["year"]))
            )
        })

    else:
        metadata_validation.append({
            "article_id": article_id,
            "title_matches": False,
            "year_matches": False
        })

metadata_validation_df = pd.DataFrame(metadata_validation)

display(metadata_validation_df)

,article_id,title_matches,year_matches
0,Applications_of_deep_learning_for_phishing_det...,True,True
1,Modeling_Hybrid_Feature_Based_Phishing_Website...,True,True
2,A_systematic_literature_review_on_phishing_web...,True,True
3,Phishing_or_Not_Phishing_A_Survey_on_the_Detec...,True,True
4,Phishing_website_detection_How_effective_are_d...,True,True


In [ ]:
# Identify the existing metadata DataFrame in the notebook.

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        if {"filename", "title", "year"}.issubset(value.columns):
            print("Metadata DataFrame:", name)
            print("Shape:", value.shape)

Metadata DataFrame: metadata_inventory
Shape: (162, 7)
Metadata DataFrame: matching_metadata
Shape: (1, 7)


In [ ]:
# Inspect the retrieved evidence
for i, passage in enumerate(test_result["evidence"], start=1):

    print(f"\n{'=' * 70}")
    print(f"EVIDENCE {i}")
    print("=" * 70)

    print("Article ID:", passage["article_id"])
    print("Title:", passage["title"])
    print("Year:", passage["year"])
    print("PDF Page:", passage["page_number"])

    print("\nSupporting Passage:")
    print(passage["text"])


EVIDENCE 1
Article ID: Applications_of_deep_learning_for_phishing_detection.pdf
Title: Applications of deep learning for phishing detection: a systematic literature review
Year: 2022.0
PDF Page: 17

Supporting Passage:
Applications of deep learning for phishing detection: a... 1473 Table 2 (continued) Study Review type Focus Description Singh and Meenu [62] Non-systematic Phishing website detection using machine learning Discusses the applications of ML techniques for phishing website detection along with various approaches protection approaches They found that the ML algorithms have achieved an approximate 99% accuracy for phishing website detection by combining 30 features Benavides et al. [12] Systematic Phishing attack solutions using Deep Learning Discusses the applications of DL techniques for phishing website detection Classified and analyzed various DL-based solutions DL algorithms have not been explored enough for phishing website detection Still, there is a need to identify 

In [ ]:
faithfulness_assessment = pd.DataFrame([
    {
        "claim": "Machine learning is a prevalent approach for phishing website detection.",
        "assessment": "Supported",
        "reason": "Evidence 3 explicitly describes machine learning as a prevalent approach."
    },
    {
        "claim": "Machine learning-based methods are generally fast and effective.",
        "assessment": "Supported",
        "reason": "Evidence 4 explicitly states this."
    },
    {
        "claim": "Machine learning methods cope well with zero-hour attacks.",
        "assessment": "Supported",
        "reason": "Evidence 4 explicitly reports this characteristic."
    },
    {
        "claim": "Phishing detection commonly uses URL information, website structure, and JavaScript features.",
        "assessment": "Supported",
        "reason": "Evidence 3 identifies all three feature types."
    },
    {
        "claim": "URL features are quick to obtain but vulnerable to manipulation.",
        "assessment": "Supported",
        "reason": "Evidence 4 describes this trade-off."
    },
    {
        "claim": "Source-code features are more robust against evasion but require downloads that can cause delays and safety issues.",
        "assessment": "Supported",
        "reason": "Evidence 4 explicitly describes these characteristics."
    },
    {
        "claim": "A study discussed in a review reported approximately 99% accuracy using 30 features.",
        "assessment": "Supported",
        "reason": "Evidence 1 attributes this finding to Singh and Meenu."
    },
    {
        "claim": "Deep learning is gaining popularity in cybersecurity and has been explored for phishing detection.",
        "assessment": "Supported",
        "reason": "Evidence 5 discusses its growing popularity and phishing detection research."
    },
    {
        "claim": "Deep-learning algorithms have not been explored enough for phishing website detection.",
        "assessment": "Supported",
        "reason": "Evidence 1 reports this conclusion from the reviewed literature."
    }
])

display(faithfulness_assessment)

,claim,assessment,reason
0,Machine learning is a prevalent approach for p...,Supported,Evidence 3 explicitly describes machine learni...
1,Machine learning-based methods are generally f...,Supported,Evidence 4 explicitly states this.
2,Machine learning methods cope well with zero-h...,Supported,Evidence 4 explicitly reports this characteris...
3,Phishing detection commonly uses URL informati...,Supported,Evidence 3 identifies all three feature types.
4,URL features are quick to obtain but vulnerabl...,Supported,Evidence 4 describes this trade-off.
5,Source-code features are more robust against e...,Supported,Evidence 4 explicitly describes these characte...
6,A study discussed in a review reported approxi...,Supported,Evidence 1 attributes this finding to Singh an...
7,Deep learning is gaining popularity in cyberse...,Supported,Evidence 5 discusses its growing popularity an...
8,Deep-learning algorithms have not been explore...,Supported,Evidence 1 reports this conclusion from the re...


10.27: Check citation validity automatically

In [ ]:
# Validate article IDs and PDF page numbers in the generated answer evidence.
import os
import pickle
import pandas as pd

# Regenerate test_result if it was cleared from the active runtime session
if "test_result" not in globals():
    print("test_result not found in memory. Regenerating from research corpus...")
    test_question = "What machine learning techniques are used for phishing website detection?"
    test_result = generate_evidence_grounded_answer(test_question)

# Safely reload page_documents if it was cleared from the current session
if "page_documents" not in globals():
    project_folder = "/content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data"
    doc_path = os.path.join(project_folder, "page_documents.pkl")
    print(f"page_documents not found in memory. Loading from {doc_path}...")
    with open(doc_path, "rb") as file:
        page_documents = pickle.load(file)

article_ids_in_corpus = {
    chunk["article_id"] for chunk in sentence_chunks
}

citation_validation = []

for passage in test_result["evidence"]:
    article_id = passage["article_id"]
    page_number = int(float(passage["page_number"]))

    matching_pages = [
        int(float(record["page_number"]))
        for record in page_documents
        if record["article_id"] == article_id
    ]

    citation_validation.append({
        "article_id": article_id,
        "article_exists": article_id in article_ids_in_corpus,
        "page_number": page_number,
        "page_exists": page_number in matching_pages
    })

citation_validation_df = pd.DataFrame(citation_validation)

print("CITATION VALIDATION")
print("-" * 50)
display(citation_validation_df)

print("\nAll article IDs valid:",
      citation_validation_df["article_exists"].all())

print("All PDF page references valid:",
      citation_validation_df["page_exists"].all())

page_documents not found in memory. Loading from /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/page_documents.pkl...
CITATION VALIDATION
--------------------------------------------------


,article_id,article_exists,page_number,page_exists
0,Applications_of_deep_learning_for_phishing_det...,True,17,True
1,Modeling_Hybrid_Feature_Based_Phishing_Website...,True,2,True
2,A_systematic_literature_review_on_phishing_web...,True,2,True
3,Phishing_or_Not_Phishing_A_Survey_on_the_Detec...,True,17,True
4,Phishing_website_detection_How_effective_are_d...,True,2,True



All article IDs valid: True
All PDF page references valid: True


10.28: Check citation metadata consistency

In [ ]:
# Identify the existing metadata DataFrame in the notebook.

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        if {"filename", "title", "year"}.issubset(value.columns):
            print("Metadata DataFrame:", name)
            print("Shape:", value.shape)

Metadata DataFrame: metadata_inventory
Shape: (162, 7)
Metadata DataFrame: matching_metadata
Shape: (1, 7)


In [ ]:
# Check whether retrieved evidence metadata matches the research inventory.

metadata_validation = []

for passage in test_result["evidence"]:
    article_id = passage["article_id"]

    matching_metadata = metadata_inventory[
        metadata_inventory["filename"] == article_id
    ]

    if len(matching_metadata) == 1:
        metadata = matching_metadata.iloc[0]

        metadata_validation.append({
            "article_id": article_id,
            "title_matches": (
                str(passage["title"]).strip()
                == str(metadata["title"]).strip()
            ),
            "year_matches": (
                int(float(passage["year"]))
                == int(float(metadata["year"]))
            )
        })
    else:
        metadata_validation.append({
            "article_id": article_id,
            "title_matches": False,
            "year_matches": False
        })

metadata_validation_df = pd.DataFrame(metadata_validation)

print("CITATION METADATA VALIDATION")
print("-" * 50)
display(metadata_validation_df)

print("\nAll titles match:",
      metadata_validation_df["title_matches"].all())

print("All publication years match:",
      metadata_validation_df["year_matches"].all())

CITATION METADATA VALIDATION
--------------------------------------------------


,article_id,title_matches,year_matches
0,Applications_of_deep_learning_for_phishing_det...,True,True
1,Modeling_Hybrid_Feature_Based_Phishing_Website...,True,True
2,A_systematic_literature_review_on_phishing_web...,True,True
3,Phishing_or_Not_Phishing_A_Survey_on_the_Detec...,True,True
4,Phishing_website_detection_How_effective_are_d...,True,True



All titles match: True
All publication years match: True


10.29: Inspect the retrieved evidence

In [ ]:
# Display retrieved passages for manual evidence-faithfulness checking.

for i, passage in enumerate(test_result["evidence"], start=1):
    print(f"\n{'=' * 70}")
    print(f"EVIDENCE {i}")
    print(f"{'=' * 70}")
    print("Article:", passage["article_id"])
    print("Title:", passage["title"])
    print("Year:", passage["year"])
    print("PDF Page:", passage["page_number"])
    print("\nSupporting Passage:\n")
    print(passage["text"])


EVIDENCE 1
Article: Applications_of_deep_learning_for_phishing_detection.pdf
Title: Applications of deep learning for phishing detection: a systematic literature review
Year: 2022.0
PDF Page: 17

Supporting Passage:

Applications of deep learning for phishing detection: a... 1473 Table 2 (continued) Study Review type Focus Description Singh and Meenu [62] Non-systematic Phishing website detection using machine learning Discusses the applications of ML techniques for phishing website detection along with various approaches protection approaches They found that the ML algorithms have achieved an approximate 99% accuracy for phishing website detection by combining 30 features Benavides et al. [12] Systematic Phishing attack solutions using Deep Learning Discusses the applications of DL techniques for phishing website detection Classified and analyzed various DL-based solutions DL algorithms have not been explored enough for phishing website detection Still, there is a need to identify on

10.29: Evidence-faithfulness assessment

In [ ]:
# Record manual evidence-faithfulness judgments for the test answer.

faithfulness_assessment = pd.DataFrame([
    {
        "claim": "Machine learning is prevalent in phishing website detection.",
        "evidence": "Evidence 3",
        "assessment": "Supported"
    },
    {
        "claim": "URL information, website structure and JavaScript features are used.",
        "evidence": "Evidence 3",
        "assessment": "Supported"
    },
    {
        "claim": "Machine learning methods can be fast and effective.",
        "evidence": "Evidence 4",
        "assessment": "Supported"
    },
    {
        "claim": "Performance depends on extracted features and training datasets.",
        "evidence": "Evidence 4",
        "assessment": "Supported"
    },
    {
        "claim": "URL features are quick but vulnerable to manipulation.",
        "evidence": "Evidence 4",
        "assessment": "Supported"
    },
    {
        "claim": "Source-code features can be more robust but introduce delays and safety issues.",
        "evidence": "Evidence 4",
        "assessment": "Supported"
    },
    {
        "claim": "Singh and Meenu reported approximately 99% accuracy using 30 features.",
        "evidence": "Evidence 1",
        "assessment": "Supported"
    },
    {
        "claim": "Deep learning approaches still require further exploration.",
        "evidence": "Evidence 1",
        "assessment": "Supported"
    },
    {
        "claim": "Deep learning is gaining popularity in cybersecurity.",
        "evidence": "Evidence 5",
        "assessment": "Supported"
    }
])

display(faithfulness_assessment)

print("\nAssessment summary:")
print(faithfulness_assessment["assessment"].value_counts())

,claim,evidence,assessment
0,Machine learning is prevalent in phishing webs...,Evidence 3,Supported
1,"URL information, website structure and JavaScr...",Evidence 3,Supported
2,Machine learning methods can be fast and effec...,Evidence 4,Supported
3,Performance depends on extracted features and ...,Evidence 4,Supported
4,URL features are quick but vulnerable to manip...,Evidence 4,Supported
5,Source-code features can be more robust but in...,Evidence 4,Supported
6,Singh and Meenu reported approximately 99% acc...,Evidence 1,Supported
7,Deep learning approaches still require further...,Evidence 1,Supported
8,Deep learning is gaining popularity in cyberse...,Evidence 5,Supported



Assessment summary:
assessment
Supported    9
Name: count, dtype: int64


In [ ]:
# Check evaluation questions and existing answer-generation results.

print("Evaluation dataset shape:", evaluation_df.shape)
print("\nQuestions by category:")
display(evaluation_df["question_category"].value_counts())

print("\nAnswerability distribution:")
display(evaluation_df["answerable"].value_counts())

print("\nExisting answer-generation variables:")

for name, value in list(globals().items()):
    if any(term in name.lower() for term in [
        "answer_generation", "generated_answers", "answer_results"
    ]):
        print(name, type(value).__name__)

Evaluation dataset shape: (80, 8)

Questions by category:


,count
question_category,
Factual,20
Conceptual,20
Multi-paper Synthesis,20
Unanswerable,20



Answerability distribution:


,count
answerable,
True,60
False,20



Existing answer-generation variables:
existing_answer_results list


In [ ]:
# Inspect the existing stored answer.

print("Existing result:")
print(existing_answer_results[0])

print("\nString length:", len(existing_answer_results[0]))

Existing result:
test_result

String length: 11


In [ ]:
# Inspect previously generated answers without making new API calls.

print("Variable type:", type(existing_answer_results).__name__)
print("Number of existing results:", len(existing_answer_results))

if len(existing_answer_results) > 0:
    # Retrieve the actual dictionary object from the global namespace using its string variable name
    result_name = existing_answer_results[0]
    result_obj = globals().get(result_name)

    print(f"\nFirst result name: {result_name}")
    print("First result type:", type(result_obj).__name__)

    if isinstance(result_obj, dict):
        print("First result keys:", list(result_obj.keys()))

        print("\nFirst question:")
        print(result_obj.get("question", "Not available"))

        print("\nFirst answer preview:")
        print(str(result_obj.get("answer", ""))[:1000])
    else:
         print(f"Variable '{result_name}' is not a dictionary. Actual value: {repr(result_obj)}")

Variable type: list
Number of existing results: 1

First result name: test_result
First result type: dict
First result keys: ['question', 'answer', 'evidence']

First question:
What machine learning techniques are used for phishing website detection?

First answer preview:
Based on the provided research evidence, machine learning techniques are widely used and prevalent approaches for detecting phishing websites [Article ID: A_systematic_literature_review_on_phishing_website_detection_techniques.pdf, Title: A systematic literature review on phishing website detection techniques, Year: 2023, PDF Page: 2] [Article ID: Applications_of_deep_learning_for_phishing_detection.pdf, Title: Applications of deep learning for phishing detection: a systematic literature review, Year: 2022, PDF Page: 17]. 

The characteristics and findings regarding these techniques include:

* **General Performance and Characteristics:** Machine learning-based methods are generally characterized as fast, effective, 

Inspect the evaluation questions

In [ ]:
# Inspect the first answerable evaluation question.

first_answerable = evaluation_df[
    evaluation_df["answerable"] == True
].iloc[0]

print("Question ID:", first_answerable["question_id"])
print("Category:", first_answerable["question_category"])
print("Answerable:", first_answerable["answerable"])
print("\nResearch Question:")
print(first_answerable["question"])

Question ID: Q001
Category: Factual
Answerable: True

Research Question:
How many phishing and legitimate webpages were used to evaluate CANTINA+?


In [ ]:
# Generate an answer for the first answerable evaluation question.

q001_result = generate_evidence_grounded_answer(
    first_answerable["question"]
)

print("Question ID:", first_answerable["question_id"])
print("Category:", first_answerable["question_category"])

print("\nGenerated Answer:")
print(q001_result["answer"])

print("\nNumber of Retrieved Evidence Passages:")
print(len(q001_result["evidence"]))

Question ID: Q001
Category: Factual

Generated Answer:
Based on the provided research evidence, the exact number of phishing and legitimate webpages used to evaluate CANTINA+ cannot be established, as the supplied passages do not state these specific quantities. 

However, regarding related evaluations mentioned in the literature:
* CANTINA+ achieved a 99.61% F1-Score using its proposed filters [Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf, Title: Phishing URL Detection: A Real-Case Scenario Through Login URLs, Year: 2022, PDF Page: 3].
* In contrast, earlier evaluations of CANTINA noted that it was tested on a collection of 200 URLs from three categories [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+, Year: 2011, PDF Page: 18].

Number of Retrieved Evidence Passages:
5


In [ ]:
# Inspect the reference answer and retrieved evidence for Q001.

q001_reference = evaluation_df[
    evaluation_df["question_id"] == "Q001"
].iloc[0]

print("QUESTION:")
print(q001_reference["question"])

print("\nGOLD ANSWER:")
print(q001_reference["gold_answer"])

print("\nEXPECTED EVIDENCE ARTICLE IDs:")
print(q001_reference["evidence_article_ids"])

print("\nEXPECTED EVIDENCE PAGES:")
print(q001_reference["evidence_pages"])

print("\n" + "=" * 80)
print("RETRIEVED EVIDENCE")
print("=" * 80)

for i, passage in enumerate(q001_result["evidence"], start=1):
    print(f"\nEVIDENCE {i}")
    print("Article ID:", passage["article_id"])
    print("Title:", passage["title"])
    print("Year:", passage["year"])
    print("PDF Page:", passage["page_number"])
    print("Passage:")
    print(passage["text"])
    print("-" * 80)

QUESTION:
How many phishing and legitimate webpages were used to evaluate CANTINA+?

GOLD ANSWER:
The evaluation used 8,118 phishing webpages and 4,883 legitimate webpages.

EXPECTED EVIDENCE ARTICLE IDs:
A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf

EXPECTED EVIDENCE PAGES:
1.0

RETRIEVED EVIDENCE

EVIDENCE 1
Article ID: Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf
Title: Phishing URL Detection: A Real-Case Scenario Through Login URLs
Year: 2022.0
PDF Page: 3
Passage:
CANTINA extracts five words from each website using TF- IDF and introduced them into the Google search engine. If a domain was within the n first results, the page was considered legitimate, or phishing otherwise. They obtained an accuracy of 95% with a threshold of n = 30 Google search results. Due to the use of external services like WHOIS7 and the high false-positive rate, authors proposed CANTINA+ [31]. Their new proposal achieved a 99.61% F1-Score including two

In [ ]:
# Inspect the expected evidence page for Q001.

q001_expected_id = q001_reference["evidence_article_ids"]
q001_expected_page = int(float(q001_reference["evidence_pages"]))

if isinstance(q001_expected_id, str):
    q001_expected_ids = [q001_expected_id]
else:
    q001_expected_ids = list(q001_expected_id)

print("Expected Article:", q001_expected_ids)
print("Expected PDF Page:", q001_expected_page)

print("\n" + "=" * 80)
print("EXPECTED PAGE CONTENT")
print("=" * 80)

for chunk in sentence_chunks:
    if (
        chunk["article_id"] in q001_expected_ids
        and chunk["page_number"] == q001_expected_page
    ):
        print(chunk["text"])
        print("-" * 80)

Expected Article: ['A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf']
Expected PDF Page: 1

EXPECTED PAGE CONTENT
21 CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites GUANG XIANG, JASON HONG, CAROLYN P. ROSE, and LORRIE CRANOR, Carnegie Mellon University Phishing is a plague in cyberspace. Typically, phish detection methods either use human-verified URL blacklists or exploit Web page features via machine learning techniques. However, the former is frail in terms of new phish, and the latter suffers from the scarcity of effective features and the high false positive rate (FP). To alleviate those problems, we propose a layered anti-phishing solution that aims at (1) exploiting the expressiveness of a rich set of features with machine learning to achieve a high true positive rate (TP) on novel phish, and (2) limiting the FP to a low level via filtering algorithms. Specifically, we proposed CANTINA+, the most comprehensive f

In [ ]:
# Check whether the expected Q001 evidence is present in sentence-based chunks.

q001_target_chunks = [
    chunk for chunk in sentence_chunks
    if (
        chunk["article_id"] == q001_expected_ids[0]
        and chunk["page_number"] == q001_expected_page
    )
]

print("Expected page chunks:", len(q001_target_chunks))

for i, chunk in enumerate(q001_target_chunks, start=1):
    print(f"\nChunk {i}:")
    print(chunk["text"])

    if "8118" in chunk["text"] and "4883" in chunk["text"]:
        print("\nFOUND: Both expected quantities are present in this chunk.")

Expected page chunks: 5

Chunk 1:
21 CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites GUANG XIANG, JASON HONG, CAROLYN P. ROSE, and LORRIE CRANOR, Carnegie Mellon University Phishing is a plague in cyberspace. Typically, phish detection methods either use human-verified URL blacklists or exploit Web page features via machine learning techniques. However, the former is frail in terms of new phish, and the latter suffers from the scarcity of effective features and the high false positive rate (FP). To alleviate those problems, we propose a layered anti-phishing solution that aims at (1) exploiting the expressiveness of a rich set of features with machine learning to achieve a high true positive rate (TP) on novel phish, and (2) limiting the FP to a low level via filtering algorithms. Specifically, we proposed CANTINA+, the most comprehensive feature-based approach in the literature including eight novel features, which exploits the HTML Document Object

In [ ]:
# Inspect the hybrid retrieval ranking for Q001.

q001_candidates = hybrid_retrieval(
    q001_reference["question"],
    top_k=10,
    candidate_k=20
)

print("Number of retrieved candidates:", len(q001_candidates))

print("\n" + "=" * 80)
print("HYBRID RETRIEVAL RESULTS")
print("=" * 80)

for rank, passage in enumerate(q001_candidates, start=1):
    is_expected = (
        passage["article_id"] == q001_expected_ids[0]
        and passage["page_number"] == q001_expected_page
        and "8118" in passage["text"]
        and "4883" in passage["text"]
    )

    print(f"\nRank: {rank}")
    print("Article:", passage["article_id"])
    print("Page:", passage["page_number"])
    print("Expected evidence:", is_expected)
    print("Text:", passage["text"][:500])
    print("-" * 80)

Number of retrieved candidates: 10

HYBRID RETRIEVAL RESULTS

Rank: 1
Article: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf
Page: 19
Expected evidence: False
Text: Subsequently, the webpage’s source code is crawled to extract pertinent content-based features, in addition to performing URL analysis. The phishing data repository contains the URL and content patterns observed from previously flagged phishing webpages. By assessing the similarity of patterns matched in the suspicious webpage, the algorithm determines whether it should be classified as phishing or legitimate. Zhang et al. [109] proposed CANTINA, a content-based approach for detecting phishing w
--------------------------------------------------------------------------------

Rank: 2
Article: CANTINA_A_Content_Based_Approach_to_Dectecting_Phishing_web_sites.pdf
Page: 2
Expected evidence: False
Text: Blacklists may have a higher level of accuracy, but generally require h

In [ ]:
# Compare BM25 and dense retrieval for Q001.

q001_query = q001_reference["question"]

q001_bm25_results = bm25_only_retrieval(
    q001_query,
    top_k=10
)

q001_dense_results = dense_only_retrieval(
    q001_query,
    top_k=10
)

def inspect_q001_results(results, method_name):

    print("\n" + "=" * 80)
    print(method_name)
    print("=" * 80)

    for rank, passage in enumerate(results, start=1):

        is_expected = (
            passage["article_id"] == q001_expected_ids[0]
            and passage["page_number"] == q001_expected_page
            and "8118" in passage["text"]
            and "4883" in passage["text"]
        )

        print(
            f"Rank {rank} | "
            f"Article: {passage['article_id']} | "
            f"Page: {passage['page_number']} | "
            f"Expected evidence: {is_expected}"
        )

        print("Text:", passage["text"][:300])
        print("-" * 80)


inspect_q001_results(q001_bm25_results, "BM25 RESULTS")

inspect_q001_results(q001_dense_results, "DENSE RESULTS")


BM25 RESULTS
Rank 1 | Article: Phishing_website_detection_How_effective_are_deeplearning_based_models_and_hyperparameter_optimization.pdf | Page: 10 | Expected evidence: False
Text: 10 of 19 ALMOUSA et al. 4.1 Datasets One phishing dataset obtained from Reference 35, we refer to it as Tan-dataset, it consists of 48 features related to websites. We further categorize the features into two groups and use them to build and test different deep learning models. We chose this dataset
--------------------------------------------------------------------------------
Rank 2 | Article: CANTINA_A_Content_Based_Approach_to_Dectecting_Phishing_web_sites.pdf | Page: 4 | Expected evidence: False
Text: We present our evaluation of the effectiveness of TF-IDF and the two heuristics (adding the domain name to the lexical signature and ZMP) in Section 4.1. We discovered that TF-IDF yielded fairly good accuracy (correctly labeling legitimate sites as legitimate and phishing sites as phishing), but als
---

In [ ]:
# Test retrieval using more specific terms from the expected evidence.

q001_test_query = (
    "CANTINA+ evaluation diverse spectrum of corpora "
    "8118 phish 4883 legitimate Web pages"
)

q001_test_results = hybrid_retrieval(
    q001_test_query,
    top_k=10,
    candidate_k=20
)

print("Diagnostic Query:")
print(q001_test_query)

print("\n" + "=" * 80)
print("TARGETED HYBRID RETRIEVAL RESULTS")
print("=" * 80)

for rank, passage in enumerate(q001_test_results, start=1):

    is_expected = (
        passage["article_id"] == q001_expected_ids[0]
        and passage["page_number"] == q001_expected_page
        and "8118" in passage["text"]
        and "4883" in passage["text"]
    )

    print(f"\nRank: {rank}")
    print("Article:", passage["article_id"])
    print("Page:", passage["page_number"])
    print("Expected evidence:", is_expected)
    print("Text:", passage["text"][:500])
    print("-" * 80)

Diagnostic Query:
CANTINA+ evaluation diverse spectrum of corpora 8118 phish 4883 legitimate Web pages

TARGETED HYBRID RETRIEVAL RESULTS

Rank: 1
Article: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Page: 1
Expected evidence: True
Text: Moreover, we designed two filters to help reduce FP and achieve runtime speedup. The first is a near-duplicate phish detector that uses hashing to catch highly similar phish. The second is a login form filter, which directly classifies Web pages with no identified login form as legitimate. We extensively evaluated CANTINA+ with two methods on a diverse spectrum of corpora with 8118 phish and 4883 legitimate Web pages. In the randomized evaluation, CANTINA+ achieved over 92% TP on unique testing 
--------------------------------------------------------------------------------

Rank: 2
Article: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Page: 26
Expected evidence: False
Text: Signific

In [ ]:
# Generate a diagnostic answer using the targeted query.

q001_diagnostic_result = generate_evidence_grounded_answer(
    q001_test_query
)

print("Diagnostic Question:")
print(q001_test_query)

print("\nGenerated Answer:")
print(q001_diagnostic_result["answer"])

print("\nRetrieved Evidence:")
for i, passage in enumerate(
    q001_diagnostic_result["evidence"], start=1
):
    print(f"\nEvidence {i}")
    print("Article:", passage["article_id"])
    print("Title:", passage["title"])
    print("Year:", passage["year"])
    print("Page:", passage["page_number"])
    print("Text:", passage["text"][:500])

Diagnostic Question:
CANTINA+ evaluation diverse spectrum of corpora 8118 phish 4883 legitimate Web pages

Generated Answer:
Based on the provided research evidence, CANTINA+ was extensively evaluated across a diverse spectrum of corpora consisting of 8,118 phish and 4,883 legitimate Web pages [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+, Year: 2011, PDF Page: 1]. 

The evaluation of CANTINA+ on this dataset was conducted using two primary methods [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+, Year: 2011, PDF Page: 1]:

1. **Randomized Evaluation:** 
   - Training and testing phish were randomly selected [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+, Year: 2011, PDF Page: 26].
   - CANTINA+ achieved over 92% true positive (TP) rate on unique testing phish, over 99% TP on near-duplicate testing phish, and approximately 0.4% false posit

In [ ]:
# Record Q001 diagnostic findings.

q001_diagnostic_record = {
    "question_id": "Q001",
    "original_retrieval_success": False,
    "targeted_retrieval_success": True,
    "expected_evidence_rank": 1,
    "expected_article_id": q001_expected_ids[0],
    "expected_page": q001_expected_page,
    "generated_answer_correct": True,
    "phishing_count_correct": True,
    "legitimate_count_correct": True,
    "citation_article_correct": True,
    "citation_page_correct": True,
    "notes": (
        "Original query failed to retrieve expected evidence in top 10. "
        "Targeted query retrieved the correct passage at rank 1. "
        "Gemini generated the correct numerical answer and citation."
    )
}

q001_diagnostic_df = pd.DataFrame([q001_diagnostic_record])

display(q001_diagnostic_df)

,question_id,original_retrieval_success,targeted_retrieval_success,expected_evidence_rank,expected_article_id,expected_page,generated_answer_correct,phishing_count_correct,legitimate_count_correct,citation_article_correct,citation_page_correct,notes
0,Q001,False,True,1,A_Feature_Rich_Machine_Learning_Framework_for_...,1,True,True,True,True,True,Original query failed to retrieve expected evi...


Systematic answer evaluation

In [ ]:
# Check the evaluation dataset and answer generation function.

print("Evaluation dataset:", evaluation_df.shape)
print("Total questions:", len(evaluation_df))

print("\nAnswerable questions:",
      evaluation_df["answerable"].astype(bool).sum())

print("Unanswerable questions:",
      (~evaluation_df["answerable"].astype(bool)).sum())

print("\nAnswer generation function:",
      callable(generate_evidence_grounded_answer))

print("\nQ001 original result available:",
      "q001_result" in globals())

print("Q001 diagnostic result available:",
      "q001_diagnostic_result" in globals())

Evaluation dataset: (80, 8)
Total questions: 80

Answerable questions: 60
Unanswerable questions: 20

Answer generation function: True

Q001 original result available: True
Q001 diagnostic result available: True


Prepare answer generation for 60 answerable questions


In [ ]:
# Prepare the 60 answerable questions.

answerable_questions = evaluation_df[
    evaluation_df["answerable"].astype(bool)
].copy()

answerable_questions = answerable_questions.sort_values(
    "question_id"
).reset_index(drop=True)

print("Total answerable questions:",
      len(answerable_questions))

print("\nQuestions by category:")
display(
    answerable_questions["question_category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="Number of Questions")
)

print("\nFirst 10 answerable questions:")
display(
    answerable_questions[
        ["question_id", "question_category", "question"]
    ].head(10)
)

Total answerable questions: 60

Questions by category:


,Category,Number of Questions
0,Factual,20
1,Conceptual,20
2,Multi-paper Synthesis,20



First 10 answerable questions:


,question_id,question_category,question
0,Q001,Factual,How many phishing and legitimate webpages were...
1,Q002,Factual,What true-positive and false-positive rates di...
2,Q003,Factual,Which cryptographic protocols are discussed fo...
3,Q004,Factual,What three feature-fusion deep learning models...
4,Q005,Factual,Which two datasets are used to evaluate the fe...
5,Q006,Factual,How is reconstruction error used in the autoen...
6,Q007,Factual,Which three deep learning components are combi...
7,Q008,Factual,Which dataset is described as widely used for ...
8,Q009,Factual,Which three emerging approaches to phishing we...
9,Q010,Factual,How many scientific papers did the systematic ...


In [ ]:
import os
import pickle

# Define the persistent output location.
answer_results_path = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data/"
    "answer_generation_results.pkl"
)

# Initialize results using the ORIGINAL Q001 answer.
answer_generation_results = {
    "Q001": q001_result
}

# Save the initial checkpoint.
with open(answer_results_path, "wb") as file:
    pickle.dump(answer_generation_results, file)

print("Answer-generation results initialized.")
print("Saved questions:", list(answer_generation_results.keys()))
print("Saved file:", answer_results_path)
print("File exists:", os.path.exists(answer_results_path))
print("File size:", round(os.path.getsize(answer_results_path) / 1024, 2), "KB")

Answer-generation results initialized.
Saved questions: ['Q001']
Saved file: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/answer_generation_results.pkl
File exists: True
File size: 6.65 KB


Generate answers for Q002–Q060

In [ ]:
import time
import pickle

# Generate answers for the remaining questions.
generation_errors = []

for index, row in answerable_questions.iterrows():

    question_id = row["question_id"]

    # Skip questions already generated.
    if question_id in answer_generation_results:
        continue

    question = row["question"]

    print(f"\nGenerating {question_id} ({index + 1}/60)")
    print("Question:", question)

    try:
        result = generate_evidence_grounded_answer(question)

        answer_generation_results[question_id] = result

        # Save checkpoint after every successful generation.
        with open(answer_results_path, "wb") as file:
            pickle.dump(answer_generation_results, file)

        print("Status: Successfully generated and saved")

    except Exception as e:

        generation_errors.append({
            "question_id": question_id,
            "error": str(e)
        })

        print("Status: Generation failed")
        print("Error:", str(e))

    time.sleep(1)

print("\n" + "=" * 60)
print("ANSWER GENERATION SUMMARY")
print("=" * 60)

print("Successfully generated:",
      len(answer_generation_results))

print("Generation errors:",
      len(generation_errors))

print("Checkpoint:", answer_results_path)


Generating Q002 (2/60)
Question: What true-positive and false-positive rates did CANTINA+ report in its randomized evaluation?
Status: Successfully generated and saved

Generating Q003 (3/60)
Question: Which cryptographic protocols are discussed for validating sender identity and data integrity in the hybrid phishing email detection approach?
Status: Successfully generated and saved

Generating Q004 (4/60)
Question: What three feature-fusion deep learning models are proposed for network intrusion detection by Ayantayo et al. (2023)?
Status: Successfully generated and saved

Generating Q005 (5/60)
Question: Which two datasets are used to evaluate the feature-fusion models in Ayantayo et al. (2023)?
Status: Successfully generated and saved

Generating Q006 (6/60)
Question: How is reconstruction error used in the autoencoder-based anomaly detection method by Torabi et al. (2023)?
Status: Successfully generated and saved

Generating Q007 (7/60)
Question: Which three deep learning componen

In [ ]:
# Verify all 60 generated answer records.

expected_ids = set(answerable_questions["question_id"])
saved_ids = set(answer_generation_results.keys())

missing_ids = expected_ids - saved_ids

generation_quality_issues = []

for question_id in expected_ids.intersection(saved_ids):

    result = answer_generation_results[question_id]

    answer = result.get("answer", "")
    evidence = result.get("evidence", [])

    if not answer or not answer.strip():
        generation_quality_issues.append(
            (question_id, "Empty answer")
        )

    elif answer.startswith("Gemini API request failed:"):
        generation_quality_issues.append(
            (question_id, "API failure stored as answer")
        )

    if not evidence:
        generation_quality_issues.append(
            (question_id, "No retrieved evidence")
        )

print("Expected questions:", len(expected_ids))
print("Saved results:", len(saved_ids))
print("Missing questions:", len(missing_ids))
print("Generation quality issues:", len(generation_quality_issues))

if missing_ids:
    print("\nMissing IDs:", sorted(missing_ids))

if generation_quality_issues:
    print("\nIssues:")
    for issue in generation_quality_issues:
        print(issue)
else:
    print("\nAll generated records passed these basic checks.")

Expected questions: 60
Saved results: 60
Missing questions: 0
Generation quality issues: 0

All generated records passed these basic checks.


In [ ]:
# Prepare the 20 unanswerable evaluation questions.

unanswerable_questions = evaluation_df[
    ~evaluation_df["answerable"].astype(bool)
].copy()

unanswerable_questions = unanswerable_questions.sort_values(
    "question_id"
).reset_index(drop=True)

print("Total unanswerable questions:",
      len(unanswerable_questions))

print("\nQuestions by category:")
display(
    unanswerable_questions["question_category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="Number of Questions")
)

print("\nUnanswerable questions:")
display(
    unanswerable_questions[
        ["question_id", "question_category", "question"]
    ]
)

Total unanswerable questions: 20

Questions by category:


,Category,Number of Questions
0,Unanswerable,20



Unanswerable questions:


,question_id,question_category,question
0,Q061,Unanswerable,What was the exact financial loss caused by ph...
1,Q062,Unanswerable,How many phishing websites were reported in Bh...
2,Q063,Unanswerable,Which Bhutanese government agency recorded the...
3,Q064,Unanswerable,What percentage of Bhutanese internet users ex...
4,Q065,Unanswerable,How many organizations in Bhutan experienced r...
5,Q066,Unanswerable,What detection accuracy did the Practical Auto...
6,Q067,Unanswerable,What exact generalisation score did the featur...
7,Q068,Unanswerable,In the explainable CNN-based IoT intrusion det...
8,Q069,Unanswerable,What accuracy did the hybrid CNN–LSTM model ac...
9,Q070,Unanswerable,What zero-day detection rate did the RNDAE mod...


In [ ]:
import pickle

# Confirm the existing 60 answers are preserved.
print("Existing answer records:",
      len(answer_generation_results))

# Save the current checkpoint before adding new results.
with open(answer_results_path, "wb") as file:
    pickle.dump(answer_generation_results, file)

print("Checkpoint saved successfully.")

# Identify questions still requiring generation.
remaining_unanswerable = unanswerable_questions[
    ~unanswerable_questions["question_id"].isin(
        answer_generation_results.keys()
    )
].copy()

print("\nUnanswerable questions remaining:",
      len(remaining_unanswerable))

print("First question to process:",
      remaining_unanswerable.iloc[0]["question_id"])

Existing answer records: 60
Checkpoint saved successfully.

Unanswerable questions remaining: 20
First question to process: Q061


In [ ]:
import time
import pickle

# Generate answers for the 20 unanswerable questions.
unanswerable_generation_errors = []

for index, row in remaining_unanswerable.iterrows():

    question_id = row["question_id"]

    # Skip any question already generated.
    if question_id in answer_generation_results:
        continue

    question = row["question"]

    print(f"\nGenerating {question_id} ({index + 1}/20)")
    print("Question:", question)

    try:
        result = generate_evidence_grounded_answer(question)

        answer_generation_results[question_id] = result

        # Save checkpoint after every successful generation.
        with open(answer_results_path, "wb") as file:
            pickle.dump(answer_generation_results, file)

        print("Status: Successfully generated and saved")

    except Exception as e:

        unanswerable_generation_errors.append({
            "question_id": question_id,
            "error": str(e)
        })

        print("Status: Generation failed")
        print("Error:", str(e))

    time.sleep(1)

print("\n" + "=" * 60)
print("UNANSWERABLE QUESTION GENERATION SUMMARY")
print("=" * 60)

print("Total saved results:",
      len(answer_generation_results))

print("Unanswerable generation errors:",
      len(unanswerable_generation_errors))

print("Checkpoint:", answer_results_path)


Generating Q061 (1/20)
Question: What was the exact financial loss caused by phishing attacks in Bhutan in 2025?
Status: Successfully generated and saved

Generating Q062 (2/20)
Question: How many phishing websites were reported in Bhutan in 2025?
Status: Successfully generated and saved

Generating Q063 (3/20)
Question: Which Bhutanese government agency recorded the highest number of phishing incidents in 2025?
Status: Successfully generated and saved

Generating Q064 (4/20)
Question: What percentage of Bhutanese internet users experienced phishing attacks in 2025?
Status: Successfully generated and saved

Generating Q065 (5/20)
Question: How many organizations in Bhutan experienced ransomware attacks in 2025?
Status: Successfully generated and saved

Generating Q066 (6/20)
Question: What detection accuracy did the Practical Autoencoder model achieve when evaluated on real operational networks?
Status: Successfully generated and saved

Generating Q067 (7/20)
Question: What exact gene

In [ ]:
import pickle

# Reload the saved checkpoint to verify persistence.
with open(answer_results_path, "rb") as file:
    verified_results = pickle.load(file)

# Check expected question IDs.
expected_all_ids = set(evaluation_df["question_id"])
saved_all_ids = set(verified_results.keys())

missing_ids = expected_all_ids - saved_all_ids

# Check answer and evidence availability.
verification_issues = []

for question_id in sorted(expected_all_ids.intersection(saved_all_ids)):

    result = verified_results[question_id]

    answer = result.get("answer", "")
    evidence = result.get("evidence", [])

    if not answer or not answer.strip():
        verification_issues.append(
            (question_id, "Empty answer")
        )

    if not evidence:
        verification_issues.append(
            (question_id, "No retrieved evidence")
        )

    if answer.startswith("Gemini API request failed:"):
        verification_issues.append(
            (question_id, "API failure")
        )

print("Expected questions:", len(expected_all_ids))
print("Saved questions:", len(saved_all_ids))
print("Missing questions:", len(missing_ids))
print("Verification issues:", len(verification_issues))

if missing_ids:
    print("Missing IDs:", sorted(missing_ids))

if verification_issues:
    print("\nIssues:")
    for issue in verification_issues:
        print(issue)
else:
    print("\nAll 80 records passed the basic verification.")


Expected questions: 80
Saved questions: 80
Missing questions: 0
Verification issues: 0

All 80 records passed the basic verification.


inspect the evaluation data structure

In [ ]:
# Restore the verified results as the active answer-generation results.
answer_generation_results = verified_results

# Inspect the evaluation dataset.
print("EVALUATION DATASET")
print("Columns:", evaluation_df.columns.tolist())
print("Shape:", evaluation_df.shape)

# Inspect one answerable question.
print("\nSAMPLE ANSWERABLE QUESTION")
sample_id = "Q001"
sample_row = evaluation_df[
    evaluation_df["question_id"] == sample_id
].iloc[0]

print("Question ID:", sample_id)
print("Question:", sample_row["question"])
print("Reference answer:", sample_row["gold_answer"])

# Inspect its original generated result.
sample_result = answer_generation_results[sample_id]

print("\nGENERATED ANSWER")
print(sample_result["answer"])

print("\nRETRIEVED EVIDENCE")
print("Number of passages:", len(sample_result["evidence"]))

if sample_result["evidence"]:
    print("\nFirst evidence passage structure:")
    print(sample_result["evidence"][0].keys())
    print("\nFirst evidence passage:")
    print(sample_result["evidence"][0])

EVALUATION DATASET
Columns: ['question_id', 'question_category', 'question', 'gold_answer', 'answerable', 'evidence_article_ids', 'evidence_pages', 'verification_notes']
Shape: (80, 8)

SAMPLE ANSWERABLE QUESTION
Question ID: Q001
Question: How many phishing and legitimate webpages were used to evaluate CANTINA+?
Reference answer: The evaluation used 8,118 phishing webpages and 4,883 legitimate webpages.

GENERATED ANSWER
Based on the provided research evidence, the exact number of phishing and legitimate webpages used to evaluate CANTINA+ cannot be established, as the supplied passages do not state these specific quantities. 

However, regarding related evaluations mentioned in the literature:
* CANTINA+ achieved a 99.61% F1-Score using its proposed filters [Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf, Title: Phishing URL Detection: A Real-Case Scenario Through Login URLs, Year: 2022, PDF Page: 3].
* In contrast, earlier evaluations of CANTINA noted that it was 

Prepare the answer evaluation dataset

In [ ]:
# Prepare the answerable-question evaluation dataset.

answerable_eval_df = evaluation_df[
    evaluation_df["answerable"] == True
].copy()

# Attach generated answers and retrieved evidence.
answerable_eval_df["generated_answer"] = (
    answerable_eval_df["question_id"]
    .map(lambda qid: verified_results[qid]["answer"])
)

answerable_eval_df["retrieved_evidence_count"] = (
    answerable_eval_df["question_id"]
    .map(lambda qid: len(verified_results[qid]["evidence"]))
)

answerable_eval_df["retrieved_article_ids"] = (
    answerable_eval_df["question_id"]
    .map(
        lambda qid: [
            item["article_id"]
            for item in verified_results[qid]["evidence"]
        ]
    )
)

answerable_eval_df["retrieved_pages"] = (
    answerable_eval_df["question_id"]
    .map(
        lambda qid: [
            item["page_number"]
            for item in verified_results[qid]["evidence"]
        ]
    )
)

# Display the dataset summary.
print("ANSWERABLE EVALUATION DATASET")
print("Shape:", answerable_eval_df.shape)

print("\nColumns:")
print(answerable_eval_df.columns.tolist())

# Inspect Q001 as the first evaluation example.
q001_evaluation = answerable_eval_df[
    answerable_eval_df["question_id"] == "Q001"
]

print("\nQ001 EVALUATION RECORD")
print(
    q001_evaluation[
        [
            "question_id",
            "question",
            "gold_answer",
            "generated_answer",
            "retrieved_evidence_count",
            "retrieved_article_ids",
            "retrieved_pages"
        ]
    ].to_string(index=False)
)

ANSWERABLE EVALUATION DATASET
Shape: (60, 12)

Columns:
['question_id', 'question_category', 'question', 'gold_answer', 'answerable', 'evidence_article_ids', 'evidence_pages', 'verification_notes', 'generated_answer', 'retrieved_evidence_count', 'retrieved_article_ids', 'retrieved_pages']

Q001 EVALUATION RECORD
question_id                                                                  question                                                                gold_answer                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

Evaluate retrieval evidence coverage

In [ ]:
# Evaluate whether the expected evidence was retrieved.

def check_evidence_coverage(row):

    expected_articles = set(row["evidence_article_ids"])
    retrieved_articles = set(row["retrieved_article_ids"])

    expected_pages = set(row["evidence_pages"])
    retrieved_pages = set(row["retrieved_pages"])

    article_match = bool(
        expected_articles.intersection(retrieved_articles)
    )

    page_match = bool(
        expected_pages.intersection(retrieved_pages)
    )

    exact_evidence_match = any(
        article in retrieved_articles
        and page in retrieved_pages
        for article, page in zip(
            row["evidence_article_ids"],
            row["evidence_pages"]
        )
    )

    return pd.Series({
        "article_match": article_match,
        "page_match": page_match,
        "exact_evidence_match": exact_evidence_match
    })


# Apply evidence coverage checks.
coverage_results = answerable_eval_df.apply(
    check_evidence_coverage,
    axis=1
)

answerable_eval_df = pd.concat(
    [answerable_eval_df, coverage_results],
    axis=1
)

# Summarize retrieval coverage.
print("RETRIEVAL EVIDENCE COVERAGE")
print("Total answerable questions:", len(answerable_eval_df))

print(
    "Expected article retrieved:",
    answerable_eval_df["article_match"].sum()
)

print(
    "Expected page retrieved:",
    answerable_eval_df["page_match"].sum()
)

print(
    "Exact article-page evidence retrieved:",
    answerable_eval_df["exact_evidence_match"].sum()
)

print("\nCoverage percentages:")
print(
    (answerable_eval_df[
        ["article_match", "page_match", "exact_evidence_match"]
    ].mean() * 100).round(2)
)

print("\nQuestions without exact expected evidence:")
print(
    answerable_eval_df.loc[
        ~answerable_eval_df["exact_evidence_match"],
        ["question_id", "question"]
    ].to_string(index=False)
)

RETRIEVAL EVIDENCE COVERAGE
Total answerable questions: 60
Expected article retrieved: 0
Expected page retrieved: 0
Exact article-page evidence retrieved: 0

Coverage percentages:
article_match           0.0
page_match              0.0
exact_evidence_match    0.0
dtype: float64

Questions without exact expected evidence:
question_id                                                                                                                                                             question
       Q001                                                                                            How many phishing and legitimate webpages were used to evaluate CANTINA+?
       Q002                                                                        What true-positive and false-positive rates did CANTINA+ report in its randomized evaluation?
       Q003                       Which cryptographic protocols are discussed for validating sender identity and data integrity in the hybrid phis

In [ ]:
# Inspect expected and retrieved evidence formats.

for qid in ["Q001", "Q002", "Q003"]:

    row = answerable_eval_df[
        answerable_eval_df["question_id"] == qid
    ].iloc[0]

    print("\n" + "=" * 70)
    print("QUESTION:", qid)

    print("\nEXPECTED ARTICLE IDs:")
    print("Value:", repr(row["evidence_article_ids"]))
    print("Type:", type(row["evidence_article_ids"]))

    print("\nEXPECTED PAGE NUMBERS:")
    print("Value:", repr(row["evidence_pages"]))
    print("Type:", type(row["evidence_pages"]))

    print("\nRETRIEVED ARTICLE IDs:")
    print("Value:", repr(row["retrieved_article_ids"]))

    print("\nRETRIEVED PAGE NUMBERS:")
    print("Value:", repr(row["retrieved_pages"]))

    print("\nORIGINAL VERIFICATION NOTES:")
    print(repr(row["verification_notes"]))


QUESTION: Q001

EXPECTED ARTICLE IDs:
Value: 'A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf'
Type: <class 'str'>

EXPECTED PAGE NUMBERS:
Value: '1.0'
Type: <class 'str'>

RETRIEVED ARTICLE IDs:
Value: ['Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf', 'Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf', 'Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf', 'CANTINA_A_Content_Based_Approach_to_Dectecting_Phishing_web_sites.pdf', 'A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf']

RETRIEVED PAGE NUMBERS:
Value: [3, 20, 5, 4, 18]

ORIGINAL VERIFICATION NOTES:
'Based on the passage describing the CANTINA+ evaluation corpus.'

QUESTION: Q002

EXPECTED ARTICLE IDs:
Value: 'A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf'
Type: <class 'str'>

EXPECTED PAGE NUMBERS:
Value: '1.0'
T

In [ ]:
import ast
import re
import pandas as pd

# Convert expected evidence fields into lists.
def parse_evidence_values(value):

    if isinstance(value, (list, tuple, set)):
        return list(value)

    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    # Handle values stored as string representations of lists.
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, (list, tuple)):
                return list(parsed)
        except (ValueError, SyntaxError):
            pass

    # Handle multiple values separated by semicolons or newlines.
    return [
        item.strip()
        for item in re.split(r"[;\n]", value)
        if item.strip()
    ]


# Compare page numbers numerically.
def same_page(a, b):

    try:
        return float(a) == float(b)
    except (ValueError, TypeError):
        return str(a).strip() == str(b).strip()


def check_corrected_coverage(row):

    expected_articles = parse_evidence_values(
        row["evidence_article_ids"]
    )

    expected_pages = parse_evidence_values(
        row["evidence_pages"]
    )

    retrieved_articles = row["retrieved_article_ids"]
    retrieved_pages = row["retrieved_pages"]

    # Check whether an expected article was retrieved.
    article_match = any(
        article in retrieved_articles
        for article in expected_articles
    )

    # Check whether the expected article and page combination
    # was retrieved together.
    exact_match = False

    for article, page in zip(expected_articles, expected_pages):

        for retrieved_article, retrieved_page in zip(
            retrieved_articles, retrieved_pages
        ):

            if (
                article == retrieved_article
                and same_page(page, retrieved_page)
            ):
                exact_match = True
                break

        if exact_match:
            break

    return pd.Series({
        "corrected_article_match": article_match,
        "corrected_exact_evidence_match": exact_match
    })


# Apply corrected checks.
corrected_coverage = answerable_eval_df.apply(
    check_corrected_coverage,
    axis=1
)

answerable_eval_df[
    ["corrected_article_match", "corrected_exact_evidence_match"]
] = corrected_coverage

# Display results.
print("CORRECTED RETRIEVAL EVIDENCE COVERAGE")
print("Total answerable questions:", len(answerable_eval_df))

article_count = answerable_eval_df[
    "corrected_article_match"
].sum()

exact_count = answerable_eval_df[
    "corrected_exact_evidence_match"
].sum()

print(
    f"\nExpected article retrieved: {article_count}/60 "
    f"({article_count / 60 * 100:.2f}%)"
)

print(
    f"Exact expected article-page evidence retrieved: "
    f"{exact_count}/60 ({exact_count / 60 * 100:.2f}%)"
)

print("\nQuestions where the expected article was not retrieved:")

print(
    answerable_eval_df.loc[
        ~answerable_eval_df["corrected_article_match"],
        ["question_id", "question"]
    ].to_string(index=False)
)

print("\nQuestions where exact expected evidence was not retrieved:")

print(
    answerable_eval_df.loc[
        ~answerable_eval_df["corrected_exact_evidence_match"],
        ["question_id", "question"]
    ].to_string(index=False)
)

CORRECTED RETRIEVAL EVIDENCE COVERAGE
Total answerable questions: 60

Expected article retrieved: 55/60 (91.67%)
Exact expected article-page evidence retrieved: 35/60 (58.33%)

Questions where the expected article was not retrieved:
question_id                                                                                                                                        question
       Q030                                                            What is the main security role of an intrusion detection system in IoT environments?
       Q039                             Why should researchers consider dataset features when developing or evaluating network intrusion detection systems?
       Q040 According to the review of intrusion detection datasets, what security weaknesses can make computer systems more vulnerable to network attacks?
       Q047                           How do the two intrusion detection studies address the challenge of detecting evolving or unfamiliar netw

In [ ]:
# Prepare a structured answer-quality review dataset.

answer_review_df = answerable_eval_df.copy()

# Make retrieved passages readable for manual inspection.
def format_retrieved_evidence(question_id):

    evidence_list = verified_results[question_id]["evidence"]

    formatted_passages = []

    for item in evidence_list:

        formatted_passages.append(
            f"Rank: {item['rerank_rank']}\n"
            f"Article ID: {item['article_id']}\n"
            f"Title: {item['title']}\n"
            f"Year: {item['year']}\n"
            f"PDF Page: {item['page_number']}\n"
            f"Passage: {item['text']}"
        )

    return "\n\n" + ("-" * 70) + "\n\n".join(
        formatted_passages
    )


answer_review_df["retrieved_passages"] = (
    answer_review_df["question_id"].map(
        format_retrieved_evidence
    )
)

# Create blank columns for systematic quality assessment.
review_columns = [
    "correctness",
    "relevance",
    "evidence_faithfulness",
    "citation_validity",
    "review_comments"
]

for column in review_columns:
    answer_review_df[column] = ""

# Select the columns needed for the review.
answer_review_df = answer_review_df[
    [
        "question_id",
        "question_category",
        "question",
        "gold_answer",
        "generated_answer",
        "retrieved_passages",
        "correctness",
        "relevance",
        "evidence_faithfulness",
        "citation_validity",
        "review_comments"
    ]
]

# Save the review dataset.
answer_review_path = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data/answer_quality_review.csv"
)

answer_review_df.to_csv(
    answer_review_path,
    index=False
)

print("ANSWER QUALITY REVIEW DATASET")
print("Total questions:", len(answer_review_df))
print("Columns:", answer_review_df.columns.tolist())
print("\nSaved to:")
print(answer_review_path)

print("\nFirst three questions:")
display(
    answer_review_df[
        [
            "question_id",
            "question",
            "gold_answer",
            "generated_answer"
        ]
    ].head(3)
)

ANSWER QUALITY REVIEW DATASET
Total questions: 60
Columns: ['question_id', 'question_category', 'question', 'gold_answer', 'generated_answer', 'retrieved_passages', 'correctness', 'relevance', 'evidence_faithfulness', 'citation_validity', 'review_comments']

Saved to:
/content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/answer_quality_review.csv

First three questions:


,question_id,question,gold_answer,generated_answer
0,Q001,How many phishing and legitimate webpages were...,"The evaluation used 8,118 phishing webpages an...","Based on the provided research evidence, the e..."
1,Q002,What true-positive and false-positive rates di...,CANTINA+ achieved over 92% true positive on un...,"Based on the supplied research evidence, the s..."
2,Q003,Which cryptographic protocols are discussed fo...,"The protocols are SPF, DKIM, DMARC, S/MIME, an...","Based on the provided research evidence, the c..."


In [ ]:
# Save the prepared answer-quality review dataset.

answer_review_path = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data/"
    "answer_quality_review.csv"
)

answer_review_df.to_csv(
    answer_review_path,
    index=False
)

print("ANSWER QUALITY REVIEW DATASET")
print("Total questions:", len(answer_review_df))
print("\nSaved successfully to:")
print(answer_review_path)

display(
    answer_review_df[
        [
            "question_id",
            "question",
            "gold_answer",
            "generated_answer"
        ]
    ].head(3)
)

ANSWER QUALITY REVIEW DATASET
Total questions: 60

Saved successfully to:
/content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/answer_quality_review.csv


,question_id,question,gold_answer,generated_answer
0,Q001,How many phishing and legitimate webpages were...,"The evaluation used 8,118 phishing webpages an...","Based on the provided research evidence, the e..."
1,Q002,What true-positive and false-positive rates di...,CANTINA+ achieved over 92% true positive on un...,"Based on the supplied research evidence, the s..."
2,Q003,Which cryptographic protocols are discussed fo...,"The protocols are SPF, DKIM, DMARC, S/MIME, an...","Based on the provided research evidence, the c..."


Evaluate Q001 first

In [ ]:
# Inspect Q001 for manual evaluation.

q001_review = answer_review_df[
    answer_review_df["question_id"] == "Q001"
].iloc[0]

print("=" * 80)
print("QUESTION ID:", q001_review["question_id"])

print("\nQUESTION:")
print(q001_review["question"])

print("\nREFERENCE ANSWER:")
print(q001_review["gold_answer"])

print("\nGENERATED ANSWER:")
print(q001_review["generated_answer"])

print("\nRETRIEVED EVIDENCE:")
print(q001_review["retrieved_passages"])

QUESTION ID: Q001

QUESTION:
How many phishing and legitimate webpages were used to evaluate CANTINA+?

REFERENCE ANSWER:
The evaluation used 8,118 phishing webpages and 4,883 legitimate webpages.

GENERATED ANSWER:
Based on the provided research evidence, the exact number of phishing and legitimate webpages used to evaluate CANTINA+ cannot be established, as the supplied passages do not state these specific quantities. 

However, regarding related evaluations mentioned in the literature:
* CANTINA+ achieved a 99.61% F1-Score using its proposed filters [Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf, Title: Phishing URL Detection: A Real-Case Scenario Through Login URLs, Year: 2022, PDF Page: 3].
* In contrast, earlier evaluations of CANTINA noted that it was tested on a collection of 200 URLs from three categories [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+, Year: 2011, PDF Page: 18].

RETRIEVED EVIDENCE:


-----

In [ ]:
# Record the manual evaluation of Q001.

q001_scores = {
    "correctness": "Incorrect",
    "relevance": "Partially Relevant",
    "evidence_faithfulness": "Supported",
    "citation_validity": "Valid",
    "review_comments": (
        "The generated answer did not provide the reference quantities "
        "(8,118 phishing webpages and 4,883 legitimate webpages). "
        "The required passage was not retrieved. The model appropriately "
        "abstained from providing unsupported numbers. Its additional "
        "claims about the 99.61% F1-score and the earlier 200-URL "
        "evaluation are supported by the retrieved passages and cited sources."
    )
}

for column, value in q001_scores.items():
    answer_review_df.loc[
        answer_review_df["question_id"] == "Q001",
        column
    ] = value

# Save the updated evaluation dataset.
answer_review_df.to_csv(answer_review_path, index=False)

print("Q001 evaluation saved successfully.")

display(
    answer_review_df.loc[
        answer_review_df["question_id"] == "Q001",
        [
            "question_id",
            "correctness",
            "relevance",
            "evidence_faithfulness",
            "citation_validity",
            "review_comments"
        ]
    ]
)

Q001 evaluation saved successfully.


,question_id,correctness,relevance,evidence_faithfulness,citation_validity,review_comments
0,Q001,Incorrect,Partially Relevant,Supported,Valid,The generated answer did not provide the refer...


In [ ]:
# Compare original and targeted Q001 retrieval results

print("ORIGINAL Q001 QUERY")
print(q001_result["question"])

print("\nORIGINAL RETRIEVED EVIDENCE")
for rank, passage in enumerate(q001_result["evidence"], start=1):
    print(f"\nRank {rank}")
    print("Article:", passage["article_id"])
    print("Page:", passage["page_number"])
    print("Text:", passage["text"][:500])

print("\nTARGETED DIAGNOSTIC QUERY")
print(q001_diagnostic_result["question"])

print("\nTARGETED RETRIEVED EVIDENCE")
for rank, passage in enumerate(q001_diagnostic_result["evidence"], start=1):
    print(f"\nRank {rank}")
    print("Article:", passage["article_id"])
    print("Page:", passage["page_number"])
    print("Text:", passage["text"][:500])

ORIGINAL Q001 QUERY
How many phishing and legitimate webpages were used to evaluate CANTINA+?

ORIGINAL RETRIEVED EVIDENCE

Rank 1
Article: Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf
Page: 3
Text: CANTINA extracts five words from each website using TF- IDF and introduced them into the Google search engine. If a domain was within the n first results, the page was considered legitimate, or phishing otherwise. They obtained an accuracy of 95% with a threshold of n = 30 Google search results. Due to the use of external services like WHOIS7 and the high false-positive rate, authors proposed CANTINA+ [31]. Their new proposal achieved a 99.61% F1-Score including two filters: (i) a comparison of 

Rank 2
Article: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf
Page: 20
Text: The similarity between the influence vectors of both the suspicious webpage and the legitimate webpage is calculated. If the score is more than a

In [ ]:
# Test query expansion for Q001

q001_query_variants = [
    "How many phishing and legitimate webpages were used to evaluate CANTINA+?",

    "What was the size and composition of the CANTINA+ evaluation dataset?",

    "CANTINA+ experimental evaluation corpus phishing websites legitimate websites sample size",

    "Number of phishing pages and legitimate pages in the CANTINA+ evaluation dataset"
]

expected_article = (
    "A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf"
)

expected_page = 1

for query_number, query in enumerate(q001_query_variants, start=1):

    print("\n" + "=" * 80)
    print(f"QUERY {query_number}: {query}")
    print("=" * 80)

    candidates = hybrid_retrieval(
        query,
        top_k=10,
        candidate_k=20
    )

    results = rerank_results(
        query,
        candidates,
        top_k=5
    )

    for rank, passage in enumerate(results, start=1):

        article_match = passage["article_id"] == expected_article
        page_match = int(float(passage["page_number"])) == expected_page

        print(f"\nRank: {rank}")
        print("Article:", passage["article_id"])
        print("Page:", passage["page_number"])
        print("Expected evidence:", article_match and page_match)
        print("Text:", passage["text"][:350])


QUERY 1: How many phishing and legitimate webpages were used to evaluate CANTINA+?

Rank: 1
Article: Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf
Page: 3
Expected evidence: False
Text: CANTINA extracts five words from each website using TF- IDF and introduced them into the Google search engine. If a domain was within the n first results, the page was considered legitimate, or phishing otherwise. They obtained an accuracy of 95% with a threshold of n = 30 Google search results. Due to the use of external services like WHOIS7 and t

Rank: 2
Article: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf
Page: 20
Expected evidence: False
Text: The similarity between the influence vectors of both the suspicious webpage and the legitimate webpage is calculated. If the score is more than a threshold then, the webpage is marked as phishing, else legitimate. The similarity score is obtained using the complexity score and matc

In [ ]:
# Inspect the successful retrieval from Query 4

successful_passage = results[0]

print("ARTICLE:", successful_passage["article_id"])
print("PAGE:", successful_passage["page_number"])

print("\nCOMPLETE RETRIEVED PASSAGE:\n")
print(successful_passage["text"])

print("\nEXPECTED ANSWER:")
print("8,118 phishing webpages and 4,883 legitimate webpages.")

ARTICLE: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
PAGE: 1

COMPLETE RETRIEVED PASSAGE:

Moreover, we designed two filters to help reduce FP and achieve runtime speedup. The first is a near-duplicate phish detector that uses hashing to catch highly similar phish. The second is a login form filter, which directly classifies Web pages with no identified login form as legitimate. We extensively evaluated CANTINA+ with two methods on a diverse spectrum of corpora with 8118 phish and 4883 legitimate Web pages. In the randomized evaluation, CANTINA+ achieved over 92% TP on unique testing phish and over 99% TP on near-duplicate testing phish, and about 0.4% FP with 10% training phish. In the time-based evaluation, CANTINA+ also achieved over 92% TP on unique testing phish, over 99% TP on near-duplicate testing phish, and about 1.4% FP under 20% training phish with a two-week sliding window. Capable of achieving 0.4% FP and over 92% TP, our CANTINA+ has bee

In [ ]:
# Generate Q001 answer using the successful reformulated query

q001_expanded_query = (
    "Number of phishing pages and legitimate pages "
    "in the CANTINA+ evaluation dataset"
)

q001_expanded_result = generate_evidence_grounded_answer(
    q001_expanded_query,
    top_k=5
)

print("REFORMULATED QUERY:")
print(q001_expanded_result["question"])

print("\nGENERATED ANSWER:\n")
print(q001_expanded_result["answer"])

print("\nRETRIEVED EVIDENCE:")
for rank, passage in enumerate(
    q001_expanded_result["evidence"], start=1
):
    print(f"\nRank {rank}")
    print("Article:", passage["article_id"])
    print("Page:", passage["page_number"])
    print("Text:", passage["text"][:500])

REFORMULATED QUERY:
Number of phishing pages and legitimate pages in the CANTINA+ evaluation dataset

GENERATED ANSWER:

Based on the supplied research evidence, the CANTINA+ evaluation dataset contains a diverse spectrum of corpora consisting of 8,118 phishing web pages and 4,883 legitimate web pages [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+, Year: 2011, PDF Page: 1]. 

The phishing corpus is comprised of two sets collected across separate periods:
* **Phish set 1:** 6,943 phishing web pages collected over a five-month period starting in early May 2008 [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+, Year: 2011, PDF Page: 12].
* **Phish set 2:** 1,175 phishing web pages gathered from February 27, 2009, to April 2, 2009 [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+, Year: 2011, PDF Page: 12].

The legitimate web pages were collected 

In [ ]:
# Verify citation metadata for the reformulated Q001 result

for rank, passage in enumerate(
    q001_expanded_result["evidence"], start=1
):
    print(f"\nRank {rank}")
    print("Article ID:", passage["article_id"])
    print("Title:", passage.get("title"))
    print("Year:", passage.get("year"))
    print("PDF Page:", passage["page_number"])


Rank 1
Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Title: CANTINA+
Year: 2011.0
PDF Page: 1

Rank 2
Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Title: CANTINA+
Year: 2011.0
PDF Page: 12

Rank 3
Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Title: CANTINA+
Year: 2011.0
PDF Page: 14

Rank 4
Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Title: CANTINA+
Year: 2011.0
PDF Page: 12

Rank 5
Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Title: CANTINA+
Year: 2011.0
PDF Page: 18


In [ ]:
# Find the CANTINA+ paper in metadata_inventory

q001_article_id = (
    "A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf"
)

q001_metadata = metadata_inventory[
    metadata_inventory["filename"].str.contains(
        "A_Feature_Rich_Machine_Learning_Framework",
        case=False,
        na=False
    )
]

display(q001_metadata)

,filename,doi_candidate,title,authors,year,venue,crossref_status
40,A_Feature_Rich_Machine_Learning_Framework_for_...,10.1145/2019599.2019606,CANTINA+,Guang Xiang; Jason Hong; Carolyn P. Rose; Lorr...,2011.0,ACM Transactions on Information and System Sec...,Retrieved


In [ ]:
# Identify loaded document and chunk objects

for name, obj in globals().copy().items():
    if any(word in name.lower() for word in ["page", "document", "chunk"]):
        print(
            name,
            "| Type:", type(obj).__name__,
            "| Shape:", getattr(obj, "shape", "Not a DataFrame")
        )

chunks_path | Type: str | Shape: Not a DataFrame
sentence_chunks | Type: list | Shape: Not a DataFrame
page_number | Type: int | Shape: Not a DataFrame
matching_pages | Type: list | Shape: Not a DataFrame
page_documents | Type: list | Shape: Not a DataFrame
q001_expected_page | Type: int | Shape: Not a DataFrame
chunk | Type: dict | Shape: Not a DataFrame
q001_target_chunks | Type: list | Shape: Not a DataFrame
same_page | Type: function | Shape: Not a DataFrame
expected_page | Type: int | Shape: Not a DataFrame
page_match | Type: bool | Shape: Not a DataFrame


In [ ]:
# Inspect page_documents structure and locate the CANTINA+ paper

print("Total page records:", len(page_documents))

print("\nFirst record keys:")
print(page_documents[0].keys())

print("\nMatching CANTINA+ document records:")

for record in page_documents:
    record_text = str(record)

    if "A_Feature_Rich_Machine_Learning_Framework" in record_text:
        print("\nAvailable fields:", record.keys())

        for key, value in record.items():
            print(f"\n{key}:")
            print(str(value)[:1500])
        break

Total page records: 3071

First record keys:
dict_keys(['article_id', 'page_number', 'text'])

Matching CANTINA+ document records:

Available fields: dict_keys(['article_id', 'page_number', 'text'])

article_id:
A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf

page_number:
1

text:
21
CANTINA+: A Feature-Rich Machine Learning Framework for
Detecting Phishing Web Sites
GUANG XIANG, JASON HONG, CAROLYN P. ROSE, and LORRIE CRANOR,
Carnegie Mellon University
Phishing is a plague in cyberspace.
Typically, phish detection methods either use human-veriﬁed URL
blacklists or exploit Web page features via machine learning techniques. However, the former is frail in
terms of new phish, and the latter suffers from the scarcity of effective features and the high false positive
rate (FP). To alleviate those problems, we propose a layered anti-phishing solution that aims at (1) exploiting
the expressiveness of a rich set of features with machine learning to achieve a hi

In [ ]:
# Inspect the first page of the CANTINA+ paper more fully

for record in page_documents:
    if (
        record["article_id"]
        == "A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf"
        and record["page_number"] == 1
    ):
        print(record["text"][:2500])
        break

21
CANTINA+: A Feature-Rich Machine Learning Framework for
Detecting Phishing Web Sites
GUANG XIANG, JASON HONG, CAROLYN P. ROSE, and LORRIE CRANOR,
Carnegie Mellon University
Phishing is a plague in cyberspace.
Typically, phish detection methods either use human-veriﬁed URL
blacklists or exploit Web page features via machine learning techniques. However, the former is frail in
terms of new phish, and the latter suffers from the scarcity of effective features and the high false positive
rate (FP). To alleviate those problems, we propose a layered anti-phishing solution that aims at (1) exploiting
the expressiveness of a rich set of features with machine learning to achieve a high true positive rate (TP)
on novel phish, and (2) limiting the FP to a low level via ﬁltering algorithms.
Speciﬁcally, we proposed CANTINA+, the most comprehensive feature-based approach in the literature
including eight novel features, which exploits the HTML Document Object Model (DOM), search engines and
thir

In [ ]:
# Inspect the CANTINA+ metadata record

cantina_mask = metadata_inventory["filename"].str.contains(
    "A_Feature_Rich_Machine_Learning_Framework",
    case=False,
    na=False
)

display(
    metadata_inventory.loc[
        cantina_mask,
        ["filename", "title", "authors", "year", "venue", "doi_candidate"]
    ]
)

,filename,title,authors,year,venue,doi_candidate
40,A_Feature_Rich_Machine_Learning_Framework_for_...,CANTINA+,Guang Xiang; Jason Hong; Carolyn P. Rose; Lorr...,2011.0,ACM Transactions on Information and System Sec...,10.1145/2019599.2019606


In [ ]:
# Inspect the CANTINA+ chunk metadata structure

cantina_chunks = [
    chunk for chunk in sentence_chunks
    if chunk.get("article_id") ==
    "A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf"
]

print("Number of matching chunks:", len(cantina_chunks))

if cantina_chunks:
    print("\nAvailable fields:")
    print(cantina_chunks[0].keys())

    print("\nCurrent title:")
    print(cantina_chunks[0].get("title"))

    print("\nSample chunk metadata:")
    for key, value in cantina_chunks[0].items():
        if key != "text":
            print(f"{key}: {value}")

Number of matching chunks: 108

Available fields:
dict_keys(['article_id', 'title', 'authors', 'year', 'venue', 'doi', 'page_number', 'text'])

Current title:
CANTINA+

Sample chunk metadata:
article_id: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
title: CANTINA+
authors: Guang Xiang; Jason Hong; Carolyn P. Rose; Lorrie Cranor
year: 2011.0
venue: ACM Transactions on Information and System Security
doi: 10.1145/2019599.2019606
page_number: 1


In [ ]:
# Correct the CANTINA+ article title

correct_title = (
    "CANTINA+: A Feature-Rich Machine Learning Framework "
    "for Detecting Phishing Web Sites"
)

target_filename = (
    "A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf"
)

# Update metadata inventory
metadata_inventory.loc[
    metadata_inventory["filename"] == target_filename,
    "title"
] = correct_title

# Update all 108 sentence-based chunks
updated_chunks = 0

for chunk in sentence_chunks:
    if chunk["article_id"] == target_filename:
        chunk["title"] = correct_title
        updated_chunks += 1

print("Metadata title:", metadata_inventory.loc[
    metadata_inventory["filename"] == target_filename,
    "title"
].iloc[0])

print("Updated sentence chunks:", updated_chunks)

Metadata title: CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites
Updated sentence chunks: 108


In [ ]:
# Check whether fixed-size chunks contain the abbreviated title

fixed_cantina_chunks = [
    chunk for chunk in fixed_size_chunks
    if chunk.get("article_id") == target_filename
]

print("Matching fixed-size chunks:", len(fixed_cantina_chunks))

if fixed_cantina_chunks:
    print("Current title:", fixed_cantina_chunks[0].get("title"))
    print("Available fields:", fixed_cantina_chunks[0].keys())

Matching fixed-size chunks: 115
Current title: CANTINA+
Available fields: dict_keys(['article_id', 'title', 'authors', 'year', 'venue', 'doi', 'page_number', 'text'])


In [ ]:
# Check the saved fixed-size chunk file

import os

fixed_chunks_path = os.path.join(
    "/content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data",
    "fixed_size_chunks.pkl"
)

print("File exists:", os.path.exists(fixed_chunks_path))
print("File path:", fixed_chunks_path)

File exists: True
File path: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/fixed_size_chunks.pkl


In [ ]:
# Load the existing fixed-size chunks

import pickle

with open(fixed_chunks_path, "rb") as file:
    fixed_size_chunks = pickle.load(file)

fixed_cantina_chunks = [
    chunk for chunk in fixed_size_chunks
    if chunk.get("article_id") == target_filename
]

print("Total fixed-size chunks:", len(fixed_size_chunks))
print("Matching CANTINA+ chunks:", len(fixed_cantina_chunks))

if fixed_cantina_chunks:
    print("Current title:", fixed_cantina_chunks[0].get("title"))
    print("Available fields:", fixed_cantina_chunks[0].keys())

Total fixed-size chunks: 13585
Matching CANTINA+ chunks: 115
Current title: CANTINA+
Available fields: dict_keys(['article_id', 'title', 'authors', 'year', 'venue', 'doi', 'page_number', 'text'])


In [ ]:
# Update the title in all matching fixed-size chunks

updated_fixed_chunks = 0

for chunk in fixed_size_chunks:
    if chunk["article_id"] == target_filename:
        chunk["title"] = correct_title
        updated_fixed_chunks += 1

print("Updated fixed-size chunks:", updated_fixed_chunks)

# Verify both chunk datasets
print(
    "Sentence chunks:",
    sum(c["article_id"] == target_filename and c["title"] == correct_title
        for c in sentence_chunks)
)

print(
    "Fixed-size chunks:",
    sum(c["article_id"] == target_filename and c["title"] == correct_title
        for c in fixed_size_chunks)
)

Updated fixed-size chunks: 115
Sentence chunks: 108
Fixed-size chunks: 115


In [ ]:
# Save corrected metadata and chunk datasets

import pickle
import os

metadata_path = os.path.join(
    "/content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data",
    "metadata_inventory.csv"
)

sentence_chunks_path = os.path.join(
    "/content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data",
    "sentence_based_chunks.pkl"
)

# Save metadata inventory
metadata_inventory.to_csv(metadata_path, index=False)

# Save sentence-based chunks
with open(sentence_chunks_path, "wb") as file:
    pickle.dump(sentence_chunks, file)

# Save fixed-size chunks
with open(fixed_chunks_path, "wb") as file:
    pickle.dump(fixed_size_chunks, file)

print("All three files saved successfully.")

print("\nSaved files:")
for path in [metadata_path, sentence_chunks_path, fixed_chunks_path]:
    print(os.path.basename(path), "|", os.path.getsize(path), "bytes")

All three files saved successfully.

Saved files:
metadata_inventory.csv | 49957 bytes
sentence_based_chunks.pkl | 14155551 bytes
fixed_size_chunks.pkl | 15086630 bytes


In [ ]:
# Reload saved files and verify the corrected title

with open(sentence_chunks_path, "rb") as file:
    verified_sentence_chunks = pickle.load(file)

with open(fixed_chunks_path, "rb") as file:
    verified_fixed_chunks = pickle.load(file)

verified_metadata = pd.read_csv(metadata_path)

# Check metadata inventory
metadata_title = verified_metadata.loc[
    verified_metadata["filename"] == target_filename,
    "title"
].iloc[0]

# Check sentence-based chunks
sentence_titles = set(
    c["title"] for c in verified_sentence_chunks
    if c["article_id"] == target_filename
)

# Check fixed-size chunks
fixed_titles = set(
    c["title"] for c in verified_fixed_chunks
    if c["article_id"] == target_filename
)

print("Metadata title:", metadata_title)
print("Sentence chunk titles:", sentence_titles)
print("Fixed-size chunk titles:", fixed_titles)

print("\nVerification:")
print("Metadata correct:", metadata_title == correct_title)
print("Sentence chunks correct:", sentence_titles == {correct_title})
print("Fixed-size chunks correct:", fixed_titles == {correct_title})

Metadata title: CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites
Sentence chunk titles: {'CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites'}
Fixed-size chunk titles: {'CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites'}

Verification:
Metadata correct: True
Sentence chunks correct: True
Fixed-size chunks correct: True


In [ ]:
# Inspect answer_generation_results structure

print("Object type:", type(answer_generation_results).__name__)

if isinstance(answer_generation_results, dict):
    print("Number of entries:", len(answer_generation_results))
    print("\nFirst entry:")
    first_key = next(iter(answer_generation_results))
    print("Key:", first_key)
    print("Value type:", type(answer_generation_results[first_key]).__name__)
    print("Value:", str(answer_generation_results[first_key])[:1000])

elif isinstance(answer_generation_results, list):
    print("Number of entries:", len(answer_generation_results))
    print("\nFirst item type:", type(answer_generation_results[0]).__name__)
    print("First item:", str(answer_generation_results[0])[:1000])

Object type: dict
Number of entries: 80

First entry:
Key: Q001
Value type: dict
Value: {'question': 'How many phishing and legitimate webpages were used to evaluate CANTINA+?', 'answer': 'Based on the provided research evidence, the exact number of phishing and legitimate webpages used to evaluate CANTINA+ cannot be established, as the supplied passages do not state these specific quantities. \n\nHowever, regarding related evaluations mentioned in the literature:\n* CANTINA+ achieved a 99.61% F1-Score using its proposed filters [Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf, Title: Phishing URL Detection: A Real-Case Scenario Through Login URLs, Year: 2022, PDF Page: 3].\n* In contrast, earlier evaluations of CANTINA noted that it was tested on a collection of 200 URLs from three categories [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+, Year: 2011, PDF Page: 18].', 'evidence': [{'rank': 3, 'rrf_score': 0.016393442

In [ ]:
# Find generated answers and evidence containing the abbreviated title

affected_questions = []

for question_id, result in answer_generation_results.items():

    answer_text = result.get("answer", "")

    answer_has_old_title = (
        "Title: CANTINA+" in answer_text
    )

    evidence_has_old_title = any(
        passage.get("title") == "CANTINA+"
        for passage in result.get("evidence", [])
    )

    if answer_has_old_title or evidence_has_old_title:
        affected_questions.append({
            "question_id": question_id,
            "answer_has_old_title": answer_has_old_title,
            "evidence_has_old_title": evidence_has_old_title
        })

print("Total affected questions:", len(affected_questions))

for item in affected_questions:
    print(item)

Total affected questions: 4
{'question_id': 'Q001', 'answer_has_old_title': True, 'evidence_has_old_title': True}
{'question_id': 'Q002', 'answer_has_old_title': True, 'evidence_has_old_title': True}
{'question_id': 'Q023', 'answer_has_old_title': False, 'evidence_has_old_title': True}
{'question_id': 'Q061', 'answer_has_old_title': False, 'evidence_has_old_title': True}


In [ ]:
# Inspect affected answer and evidence metadata

for question_id in ["Q001", "Q002", "Q023", "Q061"]:

    result = answer_generation_results[question_id]

    print("\n" + "=" * 70)
    print("QUESTION:", question_id)
    print("=" * 70)

    print("\nGenerated answer citation:")
    for line in result["answer"].splitlines():
        if "A_Feature_Rich_Machine_Learning_Framework" in line:
            print(line)

    print("\nRetrieved evidence:")
    for passage in result["evidence"]:
        if passage.get("article_id") == target_filename:
            print({
                "title": passage.get("title"),
                "year": passage.get("year"),
                "page_number": passage.get("page_number")
            })


QUESTION: Q001

Generated answer citation:
* In contrast, earlier evaluations of CANTINA noted that it was tested on a collection of 200 URLs from three categories [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+, Year: 2011, PDF Page: 18].

Retrieved evidence:
{'title': 'CANTINA+', 'year': 2011.0, 'page_number': 18}

QUESTION: Q002

Generated answer citation:
Based on the supplied research evidence, the specific true-positive (TP) and false-positive (FP) rates reported for CANTINA+ during its randomized evaluation cannot be fully determined. While the evidence describes the performance of CANTINA+ under a "Time-Based Evaluation with a Two-Week Sliding Window" (which reports TP percentages ranging from 92.25% to 99.64% and FP percentages of 1.375% or 1.948% depending on the type of phish and login filtering) [Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+, Year: 2011, PDF Page 

In [ ]:
# Correct CANTINA+ title in generated answers and retrieved evidence

updated_answers = 0
updated_evidence = 0

for question_id in ["Q001", "Q002", "Q023", "Q061"]:

    result = answer_generation_results[question_id]

    # Correct citation title in generated answer text
    old_citation = "Title: CANTINA+"
    new_citation = f"Title: {correct_title}"

    if old_citation in result["answer"]:
        result["answer"] = result["answer"].replace(
            old_citation,
            new_citation
        )
        updated_answers += 1

    # Correct title in retrieved evidence metadata
    for passage in result["evidence"]:
        if passage.get("article_id") == target_filename:
            if passage.get("title") == "CANTINA+":
                passage["title"] = correct_title
                updated_evidence += 1

print("Generated answers corrected:", updated_answers)
print("Evidence records corrected:", updated_evidence)

Generated answers corrected: 2
Evidence records corrected: 6


In [ ]:
# Verify title consistency across all 80 generated answers

remaining_old_titles = []

for question_id, result in answer_generation_results.items():

    if "Title: CANTINA+" in result.get("answer", ""):
        remaining_old_titles.append(
            (question_id, "Generated answer")
        )

    for passage in result.get("evidence", []):
        if (
            passage.get("article_id") == target_filename
            and passage.get("title") == "CANTINA+"
        ):
            remaining_old_titles.append(
                (question_id, "Retrieved evidence")
            )

print("Total generated questions:", len(answer_generation_results))
print("Remaining abbreviated titles:", len(remaining_old_titles))

if remaining_old_titles:
    print(remaining_old_titles)
else:
    print("All CANTINA+ titles are corrected.")

Total generated questions: 80
Remaining abbreviated titles: 2
[('Q001', 'Generated answer'), ('Q002', 'Generated answer')]


In [ ]:
# Find every remaining occurrence of the abbreviated title

for question_id in ["Q001", "Q002"]:

    answer = answer_generation_results[question_id]["answer"]

    print("\n", question_id)

    for line in answer.splitlines():
        if "CANTINA+" in line:
            print(line)


 Q001
Based on the provided research evidence, the exact number of phishing and legitimate webpages used to evaluate CANTINA+ cannot be established, as the supplied passages do not state these specific quantities. 
* CANTINA+ achieved a 99.61% F1-Score using its proposed filters [Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf, Title: Phishing URL Detection: A Real-Case Scenario Through Login URLs, Year: 2022, PDF Page: 3].
* In contrast, earlier evaluations of CANTINA noted that it was tested on a collection of 200 URLs from three categories [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites, Year: 2011, PDF Page: 18].

 Q002
Based on the supplied research evidence, the specific true-positive (TP) and false-positive (FP) rates reported for CANTINA+ during its randomized evaluation cannot be fully determined. While the evidence describes the perf

In [ ]:
# Verify the four affected questions

for question_id in ["Q001", "Q002", "Q023", "Q061"]:

    result = answer_generation_results[question_id]

    print(f"\n{question_id}")

    answer_has_full_title = correct_title in result["answer"]

    evidence_titles = [
        passage["title"]
        for passage in result["evidence"]
        if passage.get("article_id") == target_filename
    ]

    evidence_correct = (
        len(evidence_titles) > 0
        and all(title == correct_title for title in evidence_titles)
    )

    print("Answer contains full title:", answer_has_full_title)
    print("Evidence titles:", evidence_titles)
    print("Evidence metadata correct:", evidence_correct)


Q001
Answer contains full title: True
Evidence titles: ['CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites']
Evidence metadata correct: True

Q002
Answer contains full title: True
Evidence titles: ['CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites', 'CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites', 'CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites']
Evidence metadata correct: True

Q023
Answer contains full title: False
Evidence titles: ['CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites']
Evidence metadata correct: True

Q061
Answer contains full title: False
Evidence titles: ['CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites']
Evidence metadata correct: True


In [ ]:
# Save corrected answer-generation results

answer_results_path = (
    "/content/drive/MyDrive/NLP-Project/CyberThreatResearch/"
    "processed_data/answer_generation_results.pkl"
)

with open(answer_results_path, "wb") as file:
    pickle.dump(answer_generation_results, file)

print("Answer-generation results saved successfully.")
print("Total questions saved:", len(answer_generation_results))
print("File size:", os.path.getsize(answer_results_path), "bytes")

Answer-generation results saved successfully.
Total questions saved: 80
File size: 553444 bytes


In [ ]:
# Reload and verify saved answer-generation results

with open(answer_results_path, "rb") as file:
    verified_answer_results = pickle.load(file)

print("Total saved questions:", len(verified_answer_results))

for question_id in ["Q001", "Q002", "Q023", "Q061"]:

    result = verified_answer_results[question_id]

    evidence_titles = [
        p["title"]
        for p in result["evidence"]
        if p.get("article_id") == target_filename
    ]

    print(f"\n{question_id}")
    print("Corrected evidence metadata:",
          all(t == correct_title for t in evidence_titles)
          if evidence_titles else "No matching evidence")

Total saved questions: 80

Q001
Corrected evidence metadata: True

Q002
Corrected evidence metadata: True

Q023
Corrected evidence metadata: True

Q061
Corrected evidence metadata: True


In [ ]:
# Inspect the answer-quality review dataset

print("Dataset shape:", answer_review_df.shape)
print("Columns:", answer_review_df.columns.tolist())

print("\nQuestions whose generated answers mention the CANTINA+ article:")

for _, row in answer_review_df.iterrows():
    if target_filename in str(row["generated_answer"]):
        print(row["question_id"])

Dataset shape: (60, 11)
Columns: ['question_id', 'question_category', 'question', 'gold_answer', 'generated_answer', 'retrieved_passages', 'correctness', 'relevance', 'evidence_faithfulness', 'citation_validity', 'review_comments']

Questions whose generated answers mention the CANTINA+ article:
Q001
Q002


In [ ]:
# Inspect CANTINA+ references in the answer-quality review dataset

for question_id in ["Q001", "Q002"]:

    row = answer_review_df[
        answer_review_df["question_id"] == question_id
    ].iloc[0]

    print("\n", "=" * 60)
    print(question_id)
    print("=" * 60)

    print("\nGenerated answer contains full title:",
          correct_title in str(row["generated_answer"]))

    print("\nGenerated answer:")
    print(row["generated_answer"])

    print("\nRetrieved passages containing CANTINA+ article:")
    passages = row["retrieved_passages"]

    if isinstance(passages, list):
        for passage in passages:
            if passage.get("article_id") == target_filename:
                print({
                    "title": passage.get("title"),
                    "page_number": passage.get("page_number")
                })
    else:
        print("Retrieved passages type:", type(passages).__name__)


Q001

Generated answer contains full title: False

Generated answer:
Based on the provided research evidence, the exact number of phishing and legitimate webpages used to evaluate CANTINA+ cannot be established, as the supplied passages do not state these specific quantities. 

However, regarding related evaluations mentioned in the literature:
* CANTINA+ achieved a 99.61% F1-Score using its proposed filters [Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf, Title: Phishing URL Detection: A Real-Case Scenario Through Login URLs, Year: 2022, PDF Page: 3].
* In contrast, earlier evaluations of CANTINA noted that it was tested on a collection of 200 URLs from three categories [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+, Year: 2011, PDF Page: 18].

Retrieved passages containing CANTINA+ article:
Retrieved passages type: str

Q002

Generated answer contains full title: False

Generated answer:
Based on the supplied rese

In [ ]:
# Update Q001 and Q002 generated answers in the review dataset

for question_id in ["Q001", "Q002"]:

    corrected_answer = verified_answer_results[question_id]["answer"]

    answer_review_df.loc[
        answer_review_df["question_id"] == question_id,
        "generated_answer"
    ] = corrected_answer

print("Updated generated answers in review dataset.")

for question_id in ["Q001", "Q002"]:

    row = answer_review_df[
        answer_review_df["question_id"] == question_id
    ].iloc[0]

    print(
        question_id,
        "| Full title present:",
        correct_title in row["generated_answer"]
    )

Updated generated answers in review dataset.
Q001 | Full title present: True
Q002 | Full title present: True


In [ ]:
# Check retrieved-passage text for outdated CANTINA+ metadata

for question_id in ["Q001", "Q002"]:

    row = answer_review_df[
        answer_review_df["question_id"] == question_id
    ].iloc[0]

    passages_text = str(row["retrieved_passages"])

    print("\n", question_id)
    print("Contains abbreviated title:",
          "Title: CANTINA+" in passages_text)

    for line in passages_text.splitlines():
        if "A_Feature_Rich_Machine_Learning_Framework" in line:
            print(line[:500])


 Q001
Contains abbreviated title: True
Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf

 Q002
Contains abbreviated title: True
Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf


In [ ]:
for question_id in ["Q001", "Q002"]:

    row = answer_review_df[
        answer_review_df["question_id"] == question_id
    ].iloc[0]

    passages_text = str(row["retrieved_passages"])

    print(f"\n--- {question_id} ---")

    for line in passages_text.splitlines():
        if "Title:" in line:
            print(line)


--- Q001 ---
Title: Phishing URL Detection: A Real-Case Scenario Through Login URLs
Title: Phishing Webpage Detection: Unveiling the Threat Landscape and Investigating Detection Techniques
Title: Phishing Webpage Detection: Unveiling the Threat Landscape and Investigating Detection Techniques
Title: CANTINA: A Content-Based Approach to Detecting Phishing Web Sites
Title: CANTINA+

--- Q002 ---
Title: CANTINA: A Content-Based Approach to Detecting Phishing Web Sites
Title: CANTINA+
Title: Phishing URL Detection: A Real-Case Scenario Through Login URLs
Title: CANTINA+
Title: CANTINA+


In [ ]:
# Correct only the outdated CANTINA+ title entries in Q001 and Q002

old_title_entry = "Title: CANTINA+"
new_title_entry = f"Title: {correct_title}"

for question_id in ["Q001", "Q002"]:

    mask = answer_review_df["question_id"] == question_id

    original_text = answer_review_df.loc[
        mask, "retrieved_passages"
    ].iloc[0]

    corrected_text = str(original_text).replace(
        old_title_entry,
        new_title_entry
    )

    answer_review_df.loc[
        mask, "retrieved_passages"
    ] = corrected_text

    print(
        question_id,
        "| Replacements:",
        str(original_text).count(old_title_entry)
    )

# Save the corrected review dataset
answer_review_path = (
    "/content/drive/MyDrive/NLP-Project/CyberThreatResearch/"
    "processed_data/answer_quality_review.csv"
)

answer_review_df.to_csv(answer_review_path, index=False)

print("\nCorrected review dataset saved successfully.")

Q001 | Replacements: 1
Q002 | Replacements: 3

Corrected review dataset saved successfully.


In [ ]:
import pandas as pd

# Reload the saved review dataset
answer_review_path = (
    "/content/drive/MyDrive/NLP-Project/CyberThreatResearch/"
    "processed_data/answer_quality_review.csv"
)

verified_review_df = pd.read_csv(answer_review_path)

# Check Q001 and Q002
for question_id in ["Q001", "Q002"]:

    row = verified_review_df[
        verified_review_df["question_id"] == question_id
    ].iloc[0]

    passages = str(row["retrieved_passages"])

    print(f"\n--- {question_id} ---")
    print("Correct full title present:", correct_title in passages)
    print("Old abbreviated title remains:",
          "Title: CANTINA+" in passages.replace(
              f"Title: {correct_title}", ""
          ))

# Check Q001 manual evaluation
q001 = verified_review_df[
    verified_review_df["question_id"] == "Q001"
].iloc[0]

print("\n--- Q001 Evaluation ---")
print("Correctness:", q001["correctness"])
print("Relevance:", q001["relevance"])
print("Evidence faithfulness:", q001["evidence_faithfulness"])
print("Citation validity:", q001["citation_validity"])
print("Review comments:", q001["review_comments"])

print("\nDataset shape:", verified_review_df.shape)


--- Q001 ---
Correct full title present: True
Old abbreviated title remains: False

--- Q002 ---
Correct full title present: True
Old abbreviated title remains: False

--- Q001 Evaluation ---
Correctness: Incorrect
Relevance: Partially Relevant
Evidence faithfulness: Supported
Citation validity: Valid
Review comments: The generated answer did not provide the reference quantities (8,118 phishing webpages and 4,883 legitimate webpages). The required passage was not retrieved. The model appropriately abstained from providing unsupported numbers. Its additional claims about the 99.61% F1-score and the earlier 200-URL evaluation are supported by the retrieved passages and cited sources.

Dataset shape: (60, 11)


In [ ]:
# Display Q002 for manual answer-quality evaluation

q002 = verified_review_df[
    verified_review_df["question_id"] == "Q002"
].iloc[0]

print("QUESTION ID:", q002["question_id"])
print("CATEGORY:", q002["question_category"])

print("\nQUESTION:")
print(q002["question"])

print("\nREFERENCE ANSWER:")
print(q002["gold_answer"])

print("\nGENERATED ANSWER:")
print(q002["generated_answer"])

print("\nRETRIEVED PASSAGES:")
print(q002["retrieved_passages"])

QUESTION ID: Q002
CATEGORY: Factual

QUESTION:
What true-positive and false-positive rates did CANTINA+ report in its randomized evaluation?

REFERENCE ANSWER:
CANTINA+ achieved over 92% true positive on unique testing phishing pages, over 99% on near-duplicate testing phishing pages, and about 0.4% false positive with 10% training phishing pages.

GENERATED ANSWER:
Based on the supplied research evidence, the specific true-positive (TP) and false-positive (FP) rates reported for CANTINA+ during its randomized evaluation cannot be fully determined. While the evidence describes the performance of CANTINA+ under a "Time-Based Evaluation with a Two-Week Sliding Window" (which reports TP percentages ranging from 92.25% to 99.64% and FP percentages of 1.375% or 1.948% depending on the type of phish and login filtering) [Article ID: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing

In [ ]:
# Save the manual evaluation for Q002

q002_ratings = {
    "correctness": "Incorrect",
    "relevance": "Partially Relevant",
    "evidence_faithfulness": "Supported",
    "citation_validity": "Valid",
    "review_comments": (
        "The generated answer did not provide the reference results "
        "for the randomized evaluation. It instead reported TP and FP "
        "rates from the time-based evaluation on PDF page 16 and correctly "
        "distinguished these from the randomized evaluation. The retrieved "
        "passages included a discussion of randomized evaluation on page 18 "
        "and a reference to Table IV on page 17, but did not contain the "
        "required numerical results. The additional figures were accurately "
        "reported and supported by the cited source. The answer appropriately "
        "avoided presenting time-based results as randomized results."
    )
}

for column, value in q002_ratings.items():
    verified_review_df.loc[
        verified_review_df["question_id"] == "Q002",
        column
    ] = value

# Save the updated review dataset
verified_review_df.to_csv(answer_review_path, index=False)

print("Q002 evaluation saved successfully.")

print(
    verified_review_df.loc[
        verified_review_df["question_id"] == "Q002",
        ["correctness", "relevance",
         "evidence_faithfulness", "citation_validity"]
    ].to_string(index=False)
)

Q002 evaluation saved successfully.
correctness          relevance evidence_faithfulness citation_validity
  Incorrect Partially Relevant             Supported             Valid


In [ ]:
# Display the remaining factual questions (Q003–Q020)

batch1 = verified_review_df[
    verified_review_df["question_id"].between("Q003", "Q020")
]

for _, row in batch1.iterrows():

    print("\n" + "=" * 90)
    print("QUESTION ID:", row["question_id"])
    print("QUESTION:", row["question"])
    print("\nREFERENCE ANSWER:")
    print(row["gold_answer"])
    print("\nGENERATED ANSWER:")
    print(row["generated_answer"])
    print("\nRETRIEVED PASSAGES:")
    print(row["retrieved_passages"])


QUESTION ID: Q003
QUESTION: Which cryptographic protocols are discussed for validating sender identity and data integrity in the hybrid phishing email detection approach?

REFERENCE ANSWER:
The protocols are SPF, DKIM, DMARC, S/MIME, and PGP.

GENERATED ANSWER:
Based on the provided research evidence, the cryptographic protocols discussed for validating sender identity and confirming data integrity within the hybrid phishing email detection approach include SPF, DKIM, DMARC, S/MIME, and PGP [Article ID: A_Hybrid_Approach_for_Phishing_Email_Detection_using_AI_and_Cryptography.pdf, Title: A Hybrid Approach for Phishing Email Detection using AI and Cryptography, Year: 2025, PDF Page: 1]. 

Additionally, the study notes that cryptographic methods such as SPF, DKIM, and DMARC work in unison to secure the integrity of emails while in transit and to verify the sender's identity, preventing sender impersonation and domain spoofing [Article ID: A_Hybrid_Approach_for_Phishing_Email_Detection_us

In [ ]:

# Save the manual evaluations for Q003–Q020.
# Existing evaluations for Q001–Q002 will remain unchanged.

assessments = {
    "Q003": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly identifies all five protocols: SPF, DKIM, DMARC, S/MIME, and PGP. The list and additional explanations are supported by the retrieved passages and citations."
    },
    "Q004": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly identifies early-fusion, late-fusion, and late-ensemble learning as the three proposed models. Its explanations are supported by the retrieved passages on PDF pages 1, 4, and 9."
    },
    "Q005": {
        "correctness": "Incorrect",
        "relevance": "Partially Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer did not identify both datasets used in the study: UNSW-NB15 and NSL-KDD. It mentioned UNSW-NB15 results and datasets discussed in related work, but the supplied passages did not clearly establish both study datasets. It appropriately avoided inventing the missing information."
    },
    "Q006": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly explains that reconstruction error is used as an anomaly detection and classification metric. Its explanation of threshold-based classification and feature-level reconstruction errors is supported by the retrieved passages on PDF pages 1 and 2."
    },
    "Q007": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly describes the autoencoder, CNN, and LSTM-related methods requested. The reported details are consistent with and supported by the retrieved research evidence."
    },
    "Q008": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly identifies the KDD Cup 99 dataset and provides information consistent with the retrieved passages. Its claims are supported by the cited research evidence."
    },
    "Q009": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly covers the graph-based, large language model, and phishing-kit-related concepts requested. The explanations are supported by the retrieved passages and cited sources."
    },
    "Q010": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly reports that the systematic review examined 80 papers. This figure is supported by the retrieved passage describing the review's scope."
    },
    "Q011": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly identifies the three stages of the DNN approach requested. The explanation is consistent with the retrieved research evidence and its citations."
    },
    "Q012": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly addresses zero-day attacks, out-of-distribution inputs, and adversarial evasion. Its claims are supported by the retrieved passages."
    },
    "Q013": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly identifies the autoencoder and restricted Boltzmann machine (RBM) methods requested. The information is supported by the retrieved research evidence."
    },
    "Q014": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly reports the performance values of 91.86% and 92.12%. These figures are supported by the retrieved passages and cited source."
    },
    "Q015": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly identifies ToN-IoT and CIC-IoT 2023 as the two datasets used. The dataset information is supported by the retrieved research passages."
    },
    "Q016": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly reports 99.42% accuracy on the ToN-IoT dataset. The result is supported by the retrieved passages and cited source."
    },
    "Q017": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly reports 99.44% accuracy on CIC-IoT 2023. The result is supported by the retrieved abstract on PDF page 1 and the comparison table on page 16. The additional AUC value of 0.9988 is also supported."
    },
    "Q018": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly reports both AUC values: 0.9997 for ToN-IoT and 0.9988 for CIC-IoT 2023. Both values are explicitly supported by the retrieved passages on PDF pages 1 and 17."
    },
    "Q019": {
        "correctness": "Partially Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer identifies dropout, L2 weight regularization, and Gaussian noise, but lists batch normalization instead of label smoothing among its four main techniques. It does mention label smoothing separately in a note. All these techniques are discussed in the retrieved evidence, but the main list does not accurately match the four techniques in the reference answer."
    },
    "Q020": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly reports that machine learning techniques were applied in 57 studies. This figure is explicitly supported by the retrieved passage on PDF page 1. The additional information about other reviews is also consistent with the retrieved evidence."
    }
}

# Check that every question exists exactly once before updating.
for qid in assessments:
    matches = (verified_review_df["question_id"].astype(str) == qid).sum()
    assert matches == 1, f"{qid} appears {matches} times in the review dataset."

# Update only Q003–Q020.
for qid, review in assessments.items():
    mask = verified_review_df["question_id"].astype(str) == qid
    for column, value in review.items():
        verified_review_df.loc[mask, column] = value

# Save the updated review dataset.
verified_review_df.to_csv(answer_review_path, index=False)

# Reload the saved CSV to verify persistence.
verified_review_df = pd.read_csv(answer_review_path)

# Verify all Q003–Q020 ratings were saved.
review_columns = [
    "correctness",
    "relevance",
    "evidence_faithfulness",
    "citation_validity",
    "review_comments"
]

saved_batch = verified_review_df[
    verified_review_df["question_id"].astype(str).isin(assessments.keys())
]

assert len(saved_batch) == 18, "Expected 18 reviewed questions."
assert saved_batch[review_columns].notna().all().all(), "Some review fields are missing."

# Verify Q001 and Q002 remain intact.
earlier_reviews = verified_review_df[
    verified_review_df["question_id"].astype(str).isin(["Q001", "Q002"])
]

assert len(earlier_reviews) == 2
assert earlier_reviews[review_columns].notna().all().all()

print("SUCCESS: Factual question evaluations saved and verified.")
print("Q001–Q002: Existing evaluations preserved.")
print("Q003–Q020: 18 evaluations saved.")
print("Total Factual questions reviewed:", len(
    verified_review_df[
        verified_review_df["question_id"].astype(str).isin(
            [f"Q{i:03d}" for i in range(1, 21)]
        )
    ]
))

display(
    verified_review_df[
        verified_review_df["question_id"].astype(str).isin(
            [f"Q{i:03d}" for i in range(1, 21)]
        )
    ][["question_id"] + review_columns]
)


SUCCESS: Factual question evaluations saved and verified.
Q001–Q002: Existing evaluations preserved.
Q003–Q020: 18 evaluations saved.
Total Factual questions reviewed: 20


,question_id,correctness,relevance,evidence_faithfulness,citation_validity,review_comments
0,Q001,Incorrect,Partially Relevant,Supported,Valid,The generated answer did not provide the refer...
1,Q002,Incorrect,Partially Relevant,Supported,Valid,The generated answer did not provide the refer...
2,Q003,Correct,Relevant,Supported,Valid,The answer correctly identifies all five proto...
3,Q004,Correct,Relevant,Supported,Valid,"The answer correctly identifies early-fusion, ..."
4,Q005,Incorrect,Partially Relevant,Supported,Valid,The answer did not identify both datasets used...
5,Q006,Correct,Relevant,Supported,Valid,The answer correctly explains that reconstruct...
6,Q007,Correct,Relevant,Supported,Valid,The answer correctly describes the autoencoder...
7,Q008,Correct,Relevant,Supported,Valid,The answer correctly identifies the KDD Cup 99...
8,Q009,Correct,Relevant,Supported,Valid,"The answer correctly covers the graph-based, l..."
9,Q010,Correct,Relevant,Supported,Valid,The answer correctly reports that the systemat...


In [ ]:

# Display all 20 Conceptual questions (Q021–Q040)
# using the actual structure of answer_generation_results.

conceptual_batch = [f"Q{i:03d}" for i in range(21, 41)]

for qid in conceptual_batch:

    result = answer_generation_results[qid]

    question = evaluation_df[
        evaluation_df["question_id"].astype(str) == qid
    ].iloc[0]

    print("\n" + "=" * 100)
    print(f"QUESTION ID: {qid}")
    print(f"CATEGORY: {question['question_category']}")

    print("\nQUESTION:")
    print(question["question"])

    print("\nGOLD ANSWER:")
    print(question["gold_answer"])

    print("\nGENERATED ANSWER:")
    print(result["answer"])

    print("\nRETRIEVED EVIDENCE:")

    evidence = result.get("evidence", [])

    if evidence:
        for i, item in enumerate(evidence, 1):
            print(f"\n--- Evidence {i} ---")
            print(f"Article: {item.get('title', 'Unknown')}")
            print(f"Article ID: {item.get('article_id', 'Unknown')}")
            print(f"Year: {item.get('year', 'Unknown')}")
            print(f"PDF Page: {item.get('page_number', 'Unknown')}")
            print(f"Text: {item.get('text', '')}")
    else:
        print("No retrieved evidence available.")

print("\n" + "=" * 100)
print("Completed: Q021–Q040 displayed successfully.")



QUESTION ID: Q021
CATEGORY: Conceptual

QUESTION:
Why does the PhishBench framework evaluate phishing detection features under identical experimental conditions?

GOLD ANSWER:
Using the same system specifications, datasets, classifiers, and evaluation metrics allows existing phishing detection features to be compared systematically and fairly.

GENERATED ANSWER:
Based on the supplied research evidence, the PhishBench framework evaluates and compares existing phishing detection features under identical experimental conditions (such as unified system specifications, datasets, classifiers, and evaluation metrics) in order to enable a systematic and thorough evaluation of these features [Article ID: An_In-Depth_Benchmarking_and_Evaluation_of_Phishing_Detection_Research_for_Security_Needs.pdf, Title: An In-Depth Benchmarking and Evaluation of Phishing Detection Research for Security Needs, Year: 2020, PDF Page: 1].

RETRIEVED EVIDENCE:

--- Evidence 1 ---
Article: An In-Depth Benchmarking 

In [ ]:
import pandas as pd

# Conceptual evaluation ratings and review comments
conceptual_reviews = {
    "Q021": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The generated answer was judged correct and relevant, with its claims supported by the retrieved evidence."
    },
    "Q022": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The generated answer was judged correct and relevant, with its claims supported by the retrieved evidence."
    },
    "Q023": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The generated answer was judged correct and relevant, with its claims supported by the retrieved evidence."
    },
    "Q024": {
        "correctness": "Partially Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer was partially correct because it omitted the explanation of black-box models and the importance of trust. Its reported claims were supported by the retrieved evidence."
    },
    "Q025": {
        "correctness": "Partially Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer was partially correct because it omitted the importance of outdated and imbalanced datasets. Its reported claims were supported by the retrieved evidence."
    },
    "Q026": {
        "correctness": "Partially Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer was partially correct because it omitted important aspects involving long-distance dependencies, contextual information, and feature-extraction challenges. Its reported claims were supported by the retrieved evidence."
    },
    "Q027": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The generated answer was judged correct and relevant, with its claims supported by the retrieved evidence."
    },
    "Q028": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The generated answer was judged correct and relevant, with its claims supported by the retrieved evidence."
    },
    "Q029": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The generated answer was judged correct and relevant, with its claims supported by the retrieved evidence."
    },
    "Q030": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The generated answer was judged correct and relevant, with its claims supported by the retrieved evidence."
    },
    "Q031": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly explains that autoencoders can detect unknown or zero-day attacks while reducing the need for laborious data labelling. Its additional explanations about typical behaviour, reconstruction error, and hybrid models are also supported by the retrieved evidence."
    },
    "Q032": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly identifies the considerable time and effort required to find optimal autoencoder architectures and hyperparameter settings. Its discussion of repeated experiments, development delays, and architectural decisions is supported by the retrieved passages."
    },
    "Q033": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly explains that pruning removes relatively ineffective neurons, reducing model parameters, computational complexity, and model size while maintaining similar attack detection quality. These explanations are supported by the retrieved evidence."
    },
    "Q034": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly explains that the separate classifier performs attack-type inference while allowing the pruned and pretrained encoder to be reused. It accurately explains how this avoids full retraining from scratch."
    },
    "Q035": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly explains the local-minima problem in conventional back-propagation and how the Artificial Bee Colony algorithm searches for optimal network weights. The proposed vectorized and parallelized approach helps reduce training time, as supported by the retrieved evidence."
    },
    "Q036": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly explains the importance of datasets for training and evaluating IDS, comparing system performance, reducing overfitting, and representing modern attack scenarios. These explanations are supported by the retrieved passages."
    },
    "Q037": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly explains that good historical network data supports IDS development and helps models learn patterns that distinguish normal from attack traffic. Its discussion of specialized environments and evolving attack patterns is also supported."
    },
    "Q038": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly explains how traffic analysis tools support dataset creation through traffic acquisition, transformation, examination, storage, and annotation. It also correctly identifies the six tool categories and their possible combination."
    },
    "Q039": {
        "correctness": "Correct",
        "relevance": "Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "Valid",
        "review_comments": "The answer correctly explains why dataset features matter for dataset suitability, model effectiveness, realistic attack representation, and adapting to changing network conditions. These points are supported by the retrieved passages."
    },
    "Q040": {
        "correctness": "Incorrect",
        "relevance": "Partially Relevant",
        "evidence_faithfulness": "Supported",
        "citation_validity": "No Citation",
        "review_comments": "The answer did not identify the security weaknesses stated in the reference answer: system loopholes, ineffective security policies, and a lack of knowledge about attacks and cybercrime. The retrieved passages did not contain these details, so the model appropriately abstained rather than inventing an answer."
    }
}

# Update only the Conceptual questions
for question_id, ratings in conceptual_reviews.items():
    mask = verified_review_df["question_id"] == question_id

    if mask.sum() != 1:
        raise ValueError(f"Expected exactly one row for {question_id}")

    for column, value in ratings.items():
        verified_review_df.loc[mask, column] = value

# Save the complete review dataset
verified_review_df.to_csv(answer_review_path, index=False)

# Reload the saved file to verify
verified_review_df = pd.read_csv(answer_review_path)

# Verify Conceptual evaluations
conceptual_ids = [f"Q{i:03d}" for i in range(21, 41)]

conceptual_check = verified_review_df[
    verified_review_df["question_id"].isin(conceptual_ids)
].sort_values("question_id")

rating_columns = [
    "correctness",
    "relevance",
    "evidence_faithfulness",
    "citation_validity",
    "review_comments"
]

completed_count = conceptual_check[rating_columns].notna().all(axis=1).sum()

print("SUCCESS: Conceptual evaluations saved and reloaded.")
print("Conceptual questions:", len(conceptual_check))
print("Completed evaluations:", completed_count)

# Verify previous Factual evaluations remain intact
factual_check = verified_review_df[
    verified_review_df["question_id"].isin(
        [f"Q{i:03d}" for i in range(1, 21)]
    )
]

print("Factual evaluations preserved:", len(factual_check) == 20)
print(
    "Completed Factual evaluations:",
    factual_check[rating_columns].notna().all(axis=1).sum()
)

display(conceptual_check[
    ["question_id", "correctness", "relevance",
     "evidence_faithfulness", "citation_validity"]
])

SUCCESS: Conceptual evaluations saved and reloaded.
Conceptual questions: 20
Completed evaluations: 20
Factual evaluations preserved: True
Completed Factual evaluations: 20


,question_id,correctness,relevance,evidence_faithfulness,citation_validity
20,Q021,Correct,Relevant,Supported,Valid
21,Q022,Correct,Relevant,Supported,Valid
22,Q023,Correct,Relevant,Supported,Valid
23,Q024,Partially Correct,Relevant,Supported,Valid
24,Q025,Partially Correct,Relevant,Supported,Valid
25,Q026,Partially Correct,Relevant,Supported,Valid
26,Q027,Correct,Relevant,Supported,Valid
27,Q028,Correct,Relevant,Supported,Valid
28,Q029,Correct,Relevant,Supported,Valid
29,Q030,Correct,Relevant,Supported,Valid


In [ ]:
# Display all 20 Multi-paper Synthesis questions (Q041–Q060)
# using the actual structure of answer_generation_results.

synthesis_batch = [f"Q{i:03d}" for i in range(41, 61)]

for qid in synthesis_batch:

    result = answer_generation_results[qid]

    question = evaluation_df[
        evaluation_df["question_id"].astype(str) == qid
    ].iloc[0]

    print("\n" + "=" * 100)
    print(f"QUESTION ID: {qid}")
    print(f"CATEGORY: {question['question_category']}")

    print("\nQUESTION:")
    print(question["question"])

    print("\nGOLD ANSWER:")
    print(question["gold_answer"])

    print("\nGENERATED ANSWER:")
    print(result["answer"])

    print("\nRETRIEVED EVIDENCE:")

    evidence = result.get("evidence", [])

    if evidence:
        for i, item in enumerate(evidence, 1):
            print(f"\n--- Evidence {i} ---")
            print(f"Article: {item.get('title', 'Unknown')}")
            print(f"Article ID: {item.get('article_id', 'Unknown')}")
            print(f"Year: {item.get('year', 'Unknown')}")
            print(f"PDF Page: {item.get('page_number', 'Unknown')}")
            print(f"Text: {item.get('text', '')}")
    else:
        print("No retrieved evidence available.")

print("\n" + "=" * 100)
print("Completed: Q041–Q060 displayed successfully.")


QUESTION ID: Q041
CATEGORY: Multi-paper Synthesis

QUESTION:
How do the classification scopes of the X-URL framework and the intelligent malicious URL detection system differ?

GOLD ANSWER:
The X-URL framework performs four-class classification covering benign, defacement, malware, and phishing URLs. The intelligent malicious URL system uses a two-stage approach: first distinguishing benign URLs from malware, then classifying URLs into five classes: benign, spam, phishing, malware, and defacement.

GENERATED ANSWER:
Based on the supplied research evidence, the provided texts describe an "intelligent Identification and Classification System for Malicious URLs" [Article ID: An_intelligent_identification_and_classification_system_for_malicious_uniform_resource_locators.pdf, Title: An intelligent identification and classification system for malicious uniform resource URLs (URLs), Year: 2023, PDF Page Number: 3]. 

Regarding the classification scope of the intelligent malicious URL detecti

In [ ]:
# Save Multi-paper Synthesis evaluations (Q041–Q060)
# Preserve all previously saved evaluations (Q001–Q040).

synthesis_evaluations = {
    "Q041": ("Incorrect", "Partially Relevant", "Supported", "Valid",
             "Correctly describes the intelligent malicious URL system's two-stage classification and five categories, but cannot compare it with X-URL because the relevant X-URL evidence is missing."),

    "Q042": ("Incorrect", "Partially Relevant", "Supported", "Valid",
             "Correctly identifies the four ensemble approaches used by the intelligent URL system, but omits X-URL's character-level BiLSTM approach."),

    "Q043": ("Incorrect", "Partially Relevant", "Supported", "No Citation",
             "Abstains because X-URL evidence is missing, but fails to use the available systematic review evidence describing the prevalence of supervised deep learning."),

    "Q044": ("Incorrect", "Relevant", "Partially Supported", "Partially Valid",
             "Provides several correct metrics for the intelligent URL system but misses X-URL's evaluation measures and the detection-time measure. The cited source does not establish X-URL's actual metrics."),

    "Q045": ("Incorrect", "Relevant", "Supported", "Valid",
             "Correctly discusses the systematic review and other deep-learning research, but omits X-URL's BiLSTM architecture, four-class classification, and 98.53% mean cross-validation accuracy."),

    "Q046": ("Incorrect", "Relevant", "Supported", "Valid",
             "Correctly identifies the three traffic domains evaluated by the 2026 study but does not identify the 2024 study's combined network-and-host traffic inputs."),

    "Q047": ("Incorrect", "Partially Relevant", "Supported", "Valid",
             "Discusses zero-day and unfamiliar threat detection using other research papers but does not address the two specific intrusion detection studies requested."),

    "Q048": ("Incorrect", "Partially Relevant", "Supported", "Valid",
             "Provides a detailed and supported explanation of the 2026 CNN-LSTM architecture but does not compare it with the 2024 CNN-based study."),

    "Q049": ("Incorrect", "Partially Relevant", "Supported", "Valid",
             "Correctly describes the NSL-KDD evaluation protocols but does not cover the other study's three datasets or cross-dataset evaluation."),

    "Q050": ("Incorrect", "Partially Relevant", "Supported", "Valid",
             "Identifies the 2026 study's datasets and deployment context but provides none of the requested accuracy, F1-score, or AUC values and cannot establish the 2024 study's performance."),

    "Q051": ("Correct", "Relevant", "Partially Supported", "Partially Valid",
             "Correctly distinguishes AI2DS normal-data reconstruction and adaptive thresholding from LSTM temporal modelling. However, the MSE-thresholding explanation partly relies on a separate maritime autoencoder study."),

    "Q052": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly distinguishes temporal patterns learned by the LSTM autoencoder from packet-based graph representations and structural and contextual patterns learned by GTAE-IDS."),

    "Q053": ("Partially Correct", "Relevant", "Supported", "Valid",
             "Identifies GTAE-IDS limitations involving labelled graph data, computational requirements, and novel attacks, but omits the LSTM study's specific concerns about signature-based detection and false positives."),

    "Q054": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly explains AI2DS normal-behaviour learning and adaptive thresholding alongside GTAE-IDS unsupervised learning of benign traffic's structural and global patterns."),

    "Q055": ("Incorrect", "Partially Relevant", "Supported", "Valid",
             "Explains why results from different experimental settings should not be directly compared, but omits AI2DS's reported performance improvements and GTAE-IDS's accuracy above 98%."),

    "Q056": ("Incorrect", "Partially Relevant", "Supported", "Valid",
             "Correctly describes the 2026 survey's 124 datasets and broad scope but substitutes another 2026 dataset review for the requested 2025 Advanced IDS benchmark study."),

    "Q057": ("Incorrect", "Partially Relevant", "Supported", "Valid",
             "Provides relevant dataset-quality and benchmarking concerns but does not establish the specific comparison with Advanced IDS, including outdated datasets and the lack of standardized benchmarks."),

    "Q058": ("Incorrect", "Partially Relevant", "Supported", "Valid",
             "Correctly explains the 1996 testing methodology but omits Advanced IDS's 14 datasets, training-validation-test splits, and Macro F1 comparison."),

    "Q059": ("Incorrect", "Partially Relevant", "Supported", "Valid",
             "Correctly discusses dataset selection, application-oriented taxonomy, quality, bias, leakage, and reproducibility but does not connect these findings to Advanced IDS's specific benchmarking concerns."),

    "Q060": ("Incorrect", "Relevant", "Supported", "Valid",
             "Correctly explains the survey's recommendations and SHIELD's role in reproducibility but omits Advanced IDS's preprocessed datasets and standardized data splits.")
}

# Update only Q041–Q060.
for qid, ratings in synthesis_evaluations.items():

    mask = verified_review_df["question_id"].astype(str) == qid

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one record for {qid}, found {mask.sum()}."
        )

    verified_review_df.loc[mask, [
        "correctness",
        "relevance",
        "evidence_faithfulness",
        "citation_validity",
        "review_comments"
    ]] = ratings

# Save the updated evaluation dataset.
verified_review_df.to_csv(answer_review_path, index=False)

# Reload and verify the saved file.
verified_review_df = pd.read_csv(answer_review_path)

synthesis_ids = [f"Q{i:03d}" for i in range(41, 61)]

synthesis_check = verified_review_df[
    verified_review_df["question_id"].astype(str).isin(synthesis_ids)
]

completed = synthesis_check[
    synthesis_check[
        ["correctness", "relevance", "evidence_faithfulness", "citation_validity"]
    ].notna().all(axis=1)
]

factual_count = verified_review_df[
    verified_review_df["question_id"].astype(str).isin(
        [f"Q{i:03d}" for i in range(1, 21)]
    )
]["correctness"].notna().sum()

conceptual_count = verified_review_df[
    verified_review_df["question_id"].astype(str).isin(
        [f"Q{i:03d}" for i in range(21, 41)]
    )
]["correctness"].notna().sum()

print("SUCCESS: Multi-paper Synthesis evaluations saved and reloaded.")
print("Multi-paper Synthesis questions:", len(synthesis_check))
print("Completed evaluations:", len(completed))
print("Factual evaluations preserved:", factual_count == 20)
print("Completed Factual evaluations:", factual_count)
print("Conceptual evaluations preserved:", conceptual_count == 20)
print("Completed Conceptual evaluations:", conceptual_count)

assert len(synthesis_check) == 20
assert len(completed) == 20
assert factual_count == 20
assert conceptual_count == 20

print("\nAll three evaluation categories are saved and verified.")

SUCCESS: Multi-paper Synthesis evaluations saved and reloaded.
Multi-paper Synthesis questions: 20
Completed evaluations: 20
Factual evaluations preserved: True
Completed Factual evaluations: 20
Conceptual evaluations preserved: True
Completed Conceptual evaluations: 20

All three evaluation categories are saved and verified.


Q061–Q080 evaluation

Display the unanswerable questions

In [ ]:

# Step 1: Inspect the 20 unanswerable evaluation questions

unanswerable_eval_df = evaluation_df[
    evaluation_df["question_id"].str.extract(r"(\d+)")[0].astype(int).between(61, 80)
].copy()

print("Number of unanswerable questions:", len(unanswerable_eval_df))
print("\nQuestion details:\n")

display(
    unanswerable_eval_df[
        ["question_id", "question", "answerable", "verification_notes"]
    ].reset_index(drop=True)
)

print("\nAnswer generation results:")
print("Object type:", type(answer_generation_results))

if isinstance(answer_generation_results, dict):
    print("Total saved results:", len(answer_generation_results))
    print("Sample result keys:", list(answer_generation_results.keys())[:3])


Number of unanswerable questions: 20

Question details:



,question_id,question,answerable,verification_notes
0,Q061,What was the exact financial loss caused by ph...,False,Hybrid retrieval returned general phishing inf...
1,Q062,How many phishing websites were reported in Bh...,False,Retrieved passages included global phishing st...
2,Q063,Which Bhutanese government agency recorded the...,False,Retrieval returned general and international p...
3,Q064,What percentage of Bhutanese internet users ex...,False,Retrieved passages contained general or global...
4,Q065,How many organizations in Bhutan experienced r...,False,Retrieval returned general phishing and IoT at...
5,Q066,What detection accuracy did the Practical Auto...,False,The Practical Autoencoder paper discusses the ...
6,Q067,What exact generalisation score did the featur...,False,The target paper identifies further investigat...
7,Q068,In the explainable CNN-based IoT intrusion det...,False,The target study identifies evaluation across ...
8,Q069,What accuracy did the hybrid CNN–LSTM model ac...,False,The retrieved target-paper passage reports 99....
9,Q070,What zero-day detection rate did the RNDAE mod...,False,The RNDAE paper identifies zero-day capability...



Answer generation results:
Object type: <class 'dict'>
Total saved results: 80
Sample result keys: ['Q001', 'Q002', 'Q003']


In [ ]:

# Step 2: Inspect the structure of one saved generation result

sample_result = answer_generation_results["Q061"]

print("Result type:", type(sample_result))

if isinstance(sample_result, dict):
    print("\nAvailable fields:")
    for key, value in sample_result.items():
        print(f"\n{key}:")
        print("Type:", type(value))
        print("Preview:", str(value)[:300])
else:
    print("\nSample result:")
    print(str(sample_result)[:1500])


Result type: <class 'dict'>

Available fields:

question:
Type: <class 'str'>
Preview: What was the exact financial loss caused by phishing attacks in Bhutan in 2025?

answer:
Type: <class 'str'>
Preview: Based on the supplied research evidence, there is no information provided regarding financial losses caused by phishing attacks in Bhutan, nor are there any statistics or mentions of Bhutan for the year 2025. Therefore, the exact financial loss caused by phishing attacks in Bhutan in 2025 cannot be 

evidence:
Type: <class 'list'>
Preview: [{'rank': 1, 'rrf_score': 0.029906956136464335, 'article_id': 'A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf', 'title': 'CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites', 'year': 2011.0, 'page_number': 2, 'text': '21:2 G. Xiang et


In [ ]:

# Step 3: Display generated answers and retrieved evidence
# for the first 10 unanswerable questions (Q061-Q070)

for q_num in range(61, 81):
    qid = f"Q{q_num:03d}"
    result = answer_generation_results[qid]

    print("\n" + "=" * 100)
    print(qid)
    print("=" * 100)

    print("\nQUESTION:")
    print(result["question"])

    print("\nGENERATED ANSWER:")
    print(result["answer"])

    print("\nTOP 3 RETRIEVED EVIDENCE PASSAGES:")

    for passage in result.get("evidence", [])[:3]:
        print(f"\nRank: {passage.get('rank')}")
        print(f"Article: {passage.get('title')}")
        print(f"Year: {passage.get('year')}")
        print(f"Page: {passage.get('page_number')}")
        print(f"Evidence: {passage.get('text', '')[:500]}")



Q061

QUESTION:
What was the exact financial loss caused by phishing attacks in Bhutan in 2025?

GENERATED ANSWER:
Based on the supplied research evidence, there is no information provided regarding financial losses caused by phishing attacks in Bhutan, nor are there any statistics or mentions of Bhutan for the year 2025. Therefore, the exact financial loss caused by phishing attacks in Bhutan in 2025 cannot be established from the available evidence.

TOP 3 RETRIEVED EVIDENCE PASSAGES:

Rank: 1
Article: CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web Sites
Year: 2011.0
Page: 2
Evidence: 21:2 G. Xiang et al. 1. INTRODUCTION Phishing is a form of identity theft, in which criminals build replicas of target Web sites and lure unsuspecting victims to disclose their sensitive information like passwords, personal identification numbers (PINs), etc. According to one survey by the Gartner Group [McCall 2007], phishing attacks in the United States caused $3.2 bil

In [ ]:

# Verify the expected unanswerability of Q061-Q070

for _, row in unanswerable_eval_df[
    unanswerable_eval_df["question_id"].isin(
        [f"Q{i:03d}" for i in range(61, 71)]
    )
].iterrows():

    print("\n" + "=" * 80)
    print(row["question_id"])
    print("\nQUESTION:")
    print(row["question"])
    print("\nGOLD ANSWER / EXPECTED RESPONSE:")
    print(row["gold_answer"])
    print("\nVERIFICATION NOTES:")
    print(row["verification_notes"])



Q061

QUESTION:
What was the exact financial loss caused by phishing attacks in Bhutan in 2025?

GOLD ANSWER / EXPECTED RESPONSE:
The supplied corpus does not establish the exact financial loss caused by phishing attacks in Bhutan in 2025.

VERIFICATION NOTES:
Hybrid retrieval returned general phishing information and historical or international statistics, but no Bhutan-specific financial-loss figure for 2025. A corpus keyword scan found Bhutan-related matches only in document download footers, not research findings.

Q062

QUESTION:
How many phishing websites were reported in Bhutan in 2025?

GOLD ANSWER / EXPECTED RESPONSE:
The supplied corpus does not report the number of phishing websites reported in Bhutan in 2025.

VERIFICATION NOTES:
Retrieved passages included global phishing statistics but did not establish a Bhutan-specific count for 2025. Bhutan-related keyword matches in the corpus were document download footers.

Q063

QUESTION:
Which Bhutanese government agency recorded

In [ ]:

# Verify the expected unanswerability of Q071-Q080

for _, row in unanswerable_eval_df[
    unanswerable_eval_df["question_id"].isin(
        [f"Q{i:03d}" for i in range(71, 81)]
    )
].iterrows():

    print("\n" + "=" * 80)
    print(row["question_id"])
    print("\nQUESTION:")
    print(row["question"])
    print("\nGOLD ANSWER / EXPECTED RESPONSE:")
    print(row["gold_answer"])
    print("\nVERIFICATION NOTES:")
    print(row["verification_notes"])



Q071

QUESTION:
What was the 95th-percentile end-to-end inference latency of the X-URL model under live gateway or DNS traffic?

GOLD ANSWER / EXPECTED RESPONSE:
The supplied corpus does not report the 95th-percentile end-to-end inference latency of X-URL under live gateway or DNS traffic.

VERIFICATION NOTES:
The X-URL paper identifies production latency, throughput, memory under load and live gateway or DNS integration as unevaluated. No 95th-percentile end-to-end latency measurement is reported.

Q072

QUESTION:
What adversarial robustness score did X-URL achieve against Unicode homoglyph and character-edit attacks?

GOLD ANSWER / EXPECTED RESPONSE:
The supplied corpus does not report an adversarial robustness score for X-URL against Unicode homoglyph or character-edit attacks.

VERIFICATION NOTES:
The X-URL paper states that Unicode homoglyph substitutions and character insertion, deletion or permutation attacks were not tested.

Q073

QUESTION:
What performance did X-URL achieve 

In [ ]:

import pandas as pd

review_path = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data/answer_quality_review.csv"
)

# Load the existing 60-question review
existing_review_df = pd.read_csv(review_path)

print("EXISTING REVIEW")
print("Shape:", existing_review_df.shape)
print("Columns:", existing_review_df.columns.tolist())

print("\nSAMPLE EXISTING REVIEW ROW:")
display(existing_review_df.head(1))

# Inspect one unanswerable answer-generation record
print("\nQ061 GENERATION RESULT STRUCTURE:")
print("Fields:", answer_generation_results["Q061"].keys())

print("\nQ061 EVIDENCE STRUCTURE:")
if answer_generation_results["Q061"]["evidence"]:
    print(answer_generation_results["Q061"]["evidence"][0].keys())
    display(pd.DataFrame(answer_generation_results["Q061"]["evidence"]).head(1))


EXISTING REVIEW
Shape: (60, 11)
Columns: ['question_id', 'question_category', 'question', 'gold_answer', 'generated_answer', 'retrieved_passages', 'correctness', 'relevance', 'evidence_faithfulness', 'citation_validity', 'review_comments']

SAMPLE EXISTING REVIEW ROW:


,question_id,question_category,question,gold_answer,generated_answer,retrieved_passages,correctness,relevance,evidence_faithfulness,citation_validity,review_comments
0,Q001,Factual,How many phishing and legitimate webpages were...,"The evaluation used 8,118 phishing webpages an...","Based on the provided research evidence, the e...",\n\n------------------------------------------...,Incorrect,Partially Relevant,Supported,Valid,The generated answer did not provide the refer...



Q061 GENERATION RESULT STRUCTURE:
Fields: dict_keys(['question', 'answer', 'evidence'])

Q061 EVIDENCE STRUCTURE:
dict_keys(['rank', 'rrf_score', 'article_id', 'title', 'year', 'page_number', 'text', 'cross_encoder_score', 'rerank_rank'])


,rank,rrf_score,article_id,title,year,page_number,text,cross_encoder_score,rerank_rank
0,1,0.029907,A_Feature_Rich_Machine_Learning_Framework_for_...,CANTINA+: A Feature-Rich Machine Learning Fram...,2011.0,2,21:2 G. Xiang et al. 1. INTRODUCTION Phishing ...,0.533015,1


In [ ]:

import os
import shutil
from datetime import datetime
import pandas as pd
from pandas.testing import assert_frame_equal

# --------------------------------------------------
# 1. Preserve the original 60-question review
# --------------------------------------------------

original_review_df = pd.read_csv(review_path).copy()

# Create a timestamped backup before updating
backup_path = review_path.replace(
    ".csv",
    f"_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
)

shutil.copy2(review_path, backup_path)
print("Backup created:", backup_path)


# --------------------------------------------------
# 2. Define the manual assessments for Q061-Q080
# --------------------------------------------------

assessments = {
    "Q061": ("Correct", "Relevant", "Supported", "No Citation",
             "Correctly abstained from providing an unsupported Bhutan-specific phishing financial-loss figure."),
    "Q062": ("Correct", "Relevant", "Supported", "No Citation",
             "Correctly abstained from providing an unsupported Bhutan-specific phishing website count."),
    "Q063": ("Correct", "Relevant", "Supported", "No Citation",
             "Correctly stated that the corpus does not identify a Bhutanese government agency ranking."),
    "Q064": ("Correct", "Relevant", "Supported", "No Citation",
             "Correctly abstained from providing an unsupported Bhutan-specific phishing prevalence percentage."),
    "Q065": ("Correct", "Relevant", "Supported", "No Citation",
             "Correctly abstained from providing an unsupported Bhutan-specific ransomware incident count."),
    "Q066": ("Correct", "Relevant", "Supported", "No Citation",
             "Correctly identified that real-operational-network detection accuracy was not established."),
    "Q067": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly identified that the requested generalisation score for previously unseen attack characteristics was not reported."),
    "Q068": ("Correct", "Partially Relevant", "Supported", "Valid",
             "Correctly abstained from the requested target-study result. Related-study metrics were clearly distinguished from the specified evaluation."),
    "Q069": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly distinguished the reported single-day subset result from the requested multi-day evaluation."),
    "Q070": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly identified zero-day assessment on real-world backbone traffic as unevaluated future work."),
    "Q071": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly identified that p95 latency under live gateway or DNS traffic was not measured."),
    "Q072": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly identified that Unicode homoglyph and character-edit adversarial attacks were not tested."),
    "Q073": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly abstained from claiming temporal generalisation; random splitting does not establish performance on unseen future periods."),
    "Q074": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly identified the absence of expert or security-analyst agreement measurements for the SHAP explanations."),
    "Q075": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly identified that binary evaluation does not establish misclassification percentages for individual attack families."),
    "Q076": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly distinguished the target model's benchmark results from additional real-world deployment results."),
    "Q077": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly avoided generalising the autoencoder-depth findings beyond the configurations and datasets tested."),
    "Q078": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly distinguished a suggested hardware improvement from a measured inference-speed improvement."),
    "Q079": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly distinguished benchmark evaluation from zero-day evaluation on attack classes excluded during training."),
    "Q080": ("Correct", "Relevant", "Supported", "Valid",
             "Correctly identified cross-network adaptation to previously unseen traffic as future work.")
}


# --------------------------------------------------
# 3. Prepare the 20 new review rows
# --------------------------------------------------

new_rows = []

for i in range(61, 81):
    qid = f"Q{i:03d}"

    question_row = unanswerable_eval_df[
        unanswerable_eval_df["question_id"] == qid
    ].iloc[0]

    generation = answer_generation_results[qid]

    # Format retrieved evidence for the review CSV
    evidence_parts = []

    for item in generation["evidence"]:
        evidence_parts.append(
            f"Rank: {item.get('rank', '')}\n"
            f"Article ID: {item.get('article_id', '')}\n"
            f"Title: {item.get('title', '')}\n"
            f"Year: {item.get('year', '')}\n"
            f"Page: {item.get('page_number', '')}\n"
            f"Evidence: {item.get('text', '')}"
        )

    retrieved_passages = "\n\n" + ("-" * 60) + "\n\n".join(
        evidence_parts
    )

    correctness, relevance, faithfulness, citation, comments = (
        assessments[qid]
    )

    new_rows.append({
        "question_id": qid,
        "question_category": question_row["question_category"],
        "question": question_row["question"],
        "gold_answer": question_row["gold_answer"],
        "generated_answer": generation["answer"],
        "retrieved_passages": retrieved_passages,
        "correctness": correctness,
        "relevance": relevance,
        "evidence_faithfulness": faithfulness,
        "citation_validity": citation,
        "review_comments": comments,
        "abstention_success": "Yes"
    })

new_review_df = pd.DataFrame(new_rows)


# --------------------------------------------------
# 4. Combine the original 60 and new 20 reviews
# --------------------------------------------------

combined_review_df = original_review_df.copy()

# Add the new abstention column without changing
# any of the original 60 assessments
combined_review_df["abstention_success"] = pd.NA

combined_review_df = pd.concat(
    [combined_review_df, new_review_df],
    ignore_index=True
)


# --------------------------------------------------
# 5. Verify before saving
# --------------------------------------------------

assert combined_review_df.shape[0] == 80
assert combined_review_df["question_id"].nunique() == 80
assert set(combined_review_df["question_id"]) == {
    f"Q{i:03d}" for i in range(1, 81)
}

assert combined_review_df["abstention_success"].iloc[:60].isna().all()
assert (
    combined_review_df["abstention_success"].iloc[60:] == "Yes"
).all()

# Confirm that all original 60 assessments are unchanged
assert_frame_equal(
    combined_review_df[original_review_df.columns].iloc[:60]
        .reset_index(drop=True),
    original_review_df.reset_index(drop=True),
    check_dtype=False
)

# Confirm that all 20 new assessments are included
assert len(new_review_df) == 20


# --------------------------------------------------
# 6. Save the combined review
# --------------------------------------------------

combined_review_df.to_csv(review_path, index=False)

print("\nCombined review saved successfully.")
print("File:", review_path)
print("Backup:", backup_path)


# --------------------------------------------------
# 7. Reload and verify the saved file
# --------------------------------------------------

saved_review_df = pd.read_csv(review_path)

print("\nFINAL VERIFICATION")
print("Shape:", saved_review_df.shape)
print("Unique question IDs:", saved_review_df["question_id"].nunique())
print("Original questions:", saved_review_df["question_id"].iloc[:60].min(),
      "to", saved_review_df["question_id"].iloc[:60].max())
print("New questions:", saved_review_df["question_id"].iloc[60],
      "to", saved_review_df["question_id"].iloc[-1])

print("\nAbstention assessment:")
print(saved_review_df["abstention_success"].value_counts(dropna=False))

print("\nUnanswerable-question ratings:")
display(
    saved_review_df[
        saved_review_df["question_id"].between("Q061", "Q080")
    ][[
        "question_id",
        "correctness",
        "relevance",
        "evidence_faithfulness",
        "citation_validity",
        "abstention_success"
    ]]
)

# Final integrity check
assert saved_review_df.shape == (80, 12)
assert saved_review_df["question_id"].nunique() == 80

print("\nSUCCESS: All 80 assessments are saved and verified.")


Backup created: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/answer_quality_review_backup_20261120_085913.csv

Combined review saved successfully.
File: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/answer_quality_review.csv
Backup: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/answer_quality_review_backup_20261120_085913.csv

FINAL VERIFICATION
Shape: (80, 12)
Unique question IDs: 80
Original questions: Q001 to Q060
New questions: Q061 to Q080

Abstention assessment:
abstention_success
NaN    60
Yes    20
Name: count, dtype: int64

Unanswerable-question ratings:


,question_id,correctness,relevance,evidence_faithfulness,citation_validity,abstention_success
60,Q061,Correct,Relevant,Supported,No Citation,Yes
61,Q062,Correct,Relevant,Supported,No Citation,Yes
62,Q063,Correct,Relevant,Supported,No Citation,Yes
63,Q064,Correct,Relevant,Supported,No Citation,Yes
64,Q065,Correct,Relevant,Supported,No Citation,Yes
65,Q066,Correct,Relevant,Supported,No Citation,Yes
66,Q067,Correct,Relevant,Supported,Valid,Yes
67,Q068,Correct,Partially Relevant,Supported,Valid,Yes
68,Q069,Correct,Relevant,Supported,Valid,Yes
69,Q070,Correct,Relevant,Supported,Valid,Yes



SUCCESS: All 80 assessments are saved and verified.


overall answer-quality analysis

In [ ]:

import pandas as pd

review_path = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data/answer_quality_review.csv"
)

# Load the completed manual evaluation
review_df = pd.read_csv(review_path)

print("DATASET OVERVIEW")
print("Shape:", review_df.shape)
print("Columns:", review_df.columns.tolist())

print("\nQUESTION CATEGORY DISTRIBUTION")
print(review_df["question_category"].value_counts())

print("\nCORRECTNESS DISTRIBUTION")
print(review_df["correctness"].value_counts(dropna=False))

print("\nRELEVANCE DISTRIBUTION")
print(review_df["relevance"].value_counts(dropna=False))

print("\nEVIDENCE FAITHFULNESS DISTRIBUTION")
print(review_df["evidence_faithfulness"].value_counts(dropna=False))

print("\nCITATION VALIDITY DISTRIBUTION")
print(review_df["citation_validity"].value_counts(dropna=False))

print("\nABSTENTION SUCCESS")
print(review_df["abstention_success"].value_counts(dropna=False))


DATASET OVERVIEW
Shape: (80, 12)
Columns: ['question_id', 'question_category', 'question', 'gold_answer', 'generated_answer', 'retrieved_passages', 'correctness', 'relevance', 'evidence_faithfulness', 'citation_validity', 'review_comments', 'abstention_success']

QUESTION CATEGORY DISTRIBUTION
question_category
Factual                  20
Conceptual               20
Multi-paper Synthesis    20
Unanswerable             20
Name: count, dtype: int64

CORRECTNESS DISTRIBUTION
correctness
Correct              55
Incorrect            20
Partially Correct     5
Name: count, dtype: int64

RELEVANCE DISTRIBUTION
relevance
Relevant              63
Partially Relevant    17
Name: count, dtype: int64

EVIDENCE FAITHFULNESS DISTRIBUTION
evidence_faithfulness
Supported              78
Partially Supported     2
Name: count, dtype: int64

CITATION VALIDITY DISTRIBUTION
citation_validity
Valid              70
No Citation         8
Partially Valid     2
Name: count, dtype: int64

ABSTENTION SUCCESS
abste

Analyse performance by question category

In [ ]:

# Category-wise answer quality analysis

criteria = [
    "correctness",
    "relevance",
    "evidence_faithfulness",
    "citation_validity"
]

for criterion in criteria:
    print("\n" + "=" * 65)
    print(criterion.upper())

    category_table = pd.crosstab(
        review_df["question_category"],
        review_df[criterion]
    )

    # Convert counts into percentages within each category
    category_percentages = (
        pd.crosstab(
            review_df["question_category"],
            review_df[criterion],
            normalize="index"
        ) * 100
    ).round(2)

    print("\nCounts:")
    display(category_table)

    print("\nPercentages within each category:")
    display(category_percentages)

# Separate abstention performance
print("\n" + "=" * 65)
print("UNANSWERABLE QUESTION ABSTENTION")

unanswerable_df = review_df[
    review_df["question_category"] == "Unanswerable"
]

print(
    "Successful abstentions:",
    (unanswerable_df["abstention_success"] == "Yes").sum(),
    "/",
    len(unanswerable_df)
)



CORRECTNESS

Counts:


correctness,Correct,Incorrect,Partially Correct
question_category,,,
Conceptual,16,1,3
Factual,16,3,1
Multi-paper Synthesis,3,16,1
Unanswerable,20,0,0



Percentages within each category:


correctness,Correct,Incorrect,Partially Correct
question_category,,,
Conceptual,80.0,5.0,15.0
Factual,80.0,15.0,5.0
Multi-paper Synthesis,15.0,80.0,5.0
Unanswerable,100.0,0.0,0.0



RELEVANCE

Counts:


relevance,Partially Relevant,Relevant
question_category,,
Conceptual,1,19
Factual,3,17
Multi-paper Synthesis,12,8
Unanswerable,1,19


... (7 outputs left out)

citation_validity,No Citation,Partially Valid,Valid
question_category,,,
Conceptual,1,0,19
Factual,0,0,20
Multi-paper Synthesis,1,2,17
Unanswerable,6,0,14



Percentages within each category:


citation_validity,No Citation,Partially Valid,Valid
question_category,,,
Conceptual,5.0,0.0,95.0
Factual,0.0,0.0,100.0
Multi-paper Synthesis,5.0,10.0,85.0
Unanswerable,30.0,0.0,70.0



UNANSWERABLE QUESTION ABSTENTION
Successful abstentions: 20 / 20


Identify the synthesis failure cases

In [ ]:

# Examine incorrect and partially correct synthesis answers

synthesis_failures = review_df[
    (review_df["question_category"] == "Multi-paper Synthesis") &
    (review_df["correctness"].isin(["Incorrect", "Partially Correct"]))
].copy()

print("SYNTHESIS ANSWERS REQUIRING IMPROVEMENT")
print("Number of cases:", len(synthesis_failures))

display(
    synthesis_failures[
        [
            "question_id",
            "correctness",
            "relevance",
            "evidence_faithfulness",
            "citation_validity",
            "review_comments"
        ]
    ].sort_values("question_id")
)


SYNTHESIS ANSWERS REQUIRING IMPROVEMENT
Number of cases: 17


,question_id,correctness,relevance,evidence_faithfulness,citation_validity,review_comments
40,Q041,Incorrect,Partially Relevant,Supported,Valid,Correctly describes the intelligent malicious ...
41,Q042,Incorrect,Partially Relevant,Supported,Valid,Correctly identifies the four ensemble approac...
42,Q043,Incorrect,Partially Relevant,Supported,No Citation,"Abstains because X-URL evidence is missing, bu..."
43,Q044,Incorrect,Relevant,Partially Supported,Partially Valid,Provides several correct metrics for the intel...
44,Q045,Incorrect,Relevant,Supported,Valid,Correctly discusses the systematic review and ...
45,Q046,Incorrect,Relevant,Supported,Valid,Correctly identifies the three traffic domains...
46,Q047,Incorrect,Partially Relevant,Supported,Valid,Discusses zero-day and unfamiliar threat detec...
47,Q048,Incorrect,Partially Relevant,Supported,Valid,Provides a detailed and supported explanation ...
48,Q049,Incorrect,Partially Relevant,Supported,Valid,Correctly describes the NSL-KDD evaluation pro...
49,Q050,Incorrect,Partially Relevant,Supported,Valid,Identifies the 2026 study's datasets and deplo...


Separate answer-generation performance from abstention

In [ ]:

# Answer-generation quality for answerable questions only

answerable_df = review_df[
    review_df["question_category"] != "Unanswerable"
].copy()

total = len(answerable_df)

print("ANSWERABLE QUESTION PERFORMANCE")
print("Total questions:", total)

for column in [
    "correctness",
    "relevance",
    "evidence_faithfulness",
    "citation_validity"
]:
    print(f"\n{column.upper()}")

    counts = answerable_df[column].value_counts(dropna=False)

    results = pd.DataFrame({
        "Count": counts,
        "Percentage": (counts / total * 100).round(2)
    })

    display(results)

# Correctness summary
fully_correct = (answerable_df["correctness"] == "Correct").sum()
partially_correct = (
    answerable_df["correctness"] == "Partially Correct"
).sum()
incorrect = (answerable_df["correctness"] == "Incorrect").sum()

print("\nCORRECTNESS SUMMARY")
print(f"Fully correct: {fully_correct}/{total} ({fully_correct/total*100:.2f}%)")
print(f"Partially correct: {partially_correct}/{total} ({partially_correct/total*100:.2f}%)")
print(f"Incorrect: {incorrect}/{total} ({incorrect/total*100:.2f}%)")


ANSWERABLE QUESTION PERFORMANCE
Total questions: 60

CORRECTNESS


,Count,Percentage
correctness,,
Correct,35,58.33
Incorrect,20,33.33
Partially Correct,5,8.33



RELEVANCE


,Count,Percentage
relevance,,
Relevant,44,73.33
Partially Relevant,16,26.67



EVIDENCE_FAITHFULNESS


,Count,Percentage
evidence_faithfulness,,
Supported,58,96.67
Partially Supported,2,3.33



CITATION_VALIDITY


,Count,Percentage
citation_validity,,
Valid,56,93.33
No Citation,2,3.33
Partially Valid,2,3.33



CORRECTNESS SUMMARY
Fully correct: 35/60 (58.33%)
Partially correct: 5/60 (8.33%)
Incorrect: 20/60 (33.33%)


Identify answer-generation failure patterns

In [ ]:

# Identify all answerable questions that were not fully correct

answer_failures = answerable_df[
    answerable_df["correctness"].isin(
        ["Incorrect", "Partially Correct"]
    )
].copy()

print("ANSWER-GENERATION FAILURE ANALYSIS")
print("Total cases requiring improvement:", len(answer_failures))

print("\nDISTRIBUTION BY QUESTION CATEGORY:")
display(
    answer_failures["question_category"]
    .value_counts()
    .rename_axis("Question Category")
    .reset_index(name="Failure Count")
)

print("\nDISTRIBUTION BY CORRECTNESS:")
display(
    answer_failures["correctness"]
    .value_counts()
    .rename_axis("Correctness")
    .reset_index(name="Count")
)

print("\nDETAILED FAILURE COMMENTS:")
display(
    answer_failures[
        [
            "question_id",
            "question_category",
            "correctness",
            "relevance",
            "evidence_faithfulness",
            "citation_validity",
            "review_comments"
        ]
    ].sort_values("question_id")
)


ANSWER-GENERATION FAILURE ANALYSIS
Total cases requiring improvement: 25

DISTRIBUTION BY QUESTION CATEGORY:


,Question Category,Failure Count
0,Multi-paper Synthesis,17
1,Factual,4
2,Conceptual,4



DISTRIBUTION BY CORRECTNESS:


,Correctness,Count
0,Incorrect,20
1,Partially Correct,5



DETAILED FAILURE COMMENTS:


,question_id,question_category,correctness,relevance,evidence_faithfulness,citation_validity,review_comments
0,Q001,Factual,Incorrect,Partially Relevant,Supported,Valid,The generated answer did not provide the refer...
1,Q002,Factual,Incorrect,Partially Relevant,Supported,Valid,The generated answer did not provide the refer...
4,Q005,Factual,Incorrect,Partially Relevant,Supported,Valid,The answer did not identify both datasets used...
18,Q019,Factual,Partially Correct,Relevant,Supported,Valid,"The answer identifies dropout, L2 weight regul..."
23,Q024,Conceptual,Partially Correct,Relevant,Supported,Valid,The answer was partially correct because it om...
24,Q025,Conceptual,Partially Correct,Relevant,Supported,Valid,The answer was partially correct because it om...
25,Q026,Conceptual,Partially Correct,Relevant,Supported,Valid,The answer was partially correct because it om...
39,Q040,Conceptual,Incorrect,Partially Relevant,Supported,No Citation,The answer did not identify the security weakn...
40,Q041,Multi-paper Synthesis,Incorrect,Partially Relevant,Supported,Valid,Correctly describes the intelligent malicious ...
41,Q042,Multi-paper Synthesis,Incorrect,Partially Relevant,Supported,Valid,Correctly identifies the four ensemble approac...


Determine whether retrieval contributed to the failures

In [ ]:

# Inspect expected and retrieved evidence formats

print("EXPECTED EVIDENCE FORMAT")
sample_expected = evaluation_df[
    evaluation_df["question_id"] == "Q041"
]["evidence_article_ids"].iloc[0]

print("Value:", sample_expected)
print("Data type:", type(sample_expected))

print("\nRETRIEVED EVIDENCE FORMAT")
sample_retrieved = answer_generation_results["Q041"]["evidence"]

print("Number of passages:", len(sample_retrieved))

if sample_retrieved:
    print("Retrieved article ID:", sample_retrieved[0]["article_id"])
    print("Data type:", type(sample_retrieved[0]["article_id"]))


EXPECTED EVIDENCE FORMAT
Value: An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf;An_intelligent_identification_and_classification_system_for_malicious_uniform_resource_locators.pdf
Data type: <class 'str'>

RETRIEVED EVIDENCE FORMAT
Number of passages: 5
Retrieved article ID: An_intelligent_identification_and_classification_system_for_malicious_uniform_resource_locators.pdf
Data type: <class 'str'>


Compare expected and retrieved article coverage

In [ ]:

import pandas as pd

# Analyse only the 60 answerable questions
analysis_df = review_df[
    review_df["question_category"] != "Unanswerable"
].copy()

coverage_records = []

for _, row in analysis_df.iterrows():
    qid = row["question_id"]

    # Expected article IDs are separated by semicolons
    expected_value = evaluation_df.loc[
        evaluation_df["question_id"] == qid,
        "evidence_article_ids"
    ].iloc[0]

    expected_ids = {
        item.strip()
        for item in str(expected_value).split(";")
        if item.strip()
    }

    # Collect article IDs from the retrieved evidence
    retrieved_evidence = answer_generation_results[qid]["evidence"]

    retrieved_ids = {
        str(item["article_id"]).strip()
        for item in retrieved_evidence
        if item.get("article_id")
    }

    matched_ids = expected_ids.intersection(retrieved_ids)

    coverage_records.append({
        "question_id": qid,
        "question_category": row["question_category"],
        "correctness": row["correctness"],
        "expected_article_count": len(expected_ids),
        "retrieved_article_count": len(retrieved_ids),
        "matched_article_count": len(matched_ids),
        "any_expected_article_found": len(matched_ids) > 0,
        "all_expected_articles_found": (
            len(expected_ids) > 0 and expected_ids.issubset(retrieved_ids)
        ),
        "article_coverage": (
            len(matched_ids) / len(expected_ids)
            if expected_ids else 0
        )
    })

coverage_df = pd.DataFrame(coverage_records)

# Overall retrieval coverage
print("OVERALL EXPECTED ARTICLE COVERAGE")
print("Questions analysed:", len(coverage_df))
print(
    "At least one expected article retrieved:",
    f"{coverage_df['any_expected_article_found'].sum()}/60",
    f"({coverage_df['any_expected_article_found'].mean() * 100:.2f}%)"
)
print(
    "All expected articles retrieved:",
    f"{coverage_df['all_expected_articles_found'].sum()}/60",
    f"({coverage_df['all_expected_articles_found'].mean() * 100:.2f}%)"
)
print(
    "Average expected article coverage:",
    f"{coverage_df['article_coverage'].mean() * 100:.2f}%"
)

# Compare fully correct answers with answers needing improvement
coverage_df["answer_group"] = coverage_df["correctness"].apply(
    lambda x: "Fully Correct" if x == "Correct"
    else "Partially Correct or Incorrect"
)

print("\nRETRIEVAL COVERAGE BY ANSWER QUALITY")

group_summary = coverage_df.groupby("answer_group").agg(
    Questions=("question_id", "count"),
    Any_expected_article_found=("any_expected_article_found", "sum"),
    All_expected_articles_found=("all_expected_articles_found", "sum"),
    Average_article_coverage=("article_coverage", "mean")
)

group_summary["Any_article_found_%"] = (
    group_summary["Any_expected_article_found"]
    / group_summary["Questions"] * 100
)

group_summary["All_articles_found_%"] = (
    group_summary["All_expected_articles_found"]
    / group_summary["Questions"] * 100
)

group_summary["Average_article_coverage_%"] = (
    group_summary["Average_article_coverage"] * 100
)

print(group_summary.round(2).to_string())

# Compare retrieval coverage by question category
print("\nRETRIEVAL COVERAGE BY QUESTION CATEGORY")

category_summary = coverage_df.groupby("question_category").agg(
    Questions=("question_id", "count"),
    Any_expected_article_found=("any_expected_article_found", "sum"),
    All_expected_articles_found=("all_expected_articles_found", "sum"),
    Average_article_coverage=("article_coverage", "mean")
)

category_summary["Any_article_found_%"] = (
    category_summary["Any_expected_article_found"]
    / category_summary["Questions"] * 100
)

category_summary["All_articles_found_%"] = (
    category_summary["All_expected_articles_found"]
    / category_summary["Questions"] * 100
)

category_summary["Average_article_coverage_%"] = (
    category_summary["Average_article_coverage"] * 100
)

print(category_summary.round(2).to_string())

# Identify questions where no expected article was retrieved
print("\nQUESTIONS WITH NO EXPECTED ARTICLE RETRIEVED")

print(
    coverage_df.loc[
        ~coverage_df["any_expected_article_found"],
        ["question_id", "question_category", "correctness"]
    ].to_string(index=False)
)


OVERALL EXPECTED ARTICLE COVERAGE
Questions analysed: 60
At least one expected article retrieved: 55/60 (91.67%)
All expected articles retrieved: 44/60 (73.33%)
Average expected article coverage: 82.50%

RETRIEVAL COVERAGE BY ANSWER QUALITY
                                Questions  Any_expected_article_found  All_expected_articles_found  Average_article_coverage  Any_article_found_%  All_articles_found_%  Average_article_coverage_%
answer_group                                                                                                                                                                                       
Fully Correct                          35                          33                           33                      0.94                94.29                 94.29                       94.29
Partially Correct or Incorrect         25                          22                           11                      0.66                88.00                 44.00    

Identify which expected articles are missing

In [ ]:

# Examine missing expected articles in answers needing improvement

failure_df = coverage_df[
    coverage_df["correctness"] != "Correct"
].copy()

failure_details = []

for _, row in failure_df.iterrows():
    qid = row["question_id"]

    expected_value = evaluation_df.loc[
        evaluation_df["question_id"] == qid,
        "evidence_article_ids"
    ].iloc[0]

    expected_ids = {
        item.strip()
        for item in str(expected_value).split(";")
        if item.strip()
    }

    retrieved_evidence = answer_generation_results[qid]["evidence"]

    retrieved_ids = {
        str(item["article_id"]).strip()
        for item in retrieved_evidence
        if item.get("article_id")
    }

    missing_ids = expected_ids - retrieved_ids

    question = evaluation_df.loc[
        evaluation_df["question_id"] == qid,
        "question"
    ].iloc[0]

    failure_details.append({
        "question_id": qid,
        "category": row["question_category"],
        "correctness": row["correctness"],
        "question": question,
        "expected_articles": len(expected_ids),
        "retrieved_expected_articles": len(expected_ids & retrieved_ids),
        "missing_articles": len(missing_ids),
        "missing_article_ids": "; ".join(sorted(missing_ids))
    })

failure_details_df = pd.DataFrame(failure_details)

print("MISSING EXPECTED ARTICLES IN ANSWERS NEEDING IMPROVEMENT")
print("Total questions:", len(failure_details_df))
print(
    "Questions with missing expected articles:",
    (failure_details_df["missing_articles"] > 0).sum()
)

display(
    failure_details_df[
        [
            "question_id",
            "category",
            "correctness",
            "expected_articles",
            "retrieved_expected_articles",
            "missing_articles",
            "missing_article_ids"
        ]
    ].sort_values(["category", "question_id"])
)

print("\nQUESTIONS WHERE ALL EXPECTED ARTICLES WERE RETRIEVED")
display(
    failure_details_df[
        failure_details_df["missing_articles"] == 0
    ][["question_id", "category", "correctness", "question"]]
)


MISSING EXPECTED ARTICLES IN ANSWERS NEEDING IMPROVEMENT
Total questions: 25
Questions with missing expected articles: 14


,question_id,category,correctness,expected_articles,retrieved_expected_articles,missing_articles,missing_article_ids
4,Q024,Conceptual,Partially Correct,1,1,0,
5,Q025,Conceptual,Partially Correct,1,1,0,
6,Q026,Conceptual,Partially Correct,1,1,0,
7,Q040,Conceptual,Incorrect,1,0,1,A review of the advancement in intrusion detec...
0,Q001,Factual,Incorrect,1,1,0,
1,Q002,Factual,Incorrect,1,1,0,
2,Q005,Factual,Incorrect,1,1,0,
3,Q019,Factual,Partially Correct,1,1,0,
8,Q041,Multi-paper Synthesis,Incorrect,2,1,1,An_explainable_deep_learning_framework_for_mul...
9,Q042,Multi-paper Synthesis,Incorrect,2,1,1,An_explainable_deep_learning_framework_for_mul...



QUESTIONS WHERE ALL EXPECTED ARTICLES WERE RETRIEVED


,question_id,category,correctness,question
0,Q001,Factual,Incorrect,How many phishing and legitimate webpages were...
1,Q002,Factual,Incorrect,What true-positive and false-positive rates di...
2,Q005,Factual,Incorrect,Which two datasets are used to evaluate the fe...
3,Q019,Factual,Partially Correct,Which four techniques are incorporated into th...
4,Q024,Conceptual,Partially Correct,Why is explainability important when using dee...
5,Q025,Conceptual,Partially Correct,What data-related problems motivate the develo...
6,Q026,Conceptual,Partially Correct,What limitations of traditional network intrus...
10,Q043,Multi-paper Synthesis,Incorrect,How does the systematic literature review help...
11,Q044,Multi-paper Synthesis,Incorrect,What different evaluation measures are reporte...
13,Q046,Multi-paper Synthesis,Incorrect,How do the network traffic inputs considered b...


In [ ]:

# Inspect expected pages and retrieved passage pages

sample_questions = ["Q001", "Q041", "Q043", "Q044"]

for qid in sample_questions:
    row = evaluation_df[
        evaluation_df["question_id"] == qid
    ].iloc[0]

    print("\n" + "=" * 80)
    print("QUESTION:", qid)
    print("Category:", row["question_category"])
    print("Question:", row["question"])

    expected_pages = row["evidence_pages"]
    expected_articles = row["evidence_article_ids"]

    print("\nEXPECTED EVIDENCE")
    print("Article IDs:", expected_articles)
    print("Pages:", expected_pages)
    print("Page data type:", type(expected_pages))

    print("\nACTUALLY RETRIEVED PASSAGES")

    retrieved = answer_generation_results[qid]["evidence"]

    for i, passage in enumerate(retrieved, start=1):
        print(f"\nPassage {i}")
        print("Article:", passage.get("article_id"))
        print("Page:", passage.get("page_number"))
        print("Title:", passage.get("title"))
        print("Text:", str(passage.get("text", ""))[:350].replace("\n", " "))



QUESTION: Q001
Category: Factual
Question: How many phishing and legitimate webpages were used to evaluate CANTINA+?

EXPECTED EVIDENCE
Article IDs: A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf
Pages: 1.0
Page data type: <class 'str'>

ACTUALLY RETRIEVED PASSAGES

Passage 1
Article: Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf
Page: 3
Title: Phishing URL Detection: A Real-Case Scenario Through Login URLs
Text: CANTINA extracts five words from each website using TF- IDF and introduced them into the Google search engine. If a domain was within the n first results, the page was considered legitimate, or phishing otherwise. They obtained an accuracy of 95% with a threshold of n = 30 Google search results. Due to the use of external services like WHOIS7 and t

Passage 2
Article: Phishing_Webpage_Detection_Unveiling_the_Threat_Landscape_and_Investigating_Detection_Techniques.pdf
Page: 20
Title: Phishing Webpage Detection: Unveiling the

Inspect the saved answer records

In [ ]:

# Inspect one existing answer-generation record
sample_id = "Q001"

print("ANSWER GENERATION RECORD")
print("Available fields:")
print(answer_generation_results[sample_id].keys())

print("\nGENERATED ANSWER")
for key in answer_generation_results[sample_id]:
    if key != "evidence":
        print(f"\n{key}:")
        print(answer_generation_results[sample_id][key])

print("\nEVALUATION RECORD")
display(
    review_df.loc[
        review_df["question_id"] == sample_id
    ]
)

print("\nRETRIEVED PASSAGE COUNT:")
print(len(answer_generation_results[sample_id]["evidence"]))


ANSWER GENERATION RECORD
Available fields:
dict_keys(['question', 'answer', 'evidence'])

GENERATED ANSWER

question:
How many phishing and legitimate webpages were used to evaluate CANTINA+?

answer:
Based on the provided research evidence, the exact number of phishing and legitimate webpages used to evaluate CANTINA+ cannot be established, as the supplied passages do not state these specific quantities. 

However, regarding related evaluations mentioned in the literature:
* CANTINA+ achieved a 99.61% F1-Score using its proposed filters [Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf, Title: Phishing URL Detection: A Real-Case Scenario Through Login URLs, Year: 2022, PDF Page: 3].
* In contrast, earlier evaluations of CANTINA noted that it was tested on a collection of 200 URLs from three categories [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+: A Feature-Rich Machine Learning Framework for Detecting Phishing Web S

,question_id,question_category,question,gold_answer,generated_answer,retrieved_passages,correctness,relevance,evidence_faithfulness,citation_validity,review_comments,abstention_success
0,Q001,Factual,How many phishing and legitimate webpages were...,"The evaluation used 8,118 phishing webpages an...","Based on the provided research evidence, the e...",\n\n------------------------------------------...,Incorrect,Partially Relevant,Supported,Valid,The generated answer did not provide the refer...,NaN



RETRIEVED PASSAGE COUNT:
5


Examine the remaining failure cases

In [ ]:

# Inspect generated answers and manual review comments
# for all answers that were not fully correct.

failure_ids = failure_df["question_id"].tolist()

for qid in failure_ids:
    record = answer_generation_results[qid]

    review = review_df.loc[
        review_df["question_id"] == qid
    ].iloc[0]

    print("\n" + "=" * 90)
    print("QUESTION:", qid)
    print("CATEGORY:", review["question_category"])
    print("CORRECTNESS:", review["correctness"])
    print("RELEVANCE:", review["relevance"])
    print("EVIDENCE FAITHFULNESS:", review["evidence_faithfulness"])
    print("CITATION VALIDITY:", review["citation_validity"])

    print("\nQUESTION:")
    print(record["question"])

    print("\nGENERATED ANSWER:")
    print(record["answer"])

    print("\nMANUAL REVIEW COMMENTS:")
    print(review["review_comments"])

    print("\nRETRIEVED ARTICLES:")
    for i, passage in enumerate(record["evidence"], start=1):
        print(
            f"{i}. {passage.get('article_id')} "
            f"(Page {passage.get('page_number')})"
        )



QUESTION: Q001
CATEGORY: Factual
CORRECTNESS: Incorrect
RELEVANCE: Partially Relevant
EVIDENCE FAITHFULNESS: Supported
CITATION VALIDITY: Valid

QUESTION:
How many phishing and legitimate webpages were used to evaluate CANTINA+?

GENERATED ANSWER:
Based on the provided research evidence, the exact number of phishing and legitimate webpages used to evaluate CANTINA+ cannot be established, as the supplied passages do not state these specific quantities. 

However, regarding related evaluations mentioned in the literature:
* CANTINA+ achieved a 99.61% F1-Score using its proposed filters [Phishing_URL_Detection_A_Real-Case_Scenario_Through_Login_URLs.pdf, Title: Phishing URL Detection: A Real-Case Scenario Through Login URLs, Year: 2022, PDF Page: 3].
* In contrast, earlier evaluations of CANTINA noted that it was tested on a collection of 200 URLs from three categories [A_Feature_Rich_Machine_Learning_Framework_for_detecting_phishing_web_sites.pdf, Title: CANTINA+: A Feature-Rich Machine

inspect the evidence structure

In [ ]:

# Inspect expected and retrieved evidence formats

print("EXPECTED EVIDENCE FORMAT")
sample_expected = evaluation_df[
    evaluation_df["question_id"] == "Q041"
]["evidence_article_ids"].iloc[0]

print("Value:", sample_expected)
print("Data type:", type(sample_expected))

print("\nRETRIEVED EVIDENCE FORMAT")
sample_retrieved = answer_generation_results["Q041"]["evidence"]

print("Number of passages:", len(sample_retrieved))

if sample_retrieved:
    print("Retrieved article ID:", sample_retrieved[0]["article_id"])
    print("Data type:", type(sample_retrieved[0]["article_id"]))


EXPECTED EVIDENCE FORMAT
Value: An_explainable_deep_learning_framework_for_multi_class_malicious_URL_classification_using_priority_coefficient_analysis.pdf;An_intelligent_identification_and_classification_system_for_malicious_uniform_resource_locators.pdf
Data type: <class 'str'>

RETRIEVED EVIDENCE FORMAT
Number of passages: 5
Retrieved article ID: An_intelligent_identification_and_classification_system_for_malicious_uniform_resource_locators.pdf
Data type: <class 'str'>


In [ ]:
# Inspect the structure of one retrieved passage

sample_passage = answer_generation_results["Q041"]["evidence"][0]

print("Passage keys:")
print(sample_passage.keys())

print("\nPassage contents:")
for key, value in sample_passage.items():
    if isinstance(value, str):
        print(f"\n{key}:")
        print(value[:500] if len(value) > 500 else value)
    else:
        print(f"\n{key}: {value}")

Passage keys:
dict_keys(['rank', 'rrf_score', 'article_id', 'title', 'year', 'page_number', 'text', 'cross_encoder_score', 'rerank_rank'])

Passage contents:

rank: 3

rrf_score: 0.028438886647841874

article_id:
An_intelligent_identification_and_classification_system_for_malicious_uniform_resource_locators.pdf

title:
An intelligent identification and classification system for malicious uniform resource locators (URLs)

year: 2023.0

page_number: 3

text:
Consequently, in this paper, an intelligent Identification and Classification System for Malicious URLs is proposed. One of the main goals of this system is to detect malicious URLs and identify attack types of malicious URLs attempt to launch including benign phishing, spam, malware, and defacement. Therefore, this system performs two tasks: The first challenge involves binary classification, whereas the second involves multi-label classification. The dataset includes information from five dist

cross_encoder_score: 4.44278287887573

In [ ]:

import pandas as pd

# Separate diagnostic record for the 25 answers
# that were not fully correct.
# This does not modify the original evaluation CSV.

failure_records = [
    ("Q001", "Passage-level evidence failure",
     "Correct CANTINA+ article retrieved, but the required numerical evidence was missing."),
    ("Q002", "Passage-level evidence failure",
     "Correct article retrieved, but the required randomized-evaluation rates were not present in the selected passages."),
    ("Q005", "Passage-level evidence failure",
     "Correct paper retrieved, but the passages did not clearly establish both evaluation datasets."),
    ("Q019", "Answer-generation failure",
     "The expected label-smoothing technique appeared in the answer but was not included in the main list."),
    ("Q024", "Answer-generation failure",
     "Relevant evidence retrieved, but the explanation omitted black-box models and the importance of trust."),
    ("Q025", "Provisional: passage-level or generation failure",
     "The answer omitted outdated and imbalanced datasets; passage-level evidence needs closer confirmation."),
    ("Q026", "Mixed failure",
     "Relevant papers retrieved, but important limitations were omitted from the generated explanation."),
    ("Q040", "Retrieval failure",
     "The expected article was absent, leading to an abstention without the requested details."),

    ("Q041", "Retrieval failure",
     "The intelligent URL paper was retrieved, but the expected X-URL paper was missing."),
    ("Q042", "Retrieval failure",
     "The X-URL evidence was missing, preventing the requested comparison."),
    ("Q043", "Mixed failure",
     "Expected papers appeared in retrieval, but the selected passages and generated answer did not provide the required synthesis."),
    ("Q044", "Mixed failure",
     "Both target studies were represented, but relevant X-URL evaluation evidence was missing and the answer did not complete the comparison."),
    ("Q045", "Retrieval failure",
     "The systematic review was retrieved, but the required X-URL evidence was absent."),
    ("Q046", "Passage-level evidence failure",
     "Both studies were retrieved, but the 2024 study's combined network-and-host inputs were not identified."),
    ("Q047", "Retrieval failure",
     "The answer relied on other intrusion-detection studies rather than the two requested studies."),
    ("Q048", "Retrieval failure",
     "The 2026 CNN-LSTM paper was retrieved, but the requested 2024 CNN study was not adequately represented."),
    ("Q049", "Retrieval failure",
     "The NSL-KDD study was retrieved, but the deployment-efficient study was missing from the top-five evidence."),
    ("Q050", "Retrieval failure",
     "The 2026 deployment-efficient study was retrieved, but the required 2024 study evidence was missing."),
    ("Q053", "Passage-level evidence failure",
     "Both studies were represented, but specific limitations of the LSTM-based study were omitted."),
    ("Q055", "Provisional: retrieval or passage-level failure",
     "The answer lacked the specific AI2DS and GTAE-IDS performance findings."),
    ("Q056", "Retrieval failure",
     "The 2026 survey was retrieved, but the requested 2025 Advanced IDS study was not."),
    ("Q057", "Retrieval failure",
     "Relevant general dataset concerns were retrieved, but the specific Advanced IDS comparison was unavailable."),
    ("Q058", "Retrieval failure",
     "The 1996 methodology was retrieved, but the Advanced IDS benchmark details were missing."),
    ("Q059", "Retrieval failure",
     "The 2026 survey was retrieved, but the Advanced IDS evidence needed for comparison was missing."),
    ("Q060", "Retrieval failure",
     "The survey recommendations were retrieved, but the Advanced IDS practical resources were missing.")
]

failure_diagnostic_df = pd.DataFrame(
    failure_records,
    columns=[
        "question_id",
        "primary_failure_type",
        "diagnostic_observation"
    ]
)

print("Diagnostic table shape:", failure_diagnostic_df.shape)
print("\nFailure type counts:")
print(failure_diagnostic_df["primary_failure_type"].value_counts())

print("\nDiagnostic records:")
display(failure_diagnostic_df)


Diagnostic table shape: (25, 3)

Failure type counts:
primary_failure_type
Retrieval failure                                   13
Passage-level evidence failure                       5
Mixed failure                                        3
Answer-generation failure                            2
Provisional: passage-level or generation failure     1
Provisional: retrieval or passage-level failure      1
Name: count, dtype: int64

Diagnostic records:


,question_id,primary_failure_type,diagnostic_observation
0,Q001,Passage-level evidence failure,"Correct CANTINA+ article retrieved, but the re..."
1,Q002,Passage-level evidence failure,"Correct article retrieved, but the required ra..."
2,Q005,Passage-level evidence failure,"Correct paper retrieved, but the passages did ..."
3,Q019,Answer-generation failure,The expected label-smoothing technique appeare...
4,Q024,Answer-generation failure,"Relevant evidence retrieved, but the explanati..."
5,Q025,Provisional: passage-level or generation failure,The answer omitted outdated and imbalanced dat...
6,Q026,Mixed failure,"Relevant papers retrieved, but important limit..."
7,Q040,Retrieval failure,"The expected article was absent, leading to an..."
8,Q041,Retrieval failure,"The intelligent URL paper was retrieved, but t..."
9,Q042,Retrieval failure,"The X-URL evidence was missing, preventing the..."


In [ ]:

# Check diagnostic classifications against existing article coverage

print("Coverage DataFrame columns:")
print(coverage_df.columns.tolist())

print("\nDiagnostic DataFrame columns:")
print(failure_diagnostic_df.columns.tolist())

print("\nExisting article-coverage records for the 25 failure cases:")

coverage_check = failure_diagnostic_df.merge(
    coverage_df,
    on="question_id",
    how="left"
)

display(coverage_check)


Coverage DataFrame columns:
['question_id', 'question_category', 'correctness', 'expected_article_count', 'retrieved_article_count', 'matched_article_count', 'any_expected_article_found', 'all_expected_articles_found', 'article_coverage', 'answer_group']

Diagnostic DataFrame columns:
['question_id', 'primary_failure_type', 'diagnostic_observation']

Existing article-coverage records for the 25 failure cases:


,question_id,primary_failure_type,diagnostic_observation,question_category,correctness,expected_article_count,retrieved_article_count,matched_article_count,any_expected_article_found,all_expected_articles_found,article_coverage,answer_group
0,Q001,Passage-level evidence failure,"Correct CANTINA+ article retrieved, but the re...",Factual,Incorrect,1,4,1,True,True,1.0,Partially Correct or Incorrect
1,Q002,Passage-level evidence failure,"Correct article retrieved, but the required ra...",Factual,Incorrect,1,3,1,True,True,1.0,Partially Correct or Incorrect
2,Q005,Passage-level evidence failure,"Correct paper retrieved, but the passages did ...",Factual,Incorrect,1,1,1,True,True,1.0,Partially Correct or Incorrect
3,Q019,Answer-generation failure,The expected label-smoothing technique appeare...,Factual,Partially Correct,1,3,1,True,True,1.0,Partially Correct or Incorrect
4,Q024,Answer-generation failure,"Relevant evidence retrieved, but the explanati...",Conceptual,Partially Correct,1,3,1,True,True,1.0,Partially Correct or Incorrect
5,Q025,Provisional: passage-level or generation failure,The answer omitted outdated and imbalanced dat...,Conceptual,Partially Correct,1,5,1,True,True,1.0,Partially Correct or Incorrect
6,Q026,Mixed failure,"Relevant papers retrieved, but important limit...",Conceptual,Partially Correct,1,2,1,True,True,1.0,Partially Correct or Incorrect
7,Q040,Retrieval failure,"The expected article was absent, leading to an...",Conceptual,Incorrect,1,3,0,False,False,0.0,Partially Correct or Incorrect
8,Q041,Retrieval failure,"The intelligent URL paper was retrieved, but t...",Multi-paper Synthesis,Incorrect,2,1,1,True,False,0.5,Partially Correct or Incorrect
9,Q042,Retrieval failure,"The X-URL evidence was missing, preventing the...",Multi-paper Synthesis,Incorrect,2,3,1,True,False,0.5,Partially Correct or Incorrect


In [ ]:

# Refine failure classifications using verified article coverage

def classify_failure(row):
    qid = row["question_id"]

    # No expected article was retrieved
    if not row["any_expected_article_found"]:
        return "Article-level retrieval failure"

    # Some expected articles were retrieved, but not all
    if not row["all_expected_articles_found"]:
        return "Incomplete article-level retrieval"

    # All expected articles were retrieved
    known_cases = {
        "Q001": "Passage-level evidence failure",
        "Q002": "Passage-level evidence failure",
        "Q005": "Passage-level evidence failure",
        "Q019": "Answer-generation failure",
        "Q024": "Answer-generation failure",
        "Q043": "Passage-level evidence failure",
        "Q044": "Passage-level evidence failure",
        "Q046": "Passage-level evidence failure",
    }

    if qid in known_cases:
        return known_cases[qid]

    # Preserve uncertainty where passage sufficiency is not established
    return "Unresolved: passage-level or generation failure"


coverage_refined_df = coverage_check.copy()

coverage_refined_df["refined_failure_type"] = (
    coverage_refined_df.apply(classify_failure, axis=1)
)

print("Refined diagnostic distribution:")
display(
    coverage_refined_df["refined_failure_type"]
    .value_counts()
    .rename_axis("Failure type")
    .reset_index(name="Number of questions")
)

print("\nQuestions with unresolved classifications:")
display(
    coverage_refined_df[
        coverage_refined_df["refined_failure_type"].str.startswith("Unresolved")
    ][["question_id", "diagnostic_observation"]]
)


Refined diagnostic distribution:


,Failure type,Number of questions
0,Incomplete article-level retrieval,11
1,Passage-level evidence failure,6
2,Unresolved: passage-level or generation failure,3
3,Article-level retrieval failure,3
4,Answer-generation failure,2



Questions with unresolved classifications:


,question_id,diagnostic_observation
5,Q025,The answer omitted outdated and imbalanced dat...
6,Q026,"Relevant papers retrieved, but important limit..."
18,Q053,"Both studies were represented, but specific li..."


In [ ]:

unresolved_ids = ["Q025", "Q026", "Q053"]

for qid in unresolved_ids:
    result = answer_generation_results[qid]

    print("\n" + "=" * 100)
    print("QUESTION:", qid)

    print("\nQUESTION:")
    print(result.get("question", "Not available"))

    print("\nGENERATED ANSWER:")
    print(result.get("answer", "Not available"))

    print("\nRETRIEVED EVIDENCE:")

    for passage in result.get("evidence", []):
        print("\nArticle:", passage.get("title"))
        print("Article ID:", passage.get("article_id"))
        print("Page:", passage.get("page_number"))
        print("Rerank position:", passage.get("rerank_rank"))
        print("Passage:")
        print(passage.get("text"))
        print("-" * 80)



QUESTION: Q025

QUESTION:
What data-related problems motivate the development of a data-centric framework for phishing email detection?

GENERATED ANSWER:
Based on the supplied research evidence, the specific data-related challenges or requirements that motivate data-centric or data-generation frameworks in phishing detection include:

* **The Need for Novel and Quality Data:** In machine learning approaches for phishing detection, a primary requirement is access to novel and high-quality data [Article ID: A_data_centric_and_interpretable_PCA_based_framework_for_phishing_email_detection.pdf, Title: A data centric and interpretable PCA based framework for phishing email detection, Year: 2026, PDF Page Number: 8]. 
* **Stagnant State of Existing Datasets:** Current research on phishing email detection has been characterized as relatively stagnant, in part due to the limitations of existing datasets compared to newly generated, user-specific, and qualitatively better phishing emails (suc

In [ ]:

# Finalize the three previously unresolved cases

final_classifications = {
    "Q025": "Passage-level evidence failure",
    "Q026": "Mixed failure",
    "Q053": "Passage-level evidence failure"
}

for qid, diagnosis in final_classifications.items():
    coverage_refined_df.loc[
        coverage_refined_df["question_id"] == qid,
        "refined_failure_type"
    ] = diagnosis

print("FINAL FAILURE ANALYSIS DISTRIBUTION")
print("=" * 60)

final_failure_counts = (
    coverage_refined_df["refined_failure_type"]
    .value_counts()
    .rename_axis("Failure type")
    .reset_index(name="Number of questions")
)

final_failure_counts["Percentage"] = (
    final_failure_counts["Number of questions"] / len(coverage_refined_df) * 100
).round(2)

display(final_failure_counts)

print("\nTotal diagnosed questions:", len(coverage_refined_df))
print("Unresolved cases:",
      coverage_refined_df["refined_failure_type"]
      .str.startswith("Unresolved").sum())

print("\nFinal question-level diagnostic table:")
display(
    coverage_refined_df[
        ["question_id", "question_category", "correctness",
         "article_coverage", "refined_failure_type"]
    ].sort_values("question_id")
)


FINAL FAILURE ANALYSIS DISTRIBUTION


,Failure type,Number of questions,Percentage
0,Incomplete article-level retrieval,11,44.0
1,Passage-level evidence failure,8,32.0
2,Article-level retrieval failure,3,12.0
3,Answer-generation failure,2,8.0
4,Mixed failure,1,4.0



Total diagnosed questions: 25
Unresolved cases: 0

Final question-level diagnostic table:


,question_id,question_category,correctness,article_coverage,refined_failure_type
0,Q001,Factual,Incorrect,1.0,Passage-level evidence failure
1,Q002,Factual,Incorrect,1.0,Passage-level evidence failure
2,Q005,Factual,Incorrect,1.0,Passage-level evidence failure
3,Q019,Factual,Partially Correct,1.0,Answer-generation failure
4,Q024,Conceptual,Partially Correct,1.0,Answer-generation failure
5,Q025,Conceptual,Partially Correct,1.0,Passage-level evidence failure
6,Q026,Conceptual,Partially Correct,1.0,Mixed failure
7,Q040,Conceptual,Incorrect,0.0,Article-level retrieval failure
8,Q041,Multi-paper Synthesis,Incorrect,0.5,Incomplete article-level retrieval
9,Q042,Multi-paper Synthesis,Incorrect,0.5,Incomplete article-level retrieval


In [ ]:

import os

output_dir = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data"
)

os.makedirs(output_dir, exist_ok=True)

# Save question-level diagnostic results
diagnostic_path = os.path.join(
    output_dir,
    "failure_analysis_diagnostics.csv"
)

coverage_refined_df[
    [
        "question_id",
        "question_category",
        "correctness",
        "expected_article_count",
        "retrieved_article_count",
        "matched_article_count",
        "article_coverage",
        "refined_failure_type",
        "diagnostic_observation"
    ]
].to_csv(diagnostic_path, index=False)

# Save failure category summary
summary_path = os.path.join(
    output_dir,
    "failure_analysis_summary.csv"
)

final_failure_counts.to_csv(summary_path, index=False)

# Verify both files
print("Failure analysis files saved successfully.\n")

for path in [diagnostic_path, summary_path]:
    print("File:", os.path.basename(path))
    print("Exists:", os.path.exists(path))
    print("Size:", os.path.getsize(path), "bytes")
    print()

# Verify saved diagnostic records
saved_diagnostics = pd.read_csv(diagnostic_path)
saved_summary = pd.read_csv(summary_path)

assert len(saved_diagnostics) == 25
assert saved_diagnostics["question_id"].nunique() == 25
assert saved_diagnostics["refined_failure_type"].notna().all()
assert saved_summary["Number of questions"].sum() == 25

print("VERIFICATION SUCCESSFUL")
print("All 25 diagnostic records and summary counts are saved.")


Failure analysis files saved successfully.

File: failure_analysis_diagnostics.csv
Exists: True
Size: 4558 bytes

File: failure_analysis_summary.csv
Exists: True
Size: 216 bytes

VERIFICATION SUCCESSFUL
All 25 diagnostic records and summary counts are saved.


Step 1: Consolidating the final evaluation results.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

# Project data directory
data_dir = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data"
)

# Load the completed evaluation files
answer_review_path = os.path.join(
    data_dir, "answer_quality_review.csv"
)

failure_diagnostics_path = os.path.join(
    data_dir, "failure_analysis_diagnostics.csv"
)

failure_summary_path = os.path.join(
    data_dir, "failure_analysis_summary.csv"
)

answer_review_df = pd.read_csv(answer_review_path)
failure_diagnostics_df = pd.read_csv(failure_diagnostics_path)
failure_summary_df = pd.read_csv(failure_summary_path)

# Inspect the files
print("ANSWER QUALITY REVIEW")
print("Shape:", answer_review_df.shape)
print("Columns:", answer_review_df.columns.tolist())

print("\nFAILURE ANALYSIS DIAGNOSTICS")
print("Shape:", failure_diagnostics_df.shape)
print("Columns:", failure_diagnostics_df.columns.tolist())

print("\nFAILURE ANALYSIS SUMMARY")
print("Shape:", failure_summary_df.shape)
print("Columns:", failure_summary_df.columns.tolist())

print("\nAll three evaluation files loaded successfully.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ANSWER QUALITY REVIEW
Shape: (80, 12)
Columns: ['question_id', 'question_category', 'question', 'gold_answer', 'generated_answer', 'retrieved_passages', 'correctness', 'relevance', 'evidence_faithfulness', 'citation_validity', 'review_comments', 'abstention_success']

FAILURE ANALYSIS DIAGNOSTICS
Shape: (25, 9)
Columns: ['question_id', 'question_category', 'correctness', 'expected_article_count', 'retrieved_article_count', 'matched_article_count', 'article_coverage', 'refined_failure_type', 'diagnostic_observation']

FAILURE ANALYSIS SUMMARY
Shape: (5, 3)
Columns: ['Failure type', 'Number of questions', 'Percentage']

All three evaluation files loaded successfully.


Calculate answer-generation evaluation metrics

In [ ]:

# Overall answer quality evaluation

total_questions = len(answer_review_df)

print("OVERALL ANSWER QUALITY EVALUATION")
print("=" * 55)

# Correctness
print("\n1. CORRECTNESS")
correctness_counts = answer_review_df["correctness"].value_counts()

for category in ["Correct", "Partially Correct", "Incorrect"]:
    count = correctness_counts.get(category, 0)
    percentage = (count / total_questions) * 100
    print(f"{category}: {count}/{total_questions} ({percentage:.2f}%)")

# Relevance
print("\n2. RELEVANCE")
relevance_counts = answer_review_df["relevance"].value_counts()

for category in ["Relevant", "Partially Relevant", "Irrelevant"]:
    count = relevance_counts.get(category, 0)
    percentage = (count / total_questions) * 100
    print(f"{category}: {count}/{total_questions} ({percentage:.2f}%)")

# Evidence faithfulness
print("\n3. EVIDENCE FAITHFULNESS")
evidence_counts = answer_review_df["evidence_faithfulness"].value_counts()

for category in ["Supported", "Partially Supported", "Unsupported"]:
    count = evidence_counts.get(category, 0)
    percentage = (count / total_questions) * 100
    print(f"{category}: {count}/{total_questions} ({percentage:.2f}%)")

# Citation validity
print("\n4. CITATION VALIDITY")
citation_counts = answer_review_df["citation_validity"].value_counts()

for category in ["Valid", "Partially Valid", "Invalid", "No Citation"]:
    count = citation_counts.get(category, 0)
    percentage = (count / total_questions) * 100
    print(f"{category}: {count}/{total_questions} ({percentage:.2f}%)")

# Abstention
print("\n5. ABSTENTION")
unanswerable_df = answer_review_df[
    answer_review_df["question_id"].str.match(r"Q0(6[1-9]|7[0-9]|80)")
]

successful_abstentions = (
    unanswerable_df["abstention_success"] == "Yes"
).sum()

print(
    f"Successful abstentions: "
    f"{successful_abstentions}/{len(unanswerable_df)} "
    f"({successful_abstentions / len(unanswerable_df) * 100:.2f}%)"
)


OVERALL ANSWER QUALITY EVALUATION

1. CORRECTNESS
Correct: 55/80 (68.75%)
Partially Correct: 5/80 (6.25%)
Incorrect: 20/80 (25.00%)

2. RELEVANCE
Relevant: 63/80 (78.75%)
Partially Relevant: 17/80 (21.25%)
Irrelevant: 0/80 (0.00%)

3. EVIDENCE FAITHFULNESS
Supported: 78/80 (97.50%)
Partially Supported: 2/80 (2.50%)
Unsupported: 0/80 (0.00%)

4. CITATION VALIDITY
Valid: 70/80 (87.50%)
Partially Valid: 2/80 (2.50%)
Invalid: 0/80 (0.00%)
No Citation: 8/80 (10.00%)

5. ABSTENTION
Successful abstentions: 20/20 (100.00%)


Evaluate performance by question category

In [ ]:

# Separate answerable and unanswerable questions
question_numbers = (
    answer_review_df["question_id"]
    .str.extract(r"Q(\d+)")[0]
    .astype(int)
)

answerable_review_df = answer_review_df[
    question_numbers <= 60
].copy()

unanswerable_review_df = answer_review_df[
    question_numbers > 60
].copy()

# Calculate category-wise correctness
category_correctness = pd.crosstab(
    answerable_review_df["question_category"],
    answerable_review_df["correctness"]
)

# Ensure all expected columns are present
category_correctness = category_correctness.reindex(
    columns=["Correct", "Partially Correct", "Incorrect"],
    fill_value=0
)

category_correctness["Total"] = category_correctness.sum(axis=1)

category_correctness["Fully Correct (%)"] = (
    category_correctness["Correct"] /
    category_correctness["Total"] * 100
).round(2)

print("CATEGORY-WISE ANSWER CORRECTNESS")
print("=" * 65)
display(category_correctness)

# Calculate category-wise relevance
category_relevance = pd.crosstab(
    answerable_review_df["question_category"],
    answerable_review_df["relevance"]
)

print("\nCATEGORY-WISE ANSWER RELEVANCE")
print("=" * 65)
display(category_relevance)

# Calculate category-wise evidence faithfulness
category_evidence = pd.crosstab(
    answerable_review_df["question_category"],
    answerable_review_df["evidence_faithfulness"]
)

print("\nCATEGORY-WISE EVIDENCE FAITHFULNESS")
print("=" * 65)
display(category_evidence)

# Verify question counts
print("\nDATASET VERIFICATION")
print("Answerable questions:", len(answerable_review_df))
print("Unanswerable questions:", len(unanswerable_review_df))
print("Total questions:", len(answerable_review_df) + len(unanswerable_review_df))


CATEGORY-WISE ANSWER CORRECTNESS


correctness,Correct,Partially Correct,Incorrect,Total,Fully Correct (%)
question_category,,,,,
Conceptual,16,3,1,20,80.0
Factual,16,1,3,20,80.0
Multi-paper Synthesis,3,1,16,20,15.0



CATEGORY-WISE ANSWER RELEVANCE


relevance,Partially Relevant,Relevant
question_category,,
Conceptual,1,19
Factual,3,17
Multi-paper Synthesis,12,8



CATEGORY-WISE EVIDENCE FAITHFULNESS


evidence_faithfulness,Partially Supported,Supported
question_category,,
Conceptual,0,20
Factual,0,20
Multi-paper Synthesis,2,18



DATASET VERIFICATION
Answerable questions: 60
Unanswerable questions: 20
Total questions: 80


Review the failure-analysis distribution

In [ ]:

# Load the saved failure-analysis summary
failure_summary_df = pd.read_csv(failure_summary_path)

print("FAILURE ANALYSIS SUMMARY")
print("=" * 65)

display(failure_summary_df)

# Verify total number of diagnosed failures
total_failures = failure_summary_df["Number of questions"].sum()

print("\nFAILURE ANALYSIS VERIFICATION")
print("Total diagnosed failures:", total_failures)

assert total_failures == 25, "Failure count does not match expected 25."

print("Verification successful: all 25 failures are accounted for.")

# Display the dominant failure category
dominant_failure = failure_summary_df.loc[
    failure_summary_df["Number of questions"].idxmax()
]

print("\nMOST COMMON FAILURE TYPE")
print("Type:", dominant_failure["Failure type"])
print("Questions:", dominant_failure["Number of questions"])
print("Percentage:", dominant_failure["Percentage"], "%")

FAILURE ANALYSIS SUMMARY


,Failure type,Number of questions,Percentage
0,Incomplete article-level retrieval,11,44.0
1,Passage-level evidence failure,8,32.0
2,Article-level retrieval failure,3,12.0
3,Answer-generation failure,2,8.0
4,Mixed failure,1,4.0



FAILURE ANALYSIS VERIFICATION
Total diagnosed failures: 25
Verification successful: all 25 failures are accounted for.

MOST COMMON FAILURE TYPE
Type: Incomplete article-level retrieval
Questions: 11
Percentage: 44.0 %


Create the consolidated evaluation summary

In [ ]:

# ==========================================
# CONSOLIDATED PROJECT EVALUATION SUMMARY
# ==========================================

import os
import pandas as pd

# Recalculate question groups from saved evaluation data
question_numbers = (
    answer_review_df["question_id"]
    .str.extract(r"Q(\d+)")[0]
    .astype(int)
)

answerable_df = answer_review_df[question_numbers <= 60].copy()
unanswerable_df = answer_review_df[question_numbers > 60].copy()

# Helper function for calculating percentages
def get_metric(df, column, value):
    count = (df[column] == value).sum()
    total = len(df)
    percentage = (count / total * 100) if total else 0
    return count, total, round(percentage, 2)

# Store consolidated results
summary_records = []

def add_metric(area, metric, count, total, percentage, notes=""):
    summary_records.append({
        "Evaluation Area": area,
        "Metric": metric,
        "Count": count,
        "Total": total,
        "Percentage": percentage,
        "Notes": notes
    })

# 1. Retrieval performance
add_metric(
    "Retrieval",
    "Hybrid Recall@5",
    79.17, 100, 79.17,
    "Hybrid retrieval without Cross-Encoder"
)

add_metric(
    "Retrieval",
    "Hybrid + Cross-Encoder Recall@5",
    85.83, 100, 85.83,
    "Final reranked retrieval performance"
)

add_metric(
    "Retrieval",
    "Hybrid nDCG@10",
    75.52, 100, 75.52,
    "Hybrid retrieval without Cross-Encoder"
)

add_metric(
    "Retrieval",
    "Hybrid + Cross-Encoder nDCG@10",
    81.35, 100, 81.35,
    "Final reranked retrieval performance"
)

# 2. Overall answer quality: 80 questions
for column, values, area in [
    ("correctness",
     ["Correct", "Partially Correct", "Incorrect"],
     "Overall Correctness (80 questions)"),

    ("relevance",
     ["Relevant", "Partially Relevant", "Irrelevant"],
     "Overall Relevance (80 questions)"),

    ("evidence_faithfulness",
     ["Supported", "Partially Supported", "Unsupported"],
     "Overall Evidence Faithfulness (80 questions)"),

    ("citation_validity",
     ["Valid", "Partially Valid", "Invalid", "No Citation"],
     "Overall Citation Validity (80 questions)")
]:
    for value in values:
        count, total, pct = get_metric(answer_review_df, column, value)
        add_metric(area, value, count, total, pct)

# 3. Answerable-question performance: 60 questions
for column, values, area in [
    ("correctness",
     ["Correct", "Partially Correct", "Incorrect"],
     "Answer Generation (60 answerable questions)"),

    ("relevance",
     ["Relevant", "Partially Relevant", "Irrelevant"],
     "Relevance (60 answerable questions)"),

    ("evidence_faithfulness",
     ["Supported", "Partially Supported", "Unsupported"],
     "Evidence Faithfulness (60 answerable questions)"),

    ("citation_validity",
     ["Valid", "Partially Valid", "Invalid", "No Citation"],
     "Citation Validity (60 answerable questions)")
]:
    for value in values:
        count, total, pct = get_metric(answerable_df, column, value)
        add_metric(area, value, count, total, pct)

# 4. Abstention performance
successful = (
    unanswerable_df["abstention_success"] == "Yes"
).sum()

add_metric(
    "Abstention",
    "Successful abstentions",
    successful,
    len(unanswerable_df),
    round(successful / len(unanswerable_df) * 100, 2)
)

# 5. Article coverage (previously calculated and verified)
add_metric(
    "Article Coverage",
    "At least one expected article retrieved",
    55, 60, 91.67
)

add_metric(
    "Article Coverage",
    "All expected articles retrieved",
    44, 60, 73.33
)

add_metric(
    "Article Coverage",
    "Average expected-article coverage",
    82.50, 100, 82.50,
    "Average coverage percentage across 60 answerable questions"
)

# 6. Failure analysis
for _, row in failure_summary_df.iterrows():
    add_metric(
        "Failure Analysis",
        row["Failure type"],
        int(row["Number of questions"]),
        25,
        float(row["Percentage"]),
        "Manually assigned diagnostic category"
    )

# Create the final summary dataframe
consolidated_summary_df = pd.DataFrame(summary_records)

# Save to Google Drive
summary_output_path = os.path.join(
    data_dir,
    "consolidated_evaluation_summary.csv"
)

consolidated_summary_df.to_csv(
    summary_output_path,
    index=False
)

# Verify saved file
saved_summary_df = pd.read_csv(summary_output_path)

assert len(saved_summary_df) == len(consolidated_summary_df)
assert saved_summary_df["Metric"].notna().all()

print("CONSOLIDATED EVALUATION SUMMARY")
print("=" * 65)
print("Total summary records:", len(saved_summary_df))
print("Saved file:", summary_output_path)
print("File exists:", os.path.exists(summary_output_path))
print("File size:", os.path.getsize(summary_output_path), "bytes")

print("\nVERIFICATION SUCCESSFUL")
print("All consolidated evaluation results have been saved.")

display(saved_summary_df)


CONSOLIDATED EVALUATION SUMMARY
Total summary records: 39
Saved file: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/consolidated_evaluation_summary.csv
File exists: True
File size: 2915 bytes

VERIFICATION SUCCESSFUL
All consolidated evaluation results have been saved.


,Evaluation Area,Metric,Count,Total,Percentage,Notes
0,Retrieval,Hybrid Recall@5,79.17,100,79.17,Hybrid retrieval without Cross-Encoder
1,Retrieval,Hybrid + Cross-Encoder Recall@5,85.83,100,85.83,Final reranked retrieval performance
2,Retrieval,Hybrid nDCG@10,75.52,100,75.52,Hybrid retrieval without Cross-Encoder
3,Retrieval,Hybrid + Cross-Encoder nDCG@10,81.35,100,81.35,Final reranked retrieval performance
4,Overall Correctness (80 questions),Correct,55.00,80,68.75,NaN
5,Overall Correctness (80 questions),Partially Correct,5.00,80,6.25,NaN
6,Overall Correctness (80 questions),Incorrect,20.00,80,25.00,NaN
7,Overall Relevance (80 questions),Relevant,63.00,80,78.75,NaN
8,Overall Relevance (80 questions),Partially Relevant,17.00,80,21.25,NaN
9,Overall Relevance (80 questions),Irrelevant,0.00,80,0.00,NaN


Create a category-wise evaluation CSV

In [ ]:

# ==========================================
# CATEGORY-WISE EVALUATION SUMMARY
# ==========================================

category_metrics = [
    ("correctness", [
        "Correct", "Partially Correct", "Incorrect"
    ]),
    ("relevance", [
        "Relevant", "Partially Relevant", "Irrelevant"
    ]),
    ("evidence_faithfulness", [
        "Supported", "Partially Supported", "Unsupported"
    ]),
    ("citation_validity", [
        "Valid", "Partially Valid", "Invalid", "No Citation"
    ])
]

category_records = []

for column, categories in category_metrics:

    for question_category, group in answerable_df.groupby(
        "question_category"
    ):

        total = len(group)

        for value in categories:
            count = (group[column] == value).sum()

            category_records.append({
                "Question Category": question_category,
                "Evaluation Measure": column,
                "Result": value,
                "Count": int(count),
                "Total Questions": total,
                "Percentage": round(count / total * 100, 2)
            })

category_summary_df = pd.DataFrame(category_records)

# Save the category-wise results
category_output_path = os.path.join(
    data_dir,
    "category_wise_evaluation_summary.csv"
)

category_summary_df.to_csv(
    category_output_path,
    index=False
)

# Verify saved file
saved_category_df = pd.read_csv(category_output_path)

assert len(saved_category_df) == len(category_records)
assert saved_category_df["Percentage"].between(0, 100).all()

# Verify 20 questions per category
category_sizes = answerable_df.groupby(
    "question_category"
).size()

assert (category_sizes == 20).all()

print("CATEGORY-WISE SUMMARY SAVED")
print("=" * 55)
print("File:", category_output_path)
print("Records:", len(saved_category_df))
print("Exists:", os.path.exists(category_output_path))
print("Size:", os.path.getsize(category_output_path), "bytes")

print("\nQUESTION COUNTS BY CATEGORY")
print(category_sizes)

print("\nVERIFICATION SUCCESSFUL")
print("All category-wise evaluation results are saved.")

display(saved_category_df)


CATEGORY-WISE SUMMARY SAVED
File: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/category_wise_evaluation_summary.csv
Records: 39
Exists: True
Size: 2084 bytes

QUESTION COUNTS BY CATEGORY
question_category
Conceptual               20
Factual                  20
Multi-paper Synthesis    20
dtype: int64

VERIFICATION SUCCESSFUL
All category-wise evaluation results are saved.


,Question Category,Evaluation Measure,Result,Count,Total Questions,Percentage
0,Conceptual,correctness,Correct,16,20,80.0
1,Conceptual,correctness,Partially Correct,3,20,15.0
2,Conceptual,correctness,Incorrect,1,20,5.0
3,Factual,correctness,Correct,16,20,80.0
4,Factual,correctness,Partially Correct,1,20,5.0
5,Factual,correctness,Incorrect,3,20,15.0
6,Multi-paper Synthesis,correctness,Correct,3,20,15.0
7,Multi-paper Synthesis,correctness,Partially Correct,1,20,5.0
8,Multi-paper Synthesis,correctness,Incorrect,16,20,80.0
9,Conceptual,relevance,Relevant,19,20,95.0


In [ ]:

import os

# Main project directory
project_dir = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch"
)

print("PROJECT DIRECTORY")
print(project_dir)
print("Exists:", os.path.exists(project_dir))

print("\nPROJECT STRUCTURE")
print("=" * 60)

# Display folders and files up to two levels deep
for current_path, dirs, files in os.walk(project_dir):

    relative_path = os.path.relpath(current_path, project_dir)
    depth = relative_path.count(os.sep) if relative_path != "." else 0

    if depth > 2:
        dirs[:] = []
        continue

    print(f"\n📁 {relative_path}")

    # Avoid listing individual PDF papers
    for filename in sorted(files):
        if filename.lower().endswith(".pdf"):
            continue

        file_path = os.path.join(current_path, filename)
        size_kb = os.path.getsize(file_path) / 1024

        print(f"   📄 {filename} ({size_kb:.1f} KB)")

print("\n" + "=" * 60)

# Inspect important functions currently available in Colab
print("\nRELEVANT FUNCTIONS IN CURRENT SESSION")

keywords = [
    "retrieval", "rerank", "generate",
    "answer", "search", "embed"
]

for name, obj in globals().copy().items():
    if callable(obj) and any(
        keyword in name.lower() for keyword in keywords
    ):
        print(name)

# Inspect relevant runtime variables without revealing secret values
print("\nRELEVANT RUNTIME VARIABLES")

variable_keywords = [
    "index", "model", "tokenizer",
    "chunk", "client", "embed"
]

for name, obj in globals().copy().items():
    if any(keyword in name.lower() for keyword in variable_keywords):
        if not name.startswith("_"):
            print(f"{name}: {type(obj).__name__}")


PROJECT DIRECTORY
/content/drive/MyDrive/NLP-Project/CyberThreatResearch
Exists: True

PROJECT STRUCTURE

📁 .
   📄 Research_Paper_FullText_Audit.xlsx (23.5 KB)

📁 Extracted_Text
   📄 research_papers.json (22825.7 KB)

📁 Cleaned_Text
   📄 cleaned_research_papers.json (22177.3 KB)

📁 Document_Chunks
   📄 research_paper_chunks.json (16037.5 KB)

📁 Retrieval_Indexes
   📄 chunk_metadata.json (15968.3 KB)
   📄 document_embeddings.npy (16836.1 KB)
   📄 faiss_index.bin (16836.0 KB)

📁 Extracted_Text1
   📄 research_papers.json (22825.7 KB)

📁 Cleaned_Text1
   📄 cleaned_research_papers.json (22177.3 KB)

📁 processed_data
   📄 answer_generation_results.pkl (540.5 KB)
   📄 answer_quality_review.csv (696.5 KB)
   📄 answer_quality_review_backup_20261120_085913.csv (543.4 KB)
   📄 bm25_search_test_results.csv (12.7 KB)
   📄 category_retrieval_performance.csv (0.5 KB)
   📄 category_wise_evaluation_summary.csv (2.0 KB)
   📄 consolidated_evaluation_summary.csv (2.8 KB)
   📄 cross_encoder_reranking_resul

In [ ]:

import os
from datetime import datetime

project_dir = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch"
)

print("A. FOLDER DETAILS")
print("=" * 75)

for folder in sorted(os.listdir(project_dir)):
    folder_path = os.path.join(project_dir, folder)

    if os.path.isdir(folder_path):
        modified = datetime.fromtimestamp(
            os.path.getmtime(folder_path)
        ).strftime("%Y-%m-%d %H:%M:%S")

        total_size = 0
        file_count = 0

        for root, dirs, files in os.walk(folder_path):
            for filename in files:
                path = os.path.join(root, filename)
                total_size += os.path.getsize(path)
                file_count += 1

        print(
            f"{folder:<25} "
            f"Files: {file_count:<5} "
            f"Size: {total_size / (1024**2):.2f} MB "
            f"Modified: {modified}"
        )

print("\nB. IMPORTANT NOTEBOOK PATH VARIABLES")
print("=" * 75)

path_variables = [
    "chunks_path",
    "fixed_chunks_path",
    "sentence_chunks_path"
]

for name in path_variables:
    if name in globals():
        value = globals()[name]
        print(f"{name}: {value}")
        if isinstance(value, str):
            print("  File exists:", os.path.exists(value))

print("\nC. CURRENTLY LOADED DATA OBJECTS")
print("=" * 75)

object_names = [
    "sentence_chunks",
    "verified_sentence_chunks",
    "fixed_size_chunks",
    "verified_fixed_chunks",
    "bm25_index",
    "faiss_index",
    "embedding_model",
    "reranker"
]

for name in object_names:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, (list, tuple, dict)):
            print(f"{name}: {type(obj).__name__}, count = {len(obj)}")
        else:
            print(f"{name}: {type(obj).__name__}")

        # Check FAISS index details without rebuilding it
        if hasattr(obj, "ntotal") and hasattr(obj, "d"):
            print(f"  FAISS vectors: {obj.ntotal}")
            print(f"  Vector dimensions: {obj.d}")

print("\nD. IMPORTANT FILES AND MODIFICATION DATES")
print("=" * 75)

important_files = [
    "Document_Chunks/research_paper_chunks.json",
    "Retrieval_Indexes/chunk_metadata.json",
    "Retrieval_Indexes/document_embeddings.npy",
    "Retrieval_Indexes/faiss_index.bin",
    "processed_data/sentence_based_chunks.pkl",
    "processed_data/fixed_size_chunks.pkl",
    "processed_data/sentence_based_faiss.index",
    "processed_data/answer_generation_results.pkl"
]

for relative_path in important_files:
    path = os.path.join(project_dir, relative_path)

    if os.path.exists(path):
        modified = datetime.fromtimestamp(
            os.path.getmtime(path)
        ).strftime("%Y-%m-%d %H:%M:%S")

        size = os.path.getsize(path) / (1024**2)

        print(f"\n{relative_path}")
        print(f"  Size: {size:.2f} MB")
        print(f"  Modified: {modified}")
    else:
        print(f"\n{relative_path}: NOT FOUND")

print("\nDiagnostic completed. No files were changed.")


A. FOLDER DETAILS
Cleaned_Text              Files: 1     Size: 21.66 MB Modified: 2026-11-14 05:47:55
Cleaned_Text1             Files: 1     Size: 21.66 MB Modified: 2026-11-17 17:16:48
Document_Chunks           Files: 1     Size: 15.66 MB Modified: 2026-11-14 05:49:22
Extracted_Text            Files: 1     Size: 22.29 MB Modified: 2026-11-13 20:02:36
Extracted_Text1           Files: 1     Size: 22.29 MB Modified: 2026-11-17 17:14:00
Retrieval_Indexes         Files: 3     Size: 48.48 MB Modified: 2026-11-14 09:08:45
processed_data            Files: 22    Size: 61.06 MB Modified: 2026-11-20 09:39:53

B. IMPORTANT NOTEBOOK PATH VARIABLES
chunks_path: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/sentence_based_chunks.pkl
  File exists: True
fixed_chunks_path: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/fixed_size_chunks.pkl
  File exists: True
sentence_chunks_path: /content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/sent

Audit the four experimental configurations

In [ ]:
import os
import pickle
import pandas as pd

project_dir = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch"
)

results_path = os.path.join(
    project_dir,
    "processed_data",
    "answer_generation_results.pkl"
)

with open(results_path, "rb") as f:
    saved_results = pickle.load(f)

print("SAVED ANSWER GENERATION RESULTS")
print("=" * 60)

print("Object type:", type(saved_results))

if isinstance(saved_results, dict):

    print("Number of records:", len(saved_results))
    print("Sample question IDs:", list(saved_results.keys())[:5])

    first_key = next(iter(saved_results))
    first_record = saved_results[first_key]

    print("\nFIRST RECORD STRUCTURE")
    print("Question ID:", first_key)
    print("Record type:", type(first_record))

    if isinstance(first_record, dict):
        for key, value in first_record.items():
            print(f"\n{key}: {type(value).__name__}")

            if isinstance(value, str):
                print("Sample:", value[:200])
            elif isinstance(value, (list, tuple)):
                print("Number of items:", len(value))
            elif isinstance(value, dict):
                print("Keys:", list(value.keys())[:15])

elif isinstance(saved_results, list):

    print("Number of records:", len(saved_results))

    if saved_results:
        print("\nFIRST RECORD")
        print(saved_results[0])

print("\n" + "=" * 60)

print("EXISTING EXPERIMENT RESULT FILES")

processed_dir = os.path.join(project_dir, "processed_data")

for filename in sorted(os.listdir(processed_dir)):
    if any(word in filename.lower() for word in [
        "generation", "answer", "experiment",
        "latency", "timing", "performance"
    ]):
        print(filename)

SAVED ANSWER GENERATION RESULTS
Object type: <class 'dict'>
Number of records: 80
Sample question IDs: ['Q001', 'Q002', 'Q003', 'Q004', 'Q005']

FIRST RECORD STRUCTURE
Question ID: Q001
Record type: <class 'dict'>

question: str
Sample: How many phishing and legitimate webpages were used to evaluate CANTINA+?

answer: str
Sample: Based on the provided research evidence, the exact number of phishing and legitimate webpages used to evaluate CANTINA+ cannot be established, as the supplied passages do not state these specific quan

evidence: list
Number of items: 5

EXISTING EXPERIMENT RESULT FILES
answer_generation_results.pkl
answer_quality_review.csv
answer_quality_review_backup_20261120_085913.csv
category_retrieval_performance.csv
overall_retrieval_performance.csv


Inspect the generation pipeline

In [ ]:

import inspect

print("=" * 65)
print("ANSWER GENERATION FUNCTION")
print("=" * 65)

print(inspect.signature(generate_evidence_grounded_answer))

print(inspect.getsource(generate_evidence_grounded_answer))

print("\n" + "=" * 65)
print("RETRIEVAL FUNCTION")
print("=" * 65)

print(inspect.signature(retrieval_function))

print(inspect.getsource(retrieval_function))

ANSWER GENERATION FUNCTION
(query, top_k=5)
def generate_evidence_grounded_answer(query, top_k=5):

    # Retrieve relevant passages using hybrid retrieval
    candidates = hybrid_retrieval(
        query,
        top_k=10,
        candidate_k=20
    )

    # Rerank the retrieved passages
    evidence = rerank_results(
        query,
        candidates,
        top_k=top_k
    )

    # Prepare evidence with complete article metadata
    context = ""

    for i, passage in enumerate(evidence, start=1):

        year = passage["year"]

        if pd.notna(year):
            year = int(float(year))

        context += f"""
Evidence {i}
Article ID: {passage['article_id']}
Article Title: {passage['title']}
Publication Year: {year}
PDF Page Number: {passage['page_number']}
Supporting Passage:
{passage['text']}
"""

    # Prepare the improved evidence-grounded prompt
    prompt = f"""
You are an academic research assistant.

Answer the research question using only the supplied research eviden

inspect BM25, Dense and reranking

In [219]:
import inspect

functions_to_check = {
    "BM25-only retrieval": bm25_only_retrieval,
    "Dense-only retrieval": dense_only_retrieval,
    "Cross-Encoder reranking": rerank_results
}

for name, func in functions_to_check.items():
    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    print("Signature:", inspect.signature(func))
    print(inspect.getsource(func))

test_query = evaluation_df.iloc[0]["question"]

print("\n" + "=" * 65)
print("SAMPLE OUTPUT STRUCTURES")
print("=" * 65)

for name, func in [
    ("BM25", bm25_only_retrieval),
    ("Dense", dense_only_retrieval)
]:
    try:
        result = func(test_query, top_k=5)

        print(f"\n{name}:")
        print("Result type:", type(result))
        print("Number of results:", len(result))

        if result:
            print("First result keys:", list(result[0].keys()))
            print("Article ID:", result[0].get("article_id"))
            print("Title:", result[0].get("title"))

    except Exception as e:
        print(f"{name} error: {e}")


BM25-only retrieval
Signature: (query, top_k=5)
def bm25_only_retrieval(query, top_k=5):
    query_tokens = re.findall(r"\b\w+\b", query.lower())
    scores = bm25_index.get_scores(query_tokens)

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(ranked_indices, start=1):
        chunk = sentence_chunks[index]

        results.append({
            "rank": rank,
            "bm25_score": float(scores[index]),
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "year": chunk["year"],
            "page_number": chunk["page_number"],
            "text": chunk["text"]
        })

    return results


Dense-only retrieval
Signature: (query, top_k=5)
def dense_only_retrieval(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedd

verify the saved results

In [220]:
import pickle
import os

results_path = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data/"
    "answer_generation_results.pkl"
)

print("Checking Configuration D results...")

with open(results_path, "rb") as f:
    saved_d_results = pickle.load(f)

print("Result type:", type(saved_d_results))
print("Number of saved answers:", len(saved_d_results))

if isinstance(saved_d_results, dict):
    first_key = next(iter(saved_d_results))
    print("First record keys:", saved_d_results[first_key].keys())
else:
    print("First record keys:", saved_d_results[0].keys())

print("\nChecking evaluation dataset...")
print("Total questions:", len(evaluation_df))
print("Answerable:", int(evaluation_df["answerable"].sum()))
print("Unanswerable:", int((~evaluation_df["answerable"]).sum()))

print("\nConfiguration D results are preserved.")

Checking Configuration D results...
Result type: <class 'dict'>
Number of saved answers: 80
First record keys: dict_keys(['question', 'answer', 'evidence'])

Checking evaluation dataset...
Total questions: 80
Answerable: 60
Unanswerable: 20

Configuration D results are preserved.


create a separate generation function

In [222]:
def generate_comparison_answer(query, config, top_k=5):

    # Select evidence according to the configuration
    if config == "A":
        evidence = []

    elif config == "B":
        evidence = bm25_only_retrieval(query, top_k=top_k)

    elif config == "C":
        evidence = dense_only_retrieval(query, top_k=top_k)

    else:
        raise ValueError("Configuration must be A, B, or C.")

    # Prepare evidence context
    context = ""

    for i, passage in enumerate(evidence, start=1):

        year = passage["year"]

        if pd.notna(year):
            year = int(float(year))
        else:
            year = "Not available"

        context += f"""
Evidence {i}
Article ID: {passage['article_id']}
Article Title: {passage['title']}
Publication Year: {year}
PDF Page Number: {passage['page_number']}
Supporting Passage:
{passage['text']}
"""

    # Configuration A: no retrieved evidence
    if config == "A":

        prompt = f"""
You are an academic research assistant.

Answer the following research question using your general knowledge.

STRICT REQUIREMENTS:
1. Do not pretend that you have consulted the supplied research corpus.
2. Do not invent research-paper citations, article titles, page numbers,
   statistics, or experimental findings.
3. If you are uncertain, clearly state the uncertainty.
4. Use clear academic language.
5. Distinguish established general knowledge from claims that require
   verification against research papers.

Research Question:
{query}

Write a clear and structured answer.
"""

    # Configurations B and C: answer using retrieved evidence
    else:

        prompt = f"""
You are an academic research assistant.

Answer the research question using only the supplied research evidence.

STRICT REQUIREMENTS:
1. Use only claims supported by the supplied passages.
2. Do not invent algorithms, findings, statistics, or references.
3. Every factual research claim must have a citation.
4. Cite each source separately. Never combine multiple articles
   inside one citation.
5. Follow this exact citation format:

[Article ID: filename, Title: article title, Year: year, PDF Page: page number]

6. Copy article IDs, titles, years, and page numbers exactly from
   the evidence.
7. Clearly distinguish findings reported by individual papers.
8. If evidence is insufficient, state what cannot be established.
9. Use clear academic language.

Research Question:
{query}

Research Evidence:
{context}

Write a structured answer with appropriate citations.
"""

    # Generate answer
    try:
        interaction = gemini_client.interactions.create(
            model="gemini-3.5-flash-lite",
            input=prompt
        )

        answer = interaction.output_text

    except Exception as e:
        answer = f"Gemini API request failed: {str(e)}"

    return {
        "question": query,
        "configuration": config,
        "answer": answer,
        "evidence": evidence
    }


print("Comparison answer-generation function created successfully.")

Comparison answer-generation function created successfully.


test all three configurations using Q001

In [223]:
test_query = evaluation_df.loc[
    evaluation_df["question_id"] == "Q001", "question"
].iloc[0]

test_results = {}

for config in ["A", "B", "C"]:

    print("\n" + "=" * 70)
    print(f"TESTING CONFIGURATION {config}")
    print("=" * 70)

    result = generate_comparison_answer(
        query=test_query,
        config=config,
        top_k=5
    )

    test_results[config] = result

    print("Configuration:", result["configuration"])
    print("Retrieved evidence:", len(result["evidence"]))
    print("\nGenerated answer:\n")
    print(result["answer"])


TESTING CONFIGURATION A
Configuration: A
Retrieved evidence: 0

Generated answer:

**Answer to the Research Question**

**1. Statement of Uncertainty and Non-Consultation of Corpus**
In accordance with the strict requirements, I must state that I do not have access to a specific research corpus in this session, and I am relying solely on general academic knowledge. Due to the precise nature of the dataset statistics requested, I am uncertain of the exact, definitive number of phishing and legitimate webpages used in the original evaluation of CANTINA+ without verifying the primary source text. 

**2. Established General Knowledge Regarding CANTINA+**
*   **Context:** CANTINA+ is a well-known, feature-based phishing website detection system proposed in the computer security literature as an enhancement over the original CANTINA algorithm. 
*   **Evaluation Methodology:** Like most machine learning-based phishing detection studies, the evaluation of CANTINA+ involved compiling a balance

In [224]:
test_query = evaluation_df.loc[
    evaluation_df["question_id"] == "Q061", "question"
].iloc[0]

print("Question:", test_query)

unanswerable_test_results = {}

for config in ["A", "B", "C"]:

    print("\n" + "=" * 65)
    print(f"CONFIGURATION {config}")
    print("=" * 65)

    result = generate_comparison_answer(
        query=test_query,
        config=config,
        top_k=5
    )

    unanswerable_test_results[config] = result

    print("Retrieved evidence:", len(result["evidence"]))
    print("\nAnswer:\n", result["answer"])

Question: What was the exact financial loss caused by phishing attacks in Bhutan in 2025?

CONFIGURATION A
Retrieved evidence: 0

Answer:
 **Research Question:** What was the exact financial loss caused by phishing attacks in Bhutan in 2025?

### Answer

**1. Statement of Uncertainty**
I cannot provide the exact financial loss caused by phishing attacks in Bhutan in 2025 because the year 2025 is in the future relative to the data cutoff of my training knowledge, and comprehensive, verified national financial loss data for specific cybercrime vectors for that period is not part of established general knowledge. 

**2. Established General Knowledge**
* **Nature of the Threat:** Phishing is a globally prevalent form of social engineering where attackers impersonate legitimate entities to steal sensitive information, such as login credentials and financial details. 
* **Context in Bhutan:** Like many developing nations undergoing rapid digital transformation (driven by initiatives such as 

In [225]:
import os
import pickle

checkpoint_path = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data/"
    "configuration_ABC_answer_generation_checkpoint.pkl"
)

# Load an existing checkpoint or create a new one
if os.path.exists(checkpoint_path):

    with open(checkpoint_path, "rb") as f:
        checkpoint = pickle.load(f)

    print("Existing checkpoint loaded.")

else:

    checkpoint = {
        "results": {
            "A": {},
            "B": {},
            "C": {}
        },
        "failures": []
    }

    print("New experiment checkpoint created.")

# Reuse the six test results
for config, result in test_results.items():
    checkpoint["results"][config].setdefault("Q001", result)

for config, result in unanswerable_test_results.items():
    checkpoint["results"][config].setdefault("Q061", result)

# Save initial checkpoint
with open(checkpoint_path, "wb") as f:
    pickle.dump(checkpoint, f)

print("\nStarting A/B/C answer generation...")
print("Total questions per configuration:", len(evaluation_df))

# Generate only missing answers
for config in ["A", "B", "C"]:

    print(f"\n{'=' * 60}")
    print(f"CONFIGURATION {config}")
    print(f"{'=' * 60}")

    for row in evaluation_df.itertuples(index=False):

        question_id = row.question_id

        # Skip questions already generated
        if question_id in checkpoint["results"][config]:
            continue

        try:

            result = generate_comparison_answer(
                query=row.question,
                config=config,
                top_k=5
            )

            # Do not mark API failures as completed
            if result["answer"].startswith("Gemini API request failed:"):

                checkpoint["failures"].append({
                    "question_id": question_id,
                    "configuration": config,
                    "error": result["answer"]
                })

                print(f"API ERROR: {config} - {question_id}")

            else:

                checkpoint["results"][config][question_id] = result

                print(
                    f"Completed {config} - {question_id} "
                    f"({len(checkpoint['results'][config])}/80)"
                )

        except Exception as e:

            checkpoint["failures"].append({
                "question_id": question_id,
                "configuration": config,
                "error": str(e)
            })

            print(f"ERROR: {config} - {question_id}: {e}")

        # Save progress after each question
        with open(checkpoint_path, "wb") as f:
            pickle.dump(checkpoint, f)

print("\n" + "=" * 60)
print("EXPERIMENT GENERATION SUMMARY")
print("=" * 60)

for config in ["A", "B", "C"]:
    print(
        f"Configuration {config}: "
        f"{len(checkpoint['results'][config])}/80 completed"
    )

print("Recorded API failures:", len(checkpoint["failures"]))
print("Checkpoint:", checkpoint_path)

New experiment checkpoint created.

Starting A/B/C answer generation...
Total questions per configuration: 80

CONFIGURATION A
Completed A - Q002 (3/80)
Completed A - Q003 (4/80)
Completed A - Q004 (5/80)
Completed A - Q005 (6/80)
Completed A - Q006 (7/80)
Completed A - Q007 (8/80)
Completed A - Q008 (9/80)
Completed A - Q009 (10/80)
Completed A - Q010 (11/80)
Completed A - Q011 (12/80)
Completed A - Q012 (13/80)
Completed A - Q013 (14/80)
Completed A - Q014 (15/80)
Completed A - Q015 (16/80)
Completed A - Q016 (17/80)
Completed A - Q017 (18/80)
Completed A - Q018 (19/80)
Completed A - Q019 (20/80)
Completed A - Q020 (21/80)
Completed A - Q021 (22/80)
Completed A - Q022 (23/80)
Completed A - Q023 (24/80)
Completed A - Q024 (25/80)
Completed A - Q025 (26/80)
Completed A - Q026 (27/80)
Completed A - Q027 (28/80)
Completed A - Q028 (29/80)
Completed A - Q029 (30/80)
Completed A - Q030 (31/80)
Completed A - Q031 (32/80)
Completed A - Q032 (33/80)
Completed A - Q033 (34/80)
Completed A - Q0

verify all four configurations

In [226]:
import pickle
import os

checkpoint_path = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data/"
    "configuration_ABC_answer_generation_checkpoint.pkl"
)

with open(checkpoint_path, "rb") as f:
    abc_results = pickle.load(f)

print("FINAL ANSWER GENERATION VERIFICATION")
print("=" * 50)

for config in ["A", "B", "C"]:
    results = abc_results["results"][config]
    print(f"Configuration {config}: {len(results)}/80")

    missing = set(evaluation_df["question_id"]) - set(results.keys())
    print("Missing questions:", sorted(missing))

print("\nConfiguration D:", len(saved_d_results), "/80")
print("Recorded API failures:", len(abc_results["failures"]))

print("\nTotal generated answers:",
      sum(len(abc_results["results"][c]) for c in ["A", "B", "C"])
      + len(saved_d_results))

FINAL ANSWER GENERATION VERIFICATION
Configuration A: 80/80
Missing questions: []
Configuration B: 80/80
Missing questions: []
Configuration C: 80/80
Missing questions: []

Configuration D: 80 /80
Recorded API failures: 0

Total generated answers: 320


In [227]:
import pandas as pd
import pickle
import os

# Load the completed A/B/C results
checkpoint_path = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data/"
    "configuration_ABC_answer_generation_checkpoint.pkl"
)

with open(checkpoint_path, "rb") as f:
    abc_results = pickle.load(f)

evaluation_records = []

for config in ["A", "B", "C"]:

    for question_id, result in abc_results["results"][config].items():

        question_row = evaluation_df[
            evaluation_df["question_id"] == question_id
        ].iloc[0]

        # Format retrieved evidence for manual inspection
        evidence_text = ""

        for i, passage in enumerate(result["evidence"], start=1):

            evidence_text += (
                f"\nEvidence {i}\n"
                f"Article ID: {passage['article_id']}\n"
                f"Title: {passage['title']}\n"
                f"Year: {passage['year']}\n"
                f"PDF Page: {passage['page_number']}\n"
                f"Passage: {passage['text']}\n"
            )

        evaluation_records.append({
            "question_id": question_id,
            "configuration": config,
            "question_category": question_row["question_category"],
            "question": question_row["question"],
            "gold_answer": question_row["gold_answer"],
            "answerable": question_row["answerable"],
            "generated_answer": result["answer"],
            "retrieved_evidence": evidence_text,
            "correctness": "",
            "relevance": "",
            "evidence_faithfulness": (
                "Not Applicable" if config == "A" else ""
            ),
            "citation_validity": "",
            "abstention_success": "",
            "review_notes": ""
        })

abc_evaluation_df = pd.DataFrame(evaluation_records)

# Save to a new file
output_path = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data/"
    "configuration_ABC_quality_review_template.csv"
)

abc_evaluation_df.to_csv(output_path, index=False)

print("Evaluation template created successfully.")
print("Shape:", abc_evaluation_df.shape)
print("\nAnswers by configuration:")
print(abc_evaluation_df["configuration"].value_counts())

print("\nSaved to:")
print(output_path)

Evaluation template created successfully.
Shape: (240, 14)

Answers by configuration:
configuration
A    80
B    80
C    80
Name: count, dtype: int64

Saved to:
/content/drive/MyDrive/NLP-Project/CyberThreatResearch/processed_data/configuration_ABC_quality_review_template.csv


In [231]:
for i in range(1, 81):

    question_id = f"Q{i:03d}"

    question_row = evaluation_df[
        evaluation_df["question_id"] == question_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print(f"{question_id} | {question_row['question_category']}")
    print("=" * 100)

    print("\nQUESTION:")
    print(question_row["question"])

    print("\nGOLD ANSWER:")
    print(question_row["gold_answer"])

    for config in ["A", "B", "C"]:

        result = abc_results["results"][config][question_id]

        print("\n" + "-" * 80)
        print(f"CONFIGURATION {config}")
        print("-" * 80)

        print("\nGENERATED ANSWER:")
        print(result["answer"])

        if result["evidence"]:
            print("\nRETRIEVED EVIDENCE:")

            for j, passage in enumerate(result["evidence"], 1):
                print(f"\nEvidence {j}")
                print("Article:", passage["article_id"])
                print("Title:", passage["title"])
                print("Year:", passage["year"])
                print("Page:", passage["page_number"])
                print("Passage:", passage["text"])

        else:
            print("\nNo retrieved evidence (standalone LLM).")

Streaming output truncated to the last 5000 lines.
Evidence 2
Article: Phishing_URL_detection_generalisation_using_Unsupervised_Domain_Adaptation.pdf
Title: Phishing URL detection generalisation using Unsupervised Domain Adaptation
Year: 2024.0
Page: 3
Passage: First, we explain how we adapt the general theoretical UDA framework, and then we provide specific details of our approach. 3.1. UDA theoretical framework An outline of the general UDA theoretical framework can be found in [50] where the domain Dis defined as D= {X, Y, p(x, y)}. For our phishing detection problem, each dataset represents a different domain. Therefore X⊂Rdis a subset of a d-dimensional space representing the URL input domain of a unique dataset, and X= x1, x2, ... , xnwhere n is the number of samples and each xiis a d-dimensional representation of a URL sample. For classifiers that use hand-crafted features [10,11], drepresents the total number of hand-crafted features — examples of which include the URL length o

In [232]:
import pickle
import os

checkpoint_path = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data/"
    "configuration_ABC_answer_generation_checkpoint.pkl"
)

with open(checkpoint_path, "rb") as f:
    abc_results = pickle.load(f)

for config in ["A", "B", "C"]:
    results = abc_results["results"][config]
    missing = set(evaluation_df["question_id"]) - set(results.keys())

    print(f"Configuration {config}: {len(results)}/80")
    print("Missing questions:", sorted(missing))

print("Recorded API failures:", len(abc_results["failures"]))
print("Configuration D:", len(saved_d_results), "/80")

Configuration A: 80/80
Missing questions: []
Configuration B: 80/80
Missing questions: []
Configuration C: 80/80
Missing questions: []
Recorded API failures: 0
Configuration D: 80 /80


Save the A/B/C manual evaluation ratings.

In [233]:
import os

folder = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data"
)

for file in os.listdir(folder):
    if "ABC" in file or "evaluation" in file.lower():
        print(file)

evaluation_questions_backup.csv
evaluation_questions.csv
retrieval_evaluation_results.csv
consolidated_evaluation_summary.csv
category_wise_evaluation_summary.csv
configuration_ABC_answer_generation_checkpoint.pkl
configuration_ABC_quality_review_template.csv


In [234]:
import pandas as pd

abc_review_path = (
    "/content/drive/MyDrive/NLP-Project/"
    "CyberThreatResearch/processed_data/"
    "configuration_ABC_quality_review_template.csv"
)

abc_review_df = pd.read_csv(abc_review_path)

print("Shape:", abc_review_df.shape)
print("\nColumns:")
print(abc_review_df.columns.tolist())

display(abc_review_df.head(3))

Shape: (240, 14)

Columns:
['question_id', 'configuration', 'question_category', 'question', 'gold_answer', 'answerable', 'generated_answer', 'retrieved_evidence', 'correctness', 'relevance', 'evidence_faithfulness', 'citation_validity', 'abstention_success', 'review_notes']


,question_id,configuration,question_category,question,gold_answer,answerable,generated_answer,retrieved_evidence,correctness,relevance,evidence_faithfulness,citation_validity,abstention_success,review_notes
0,Q001,A,Factual,How many phishing and legitimate webpages were...,"The evaluation used 8,118 phishing webpages an...",True,**Answer to the Research Question**\n\n**1. St...,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN
1,Q061,A,Unanswerable,What was the exact financial loss caused by ph...,The supplied corpus does not establish the exa...,False,**Research Question:** What was the exact fina...,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN
2,Q002,A,Factual,What true-positive and false-positive rates di...,CANTINA+ achieved over 92% true positive on un...,True,**Research Question:** What true-positive and ...,NaN,NaN,NaN,Not Applicable,NaN,NaN,NaN


In [235]:
rating_columns = [
    "correctness",
    "relevance",
    "evidence_faithfulness",
    "citation_validity",
    "abstention_success"
]

print(abc_review_df[rating_columns].isna().sum())

correctness              240
relevance                240
evidence_faithfulness    160
citation_validity        240
abstention_success       240
dtype: int64
